In [ ]:
import matplotlib.pyplot as plt

# Configurações globais de alta qualidade
plt.rcParams['figure.dpi'] = 600       # Qualidade da visualização no notebook
plt.rcParams['savefig.dpi'] = 600     # Qualidade do ficheiro guardado
plt.rcParams['savefig.bbox'] = 'tight' # Remove bordas brancas inúteis

# Categorias COS

categorias da COS presentes na zona de interesse

In [ ]:
# ==============================================================================
# SCRIPT MÍNIMO: VISUALIZAÇÃO DA COS E CATEGORIAS
# ==============================================================================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np # Necessário para funções de array
import os # Necessário para verificar arquivos (embora não usado diretamente no plot)

# --- 1. DEFINIÇÃO DE COORDENADAS E ARQUIVOS (SETUP) ---
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"

# Limites da área de interesse (em EPSG:3035)

# Alqueva
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

# Castelo do Bode
# norte_min, norte_max = 2017250, 2018550
# este_min, este_max = 2754250, 2754850

# Pedrógão
# norte_min, norte_max = 1848550, 1849250
# este_min, este_max = 2779050, 2779550

# --- 2. CARREGAMENTO E REPROJEÇÃO ---
try:
    # Carrega a COS e reprojeta para o Web Mercator (EPSG:3857, padrão para contextily)
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    cos_3857 = cos.to_crs(epsg=3857)
    
    # Cria uma caixa de recorte (no CRS de destino)
    clip_box_3035 = box(este_min, norte_min, este_max, norte_max)
    clip_box_3857 = gpd.GeoSeries(clip_box_3035, crs='EPSG:3035').to_crs(epsg=3857).iloc[0]
    
    # Calcula os limites de zoom a partir da caixa
    minx, miny, maxx, maxy = clip_box_3857.bounds
    pad = (maxx - minx) * 0.05
    x0, x1 = minx - pad, maxx + pad
    y0, y1 = miny - pad, maxy + pad

    # Recorta a COS para a área de interesse
    cos_clip = cos_3857.clip(clip_box_3857)

except Exception as e:
    print(f"ERRO: Não foi possível carregar a COS ou calcular limites. Verifique o COS_PATH. {e}")
    exit()

# --- 3. ANÁLISE: IMPRIMIR CATEGORIAS PRESENTES ---
cos_categories = cos_clip['COS23_n4_L'].unique()
print("\n=======================================================")
print("  CATEGORIAS COS (COS23_n4_L) PRESENTES NA ÁREA")
print("=======================================================")
for category in sorted(cos_categories):
    print(f"- {category}")
print("-------------------------------------------------------")


# ============================================================
# FIGURA 1: VISUALIZAÇÃO SIMPLES DA COS
# ============================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# Plotar a COS recortada, usando a coluna de categorias para a cor
cos_clip.plot(
    column='COS23_n4_L', 
    ax=ax, 
    cmap='tab20', # Mapa de cores diversificado
    alpha=0.7, 
    edgecolor='k', 
    linewidth=0.2, 
    legend=True, 
    legend_kwds={'loc': 'upper left', 'bbox_to_anchor': (1, 1), 'fontsize': 8}
)

# Adicionar Mapa Base para Contexto
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, crs=cos_3857.crs.to_string())

# Formatação Final
ax.set_xlim(x0, x1)
ax.set_ylim(y0, y1)
ax.set_title("Carta de Uso e Ocupação do Solo (COS) 2023 - Área Recortada", fontsize=14)
ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# SCRIPT: VISUALIZAÇÃO COS PARA 20 BARRAGENS DE BETÃO EM PORTUGAL
# ==============================================================================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box, Point
import contextily as ctx
import numpy as np

# ==============================================================================
# 1. LISTA DE BARRAGENS (Nome, Latitude, Longitude)
# ==============================================================================
# Coordenadas aproximadas do paredão da barragem (WGS84)
barragens = [
    # Bacia do Tejo / Zêzere
    {"nome": "Castelo do Bode", "lat": 39.5424, "lon": -8.3225},
    {"nome": "Cabril", "lat": 39.9169, "lon": -8.1311},
    {"nome": "Bouçã", "lat": 39.8453, "lon": -8.2439},
    {"nome": "Fratel", "lat": 39.5328, "lon": -7.7844},
    {"nome": "Belver", "lat": 39.4933, "lon": -7.9931},
    {"nome": "Santa Luzia", "lat": 40.0883, "lon": -7.8222},
    
    # Bacia do Douro
    {"nome": "Picote", "lat": 41.3683, "lon": -6.2344},
    {"nome": "Miranda", "lat": 41.4903, "lon": -6.2764},
    {"nome": "Bemposta", "lat": 41.3122, "lon": -6.4983},
    {"nome": "Baixo Sabor", "lat": 41.2722, "lon": -7.0781},
    {"nome": "Foz Tua", "lat": 41.3811, "lon": -7.4281},
    {"nome": "Crestuma-Lever", "lat": 41.0744, "lon": -8.5061},
    {"nome": "Carrapatelo", "lat": 41.1075, "lon": -8.1481},
    
    # Bacia do Cávado / Lima
    {"nome": "Alto Lindoso", "lat": 41.8711, "lon": -8.2128},
    {"nome": "Caniçada", "lat": 41.6419, "lon": -8.2253},
    {"nome": "Salamonde", "lat": 41.7222, "lon": -7.9944},
    {"nome": "Venda Nova", "lat": 41.6883, "lon": -7.9422},
    {"nome": "Alto Rabagão", "lat": 41.7372, "lon": -7.8589},
    
    # Bacia do Mondego
    {"nome": "Aguieira", "lat": 40.3419, "lon": -8.1964},
    {"nome": "Raiva", "lat": 40.3014, "lon": -8.2561},
    
    # (Opcional - Adicione o Alqueva aqui se quiser a lista completa)
    # {"nome": "Alqueva", "lat": 38.1961, "lon": -7.4964}, 
]

# Tamanho da janela de visualização (metros para cada lado do centro)
BUFFER_METROS = 800  # Total de largura será 1600m

# ==============================================================================
# 2. CARREGAMENTO GLOBAL (Executado apenas uma vez para performance)
# ==============================================================================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"

print(">>> A carregar a Carta de Ocupação do Solo (pode demorar um pouco)...")
try:
    # Carregamos a COS inteira uma vez. Se for muito pesado, podemos carregar por bbox dentro do loop,
    # mas carregar uma vez é melhor se tiver RAM suficiente.
    cos_full = gpd.read_file(COS_PATH).to_crs(epsg=3857) # Converter logo para Web Mercator para o mapa
    print(">>> COS carregada com sucesso.")
except Exception as e:
    print(f"ERRO CRÍTICO: Não foi possível carregar a COS. {e}")
    exit()

# ==============================================================================
# 3. FUNÇÃO DE ANÁLISE E PLOTAGEM
# ==============================================================================
def analisar_barragem(info_barragem, cos_gdf, buffer):
    nome = info_barragem['nome']
    lat = info_barragem['lat']
    lon = info_barragem['lon']
    
    print(f"\n--- A processar: {nome} ---")
    
    # 1. Converter Lat/Lon (WGS84) para Web Mercator (EPSG:3857)
    # Criamos um ponto e reprojetamos
    pt_wgs84 = gpd.GeoSeries([Point(lon, lat)], crs="EPSG:4326")
    pt_3857 = pt_wgs84.to_crs(epsg=3857)
    
    center_x = pt_3857.iloc[0].x
    center_y = pt_3857.iloc[0].y
    
    # 2. Definir a caixa limite (Bounding Box)
    minx, miny = center_x - buffer, center_y - buffer
    maxx, maxy = center_x + buffer, center_y + buffer
    
    # 3. Recortar a COS para esta área (Spatial Indexing torna isto rápido)
    # .cx faz um recorte espacial rápido baseado no índice
    cos_clip = cos_gdf.cx[minx:maxx, miny:maxy]
    
    if cos_clip.empty:
        print(f"AVISO: Nenhuma categoria COS encontrada para {nome} (verifique coordenadas).")
        return

    # 4. Imprimir Categorias
    cats = cos_clip['COS23_n4_L'].unique()
    print(f"Categorias presentes ({len(cats)}):")
    for c in sorted(cats):
        print(f" - {c}")

    # 5. Gerar Figura
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    
    # Plot COS
    cos_clip.plot(
        column='COS23_n4_L',
        ax=ax,
        cmap='tab20',
        alpha=0.6,
        edgecolor='k',
        linewidth=0.5,
        legend=True,
        legend_kwds={'loc': 'upper left', 'bbox_to_anchor': (1, 1), 'fontsize': 9, 'title': 'Uso do Solo'}
    )
    
    # Marcar o centro da barragem
    ax.plot(center_x, center_y, 'r+', markersize=15, markeredgewidth=2, label='Local Barragem')
    
    # Adicionar Imagem de Satélite
    try:
        ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
    except:
        print("Aviso: Não foi possível carregar o basemap (sem internet?).")

    # Formatação
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)
    ax.set_title(f"Barragem: {nome} (COS 2023)", fontsize=14, fontweight='bold')
    ax.set_axis_off()
    
    plt.tight_layout()
    plt.show()

# ==============================================================================
# 4. LOOP DE EXECUÇÃO
# ==============================================================================

# Iterar sobre a lista de barragens
for b in barragens:
    analisar_barragem(b, cos_full, BUFFER_METROS)

print("\n>>> Processamento de todas as barragens concluído.")

In [ ]:
# ==============================================================================
# SCRIPT: VISUALIZAÇÃO COS MANUAL (BBOX EXATA) + SATÉLITE
# ==============================================================================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np

# ==============================================================================
# 1. CONFIGURAÇÃO DAS BARRAGENS (Adicione as coordenadas aqui)
# ==============================================================================
# Formato: 'Nome': [este_min, este_max, norte_min, norte_max]
# Sistema de Coordenadas: EPSG:3035 (ETRS89 / LAEA Europe)

barragens_coords = {
    # --- EXEMPLO QUE FORNECEU (Castelo do Bode) ---
    "Castelo do Bode": [2754250, 2754850, 2017250, 2018550],

    # --- Outras Barragens (Valores Exemplo - SUBSTITUA PELOS SEUS VALORES REAIS) ---
    "Alqueva":         [2792250, 2793250, 1855050, 1855850], # Aprox. anterior
    "Cabril":          [2770000, 2771000, 2050000, 2051000], # Placeholder
    "Bouçã":           [2760000, 2761000, 2040000, 2041000], # Placeholder
    "Fratel":          [2800000, 2801000, 2000000, 2001000], # Placeholder
    "Belver":          [2790000, 2791000, 1990000, 1991000], # Placeholder
    "Santa Luzia":     [2780000, 2781000, 2060000, 2061000], # Placeholder
    "Picote":          [2900000, 2901000, 2180000, 2181000], # Placeholder
    "Miranda":         [2910000, 2911000, 2190000, 2191000], # Placeholder
    "Bemposta":        [2890000, 2891000, 2170000, 2171000], # Placeholder
    "Baixo Sabor":     [2850000, 2851000, 2160000, 2161000], # Placeholder
    "Foz Tua":         [2840000, 2841000, 2150000, 2151000], # Placeholder
    "Crestuma-Lever":  [2740000, 2741000, 2120000, 2121000], # Placeholder
    "Carrapatelo":     [2780000, 2781000, 2130000, 2131000], # Placeholder
    "Alto Lindoso":    [2760000, 2761000, 2220000, 2221000], # Placeholder
    "Caniçada":        [2770000, 2771000, 2200000, 2201000], # Placeholder
    "Salamonde":       [2780000, 2781000, 2210000, 2211000], # Placeholder
    "Venda Nova":      [2790000, 2791000, 2215000, 2216000], # Placeholder
    "Alto Rabagão":    [2795000, 2796000, 2225000, 2226000], # Placeholder
    "Aguieira":        [2765000, 2766000, 2080000, 2081000], # Placeholder
    "Raiva":           [2760000, 2761000, 2075000, 2076000], # Placeholder
}

# ==============================================================================
# 2. CARREGAMENTO GLOBAL
# ==============================================================================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"

print(">>> A carregar COS (uma única vez)...")
try:
    # Carregamos em EPSG:3035 (sistema nativo da COS e das suas coordenadas)
    cos_full = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    print(">>> COS carregada com sucesso.")
except Exception as e:
    print(f"ERRO: Não foi possível carregar a COS. {e}")
    exit()

# ==============================================================================
# 3. LOOP DE PROCESSAMENTO
# ==============================================================================

def processar_barragem(nome, coords):
    emin, emax, nmin, nmax = coords
    
    print(f"\n{'='*50}")
    print(f" A PROCESSAR: {nome}")
    print(f" BBox: E[{emin}-{emax}] N[{nmin}-{nmax}]")
    print(f"{'='*50}")

    # 1. Criar a caixa de recorte
    bbox_geom = box(emin, nmin, emax, nmax)
    
    # 2. Recortar a COS (Spatial Indexing)
    # Primeiro filtramos grosseiramente com .cx para performance, depois clip exato
    subset = cos_full.cx[emin:emax, nmin:nmax]
    cos_clip = subset.clip(bbox_geom)
    
    if cos_clip.empty:
        print(" -> AVISO: Nenhuma geometria COS encontrada nestas coordenadas.")
        return

    # 3. Imprimir Categorias
    cats = cos_clip['COS23_n4_L'].unique()
    print("Categorias Presentes:")
    for c in sorted(cats):
        print(f" - {c}")

    # 4. Preparar para Plotagem (Converter para Web Mercator 3857 para o Satélite)
    cos_clip_3857 = cos_clip.to_crs(epsg=3857)
    
    # Calcular limites em 3857 para o zoom do mapa
    bounds = cos_clip_3857.total_bounds
    minx, miny, maxx, maxy = bounds
    pad_x = (maxx - minx) * 0.05
    pad_y = (maxy - miny) * 0.05

    # 5. Gerar Figura
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    
    # A. Plot da COS (com transparência e cores distintas)
    cos_clip_3857.plot(
        column='COS23_n4_L',
        ax=ax,
        cmap='tab20',      # Mapa de cores variado
        alpha=0.6,         # Transparência para ver o satélite por baixo
        edgecolor='black', # Contorno fino para definição
        linewidth=0.5,
        legend=True,
        legend_kwds={'loc': 'upper left', 'bbox_to_anchor': (1, 1), 'fontsize': 8, 'title': 'Uso do Solo'}
    )
    
    # B. Adicionar Fundo de Satélite (Esri World Imagery = Google Satélite)
    try:
        ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
    except:
        print(" -> Erro ao carregar mapa base (sem internet?)")

    # C. Formatação
    ax.set_xlim(minx - pad_x, maxx + pad_x)
    ax.set_ylim(miny - pad_y, maxy + pad_y)
    ax.set_title(f"Uso do Solo (COS 2023): {nome}", fontsize=14, fontweight='bold')
    ax.set_axis_off()
    
    plt.tight_layout()
    plt.show()

# ==============================================================================
# 4. EXECUÇÃO
# ==============================================================================

# Percorrer o dicionário e gerar os mapas
for nome_barragem, coordenadas in barragens_coords.items():
    processar_barragem(nome_barragem, coordenadas)

# K-Means

## Deslocamento vertical, dV

todos os pontos das celulas quadradas

In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots apenas da barragem
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área aproximada
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

# Limites aproximados (para acelerar o processamento)
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 6. Criar grelha sobre asc_interp
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

# Criar colunas cell_x, cell_y, cell_id
asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Criar GeoDataFrame da grelha
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix,
            'cell_y': iy,
            'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# Recorte à barragem
grid_barragem = gpd.overlay(grid, barragem, how='intersection').to_crs(epsg=3857)

# ==============================
# 7. Clustering apenas células da barragem
# ==============================
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# Considerar apenas células que intersectam a barragem
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)

k = 2
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}

grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 8. Plot mapa + séries temporais
# ==============================
# Pontos centrais
points = asc_interp[asc_interp['cell_id'].isin(grid_barragem['cell_id'])].groupby('cell_id').agg(
    {'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# Plot
fig = plt.figure(figsize=(20,18))
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# Linha 1: mapa
ax_map = fig.add_subplot(gs[0, :])
grid_barragem.boundary.plot(ax=ax_map, color='black', linewidth=1.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

#gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# Linha 2: séries temporais
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min)*0.1

# Carregar temperatura
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id+1}')

    # Temperatura
    ax2 = ax.twinx()
    temp_min = df_temp['med_smooth'].min()
    temp_max = df_temp['med_smooth'].max()
    temp_visual = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_visual*(dV_max-dV_min)*0.25 + (dV_max-(dV_max-dV_min)*0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')
    ax2.set_ylabel("Temperatura (°C)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    # Eixos
    ax.set_ylim(dV_min-dV_margin, dV_max+dV_margin)
    ax2.set_ylim(dV_min-dV_margin, dV_max+dV_margin)
    ax.set_title(f'Cluster {cluster_id+1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, labels1+labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


todos os pontos, agregações, clustering e plots sejam feitos apenas com os pontos dentro da barragem, incluindo o overlay da temperatura nas séries temporais.

In [ ]:
# Escolha aqui o que quer analisar: 'dV' ou 'dH'
TARGET_VAR = 'dV'  # ou 'dV'

# número de clusters
k = 3

# grelha
grid_size = 50

# ==============================
# SCRIPT COMPLETO: Clustering e plots (MAPA + Séries SEPARADAS)
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
except Exception as e:
    print(f"Aviso: Não foi possível ler o ficheiro COS. Criando placeholder. Erro: {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw', 'theta_desc', 'alpha_desc'])

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    if denom == 0: return pd.Series({'dV': np.nan, 'dH': np.nan})
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)
asc_interp = asc_interp.dropna(subset=['dV','dH'])

# ==============================
# 6. Criar grelha e agregação
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix, 'cell_y': iy, 'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy], x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')
#agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
#agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# Agregação: Calcula a média do TARGET_VAR (dV ou dH) 
# mas guarda o resultado numa coluna chamada 'dV' para o resto do script funcionar
agg = asc_interp.groupby(['cell_x', 'cell_y', 'date'])[[TARGET_VAR]].mean().reset_index()

# Criar cell_id
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_barragem_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)
valid_cell_ids = set()
for pt in points_gdf.geometry:
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])
grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

# ==============================
# 8. Clustering
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
#k = 5
kmeans = KMeans(n_clusters=k, random_state=0, n_init='auto')
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Dados hidro-climáticos (temperatura, nível, precipitação)
# ==============================
# Aqui usamos exatamente os placeholders ou arquivos caso existam
date_range = agg_pivot.columns

# --- Temperatura ---
try:
    df_temp_raw = pd.read_excel("data/alqueva_temp.xlsx")
    df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
except FileNotFoundError:
    df_temp_raw = pd.DataFrame({
        'data': date_range,
        'med': 15 + 10*np.sin(np.pi*2*(date_range-date_range.min()).days/365) + np.random.randn(len(date_range))*2
    })
df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
temp_series = df_temp_raw.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': temp_series.index, 'med': temp_series.values})
window = 13 if len(df_temp) >= 13 else max(3, len(df_temp)//2*2+1)
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), window_length=window, polyorder=2)

# --- Nível ---
try:
    df_nivel_raw = pd.read_excel("data/alqueva_nivel.xlsx")
    df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
except FileNotFoundError:
    df_nivel_raw = pd.DataFrame({
        'data': pd.date_range(date_range.min(), date_range.max(), freq='D'),
        'nivel': 150 + 5 * np.cos(np.pi * 2 * (pd.date_range(date_range.min(), date_range.max(), freq='D') - date_range.min()).days / 365)
                 + np.random.randn(len(pd.date_range(date_range.min(), date_range.max(), freq='D'))) * 0.5
    })
df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
df_nivel_raw_monthly = df_nivel_raw.set_index('data').resample('MS').mean()
nivel_series = df_nivel_raw_monthly['nivel'].reindex(date_range).interpolate().ffill().bfill()
df_nivel = pd.DataFrame({'data': nivel_series.index, 'nivel': nivel_series.values})
window_smooth = 13 if len(date_range) >= 13 else max(3, len(date_range)//2*2+1)
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window_smooth, polyorder=2)

# --- Precipitação ---
try:
    df_prec_raw = pd.read_excel("data/prec.xlsx")
    df_prec_raw['data'] = pd.to_datetime(df_prec_raw['data'])
except FileNotFoundError:
    np.random.seed(42)
    df_prec_raw = pd.DataFrame({'data': date_range, 'prec': np.clip(np.random.normal(50, 30, len(date_range)),0,None)})
prec_series = df_prec_raw.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': prec_series.index, 'prec': prec_series.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks

# ==============================
# FIGURA 1: MAPA DE CLUSTERS (ISOLADO E COM CENTRÓIDES)
# ==============================
from matplotlib.lines import Line2D # Necessário para a legenda de centróides

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)

# --- 1. Plot da grelha de fundo ---
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1.0)

# --- 2. Plot dos Clusters e Centróides ---
for i in sorted(cluster_df['cluster'].unique()):
    subset = grid_sel[grid_sel['cluster'] == i]
    
    # Plot Clusters (Preenchimento)
    subset.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    
    # Plot Centróides
    gdf_centroids = subset.copy()
    gdf_centroids.geometry = gdf_centroids.geometry.centroid
    gdf_centroids.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)


# --- 3. Mapa Base ---
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# --- 4. Legenda ---
legend_handles_map = [plt.Rectangle((0, 0), 1, 1, fc=cluster_colors[i], alpha=0.4) for i in sorted(cluster_df['cluster'].unique())]
legend_labels_map = [f'Cluster {i+1} ({len(cluster_df[cluster_df["cluster"] == i])} células)' for i in sorted(cluster_df['cluster'].unique())]

# Adicionar Centróides à legenda
legend_handles_map.append(Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None'))
legend_labels_map.append('Centróides da Célula')

ax_map.legend(legend_handles_map, legend_labels_map, fontsize=10, loc='upper left', title=f"Clusters de dV (K={k})")
#ax_map.set_title(f"Mapa de Clusters (dV) e Centróides", fontsize=15)
ax_map.set_title(f"Mapa de Clusters ({TARGET_VAR}) e Centróides", fontsize=15)

plt.tight_layout()
plt.show()


# ==============================
# FIGURA 2: SÉRIES TEMPORAIS (CORRIGIDO: Y2 APENAS NA ÚLTIMA COLUNA)
# ==============================
print("Gerando Figura 2...")
clusters_present = sorted(cluster_df['cluster'].unique())
nc = len(clusters_present)
nr = 5

fig_series = plt.figure(figsize=(5.5 * nc, 3.0 * nr))
gs_series = fig_series.add_gridspec(nr, nc, height_ratios=[3.0] * nr)

dV_min = agg[TARGET_VAR].min()
dV_max = agg[TARGET_VAR].max()
dV_ylim = (dV_min - (dV_max-dV_min)*0.1, dV_max + (dV_max-dV_min)*0.1)

# Séries visuais
tv, tmi, tmx = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max)
nv, nmn, nmx = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max)
ptv, ptmi, ptmx = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max)

for idx, cluster_id in enumerate(clusters_present):
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)

    def plot_dV_comparison_series(row_idx, ext_data_visual, ext_data_df, ext_col_real, ext_label, ext_color, plot_as_bar=False, bar_color=None, plot_real_scale_line=False, combine_annual_prec=False, integer_ticks=False, show_background_bars=False):
        ax = fig_series.add_subplot(gs_series[row_idx, idx])
        
        # Plot dV
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7, lw=0.5)
        ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=1.0, label=f'Média Cluster {cluster_id+1}')
        
        # Plot Eixo Secundário
        ax2 = ax.twinx()
        lines2, labels2 = [], []
        ext_color_label = ext_color
        
        # Lógica de plotagem (igual à anterior)
        if combine_annual_prec:
            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            for y in df_prec['ano_hidrologico'].unique():
                g = df_prec[df_prec['ano_hidrologico'] == y]
                ax2.plot(g['data'], g['prec_acum_anual'], color='teal', lw=2)
            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            lines2 = [plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Prec. mensal (mm)'), Line2D([0],[0], color='teal', linewidth=2, label='Prec. acumulada anual (mm)')]
            ext_color_label = 'teal'

        elif plot_as_bar:
            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15, color=bar_color or ext_color, alpha=0.5)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)]
            ext_color_label = bar_color or ext_color

        elif plot_real_scale_line:
            if show_background_bars: ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real], color=ext_color, linewidth=1.5, alpha=0.85)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Prec. mensal (mm)'), Line2D([0],[0], color=ext_color, linewidth=2, label=ext_label)]
            ext_color_label = ext_color 

        else: # Escalonado
            ax2.plot(ext_data_df['data'], ext_data_visual, color=ext_color, linewidth=1.5, alpha=0.85)
            vt, rt = create_visual_ticks(ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(), dV_min, dV_max)
            ax2.set_ylim(dV_ylim); ax2.set_yticks(vt)
            
            # Só definir labels dos ticks se for a última coluna
            if idx == nc - 1:
                fmt = "{:.0f}" if integer_ticks else "{:.1f}"
                ax2.set_yticklabels([fmt.format(x) for x in rt])
            else:
                ax2.set_yticklabels([]) # Ticks vazios para outras colunas
            
            lines2 = [Line2D([0],[0], color=ext_color, linewidth=1.5, label=ext_label)]
            ext_color_label = ext_color if ext_color != 'black' else 'dimgray'

        # --- Configuração Final dos Eixos ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)
        
        # Y1 (Esquerda): Só na primeira coluna
        if idx == 0: 
            ax.set_ylabel('dV (mm)')
            ax.tick_params(axis='y', left=True, labelleft=True)
        else: 
            ax.set_yticklabels([])
            ax.tick_params(axis='y', left=False) # Remove ticks visuais

        # Y2 (Direita): Só na última coluna
        if idx == nc - 1:
            clean_label = ext_label.replace(" (Escala Real)", "") if ext_label else ""
            ax2.set_ylabel(clean_label, color=ext_color_label)
            ax2.tick_params(axis='y', labelcolor=ext_color_label, right=True)
        else:
            ax2.set_yticklabels([])
            ax2.set_ylabel('')
            ax2.tick_params(axis='y', right=False) # Remove ticks visuais

        # X-Axis
        if row_idx == nr - 1: ax.xaxis.set_major_formatter(mdates.DateFormatter("'%y"))
        else: ax.set_xticklabels([])

        ax.grid(False); ax2.grid(False)
        
        # Legenda (melhorada)
        l1, lb1 = ax.get_legend_handles_labels()
        lb2 = [l.get_label() for l in lines2]
        ax.legend(l1 + lines2, lb1 + lb2, loc='best', fontsize=8, framealpha=1.0).set_zorder(100)

        return ax

    # Chamadas
    plot_dV_comparison_series(0, tv, df_temp, 'med_smooth', 'Temperatura média (°C)', 'black', integer_ticks=True)
    plot_dV_comparison_series(1, nv, df_nivel, 'nivel_smooth', 'Nível da albufeira (m)', 'navy', integer_ticks=True)
    plot_dV_comparison_series(2, None, df_prec, 'prec', 'Precipitação mensal (mm)', 'teal', plot_as_bar=True, bar_color='teal')
    plot_dV_comparison_series(3, None, df_prec, 'prec_acum', 'Precipitação total (mm)', 'teal', plot_real_scale_line=True, show_background_bars=True)
    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual', 'Precipitação acumulada anual', 'teal', combine_annual_prec=True)

plt.tight_layout()
plt.show()


# ==============================
# FIGURA 3: Decomposição de Séries Temporais por Cluster
# (Observed / Trend / Seasonal / Residual)
# ==============================
from statsmodels.tsa.seasonal import seasonal_decompose

# Preparação
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# AJUSTE: Altura reduzida para metade (de 14 para 7)
fig_series_decomp, axes = plt.subplots(4, n_clusters, figsize=(6 * max(1, n_clusters), 7))

# --- AJUSTE FINAL DE TICKS E LABELS DA FIGURA 3 ---
for row in range(4):           # 0=Observed, 1=Trend, 2=Seasonal, 3=Residual
    for col in range(n_clusters):

        ax = axes[row, col]

        # ============================
        # 1) OBSERVED (linha 0)
        # ============================
        if row == 0:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Observed
            ax.set_xticks([])
            continue

        # ============================
        # 2) TREND / SEASONAL (linhas 1 e 2)
        # ============================
        if row in [1, 2]:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Trend e Seasonal
            ax.set_xticks([])
            continue

        # ============================
        # 3) RESIDUAL (linha 3)
        # ============================
        if row == 3:
            if col == 0:
                ax.tick_params(axis='both', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # RESIDUAL mantém ticks X
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.xaxis.set_major_locator(mdates.YearLocator())
            continue


# Forçar formato 2D
if n_clusters == 1:
    axes = axes.reshape(4, 1)

# Dicionários
cluster_mean_dict = {}
decomp_dict = {}

# Listas para limites
dv_vals, trend_vals, season_vals, resid_vals = [], [], [], []
lw = 1.5

# --- PASSO 1: Cálculos ---
for cluster_id in clusters_present:
    cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cells) == 0:
        cluster_mean_dict[cluster_id] = (pd.Series(dtype=float), pd.DataFrame())
        decomp_dict[cluster_id] = None
        continue

    cluster_data = agg_pivot.loc[agg_pivot.index.intersection(cells)]
    cluster_mean = cluster_data.mean(axis=0) if not cluster_data.empty else pd.Series(dtype=float)
    cluster_mean_dict[cluster_id] = (cluster_data, cluster_mean)

    if cluster_mean.empty:
        decomp = None
    else:
        series_for_decomp = pd.Series(cluster_mean.values, index=cluster_mean.index)
        try:
            decomp = seasonal_decompose(series_for_decomp, model='additive', period=12, extrapolate_trend='freq')
        except Exception:
            decomp = None
    decomp_dict[cluster_id] = decomp

    if not cluster_mean.empty:
        dv_vals.extend(cluster_mean.values)
    if decomp is not None:
        if decomp.trend is not None: trend_vals.extend(decomp.trend.values[~np.isnan(decomp.trend.values)])
        if decomp.seasonal is not None: season_vals.extend(decomp.seasonal.values[~np.isnan(decomp.seasonal.values)])
        if decomp.resid is not None: resid_vals.extend(decomp.resid.values[~np.isnan(decomp.resid.values)])

# --- PASSO 2: Limites ---
def safe_minmax(arr):
    if len(arr) == 0: return -1.0, 1.0
    return np.nanmin(arr), np.nanmax(arr)

def with_margin(vmin, vmax, margin=0.1):
    span = vmax - vmin
    if span == 0: span = abs(vmax) if vmax != 0 else 1.0
    m = span * margin
    return vmin - m, vmax + m

dv_ylim = with_margin(*safe_minmax(dv_vals), 0.1)
trend_ylim = with_margin(*safe_minmax(trend_vals), 0.1)
season_ylim = with_margin(*safe_minmax(season_vals), 0.1)
resid_ylim = with_margin(*safe_minmax(resid_vals), 0.1)

# --- PASSO 3: Plotagem ---
for idx, cluster_id in enumerate(clusters_present):
    cluster_data, cluster_mean = cluster_mean_dict[cluster_id]
    decomp = decomp_dict[cluster_id]
    color = cluster_colors.get(cluster_id, 'black')

    # 1. Observed
    ax0 = axes[0, idx]
    if not cluster_data.empty:
        for cid in cluster_data.index:
            ax0.plot(cluster_data.columns, cluster_data.loc[cid].values, color='lightgray', alpha=0.5, linewidth=1.5)
        ax0.plot(cluster_mean.index, cluster_mean.values, color=color, linewidth=lw, label='Média')
    ax0.set_ylim(dv_ylim)
    #ax0.set_title(f'Cluster {cluster_id + 1}', fontsize=11)
    ax0.set_title(f'Seasonal Decompose - Cluster {cluster_id + 1}', fontsize=12)
    if idx == 0: ax0.set_ylabel('Observed', fontsize=10)
    ax0.grid(False)

    # 2. Trend
    ax1 = axes[1, idx]
    if decomp is not None and decomp.trend is not None:
        ax1.plot(decomp.trend.index, decomp.trend.values, color=color, linewidth=lw)
    ax1.set_ylim(trend_ylim)
    if idx == 0: ax1.set_ylabel('Trend', fontsize=10)
    ax1.grid(False)

    # 3. Seasonal
    ax2 = axes[2, idx]
    if decomp is not None and decomp.seasonal is not None:
        ax2.plot(decomp.seasonal.index, decomp.seasonal.values, color=color, linewidth=lw)
    ax2.set_ylim(season_ylim)
    if idx == 0: ax2.set_ylabel('Seasonal', fontsize=10)
    ax2.grid(False)

    # 4. Residual
    ax3 = axes[3, idx]
    if decomp is not None and decomp.resid is not None:
        ax3.scatter(decomp.resid.index, decomp.resid.values, color=color, s=8, alpha=0.7)
        ax3.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax3.set_ylim(resid_ylim)
    if idx == 0: ax3.set_ylabel('Residual', fontsize=10)
    
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax3.xaxis.set_major_locator(mdates.YearLocator())
    ax3.grid(False)

plt.tight_layout()
plt.show()

### Castelo do Bode

In [ ]:
# Escolha aqui o que quer analisar: 'dV' ou 'dH'
TARGET_VAR = 'dV'  # ou 'dV'

# número de clusters
k = 3

# grelha
grid_size = 20

# ==============================
# SCRIPT COMPLETO: Clustering e plots (MAPA + Séries SEPARADAS)
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
except Exception as e:
    print(f"Aviso: Não foi possível ler o ficheiro COS. Criando placeholder. Erro: {e}")
    #barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, 
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2754350, 2017550, 2754750, 2018250)]}, 
    crs="EPSG:3035")

# import geopandas as gpd
# from shapely.geometry import box

# COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"

# # 1. Definir a caixa da área de interesse (Castelo do Bode)
# # Coordenadas que forneceu no exemplo
# minx, miny = 2754350, 2017550
# maxx, maxy = 2754750, 2018250
# bbox_geom = box(minx, miny, maxx, maxy)

# try:
#     # 2. Ler a COS filtrando espacialmente pela bbox
#     # Isto carrega TODAS as categorias que caiam dentro desta caixa
#     # O uso de 'bbox=' no read_file é muito mais rápido que carregar tudo e recortar depois
#     barragem = gpd.read_file(COS_PATH, bbox=bbox_geom).to_crs(epsg=3035)
    
#     # Verificação de segurança: Se o bbox na leitura falhar (algumas versões antigas),
#     # carregamos tudo e fazemos clip (mais lento, mas seguro)
#     if barragem.empty:
#         print("Aviso: Leitura otimizada vazia. Tentando recorte manual...")
#         full_cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
#         barragem = gpd.clip(full_cos, bbox_geom)

#     # 3. Imprimir quais as categorias encontradas nessa zona
#     categorias_presentes = barragem['COS23_n4_L'].unique()
#     print(f"\nSucesso! Foram encontradas {len(barragem)} geometrias.")
#     print("Categorias presentes na área:")
#     for cat in categorias_presentes:
#         print(f" - {cat}")

# except Exception as e:
#     print(f"Aviso: Não foi possível ler o ficheiro COS. Criando placeholder. Erro: {e}")
    
#     # Placeholder com a mesma área
#     barragem = gpd.GeoDataFrame(
#         {'COS23_n4_L': ['Placeholder']}, 
#         geometry=[bbox_geom], 
#         crs="EPSG:3035"
#     )

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área
# ==============================
asc_file = "data/castelo_bode_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0233_IW1_VV_2019_2023_1/EGMS_L2b_147_0233_IW1_VV_2019_2023_1.csv"
desc_file = "data/castelo_bode_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0841_IW3_VV_2019_2023_1/EGMS_L2b_052_0841_IW3_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

# ALqueva
# norte_min, norte_max = 1855050, 1855850
# este_min, este_max = 2792250, 2793250

# Pedrógão
# norte_min, norte_max = 1848550, 1849250
# este_min, este_max = 2779050, 2779550

# Castelo do Bode
norte_min, norte_max = 2017550, 2018250
este_min, este_max = 2754350, 2754750

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw', 'theta_desc', 'alpha_desc'])

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    if denom == 0: return pd.Series({'dV': np.nan, 'dH': np.nan})
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)
asc_interp = asc_interp.dropna(subset=['dV','dH'])

# ==============================
# 6. Criar grelha e agregação
# ==============================
#grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix, 'cell_y': iy, 'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy], x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# Agregação: Calcula a média do TARGET_VAR (dV ou dH) 
# mas guarda o resultado numa coluna chamada 'dV' para o resto do script funcionar
agg = asc_interp.groupby(['cell_x', 'cell_y', 'date'])[[TARGET_VAR]].mean().reset_index()

# Criar cell_id
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_barragem_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)
valid_cell_ids = set()
for pt in points_gdf.geometry:
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])
grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

# ==============================
# 8. Clustering
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
#k = 5
kmeans = KMeans(n_clusters=k, random_state=0, n_init='auto')
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Dados hidro-climáticos (temperatura, nível, precipitação)
# ==============================
# Aqui usamos exatamente os placeholders ou arquivos caso existam
date_range = agg_pivot.columns

# --- Temperatura ---
try:
    df_temp_raw = pd.read_excel("data/alqueva_temp.xlsx")
    df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
except FileNotFoundError:
    df_temp_raw = pd.DataFrame({
        'data': date_range,
        'med': 15 + 10*np.sin(np.pi*2*(date_range-date_range.min()).days/365) + np.random.randn(len(date_range))*2
    })
df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
temp_series = df_temp_raw.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': temp_series.index, 'med': temp_series.values})
window = 13 if len(df_temp) >= 13 else max(3, len(df_temp)//2*2+1)
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), window_length=window, polyorder=2)

# --- Nível ---
try:
    df_nivel_raw = pd.read_excel("data/alqueva_nivel.xlsx")
    df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
except FileNotFoundError:
    df_nivel_raw = pd.DataFrame({
        'data': pd.date_range(date_range.min(), date_range.max(), freq='D'),
        'nivel': 150 + 5 * np.cos(np.pi * 2 * (pd.date_range(date_range.min(), date_range.max(), freq='D') - date_range.min()).days / 365)
                 + np.random.randn(len(pd.date_range(date_range.min(), date_range.max(), freq='D'))) * 0.5
    })
df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
df_nivel_raw_monthly = df_nivel_raw.set_index('data').resample('MS').mean()
nivel_series = df_nivel_raw_monthly['nivel'].reindex(date_range).interpolate().ffill().bfill()
df_nivel = pd.DataFrame({'data': nivel_series.index, 'nivel': nivel_series.values})
window_smooth = 13 if len(date_range) >= 13 else max(3, len(date_range)//2*2+1)
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window_smooth, polyorder=2)

# --- Precipitação ---
try:
    df_prec_raw = pd.read_excel("data/prec.xlsx")
    df_prec_raw['data'] = pd.to_datetime(df_prec_raw['data'])
except FileNotFoundError:
    np.random.seed(42)
    df_prec_raw = pd.DataFrame({'data': date_range, 'prec': np.clip(np.random.normal(50, 30, len(date_range)),0,None)})
prec_series = df_prec_raw.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': prec_series.index, 'prec': prec_series.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks

# ==============================
# FIGURA 1: MAPA DE CLUSTERS (ISOLADO E COM CENTRÓIDES)
# ==============================
from matplotlib.lines import Line2D # Necessário para a legenda de centróides

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)

# --- 1. Plot da grelha de fundo ---
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1.0)

# --- 2. Plot dos Clusters e Centróides ---
for i in sorted(cluster_df['cluster'].unique()):
    subset = grid_sel[grid_sel['cluster'] == i]
    
    # Plot Clusters (Preenchimento)
    subset.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    
    # Plot Centróides
    gdf_centroids = subset.copy()
    gdf_centroids.geometry = gdf_centroids.geometry.centroid
    gdf_centroids.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)


# --- 3. Mapa Base ---
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# --- 4. Legenda ---
legend_handles_map = [plt.Rectangle((0, 0), 1, 1, fc=cluster_colors[i], alpha=0.4) for i in sorted(cluster_df['cluster'].unique())]
legend_labels_map = [f'Cluster {i+1} ({len(cluster_df[cluster_df["cluster"] == i])} células)' for i in sorted(cluster_df['cluster'].unique())]

# Adicionar Centróides à legenda
legend_handles_map.append(Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None'))
legend_labels_map.append('Centróides da Célula')

ax_map.legend(legend_handles_map, legend_labels_map, fontsize=10, loc='upper left', title=f"Clusters de dV (K={k})")
#ax_map.set_title(f"Mapa de Clusters (dV) e Centróides", fontsize=15)
ax_map.set_title(f"Mapa de Clusters ({TARGET_VAR}) e Centróides", fontsize=15)

plt.tight_layout()
plt.show()


# ==============================
# FIGURA 2: SÉRIES TEMPORAIS (X-AXIS CORRIGIDO)
# ==============================
print("Gerando Figura 2...")

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5 

fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

dV_min = agg[TARGET_VAR].min()
dV_max = agg[TARGET_VAR].max()
dV_ylim = (dV_min - (dV_max-dV_min)*0.1, dV_max + (dV_max-dV_min)*0.1)

# Séries visuais
tv, tmi, tmx = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, nivel_min, nivel_max = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)


for idx, cluster_id in enumerate(clusters_present):
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)

    def plot_dV_comparison_series(row_idx, ext_data_visual, ext_data_df, ext_col_real, ext_label, ext_color, plot_as_bar=False, bar_color=None, plot_real_scale_line=False, combine_annual_prec=False, integer_ticks=False, show_background_bars=False):
        ax = fig_series.add_subplot(gs_series[row_idx, idx])
        
        # --- Plot dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7, lw=0.5)
        ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=1.0, label=f'Média Cluster {cluster_id+1}')
        
        # --- Eixo secundário ---
        ax2 = ax.twinx()
        lines2, labels2 = [], []

        if combine_annual_prec:
            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            for y in df_prec['ano_hidrologico'].unique():
                g = df_prec[df_prec['ano_hidrologico'] == y]
                ax2.plot(g['data'], g['prec_acum_anual'], color='teal', linewidth=1.5, alpha=0.9)
            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Prec. mensal (mm)'),
                Line2D([0],[0], color='teal', linewidth=1.5, label='Prec. acumulada anual (mm)')
            ]
            ext_color_label = 'teal'
        
        elif plot_as_bar:
            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15, color=bar_color or ext_color, alpha=0.5)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)]
            ext_color_label = bar_color or ext_color

        elif plot_real_scale_line:
            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real], color=ext_color, linewidth=1.5, alpha=0.85)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Prec. mensal (mm)'),
                Line2D([0],[0], color=ext_color, linewidth=1.5, label=ext_label)
            ]
            ext_color_label = ext_color 

        else:
            ax2.plot(ext_data_df['data'], ext_data_visual, color=ext_color, linewidth=1.5, alpha=0.85)
            visual_ticks, real_ticks = create_visual_ticks(ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(), dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
            ax2.set_ylim(dV_ylim); ax2.set_yticks(visual_ticks)
            if integer_ticks: ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else: ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])
            lines2 = [ax2.lines[-1]]
            ext_color_label = ext_color if ext_color != 'black' else 'dimgray'

        # --- Configuração Eixos ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)
        
        # Y1 (Esquerda)
        if idx == 0: 
            ax.set_ylabel('dV (mm)')
            ax.tick_params(axis='y', left=True, labelleft=True)
        else: 
            ax.set_yticklabels([])
            ax.tick_params(axis='y', left=False)

        # Y2 (Direita)
        if idx == n_clusters - 1:
            clean_label = ext_label.replace(" (Escala Real)", "") if ext_label else ""
            ax2.set_ylabel(clean_label, color=ext_color_label)
            ax2.tick_params(axis='y', labelcolor=ext_color_label, right=True)
        else:
            ax2.set_yticklabels([])
            ax2.set_ylabel('')
            ax2.tick_params(axis='y', right=False)

        # --- Eixo X (CORREÇÃO AQUI) ---
        if row_idx == n_rows_series - 1:
            # Última linha: Formato 2023, 2024 e Ticks visíveis
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.tick_params(axis='x', bottom=True, length=5, labelbottom=True)
        else:
            # Linhas superiores: Sem texto e sem ticks
            ax.set_xticklabels([])
            ax.tick_params(axis='x', bottom=False, length=0, labelbottom=False)

        ax.grid(False); ax2.grid(False)
        
        l1, lb1 = ax.get_legend_handles_labels()
        lb2 = [l.get_label() for l in lines2]
        ax.legend(l1 + lines2, lb1 + lb2, loc='upper left', fontsize=8, framealpha=1.0).set_zorder(100)

        return ax

    # Chamadas
    plot_dV_comparison_series(0, tv, df_temp, 'med_smooth', 'Temperatura média (°C)', 'black', integer_ticks=True)
    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth', 'Nível da albufeira (m)', 'navy', integer_ticks=True)
    plot_dV_comparison_series(2, None, df_prec, 'prec', 'Precipitação mensal (mm)', 'teal', plot_as_bar=True, bar_color='teal')
    plot_dV_comparison_series(3, None, df_prec, 'prec_acum', 'Prec. acumulada total (mm)', 'teal', plot_real_scale_line=True, show_background_bars=True)
    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual', 'Prec. acumulada anual', 'teal', combine_annual_prec=True)

plt.tight_layout()
plt.show()

# ==============================
# FIGURA 3: Decomposição de Séries Temporais por Cluster
# (Observed / Trend / Seasonal / Residual)
# ==============================
from statsmodels.tsa.seasonal import seasonal_decompose

# Preparação
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# AJUSTE: Altura reduzida para metade (de 14 para 7)
fig_series_decomp, axes = plt.subplots(4, n_clusters, figsize=(6 * max(1, n_clusters), 7))

# --- AJUSTE FINAL DE TICKS E LABELS DA FIGURA 3 ---
for row in range(4):           # 0=Observed, 1=Trend, 2=Seasonal, 3=Residual
    for col in range(n_clusters):

        ax = axes[row, col]

        # ============================
        # 1) OBSERVED (linha 0)
        # ============================
        if row == 0:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Observed
            ax.set_xticks([])
            continue

        # ============================
        # 2) TREND / SEASONAL (linhas 1 e 2)
        # ============================
        if row in [1, 2]:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Trend e Seasonal
            ax.set_xticks([])
            continue

        # ============================
        # 3) RESIDUAL (linha 3)
        # ============================
        if row == 3:
            if col == 0:
                ax.tick_params(axis='both', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # RESIDUAL mantém ticks X
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.xaxis.set_major_locator(mdates.YearLocator())
            continue


# Forçar formato 2D
if n_clusters == 1:
    axes = axes.reshape(4, 1)

# Dicionários
cluster_mean_dict = {}
decomp_dict = {}

# Listas para limites
dv_vals, trend_vals, season_vals, resid_vals = [], [], [], []
lw = 1.5

# --- PASSO 1: Cálculos ---
for cluster_id in clusters_present:
    cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cells) == 0:
        cluster_mean_dict[cluster_id] = (pd.Series(dtype=float), pd.DataFrame())
        decomp_dict[cluster_id] = None
        continue

    cluster_data = agg_pivot.loc[agg_pivot.index.intersection(cells)]
    cluster_mean = cluster_data.mean(axis=0) if not cluster_data.empty else pd.Series(dtype=float)
    cluster_mean_dict[cluster_id] = (cluster_data, cluster_mean)

    if cluster_mean.empty:
        decomp = None
    else:
        series_for_decomp = pd.Series(cluster_mean.values, index=cluster_mean.index)
        try:
            decomp = seasonal_decompose(series_for_decomp, model='additive', period=12, extrapolate_trend='freq')
        except Exception:
            decomp = None
    decomp_dict[cluster_id] = decomp

    if not cluster_mean.empty:
        dv_vals.extend(cluster_mean.values)
    if decomp is not None:
        if decomp.trend is not None: trend_vals.extend(decomp.trend.values[~np.isnan(decomp.trend.values)])
        if decomp.seasonal is not None: season_vals.extend(decomp.seasonal.values[~np.isnan(decomp.seasonal.values)])
        if decomp.resid is not None: resid_vals.extend(decomp.resid.values[~np.isnan(decomp.resid.values)])

# --- PASSO 2: Limites ---
def safe_minmax(arr):
    if len(arr) == 0: return -1.0, 1.0
    return np.nanmin(arr), np.nanmax(arr)

def with_margin(vmin, vmax, margin=0.1):
    span = vmax - vmin
    if span == 0: span = abs(vmax) if vmax != 0 else 1.0
    m = span * margin
    return vmin - m, vmax + m

dv_ylim = with_margin(*safe_minmax(dv_vals), 0.1)
trend_ylim = with_margin(*safe_minmax(trend_vals), 0.1)
season_ylim = with_margin(*safe_minmax(season_vals), 0.1)
resid_ylim = with_margin(*safe_minmax(resid_vals), 0.1)

# --- PASSO 3: Plotagem ---
for idx, cluster_id in enumerate(clusters_present):
    cluster_data, cluster_mean = cluster_mean_dict[cluster_id]
    decomp = decomp_dict[cluster_id]
    color = cluster_colors.get(cluster_id, 'black')

    # 1. Observed
    ax0 = axes[0, idx]
    if not cluster_data.empty:
        for cid in cluster_data.index:
            ax0.plot(cluster_data.columns, cluster_data.loc[cid].values, color='lightgray', alpha=0.5, linewidth=1.5)
        ax0.plot(cluster_mean.index, cluster_mean.values, color=color, linewidth=lw, label='Média')
    ax0.set_ylim(dv_ylim)
    #ax0.set_title(f'Cluster {cluster_id + 1}', fontsize=11)
    ax0.set_title(f'Seasonal Decompose - Cluster {cluster_id + 1}', fontsize=12)
    if idx == 0: ax0.set_ylabel('Observed', fontsize=10)
    ax0.grid(False)

    # 2. Trend
    ax1 = axes[1, idx]
    if decomp is not None and decomp.trend is not None:
        ax1.plot(decomp.trend.index, decomp.trend.values, color=color, linewidth=lw)
    ax1.set_ylim(trend_ylim)
    if idx == 0: ax1.set_ylabel('Trend', fontsize=10)
    ax1.grid(False)

    # 3. Seasonal
    ax2 = axes[2, idx]
    if decomp is not None and decomp.seasonal is not None:
        ax2.plot(decomp.seasonal.index, decomp.seasonal.values, color=color, linewidth=lw)
    ax2.set_ylim(season_ylim)
    if idx == 0: ax2.set_ylabel('Seasonal', fontsize=10)
    ax2.grid(False)

    # 4. Residual
    ax3 = axes[3, idx]
    if decomp is not None and decomp.resid is not None:
        ax3.scatter(decomp.resid.index, decomp.resid.values, color=color, s=8, alpha=0.7)
        ax3.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax3.set_ylim(resid_ylim)
    if idx == 0: ax3.set_ylabel('Residual', fontsize=10)
    
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax3.xaxis.set_major_locator(mdates.YearLocator())
    ax3.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# Escolha aqui o que quer analisar: 'dV' ou 'dH'
TARGET_VAR = 'dV'  # ou 'dV'

# número de clusters
k = 5

# grelha
grid_size = 50

# ==============================
# SCRIPT COMPLETO: Clustering e plots (MAPA + Séries SEPARADAS)
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
except Exception as e:
    print(f"Aviso: Não foi possível ler o ficheiro COS. Criando placeholder. Erro: {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw', 'theta_desc', 'alpha_desc'])

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    if denom == 0: return pd.Series({'dV': np.nan, 'dH': np.nan})
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)
asc_interp = asc_interp.dropna(subset=['dV','dH'])

# ==============================
# 6. Criar grelha e agregação
# ==============================
#grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix, 'cell_y': iy, 'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy], x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# Agregação: Calcula a média do TARGET_VAR (dV ou dH) 
# mas guarda o resultado numa coluna chamada 'dV' para o resto do script funcionar
agg = asc_interp.groupby(['cell_x', 'cell_y', 'date'])[[TARGET_VAR]].mean().reset_index()

# Criar cell_id
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_barragem_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)
valid_cell_ids = set()
for pt in points_gdf.geometry:
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])
grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

# ==============================
# 8. Clustering
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
#k = 5
kmeans = KMeans(n_clusters=k, random_state=0, n_init='auto')
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Dados hidro-climáticos (temperatura, nível, precipitação)
# ==============================
# Aqui usamos exatamente os placeholders ou arquivos caso existam
date_range = agg_pivot.columns

# --- Temperatura ---
try:
    df_temp_raw = pd.read_excel("data/alqueva_temp.xlsx")
    df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
except FileNotFoundError:
    df_temp_raw = pd.DataFrame({
        'data': date_range,
        'med': 15 + 10*np.sin(np.pi*2*(date_range-date_range.min()).days/365) + np.random.randn(len(date_range))*2
    })
df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
temp_series = df_temp_raw.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': temp_series.index, 'med': temp_series.values})
window = 13 if len(df_temp) >= 13 else max(3, len(df_temp)//2*2+1)
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), window_length=window, polyorder=2)

# --- Nível ---
try:
    df_nivel_raw = pd.read_excel("data/alqueva_nivel.xlsx")
    df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
except FileNotFoundError:
    df_nivel_raw = pd.DataFrame({
        'data': pd.date_range(date_range.min(), date_range.max(), freq='D'),
        'nivel': 150 + 5 * np.cos(np.pi * 2 * (pd.date_range(date_range.min(), date_range.max(), freq='D') - date_range.min()).days / 365)
                 + np.random.randn(len(pd.date_range(date_range.min(), date_range.max(), freq='D'))) * 0.5
    })
df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
df_nivel_raw_monthly = df_nivel_raw.set_index('data').resample('MS').mean()
nivel_series = df_nivel_raw_monthly['nivel'].reindex(date_range).interpolate().ffill().bfill()
df_nivel = pd.DataFrame({'data': nivel_series.index, 'nivel': nivel_series.values})
window_smooth = 13 if len(date_range) >= 13 else max(3, len(date_range)//2*2+1)
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window_smooth, polyorder=2)

# --- Precipitação ---
try:
    df_prec_raw = pd.read_excel("data/prec.xlsx")
    df_prec_raw['data'] = pd.to_datetime(df_prec_raw['data'])
except FileNotFoundError:
    np.random.seed(42)
    df_prec_raw = pd.DataFrame({'data': date_range, 'prec': np.clip(np.random.normal(50, 30, len(date_range)),0,None)})
prec_series = df_prec_raw.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': prec_series.index, 'prec': prec_series.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks

# ==============================
# FIGURA 1: MAPA DE CLUSTERS (ISOLADO E COM CENTRÓIDES)
# ==============================
from matplotlib.lines import Line2D # Necessário para a legenda de centróides

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)

# --- 1. Plot da grelha de fundo ---
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1.0)

# --- 2. Plot dos Clusters e Centróides ---
for i in sorted(cluster_df['cluster'].unique()):
    subset = grid_sel[grid_sel['cluster'] == i]
    
    # Plot Clusters (Preenchimento)
    subset.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    
    # Plot Centróides
    gdf_centroids = subset.copy()
    gdf_centroids.geometry = gdf_centroids.geometry.centroid
    gdf_centroids.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)


# --- 3. Mapa Base ---
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# --- 4. Legenda ---
legend_handles_map = [plt.Rectangle((0, 0), 1, 1, fc=cluster_colors[i], alpha=0.4) for i in sorted(cluster_df['cluster'].unique())]
legend_labels_map = [f'Cluster {i+1} ({len(cluster_df[cluster_df["cluster"] == i])} células)' for i in sorted(cluster_df['cluster'].unique())]

# Adicionar Centróides à legenda
legend_handles_map.append(Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None'))
legend_labels_map.append('Centróides da Célula')

ax_map.legend(legend_handles_map, legend_labels_map, fontsize=10, loc='upper left', title=f"Clusters de dV (K={k})")
#ax_map.set_title(f"Mapa de Clusters (dV) e Centróides", fontsize=15)
ax_map.set_title(f"Mapa de Clusters ({TARGET_VAR}) e Centróides", fontsize=15)

plt.tight_layout()
plt.show()


# ==============================
# FIGURA 2: SÉRIES TEMPORAIS (X-AXIS CORRIGIDO)
# ==============================
print("Gerando Figura 2...")

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5 

fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

dV_min = agg[TARGET_VAR].min()
dV_max = agg[TARGET_VAR].max()
dV_ylim = (dV_min - (dV_max-dV_min)*0.1, dV_max + (dV_max-dV_min)*0.1)

# Séries visuais
tv, tmi, tmx = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, nivel_min, nivel_max = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)


for idx, cluster_id in enumerate(clusters_present):
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)

    def plot_dV_comparison_series(row_idx, ext_data_visual, ext_data_df, ext_col_real, ext_label, ext_color, plot_as_bar=False, bar_color=None, plot_real_scale_line=False, combine_annual_prec=False, integer_ticks=False, show_background_bars=False):
        ax = fig_series.add_subplot(gs_series[row_idx, idx])
        
        # --- Plot dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7, lw=0.5)
        ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=1.0, label=f'Média Cluster {cluster_id+1}')
        
        # --- Eixo secundário ---
        ax2 = ax.twinx()
        lines2, labels2 = [], []

        if combine_annual_prec:
            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            for y in df_prec['ano_hidrologico'].unique():
                g = df_prec[df_prec['ano_hidrologico'] == y]
                ax2.plot(g['data'], g['prec_acum_anual'], color='teal', linewidth=1.5, alpha=0.9)
            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Prec. mensal (mm)'),
                Line2D([0],[0], color='teal', linewidth=1.5, label='Prec. acumulada anual (mm)')
            ]
            ext_color_label = 'teal'
        
        elif plot_as_bar:
            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15, color=bar_color or ext_color, alpha=0.5)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)]
            ext_color_label = bar_color or ext_color

        elif plot_real_scale_line:
            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real], color=ext_color, linewidth=1.5, alpha=0.85)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Prec. mensal (mm)'),
                Line2D([0],[0], color=ext_color, linewidth=1.5, label=ext_label)
            ]
            ext_color_label = ext_color 

        else:
            ax2.plot(ext_data_df['data'], ext_data_visual, color=ext_color, linewidth=1.5, alpha=0.85)
            visual_ticks, real_ticks = create_visual_ticks(ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(), dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
            ax2.set_ylim(dV_ylim); ax2.set_yticks(visual_ticks)
            if integer_ticks: ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else: ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])
            lines2 = [ax2.lines[-1]]
            ext_color_label = ext_color if ext_color != 'black' else 'dimgray'

        # --- Configuração Eixos ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)
        
        # Y1 (Esquerda)
        if idx == 0: 
            ax.set_ylabel('dV (mm)')
            ax.tick_params(axis='y', left=True, labelleft=True)
        else: 
            ax.set_yticklabels([])
            ax.tick_params(axis='y', left=False)

        # Y2 (Direita)
        if idx == n_clusters - 1:
            clean_label = ext_label.replace(" (Escala Real)", "") if ext_label else ""
            ax2.set_ylabel(clean_label, color=ext_color_label)
            ax2.tick_params(axis='y', labelcolor=ext_color_label, right=True)
        else:
            ax2.set_yticklabels([])
            ax2.set_ylabel('')
            ax2.tick_params(axis='y', right=False)

        # --- Eixo X (CORREÇÃO AQUI) ---
        if row_idx == n_rows_series - 1:
            # Última linha: Formato 2023, 2024 e Ticks visíveis
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.tick_params(axis='x', bottom=True, length=5, labelbottom=True)
        else:
            # Linhas superiores: Sem texto e sem ticks
            ax.set_xticklabels([])
            ax.tick_params(axis='x', bottom=False, length=0, labelbottom=False)

        ax.grid(False); ax2.grid(False)
        
        l1, lb1 = ax.get_legend_handles_labels()
        lb2 = [l.get_label() for l in lines2]
        ax.legend(l1 + lines2, lb1 + lb2, loc='upper left', fontsize=8, framealpha=1.0).set_zorder(100)

        return ax

    # Chamadas
    plot_dV_comparison_series(0, tv, df_temp, 'med_smooth', 'Temperatura média (°C)', 'black', integer_ticks=True)
    plot_dV_comparison_series(1, nv, df_nivel, 'nivel_smooth', 'Nível da albufeira (m)', 'navy', integer_ticks=True)
    plot_dV_comparison_series(2, None, df_prec, 'prec', 'Precipitação mensal (mm)', 'teal', plot_as_bar=True, bar_color='teal')
    plot_dV_comparison_series(3, None, df_prec, 'prec_acum', 'Prec. acumulada total (mm)', 'teal', plot_real_scale_line=True, show_background_bars=True)
    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual', 'Prec. acumulada anual', 'teal', combine_annual_prec=True)

plt.tight_layout()
plt.show()

# ==============================
# FIGURA 3: Decomposição de Séries Temporais por Cluster
# (Observed / Trend / Seasonal / Residual)
# ==============================
from statsmodels.tsa.seasonal import seasonal_decompose

# Preparação
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# AJUSTE: Altura reduzida para metade (de 14 para 7)
fig_series_decomp, axes = plt.subplots(4, n_clusters, figsize=(6 * max(1, n_clusters), 7))

# --- AJUSTE FINAL DE TICKS E LABELS DA FIGURA 3 ---
for row in range(4):           # 0=Observed, 1=Trend, 2=Seasonal, 3=Residual
    for col in range(n_clusters):

        ax = axes[row, col]

        # ============================
        # 1) OBSERVED (linha 0)
        # ============================
        if row == 0:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Observed
            ax.set_xticks([])
            continue

        # ============================
        # 2) TREND / SEASONAL (linhas 1 e 2)
        # ============================
        if row in [1, 2]:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Trend e Seasonal
            ax.set_xticks([])
            continue

        # ============================
        # 3) RESIDUAL (linha 3)
        # ============================
        if row == 3:
            if col == 0:
                ax.tick_params(axis='both', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # RESIDUAL mantém ticks X
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.xaxis.set_major_locator(mdates.YearLocator())
            continue


# Forçar formato 2D
if n_clusters == 1:
    axes = axes.reshape(4, 1)

# Dicionários
cluster_mean_dict = {}
decomp_dict = {}

# Listas para limites
dv_vals, trend_vals, season_vals, resid_vals = [], [], [], []
lw = 1.5

# --- PASSO 1: Cálculos ---
for cluster_id in clusters_present:
    cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cells) == 0:
        cluster_mean_dict[cluster_id] = (pd.Series(dtype=float), pd.DataFrame())
        decomp_dict[cluster_id] = None
        continue

    cluster_data = agg_pivot.loc[agg_pivot.index.intersection(cells)]
    cluster_mean = cluster_data.mean(axis=0) if not cluster_data.empty else pd.Series(dtype=float)
    cluster_mean_dict[cluster_id] = (cluster_data, cluster_mean)

    if cluster_mean.empty:
        decomp = None
    else:
        series_for_decomp = pd.Series(cluster_mean.values, index=cluster_mean.index)
        try:
            decomp = seasonal_decompose(series_for_decomp, model='additive', period=12, extrapolate_trend='freq')
        except Exception:
            decomp = None
    decomp_dict[cluster_id] = decomp

    if not cluster_mean.empty:
        dv_vals.extend(cluster_mean.values)
    if decomp is not None:
        if decomp.trend is not None: trend_vals.extend(decomp.trend.values[~np.isnan(decomp.trend.values)])
        if decomp.seasonal is not None: season_vals.extend(decomp.seasonal.values[~np.isnan(decomp.seasonal.values)])
        if decomp.resid is not None: resid_vals.extend(decomp.resid.values[~np.isnan(decomp.resid.values)])

# --- PASSO 2: Limites ---
def safe_minmax(arr):
    if len(arr) == 0: return -1.0, 1.0
    return np.nanmin(arr), np.nanmax(arr)

def with_margin(vmin, vmax, margin=0.1):
    span = vmax - vmin
    if span == 0: span = abs(vmax) if vmax != 0 else 1.0
    m = span * margin
    return vmin - m, vmax + m

dv_ylim = with_margin(*safe_minmax(dv_vals), 0.1)
trend_ylim = with_margin(*safe_minmax(trend_vals), 0.1)
season_ylim = with_margin(*safe_minmax(season_vals), 0.1)
resid_ylim = with_margin(*safe_minmax(resid_vals), 0.1)

# --- PASSO 3: Plotagem ---
for idx, cluster_id in enumerate(clusters_present):
    cluster_data, cluster_mean = cluster_mean_dict[cluster_id]
    decomp = decomp_dict[cluster_id]
    color = cluster_colors.get(cluster_id, 'black')

    # 1. Observed
    ax0 = axes[0, idx]
    if not cluster_data.empty:
        for cid in cluster_data.index:
            ax0.plot(cluster_data.columns, cluster_data.loc[cid].values, color='lightgray', alpha=0.5, linewidth=1.5)
        ax0.plot(cluster_mean.index, cluster_mean.values, color=color, linewidth=lw, label='Média')
    ax0.set_ylim(dv_ylim)
    #ax0.set_title(f'Cluster {cluster_id + 1}', fontsize=11)
    ax0.set_title(f'Seasonal Decompose - Cluster {cluster_id + 1}', fontsize=12)
    if idx == 0: ax0.set_ylabel('Observed', fontsize=10)
    ax0.grid(False)

    # 2. Trend
    ax1 = axes[1, idx]
    if decomp is not None and decomp.trend is not None:
        ax1.plot(decomp.trend.index, decomp.trend.values, color=color, linewidth=lw)
    ax1.set_ylim(trend_ylim)
    if idx == 0: ax1.set_ylabel('Trend', fontsize=10)
    ax1.grid(False)

    # 3. Seasonal
    ax2 = axes[2, idx]
    if decomp is not None and decomp.seasonal is not None:
        ax2.plot(decomp.seasonal.index, decomp.seasonal.values, color=color, linewidth=lw)
    ax2.set_ylim(season_ylim)
    if idx == 0: ax2.set_ylabel('Seasonal', fontsize=10)
    ax2.grid(False)

    # 4. Residual
    ax3 = axes[3, idx]
    if decomp is not None and decomp.resid is not None:
        ax3.scatter(decomp.resid.index, decomp.resid.values, color=color, s=8, alpha=0.7)
        ax3.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax3.set_ylim(resid_ylim)
    if idx == 0: ax3.set_ylabel('Residual', fontsize=10)
    
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax3.xaxis.set_major_locator(mdates.YearLocator())
    ax3.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# Escolha aqui o que quer analisar: 'dV' ou 'dH'
TARGET_VAR = 'dV'  # ou 'dV'

# número de clusters
k = 4

# grelha
grid_size = 100

# ==============================
# SCRIPT COMPLETO: Clustering e plots (MAPA + Séries SEPARADAS)
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
except Exception as e:
    print(f"Aviso: Não foi possível ler o ficheiro COS. Criando placeholder. Erro: {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw', 'theta_desc', 'alpha_desc'])

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    if denom == 0: return pd.Series({'dV': np.nan, 'dH': np.nan})
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)
asc_interp = asc_interp.dropna(subset=['dV','dH'])

# ==============================
# 6. Criar grelha e agregação
# ==============================
#grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix, 'cell_y': iy, 'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy], x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# Agregação: Calcula a média do TARGET_VAR (dV ou dH) 
# mas guarda o resultado numa coluna chamada 'dV' para o resto do script funcionar
# --- CORREÇÃO AQUI ---
# 1. Agregamos para uma coluna temporária chamada 'val'
agg = asc_interp.groupby(['cell_x', 'cell_y', 'date'])[[TARGET_VAR]].mean().reset_index()

# Criar cell_id
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_barragem_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)
valid_cell_ids = set()
for pt in points_gdf.geometry:
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])
grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

# ==============================
# 8. Clustering
# ==============================
#agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)

agg_pivot = agg.pivot(index='cell_id', columns='date', values=TARGET_VAR).fillna(0)
#k = 3
kmeans = KMeans(n_clusters=k, random_state=0, n_init='auto')
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Dados hidro-climáticos (temperatura, nível, precipitação)
# ==============================
# Aqui usamos exatamente os placeholders ou arquivos caso existam
date_range = agg_pivot.columns

# --- Temperatura ---
try:
    df_temp_raw = pd.read_excel("data/alqueva_temp.xlsx")
    df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
except FileNotFoundError:
    df_temp_raw = pd.DataFrame({
        'data': date_range,
        'med': 15 + 10*np.sin(np.pi*2*(date_range-date_range.min()).days/365) + np.random.randn(len(date_range))*2
    })
df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
temp_series = df_temp_raw.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': temp_series.index, 'med': temp_series.values})
window = 13 if len(df_temp) >= 13 else max(3, len(df_temp)//2*2+1)
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), window_length=window, polyorder=2)

# --- Nível ---
try:
    df_nivel_raw = pd.read_excel("data/alqueva_nivel.xlsx")
    df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
except FileNotFoundError:
    df_nivel_raw = pd.DataFrame({
        'data': pd.date_range(date_range.min(), date_range.max(), freq='D'),
        'nivel': 150 + 5 * np.cos(np.pi * 2 * (pd.date_range(date_range.min(), date_range.max(), freq='D') - date_range.min()).days / 365)
                 + np.random.randn(len(pd.date_range(date_range.min(), date_range.max(), freq='D'))) * 0.5
    })
df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
df_nivel_raw_monthly = df_nivel_raw.set_index('data').resample('MS').mean()
nivel_series = df_nivel_raw_monthly['nivel'].reindex(date_range).interpolate().ffill().bfill()
df_nivel = pd.DataFrame({'data': nivel_series.index, 'nivel': nivel_series.values})
window_smooth = 13 if len(date_range) >= 13 else max(3, len(date_range)//2*2+1)
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window_smooth, polyorder=2)

# --- Precipitação ---
try:
    df_prec_raw = pd.read_excel("data/prec.xlsx")
    df_prec_raw['data'] = pd.to_datetime(df_prec_raw['data'])
except FileNotFoundError:
    np.random.seed(42)
    df_prec_raw = pd.DataFrame({'data': date_range, 'prec': np.clip(np.random.normal(50, 30, len(date_range)),0,None)})
prec_series = df_prec_raw.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': prec_series.index, 'prec': prec_series.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks

# ==============================
# FIGURA 1: MAPA DE CLUSTERS (ISOLADO E COM CENTRÓIDES)
# ==============================
from matplotlib.lines import Line2D # Necessário para a legenda de centróides

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)

# --- 1. Plot da grelha de fundo ---
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1.0)

# --- 2. Plot dos Clusters e Centróides ---
for i in sorted(cluster_df['cluster'].unique()):
    subset = grid_sel[grid_sel['cluster'] == i]
    
    # Plot Clusters (Preenchimento)
    subset.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    
    # Plot Centróides
    gdf_centroids = subset.copy()
    gdf_centroids.geometry = gdf_centroids.geometry.centroid
    gdf_centroids.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)


# --- 3. Mapa Base ---
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# --- 4. Legenda ---
legend_handles_map = [plt.Rectangle((0, 0), 1, 1, fc=cluster_colors[i], alpha=0.4) for i in sorted(cluster_df['cluster'].unique())]
legend_labels_map = [f'Cluster {i+1} ({len(cluster_df[cluster_df["cluster"] == i])} células)' for i in sorted(cluster_df['cluster'].unique())]

# Adicionar Centróides à legenda
legend_handles_map.append(Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None'))
legend_labels_map.append('Centróides da Célula')

ax_map.legend(legend_handles_map, legend_labels_map, fontsize=10, loc='upper left', title=f"Clusters de dV (K={k})")
#ax_map.set_title(f"Mapa de Clusters (dV) e Centróides", fontsize=15)
ax_map.set_title(f"Mapa de Clusters ({TARGET_VAR}) e Centróides", fontsize=15)

plt.tight_layout()
plt.show()


# ==============================
# FIGURA 2: SÉRIES TEMPORAIS (LEGENDAS CORRIGIDAS)
# ==============================
print("Gerando Figura 2...")
from matplotlib.lines import Line2D 

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5 

fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

dV_min = agg[TARGET_VAR].min()
dV_max = agg[TARGET_VAR].max()
dV_ylim = (dV_min - (dV_max-dV_min)*0.1, dV_max + (dV_max-dV_min)*0.1)

# Séries visuais
tv, tmi, tmx = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, nivel_min, nivel_max = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)


for idx, cluster_id in enumerate(clusters_present):
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)

    def plot_dV_comparison_series(row_idx, ext_data_visual, ext_data_df, ext_col_real, ext_label, ext_color, plot_as_bar=False, bar_color=None, plot_real_scale_line=False, combine_annual_prec=False, integer_ticks=False, show_background_bars=False):
        ax = fig_series.add_subplot(gs_series[row_idx, idx])
        
        # --- Plot dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7, lw=0.5)
        ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=1.0, label=f'Média Cluster {cluster_id+1}')
        
        # --- Eixo secundário ---
        ax2 = ax.twinx()
        lines2, labels2 = [], []

        if combine_annual_prec:
            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            for y in df_prec['ano_hidrologico'].unique():
                g = df_prec[df_prec['ano_hidrologico'] == y]
                ax2.plot(g['data'], g['prec_acum_anual'], color='teal', linewidth=1.5, alpha=0.9)
            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                Line2D([0],[0], color='teal', linewidth=1.5, label='Precipitação acumulada anual (mm)')
            ]
            ext_color_label = 'teal'
        
        elif plot_as_bar:
            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15, color=bar_color or ext_color, alpha=0.5)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)]
            ext_color_label = bar_color or ext_color

        elif plot_real_scale_line:
            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real], color=ext_color, linewidth=1.5, alpha=0.85)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                Line2D([0],[0], color=ext_color, linewidth=1.5, label=ext_label)
            ]
            ext_color_label = ext_color 

        else:
            # --- CORREÇÃO AQUI: Adicionado label e criado proxy manual ---
            ax2.plot(ext_data_df['data'], ext_data_visual, color=ext_color, linewidth=1.5, alpha=0.85, label=ext_label)
            
            visual_ticks, real_ticks = create_visual_ticks(
                ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(), 
                dV_min, dV_max, scale_factor=0.20, offset_factor=0.30
            )
            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(visual_ticks)
            
            if integer_ticks:
                ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else:
                ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])
            
            # Criar linha manual para a legenda para garantir que o texto aparece
            lines2 = [Line2D([0],[0], color=ext_color, linewidth=1.5, label=ext_label)]
            ext_color_label = ext_color if ext_color != 'black' else 'dimgray'

        # --- Eixos ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)
        if idx == 0: 
            ax.set_ylabel('dV (mm)')
            ax.tick_params(axis='y', left=True, labelleft=True)
        else: 
            ax.set_yticklabels([])
            ax.tick_params(axis='y', left=False)

        # Y2 label
        if idx == n_clusters - 1:
            clean_label = ext_label.replace(" (Escala Real)", "") if ext_label else ""
            ax2.set_ylabel(clean_label, color=ext_color_label)
            ax2.tick_params(axis='y', labelcolor=ext_color_label, right=True)
        else:
            ax2.set_yticklabels([])
            ax2.set_ylabel('')
            ax2.tick_params(axis='y', right=False)

        # x-axis
        if row_idx == n_rows_series - 1: 
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.tick_params(axis='x', bottom=True, length=5, labelbottom=True)
        else: 
            ax.set_xticklabels([])
            ax.tick_params(axis='x', bottom=False, length=0, labelbottom=False)

        ax.grid(False); ax2.grid(False)
        
        # Legenda
        l1, lb1 = ax.get_legend_handles_labels()
        # Usamos os labels definidos manualmente nos blocos if/else
        lb2 = [l.get_label() for l in lines2] 
        
        # Loc 'upper left' para evitar tapar dados em baixo, ou 'best'
        ax.legend(l1 + lines2, lb1 + lb2, loc='upper left', fontsize=8, framealpha=1.0, facecolor='white', edgecolor='lightgray').set_zorder(100)

        return ax

    # Chamadas
    #plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth', 'Temperatura média (°C)', 'black', integer_ticks=True)
    plot_dV_comparison_series(0, tv, df_temp, 'med_smooth', 'Temperatura média (°C)', 'black', integer_ticks=True)
    plot_dV_comparison_series(1, nv, df_nivel, 'nivel_smooth', 'Nível da albufeira (m)', 'navy', integer_ticks=True)
    plot_dV_comparison_series(2, None, df_prec, 'prec', 'Precipitação Mensal (mm)', 'teal', plot_as_bar=True, bar_color='teal')
    plot_dV_comparison_series(3, None, df_prec, 'prec_acum', 'Precipitação Total (mm)', 'teal', plot_real_scale_line=True, show_background_bars=True)
    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual', 'Precipitação acumulada anual', 'teal', combine_annual_prec=True)

plt.tight_layout()
plt.show()

# ==============================
# FIGURA 3: Decomposição de Séries Temporais por Cluster
# (Observed / Trend / Seasonal / Residual)
# ==============================
from statsmodels.tsa.seasonal import seasonal_decompose

# Preparação
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# AJUSTE: Altura reduzida para metade (de 14 para 7)
fig_series_decomp, axes = plt.subplots(4, n_clusters, figsize=(6 * max(1, n_clusters), 7))

# --- AJUSTE FINAL DE TICKS E LABELS DA FIGURA 3 ---
for row in range(4):           # 0=Observed, 1=Trend, 2=Seasonal, 3=Residual
    for col in range(n_clusters):

        ax = axes[row, col]

        # ============================
        # 1) OBSERVED (linha 0)
        # ============================
        if row == 0:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Observed
            ax.set_xticks([])
            continue

        # ============================
        # 2) TREND / SEASONAL (linhas 1 e 2)
        # ============================
        if row in [1, 2]:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Trend e Seasonal
            ax.set_xticks([])
            continue

        # ============================
        # 3) RESIDUAL (linha 3)
        # ============================
        if row == 3:
            if col == 0:
                ax.tick_params(axis='both', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # RESIDUAL mantém ticks X
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.xaxis.set_major_locator(mdates.YearLocator())
            continue


# Forçar formato 2D
if n_clusters == 1:
    axes = axes.reshape(4, 1)

# Dicionários
cluster_mean_dict = {}
decomp_dict = {}

# Listas para limites
dv_vals, trend_vals, season_vals, resid_vals = [], [], [], []
lw = 1.5

# --- PASSO 1: Cálculos ---
for cluster_id in clusters_present:
    cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cells) == 0:
        cluster_mean_dict[cluster_id] = (pd.Series(dtype=float), pd.DataFrame())
        decomp_dict[cluster_id] = None
        continue

    cluster_data = agg_pivot.loc[agg_pivot.index.intersection(cells)]
    cluster_mean = cluster_data.mean(axis=0) if not cluster_data.empty else pd.Series(dtype=float)
    cluster_mean_dict[cluster_id] = (cluster_data, cluster_mean)

    if cluster_mean.empty:
        decomp = None
    else:
        series_for_decomp = pd.Series(cluster_mean.values, index=cluster_mean.index)
        try:
            decomp = seasonal_decompose(series_for_decomp, model='additive', period=12, extrapolate_trend='freq')
        except Exception:
            decomp = None
    decomp_dict[cluster_id] = decomp

    if not cluster_mean.empty:
        dv_vals.extend(cluster_mean.values)
    if decomp is not None:
        if decomp.trend is not None: trend_vals.extend(decomp.trend.values[~np.isnan(decomp.trend.values)])
        if decomp.seasonal is not None: season_vals.extend(decomp.seasonal.values[~np.isnan(decomp.seasonal.values)])
        if decomp.resid is not None: resid_vals.extend(decomp.resid.values[~np.isnan(decomp.resid.values)])

# --- PASSO 2: Limites ---
def safe_minmax(arr):
    if len(arr) == 0: return -1.0, 1.0
    return np.nanmin(arr), np.nanmax(arr)

def with_margin(vmin, vmax, margin=0.1):
    span = vmax - vmin
    if span == 0: span = abs(vmax) if vmax != 0 else 1.0
    m = span * margin
    return vmin - m, vmax + m

dv_ylim = with_margin(*safe_minmax(dv_vals), 0.1)
trend_ylim = with_margin(*safe_minmax(trend_vals), 0.1)
season_ylim = with_margin(*safe_minmax(season_vals), 0.1)
resid_ylim = with_margin(*safe_minmax(resid_vals), 0.1)

# --- PASSO 3: Plotagem ---
for idx, cluster_id in enumerate(clusters_present):
    cluster_data, cluster_mean = cluster_mean_dict[cluster_id]
    decomp = decomp_dict[cluster_id]
    color = cluster_colors.get(cluster_id, 'black')

    # 1. Observed
    ax0 = axes[0, idx]
    if not cluster_data.empty:
        for cid in cluster_data.index:
            ax0.plot(cluster_data.columns, cluster_data.loc[cid].values, color='lightgray', alpha=0.5, linewidth=1.5)
        ax0.plot(cluster_mean.index, cluster_mean.values, color=color, linewidth=lw, label='Média')
    ax0.set_ylim(dv_ylim)
    #ax0.set_title(f'Cluster {cluster_id + 1}', fontsize=11)
    ax0.set_title(f'Seasonal Decompose - Cluster {cluster_id + 1}', fontsize=12)
    if idx == 0: ax0.set_ylabel('Observed', fontsize=10)
    ax0.grid(False)

    # 2. Trend
    ax1 = axes[1, idx]
    if decomp is not None and decomp.trend is not None:
        ax1.plot(decomp.trend.index, decomp.trend.values, color=color, linewidth=lw)
    ax1.set_ylim(trend_ylim)
    if idx == 0: ax1.set_ylabel('Trend', fontsize=10)
    ax1.grid(False)

    # 3. Seasonal
    ax2 = axes[2, idx]
    if decomp is not None and decomp.seasonal is not None:
        ax2.plot(decomp.seasonal.index, decomp.seasonal.values, color=color, linewidth=lw)
    ax2.set_ylim(season_ylim)
    if idx == 0: ax2.set_ylabel('Seasonal', fontsize=10)
    ax2.grid(False)

    # 4. Residual
    ax3 = axes[3, idx]
    if decomp is not None and decomp.resid is not None:
        ax3.scatter(decomp.resid.index, decomp.resid.values, color=color, s=8, alpha=0.7)
        ax3.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax3.set_ylim(resid_ylim)
    if idx == 0: ax3.set_ylabel('Residual', fontsize=10)
    
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax3.xaxis.set_major_locator(mdates.YearLocator())
    ax3.grid(False)

plt.tight_layout()
plt.show()

## Deslocamento horizontal, dH

In [ ]:
# Escolha aqui o que quer analisar: 'dV' ou 'dH'
TARGET_VAR = 'dV'  # ou 'dV'

# número de clusters
k = 3

# grelha
grid_size = 20

# ==============================
# SCRIPT COMPLETO: Clustering e plots (MAPA + Séries SEPARADAS)
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
# COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
# try:
#     cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
#     barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
# except Exception as e:
#     print(f"Aviso: Não foi possível ler o ficheiro COS. Criando placeholder. Erro: {e}")
#     barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'].isin([
        'Infraestruturas de produção de energia hídrica', 
        'Equipamentos culturais',
        'Florestas de azinheira',
        'Matos',
        'Pastagens melhoradas',
        'Rede rodoviária',
        'Superfícies silvopastoris de azinheira'
    ])]
except Exception as e:
    print(f"Aviso: Usando placeholder para área. {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")


# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=15, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=3, distance_upper_bound=radius)
        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw', 'theta_desc', 'alpha_desc'])

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    if denom == 0: return pd.Series({'dV': np.nan, 'dH': np.nan})
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)
asc_interp = asc_interp.dropna(subset=['dV','dH'])

# ==============================
# 6. Criar grelha e agregação
# ==============================
#grid_size = 50 # (Ou o valor que desejar)
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix, 'cell_y': iy, 'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy], x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# --- CORREÇÃO AQUI ---
# 1. Agregamos para uma coluna temporária chamada 'val'
agg = asc_interp.groupby(['cell_x', 'cell_y', 'date'])[[TARGET_VAR]].mean().reset_index()

# Criar cell_id
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_barragem_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)
valid_cell_ids = set()
for pt in points_gdf.geometry:
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])
grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

# ==============================
# 8. Clustering
# ==============================

# Agora o pivot vai funcionar porque a coluna no 'agg' tem o mesmo nome que TARGET_VAR
agg_pivot = agg.pivot(index='cell_id', columns='date', values=TARGET_VAR).fillna(0)

kmeans = KMeans(n_clusters=k, random_state=0, n_init='auto')
cluster_labels = kmeans.fit_predict(agg_pivot)
#k = 3
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Dados hidro-climáticos (temperatura, nível, precipitação)
# ==============================
# Aqui usamos exatamente os placeholders ou arquivos caso existam
date_range = agg_pivot.columns

# --- Temperatura ---
try:
    df_temp_raw = pd.read_excel("data/alqueva_temp.xlsx")
    df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
except FileNotFoundError:
    df_temp_raw = pd.DataFrame({
        'data': date_range,
        'med': 15 + 10*np.sin(np.pi*2*(date_range-date_range.min()).days/365) + np.random.randn(len(date_range))*2
    })
df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
temp_series = df_temp_raw.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': temp_series.index, 'med': temp_series.values})
window = 13 if len(df_temp) >= 13 else max(3, len(df_temp)//2*2+1)
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), window_length=window, polyorder=2)

# --- Nível ---
try:
    df_nivel_raw = pd.read_excel("data/alqueva_nivel.xlsx")
    df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
except FileNotFoundError:
    df_nivel_raw = pd.DataFrame({
        'data': pd.date_range(date_range.min(), date_range.max(), freq='D'),
        'nivel': 150 + 5 * np.cos(np.pi * 2 * (pd.date_range(date_range.min(), date_range.max(), freq='D') - date_range.min()).days / 365)
                 + np.random.randn(len(pd.date_range(date_range.min(), date_range.max(), freq='D'))) * 0.5
    })
df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
df_nivel_raw_monthly = df_nivel_raw.set_index('data').resample('MS').mean()
nivel_series = df_nivel_raw_monthly['nivel'].reindex(date_range).interpolate().ffill().bfill()
df_nivel = pd.DataFrame({'data': nivel_series.index, 'nivel': nivel_series.values})
window_smooth = 13 if len(date_range) >= 13 else max(3, len(date_range)//2*2+1)
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window_smooth, polyorder=2)

# --- Precipitação ---
try:
    df_prec_raw = pd.read_excel("data/prec.xlsx")
    df_prec_raw['data'] = pd.to_datetime(df_prec_raw['data'])
except FileNotFoundError:
    np.random.seed(42)
    df_prec_raw = pd.DataFrame({'data': date_range, 'prec': np.clip(np.random.normal(50, 30, len(date_range)),0,None)})
prec_series = df_prec_raw.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': prec_series.index, 'prec': prec_series.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks

# ==============================
# FIGURA 1: MAPA DE CLUSTERS (ISOLADO E COM CENTRÓIDES)
# ==============================
from matplotlib.lines import Line2D # Necessário para a legenda de centróides

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)

# --- 1. Plot da grelha de fundo ---
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1.0)

# --- 2. Plot dos Clusters e Centróides ---
for i in sorted(cluster_df['cluster'].unique()):
    subset = grid_sel[grid_sel['cluster'] == i]
    
    # Plot Clusters (Preenchimento)
    subset.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    
    # Plot Centróides
    gdf_centroids = subset.copy()
    gdf_centroids.geometry = gdf_centroids.geometry.centroid
    gdf_centroids.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)


# --- 3. Mapa Base ---
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# --- 4. Legenda ---
legend_handles_map = [plt.Rectangle((0, 0), 1, 1, fc=cluster_colors[i], alpha=0.4) for i in sorted(cluster_df['cluster'].unique())]
legend_labels_map = [f'Cluster {i+1} ({len(cluster_df[cluster_df["cluster"] == i])} células)' for i in sorted(cluster_df['cluster'].unique())]

# Adicionar Centróides à legenda
legend_handles_map.append(Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None'))
legend_labels_map.append('Centróides da Célula')

ax_map.legend(legend_handles_map, legend_labels_map, fontsize=10, loc='upper left', title=f"Clusters de dV (K={k})")
#ax_map.set_title(f"Mapa de Clusters '(dV)' e Centróides", fontsize=15)
ax_map.set_title(f"Mapa de Clusters ({TARGET_VAR}) e Centróides", fontsize=15)

plt.tight_layout()
plt.show()


# ==============================
# FIGURA 2: SÉRIES TEMPORAIS (LEGENDAS CORRIGIDAS)
# ==============================
print("Gerando Figura 2...")
from matplotlib.lines import Line2D 

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5 

fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

dV_min = agg[TARGET_VAR].min()
dV_max = agg[TARGET_VAR].max()
dV_ylim = (dV_min - (dV_max-dV_min)*0.1, dV_max + (dV_max-dV_min)*0.1)

# --- Séries visuais ---
#temp_visual, _, _ = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
#nivel_visual, _, _ = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
#prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)

# Séries visuais
tv, tmi, tmx = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, nivel_min, nivel_max = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)


for idx, cluster_id in enumerate(clusters_present):
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)

    def plot_dV_comparison_series(row_idx, ext_data_visual, ext_data_df, ext_col_real, ext_label, ext_color, plot_as_bar=False, bar_color=None, plot_real_scale_line=False, combine_annual_prec=False, integer_ticks=False, show_background_bars=False):
        ax = fig_series.add_subplot(gs_series[row_idx, idx])
        
        # --- Plot dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7, lw=0.5)
        ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=1.0, label=f'Média Cluster {cluster_id+1}')
        
        # --- Eixo secundário ---
        ax2 = ax.twinx()
        lines2, labels2 = [], []

        if combine_annual_prec:
            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            for y in df_prec['ano_hidrologico'].unique():
                g = df_prec[df_prec['ano_hidrologico'] == y]
                ax2.plot(g['data'], g['prec_acum_anual'], color='teal', linewidth=1.5, alpha=0.9)
            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                Line2D([0],[0], color='teal', linewidth=1.5, label='Precipitação acumulada anual (mm)')
            ]
            ext_color_label = 'teal'
        
        elif plot_as_bar:
            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15, color=bar_color or ext_color, alpha=0.5)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)]
            ext_color_label = bar_color or ext_color

        elif plot_real_scale_line:
            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real], color=ext_color, linewidth=1.5, alpha=0.85)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                Line2D([0],[0], color=ext_color, linewidth=1.5, label=ext_label)
            ]
            ext_color_label = ext_color 

        else:
            # --- CORREÇÃO AQUI: Adicionado label e criado proxy manual ---
            ax2.plot(ext_data_df['data'], ext_data_visual, color=ext_color, linewidth=1.5, alpha=0.85, label=ext_label)
            
            visual_ticks, real_ticks = create_visual_ticks(
                ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(), 
                dV_min, dV_max, scale_factor=0.20, offset_factor=0.30
            )
            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(visual_ticks)
            
            if integer_ticks:
                ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else:
                ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])
            
            # Criar linha manual para a legenda para garantir que o texto aparece
            lines2 = [Line2D([0],[0], color=ext_color, linewidth=1.5, label=ext_label)]
            ext_color_label = ext_color if ext_color != 'black' else 'dimgray'

        # --- Eixos ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)
        if idx == 0: 
            ax.set_ylabel('dV (mm)')
            ax.tick_params(axis='y', left=True, labelleft=True)
        else: 
            ax.set_yticklabels([])
            ax.tick_params(axis='y', left=False)

        # Y2 label
        if idx == n_clusters - 1:
            clean_label = ext_label.replace(" (Escala Real)", "") if ext_label else ""
            ax2.set_ylabel(clean_label, color=ext_color_label)
            ax2.tick_params(axis='y', labelcolor=ext_color_label, right=True)
        else:
            ax2.set_yticklabels([])
            ax2.set_ylabel('')
            ax2.tick_params(axis='y', right=False)

        # x-axis
        if row_idx == n_rows_series - 1: 
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.tick_params(axis='x', bottom=True, length=5, labelbottom=True)
        else: 
            ax.set_xticklabels([])
            ax.tick_params(axis='x', bottom=False, length=0, labelbottom=False)

        ax.grid(False); ax2.grid(False)
        
        # Legenda
        l1, lb1 = ax.get_legend_handles_labels()
        # Usamos os labels definidos manualmente nos blocos if/else
        lb2 = [l.get_label() for l in lines2] 
        
        # Loc 'upper left' para evitar tapar dados em baixo, ou 'best'
        ax.legend(l1 + lines2, lb1 + lb2, loc='upper left', fontsize=8, framealpha=1.0, facecolor='white', edgecolor='lightgray').set_zorder(100)

        return ax

    # Chamadas
    #plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth', 'Temperatura média (°C)', 'black', integer_ticks=True)
    plot_dV_comparison_series(0, tv, df_temp, 'med_smooth', 'Temperatura média (°C)', 'black', integer_ticks=True)
    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth', 'Nível da albufeira (m)', 'navy', integer_ticks=True)
    plot_dV_comparison_series(2, None, df_prec, 'prec', 'Precipitação Mensal (mm)', 'teal', plot_as_bar=True, bar_color='teal')
    plot_dV_comparison_series(3, None, df_prec, 'prec_acum', 'Precipitação Total (mm)', 'teal', plot_real_scale_line=True, show_background_bars=True)
    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual', 'Precipitação acumulada anual', 'teal', combine_annual_prec=True)

plt.tight_layout()
plt.show()

# ==============================
# FIGURA 3: Decomposição de Séries Temporais por Cluster
# (Observed / Trend / Seasonal / Residual)
# ==============================
from statsmodels.tsa.seasonal import seasonal_decompose

# Preparação
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# AJUSTE: Altura reduzida para metade (de 14 para 7)
fig_series_decomp, axes = plt.subplots(4, n_clusters, figsize=(6 * max(1, n_clusters), 7))

# --- AJUSTE FINAL DE TICKS E LABELS DA FIGURA 3 ---
for row in range(4):           # 0=Observed, 1=Trend, 2=Seasonal, 3=Residual
    for col in range(n_clusters):

        ax = axes[row, col]

        # ============================
        # 1) OBSERVED (linha 0)
        # ============================
        if row == 0:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Observed
            ax.set_xticks([])
            continue

        # ============================
        # 2) TREND / SEASONAL (linhas 1 e 2)
        # ============================
        if row in [1, 2]:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Trend e Seasonal
            ax.set_xticks([])
            continue

        # ============================
        # 3) RESIDUAL (linha 3)
        # ============================
        if row == 3:
            if col == 0:
                ax.tick_params(axis='both', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # RESIDUAL mantém ticks X
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.xaxis.set_major_locator(mdates.YearLocator())
            continue


# Forçar formato 2D
if n_clusters == 1:
    axes = axes.reshape(4, 1)

# Dicionários
cluster_mean_dict = {}
decomp_dict = {}

# Listas para limites
dv_vals, trend_vals, season_vals, resid_vals = [], [], [], []
lw = 1.5

# --- PASSO 1: Cálculos ---
for cluster_id in clusters_present:
    cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cells) == 0:
        cluster_mean_dict[cluster_id] = (pd.Series(dtype=float), pd.DataFrame())
        decomp_dict[cluster_id] = None
        continue

    cluster_data = agg_pivot.loc[agg_pivot.index.intersection(cells)]
    cluster_mean = cluster_data.mean(axis=0) if not cluster_data.empty else pd.Series(dtype=float)
    cluster_mean_dict[cluster_id] = (cluster_data, cluster_mean)

    if cluster_mean.empty:
        decomp = None
    else:
        series_for_decomp = pd.Series(cluster_mean.values, index=cluster_mean.index)
        try:
            decomp = seasonal_decompose(series_for_decomp, model='additive', period=12, extrapolate_trend='freq')
        except Exception:
            decomp = None
    decomp_dict[cluster_id] = decomp

    if not cluster_mean.empty:
        dv_vals.extend(cluster_mean.values)
    if decomp is not None:
        if decomp.trend is not None: trend_vals.extend(decomp.trend.values[~np.isnan(decomp.trend.values)])
        if decomp.seasonal is not None: season_vals.extend(decomp.seasonal.values[~np.isnan(decomp.seasonal.values)])
        if decomp.resid is not None: resid_vals.extend(decomp.resid.values[~np.isnan(decomp.resid.values)])

# --- PASSO 2: Limites ---
def safe_minmax(arr):
    if len(arr) == 0: return -1.0, 1.0
    return np.nanmin(arr), np.nanmax(arr)

def with_margin(vmin, vmax, margin=0.1):
    span = vmax - vmin
    if span == 0: span = abs(vmax) if vmax != 0 else 1.0
    m = span * margin
    return vmin - m, vmax + m

dv_ylim = with_margin(*safe_minmax(dv_vals), 0.1)
trend_ylim = with_margin(*safe_minmax(trend_vals), 0.1)
season_ylim = with_margin(*safe_minmax(season_vals), 0.1)
resid_ylim = with_margin(*safe_minmax(resid_vals), 0.1)

# --- PASSO 3: Plotagem ---
for idx, cluster_id in enumerate(clusters_present):
    cluster_data, cluster_mean = cluster_mean_dict[cluster_id]
    decomp = decomp_dict[cluster_id]
    color = cluster_colors.get(cluster_id, 'black')

    # 1. Observed
    ax0 = axes[0, idx]
    if not cluster_data.empty:
        for cid in cluster_data.index:
            ax0.plot(cluster_data.columns, cluster_data.loc[cid].values, color='lightgray', alpha=0.5, linewidth=1.5)
        ax0.plot(cluster_mean.index, cluster_mean.values, color=color, linewidth=lw, label='Média')
    ax0.set_ylim(dv_ylim)
    #ax0.set_title(f'Cluster {cluster_id + 1}', fontsize=11)
    ax0.set_title(f'Seasonal Decompose - Cluster {cluster_id + 1}', fontsize=12)
    if idx == 0: ax0.set_ylabel('Observed', fontsize=10)
    ax0.grid(False)

    # 2. Trend
    ax1 = axes[1, idx]
    if decomp is not None and decomp.trend is not None:
        ax1.plot(decomp.trend.index, decomp.trend.values, color=color, linewidth=lw)
    ax1.set_ylim(trend_ylim)
    if idx == 0: ax1.set_ylabel('Trend', fontsize=10)
    ax1.grid(False)

    # 3. Seasonal
    ax2 = axes[2, idx]
    if decomp is not None and decomp.seasonal is not None:
        ax2.plot(decomp.seasonal.index, decomp.seasonal.values, color=color, linewidth=lw)
    ax2.set_ylim(season_ylim)
    if idx == 0: ax2.set_ylabel('Seasonal', fontsize=10)
    ax2.grid(False)

    # 4. Residual
    ax3 = axes[3, idx]
    if decomp is not None and decomp.resid is not None:
        ax3.scatter(decomp.resid.index, decomp.resid.values, color=color, s=8, alpha=0.7)
        ax3.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax3.set_ylim(resid_ylim)
    if idx == 0: ax3.set_ylabel('Residual', fontsize=10)
    
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax3.xaxis.set_major_locator(mdates.YearLocator())
    ax3.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import numpy as np

# 1. Definições Globais de Estilo
LINE_WIDTH = 0.4
LINE_COLOR = '#333333' 
MY_CMAP = plt.cm.RdBu # RdBu padrão: Azul é Positivo (+1), Vermelho é Negativo (-1)

sns.set_context("paper", font_scale=0.9)
sns.set_style("white")
plt.rcParams['axes.linewidth'] = LINE_WIDTH

# --- Dados (Certifica-te que estas variáveis existem no teu ambiente) ---
CLUSTER_ALVO_ID = 1 
c_cells = cluster_df[cluster_df['cluster'] == CLUSTER_ALVO_ID]['cell_id']
dV_mean = agg_pivot.loc[c_cells].mean()
df_plot = pd.DataFrame({
    'dV': dV_mean,
    'Temp': df_temp.set_index('data')['med_smooth'].reindex(dV_mean.index),
    'Niv': df_nivel.set_index('data')['nivel_smooth'].reindex(dV_mean.index),
    'Prec': df_prec.set_index('data')['prec'].reindex(dV_mean.index),
    'P_Ac': df_prec.set_index('data')['prec_acum'].reindex(dV_mean.index),
    'P_An': df_prec.set_index('data')['prec_acum_anual'].reindex(dV_mean.index)
}).dropna()

cols = df_plot.columns.tolist()
n_vars = len(cols)

# 2. Criar o PairGrid
g = sns.PairGrid(df_plot, vars=cols, diag_sharey=False, height=1.0, aspect=1.0)

# --- Funções de Desenho ---
def cor_func(x, y, **kwargs):
    r, p = pearsonr(x, y)
    ax = plt.gca()
    # Mapeamento: r=1 -> Azul, r=-1 -> Vermelho
    color_val = (r + 1) / 2
    facecolor = MY_CMAP(color_val)
    ax.set_facecolor((*facecolor[:3], 0.5))
    ax.annotate(f"{r:.2f}", xy=(0.5, 0.5), xycoords=ax.transAxes, 
                ha='center', va='center', fontsize=8, fontweight='normal')

g.map_diag(sns.histplot, kde=True, color="#2c3e50", alpha=0.2, edgecolor='white', linewidth=0.3, line_kws={'linewidth': 0.7})
# --- PARTE INFERIOR: Scatter Plots ---
g.map_lower(sns.regplot, ci=None, 
            scatter_kws={
                's': 3,                # Tamanho do ponto
                'alpha': 1.0,          # Opacidade total (sem transparência)
                'color': 'black',      # Cor base
                'facecolor': 'black',  # Preenchimento preto sólido
                'edgecolor': 'black',  # Contorno preto sólido
                'linewidths': 0        # Remove qualquer largura de linha de bordo para evitar reflexos
            }, 
            line_kws={'color': 'red', 'linewidth': 0.7})
g.map_upper(cor_func)

# 3. Uniformização Total das Linhas e Padding
for i in range(n_vars):
    for j in range(n_vars):
        ax = g.axes[i, j]
        ax.set_xlabel(""); ax.set_ylabel("")
        
        # Ajuste de Padding interno (Respiro de 25%)
        if i != j:
            x_min, x_max = df_plot[cols[j]].min(), df_plot[cols[j]].max()
            y_min, y_max = df_plot[cols[i]].min(), df_plot[cols[i]].max()
            x_range, y_range = x_max - x_min, y_max - y_min
            ax.set_xlim(x_min - 0.3 * x_range, x_max + 0.3 * x_range)
            ax.set_ylim(y_min - 0.3 * y_range, y_max + 0.3 * y_range)

        # Labels apenas nas extremidades
        if j != 0: ax.set_yticklabels([])
        if i != n_vars - 1: ax.set_xticklabels([])
        
        ax.tick_params(labelsize=6, direction='in', pad=1, width=LINE_WIDTH, color=LINE_COLOR)
        
        # Forçar todas as linhas da grelha
        for edge in ['top', 'bottom', 'left', 'right']:
            ax.spines[edge].set_visible(True)
            ax.spines[edge].set_linewidth(LINE_WIDTH)
            ax.spines[edge].set_color(LINE_COLOR)
        
        # Identificação na Diagonal em Vermelho (Sem Bold)
        if i == j:
            ax.annotate(cols[i], xy=(0.05, 0.90), xycoords='axes fraction', 
                        ha='left', va='top', fontsize=8, fontweight='normal', color='red',
                        bbox=dict(facecolor='white', alpha=0.4, edgecolor='none', pad=0))

# 4. Ajuste de Layout para Colagem
plt.subplots_adjust(hspace=0, wspace=0, left=0.1, right=0.85, bottom=0.18, top=0.95)

# 5. Colorbar Corrigida (Azul=Positivo, Vermelho=Negativo)
cax = g.fig.add_axes([0.87, 0.18, 0.02, 0.77]) 
sm = plt.cm.ScalarMappable(cmap=MY_CMAP, norm=plt.Normalize(-1, 1))
cbar = g.fig.colorbar(sm, cax=cax)
cbar.set_ticks([-1, 0, 1])
cbar.outline.set_linewidth(LINE_WIDTH)
cbar.outline.set_edgecolor(LINE_COLOR)
cbar.set_label('Correlation coefficient', rotation=270, labelpad=15, fontsize=9)
cbar.ax.tick_params(labelsize=7, width=LINE_WIDTH, color=LINE_COLOR)

# 6. Legenda Inferior (Recuperada e Colada)
ax_table = g.fig.add_axes([0.1, 0.06, 0.75, 0.12]) 
ax_table.axis('off')

legend_data = [
    ["dV: Vertical Displacement", "Temp: Temperature", "Niv: Reservoir Level"],
    ["Prec: Daily Precipitation", "P_Ac: Accumulated Prec.", "P_An: Annual Acc. Prec."]
]

table = ax_table.table(cellText=legend_data, loc='upper center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.4) 

for (row, col), cell in table.get_celld().items():
    cell.set_linewidth(LINE_WIDTH)
    cell.set_edgecolor(LINE_COLOR)
    cell.get_text().set_fontweight('normal')

# 7. Finalização
plt.savefig("Matriz_Correlacao_Final_Final.png", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import numpy as np
import pandas as pd

# 1. Definições Globais de Estilo
LINE_WIDTH = 0.4
LINE_COLOR = '#333333' 
MY_CMAP = plt.cm.RdBu 

sns.set_context("paper", font_scale=0.9)
sns.set_style("white")
plt.rcParams['axes.linewidth'] = LINE_WIDTH

# Identificar todos os clusters únicos
todos_clusters = sorted(cluster_df['cluster'].unique())

# ==============================================================================
# CICLO PARA EXIBIÇÃO DE TODOS OS CLUSTERS
# ==============================================================================
for cluster_id in todos_clusters:
    # 2. Filtrar dados para o cluster atual
    c_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id']
    dV_mean = agg_pivot.loc[c_cells].mean()
    
    df_plot = pd.DataFrame({
        'dV': dV_mean,
        'Temp': df_temp.set_index('data')['med_smooth'].reindex(dV_mean.index),
        'Niv': df_nivel.set_index('data')['nivel_smooth'].reindex(dV_mean.index),
        'Prec': df_prec.set_index('data')['prec'].reindex(dV_mean.index),
        'P_Ac': df_prec.set_index('data')['prec_acum'].reindex(dV_mean.index),
        'P_An': df_prec.set_index('data')['prec_acum_anual'].reindex(dV_mean.index)
    }).dropna()

    cols = df_plot.columns.tolist()
    n_vars = len(cols)

    # 3. Criar o PairGrid
    g = sns.PairGrid(df_plot, vars=cols, diag_sharey=False, height=1.1, aspect=1.0)

    # --- Funções de Desenho ---
    def cor_func(x, y, **kwargs):
        r, p = pearsonr(x, y)
        ax = plt.gca()
        color_val = (r + 1) / 2
        facecolor = MY_CMAP(color_val)
        ax.set_facecolor((*facecolor[:3], 0.5))
        ax.annotate(f"{r:.2f}", xy=(0.5, 0.5), xycoords=ax.transAxes, 
                    ha='center', va='center', fontsize=8)

    g.map_diag(sns.histplot, kde=True, color="#2c3e50", alpha=0.2, edgecolor='white', linewidth=0.3)
    g.map_lower(sns.regplot, ci=None, 
                scatter_kws={'s': 3, 'alpha': 1.0, 'color': 'black', 'facecolor': 'black', 'edgecolor': 'black', 'linewidths': 0}, 
                line_kws={'color': 'red', 'linewidth': 0.7})
    g.map_upper(cor_func)

    # 4. Ajustes Finos
    for i in range(n_vars):
        for j in range(n_vars):
            ax = g.axes[i, j]
            ax.set_xlabel(""); ax.set_ylabel("")
            
            if i != j:
                x_min, x_max = df_plot[cols[j]].min(), df_plot[cols[j]].max()
                y_min, y_max = df_plot[cols[i]].min(), df_plot[cols[i]].max()
                x_range, y_range = x_max - x_min, y_max - y_min
                ax.set_xlim(x_min - 0.3 * x_range, x_max + 0.3 * x_range)
                ax.set_ylim(y_min - 0.3 * y_range, y_max + 0.3 * y_range)

            if j != 0: ax.set_yticklabels([])
            if i != n_vars - 1: ax.set_xticklabels([])
            
            ax.tick_params(labelsize=6, direction='in', pad=1, width=LINE_WIDTH, color=LINE_COLOR)
            
            for edge in ['top', 'bottom', 'left', 'right']:
                ax.spines[edge].set_visible(True)
                ax.spines[edge].set_linewidth(LINE_WIDTH)
                ax.spines[edge].set_color(LINE_COLOR)
            
            if i == j:
                y_min, y_max = ax.get_ylim()
                ax.set_ylim(y_min, y_max * 1.3)
                ax.annotate(cols[i], xy=(0.05, 0.90), xycoords='axes fraction', 
                            ha='left', va='top', fontsize=8, color='red',
                            bbox=dict(facecolor='white', alpha=0.4, edgecolor='none', pad=0))

    # 5. Título e Layout
    g.fig.suptitle(f"Matriz de Correlação: Cluster {cluster_id + 1}", fontsize=12, y=0.98, color='#2c3e50')
    plt.subplots_adjust(hspace=0, wspace=0, left=0.1, right=0.85, bottom=0.18, top=0.94)

    # Colorbar
    cax = g.fig.add_axes([0.87, 0.18, 0.02, 0.76]) 
    sm = plt.cm.ScalarMappable(cmap=MY_CMAP, norm=plt.Normalize(-1, 1))
    cbar = g.fig.colorbar(sm, cax=cax)
    cbar.set_ticks([-1, 0, 1])
    cbar.outline.set_linewidth(LINE_WIDTH)
    cbar.set_label('Correlation coefficient', rotation=270, labelpad=12, fontsize=8)

    # Legenda Inferior
    ax_table = g.fig.add_axes([0.1, 0.06, 0.75, 0.10]) 
    ax_table.axis('off')
    legend_data = [
        ["dV: Vertical Displacement", "Temp: Temperature", "Niv: Reservoir Level"],
        ["Prec: Daily Precipitation", "P_Ac: Accumulated Prec.", "P_An: Annual Acc. Prec."]
    ]
    table = ax_table.table(cellText=legend_data, loc='center', cellLoc='left')
    table.auto_set_font_size(False)
    table.set_fontsize(7)
    table.scale(1, 1.3)
    
    for key, cell in table.get_celld().items():
        cell.set_linewidth(LINE_WIDTH)
        cell.set_edgecolor(LINE_COLOR)

    # Exibir a figura atual
    plt.show()

### Superfícies silvopastoris de azinheira

In [ ]:
# Escolha aqui o que quer analisar: 'dV' ou 'dH'
TARGET_VAR = 'dH'  # ou 'dV'

# número de clusters
k = 4

# grelha
grid_size = 100

# ==============================
# SCRIPT COMPLETO: Clustering e plots (MAPA + Séries SEPARADAS)
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates
from shapely.strtree import STRtree

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    #barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
    barragem = cos[cos['COS23_n4_L'] == 'Superfícies silvopastoris de azinheira']
except Exception as e:
    print(f"Aviso: Não foi possível ler o ficheiro COS. Criando placeholder. Erro: {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw', 'theta_desc', 'alpha_desc'])

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    if denom == 0: return pd.Series({'dV': np.nan, 'dH': np.nan})
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)
asc_interp = asc_interp.dropna(subset=['dV','dH'])

# ==============================
# 6. Criar grelha e agregação
# ==============================
#grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix, 'cell_y': iy, 'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy], x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# Agregação: Calcula a média do TARGET_VAR (dV ou dH) 
# mas guarda o resultado numa coluna chamada 'dV' para o resto do script funcionar
# Agregação: calcula a média do TARGET_VAR
agg = asc_interp.groupby(['cell_x', 'cell_y', 'date'])[[TARGET_VAR]].mean().reset_index()

# Criar cell_id
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 7. Recorte pela barragem mantendo células com pontos
# ==============================
points_gdf = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_barragem_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

cell_polygons = grid_barragem_recort.geometry.values
cell_ids = grid_barragem_recort['cell_id'].values
tree = STRtree(cell_polygons)
valid_cell_ids = set()
for pt in points_gdf.geometry:
    idxs = tree.query(pt)
    for idx in idxs:
        poly = cell_polygons[idx]
        if poly.contains(pt):
            valid_cell_ids.add(cell_ids[idx])
grid_barragem = grid_barragem_recort[grid_barragem_recort['cell_id'].isin(valid_cell_ids)]
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

# ==============================
# 8. Clustering
# ==============================
# Agora o pivot vai funcionar porque a coluna no 'agg' tem o mesmo nome que TARGET_VAR
agg_pivot = agg.pivot(index='cell_id', columns='date', values=TARGET_VAR).fillna(0)

kmeans = KMeans(n_clusters=k, random_state=0, n_init='auto')
cluster_labels = kmeans.fit_predict(agg_pivot)
#k = 3
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 9. Dados hidro-climáticos (temperatura, nível, precipitação)
# ==============================
# Aqui usamos exatamente os placeholders ou arquivos caso existam
date_range = agg_pivot.columns

# --- Temperatura ---
try:
    df_temp_raw = pd.read_excel("data/alqueva_temp.xlsx")
    df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
except FileNotFoundError:
    df_temp_raw = pd.DataFrame({
        'data': date_range,
        'med': 15 + 10*np.sin(np.pi*2*(date_range-date_range.min()).days/365) + np.random.randn(len(date_range))*2
    })
df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
temp_series = df_temp_raw.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': temp_series.index, 'med': temp_series.values})
window = 13 if len(df_temp) >= 13 else max(3, len(df_temp)//2*2+1)
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), window_length=window, polyorder=2)

# --- Nível ---
try:
    df_nivel_raw = pd.read_excel("data/alqueva_nivel.xlsx")
    df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
except FileNotFoundError:
    df_nivel_raw = pd.DataFrame({
        'data': pd.date_range(date_range.min(), date_range.max(), freq='D'),
        'nivel': 150 + 5 * np.cos(np.pi * 2 * (pd.date_range(date_range.min(), date_range.max(), freq='D') - date_range.min()).days / 365)
                 + np.random.randn(len(pd.date_range(date_range.min(), date_range.max(), freq='D'))) * 0.5
    })
df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
df_nivel_raw_monthly = df_nivel_raw.set_index('data').resample('MS').mean()
nivel_series = df_nivel_raw_monthly['nivel'].reindex(date_range).interpolate().ffill().bfill()
df_nivel = pd.DataFrame({'data': nivel_series.index, 'nivel': nivel_series.values})
window_smooth = 13 if len(date_range) >= 13 else max(3, len(date_range)//2*2+1)
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window_smooth, polyorder=2)

# --- Precipitação ---
try:
    df_prec_raw = pd.read_excel("data/prec.xlsx")
    df_prec_raw['data'] = pd.to_datetime(df_prec_raw['data'])
except FileNotFoundError:
    np.random.seed(42)
    df_prec_raw = pd.DataFrame({'data': date_range, 'prec': np.clip(np.random.normal(50, 30, len(date_range)),0,None)})
prec_series = df_prec_raw.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': prec_series.index, 'prec': prec_series.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks

# ==============================
# FIGURA 1: MAPA DE CLUSTERS (ISOLADO E COM CENTRÓIDES)
# ==============================
from matplotlib.lines import Line2D # Necessário para a legenda de centróides

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)

# --- 1. Plot da grelha de fundo ---
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1.0)

# --- 2. Plot dos Clusters e Centróides ---
for i in sorted(cluster_df['cluster'].unique()):
    subset = grid_sel[grid_sel['cluster'] == i]
    
    # Plot Clusters (Preenchimento)
    subset.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    
    # Plot Centróides
    gdf_centroids = subset.copy()
    gdf_centroids.geometry = gdf_centroids.geometry.centroid
    gdf_centroids.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)


# --- 3. Mapa Base ---
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# --- 4. Legenda ---
legend_handles_map = [plt.Rectangle((0, 0), 1, 1, fc=cluster_colors[i], alpha=0.4) for i in sorted(cluster_df['cluster'].unique())]
legend_labels_map = [f'Cluster {i+1} ({len(cluster_df[cluster_df["cluster"] == i])} células)' for i in sorted(cluster_df['cluster'].unique())]

# Adicionar Centróides à legenda
legend_handles_map.append(Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None'))
legend_labels_map.append('Centróides da Célula')

ax_map.legend(legend_handles_map, legend_labels_map, fontsize=10, loc='upper left', title=f"Clusters de dV (K={k})")
#ax_map.set_title(f"Mapa de Clusters (dV) e Centróides", fontsize=15)
ax_map.set_title(f"Mapa de Clusters ({TARGET_VAR}) e Centróides", fontsize=15)

plt.tight_layout()
plt.show()


# ==============================
# FIGURA 2: SÉRIES TEMPORAIS (LEGENDAS CORRIGIDAS)
# ==============================
print("Gerando Figura 2...")
from matplotlib.lines import Line2D 

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5 

fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

dV_min = agg[TARGET_VAR].min()
dV_max = agg[TARGET_VAR].max()
dV_ylim = (dV_min - (dV_max-dV_min)*0.1, dV_max + (dV_max-dV_min)*0.1)

# Séries visuais
tv, tmi, tmx = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, nivel_min, nivel_max = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)


for idx, cluster_id in enumerate(clusters_present):
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)

    def plot_dV_comparison_series(row_idx, ext_data_visual, ext_data_df, ext_col_real, ext_label, ext_color, plot_as_bar=False, bar_color=None, plot_real_scale_line=False, combine_annual_prec=False, integer_ticks=False, show_background_bars=False):
        ax = fig_series.add_subplot(gs_series[row_idx, idx])
        
        # --- Plot dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7, lw=0.5)
        ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=1.0, label=f'Média Cluster {cluster_id+1}')
        
        # --- Eixo secundário ---
        ax2 = ax.twinx()
        lines2, labels2 = [], []

        if combine_annual_prec:
            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            for y in df_prec['ano_hidrologico'].unique():
                g = df_prec[df_prec['ano_hidrologico'] == y]
                ax2.plot(g['data'], g['prec_acum_anual'], color='teal', linewidth=1.5, alpha=0.9)
            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                Line2D([0],[0], color='teal', linewidth=1.5, label='Precipitação acumulada anual (mm)')
            ]
            ext_color_label = 'teal'
        
        elif plot_as_bar:
            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15, color=bar_color or ext_color, alpha=0.5)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)]
            ext_color_label = bar_color or ext_color

        elif plot_real_scale_line:
            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)
            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real], color=ext_color, linewidth=1.5, alpha=0.85)
            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                Line2D([0],[0], color=ext_color, linewidth=1.5, label=ext_label)
            ]
            ext_color_label = ext_color 

        else:
            # --- CORREÇÃO AQUI: Adicionado label e criado proxy manual ---
            ax2.plot(ext_data_df['data'], ext_data_visual, color=ext_color, linewidth=1.5, alpha=0.85, label=ext_label)
            
            visual_ticks, real_ticks = create_visual_ticks(
                ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(), 
                dV_min, dV_max, scale_factor=0.20, offset_factor=0.30
            )
            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(visual_ticks)
            
            if integer_ticks:
                ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else:
                ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])
            
            # Criar linha manual para a legenda para garantir que o texto aparece
            lines2 = [Line2D([0],[0], color=ext_color, linewidth=1.5, label=ext_label)]
            ext_color_label = ext_color if ext_color != 'black' else 'dimgray'

        # --- Eixos ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)
        if idx == 0: 
            ax.set_ylabel('dV (mm)')
            ax.tick_params(axis='y', left=True, labelleft=True)
        else: 
            ax.set_yticklabels([])
            ax.tick_params(axis='y', left=False)

        # Y2 label
        if idx == n_clusters - 1:
            clean_label = ext_label.replace(" (Escala Real)", "") if ext_label else ""
            ax2.set_ylabel(clean_label, color=ext_color_label)
            ax2.tick_params(axis='y', labelcolor=ext_color_label, right=True)
        else:
            ax2.set_yticklabels([])
            ax2.set_ylabel('')
            ax2.tick_params(axis='y', right=False)

        # x-axis
        if row_idx == n_rows_series - 1: 
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.tick_params(axis='x', bottom=True, length=5, labelbottom=True)
        else: 
            ax.set_xticklabels([])
            ax.tick_params(axis='x', bottom=False, length=0, labelbottom=False)

        ax.grid(False); ax2.grid(False)
        
        # Legenda
        l1, lb1 = ax.get_legend_handles_labels()
        # Usamos os labels definidos manualmente nos blocos if/else
        lb2 = [l.get_label() for l in lines2] 
        
        # Loc 'upper left' para evitar tapar dados em baixo, ou 'best'
        ax.legend(l1 + lines2, lb1 + lb2, loc='upper left', fontsize=8, framealpha=1.0, facecolor='white', edgecolor='lightgray').set_zorder(100)

        return ax

    # Chamadas
    #plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth', 'Temperatura média (°C)', 'black', integer_ticks=True)
    plot_dV_comparison_series(0, tv, df_temp, 'med_smooth', 'Temperatura média (°C)', 'black', integer_ticks=True)
    plot_dV_comparison_series(1, nv, df_nivel, 'nivel_smooth', 'Nível da albufeira (m)', 'navy', integer_ticks=True)
    plot_dV_comparison_series(2, None, df_prec, 'prec', 'Precipitação Mensal (mm)', 'teal', plot_as_bar=True, bar_color='teal')
    plot_dV_comparison_series(3, None, df_prec, 'prec_acum', 'Precipitação Total (mm)', 'teal', plot_real_scale_line=True, show_background_bars=True)
    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual', 'Precipitação acumulada anual', 'teal', combine_annual_prec=True)

plt.tight_layout()
plt.show()

# ==============================
# FIGURA 3: Decomposição de Séries Temporais por Cluster
# (Observed / Trend / Seasonal / Residual)
# ==============================
from statsmodels.tsa.seasonal import seasonal_decompose

# Preparação
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# AJUSTE: Altura reduzida para metade (de 14 para 7)
fig_series_decomp, axes = plt.subplots(4, n_clusters, figsize=(6 * max(1, n_clusters), 7))

# --- AJUSTE FINAL DE TICKS E LABELS DA FIGURA 3 ---
for row in range(4):           # 0=Observed, 1=Trend, 2=Seasonal, 3=Residual
    for col in range(n_clusters):

        ax = axes[row, col]

        # ============================
        # 1) OBSERVED (linha 0)
        # ============================
        if row == 0:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Observed
            ax.set_xticks([])
            continue

        # ============================
        # 2) TREND / SEASONAL (linhas 1 e 2)
        # ============================
        if row in [1, 2]:
            if col == 0:
                ax.tick_params(axis='y', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # REMOVE TICKS X em todos os Trend e Seasonal
            ax.set_xticks([])
            continue

        # ============================
        # 3) RESIDUAL (linha 3)
        # ============================
        if row == 3:
            if col == 0:
                ax.tick_params(axis='both', labelsize=10)
            else:
                ax.set_yticks([])
                ax.set_ylabel("")
            # RESIDUAL mantém ticks X
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.xaxis.set_major_locator(mdates.YearLocator())
            continue


# Forçar formato 2D
if n_clusters == 1:
    axes = axes.reshape(4, 1)

# Dicionários
cluster_mean_dict = {}
decomp_dict = {}

# Listas para limites
dv_vals, trend_vals, season_vals, resid_vals = [], [], [], []
lw = 1.5

# --- PASSO 1: Cálculos ---
for cluster_id in clusters_present:
    cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cells) == 0:
        cluster_mean_dict[cluster_id] = (pd.Series(dtype=float), pd.DataFrame())
        decomp_dict[cluster_id] = None
        continue

    cluster_data = agg_pivot.loc[agg_pivot.index.intersection(cells)]
    cluster_mean = cluster_data.mean(axis=0) if not cluster_data.empty else pd.Series(dtype=float)
    cluster_mean_dict[cluster_id] = (cluster_data, cluster_mean)

    if cluster_mean.empty:
        decomp = None
    else:
        series_for_decomp = pd.Series(cluster_mean.values, index=cluster_mean.index)
        try:
            decomp = seasonal_decompose(series_for_decomp, model='additive', period=12, extrapolate_trend='freq')
        except Exception:
            decomp = None
    decomp_dict[cluster_id] = decomp

    if not cluster_mean.empty:
        dv_vals.extend(cluster_mean.values)
    if decomp is not None:
        if decomp.trend is not None: trend_vals.extend(decomp.trend.values[~np.isnan(decomp.trend.values)])
        if decomp.seasonal is not None: season_vals.extend(decomp.seasonal.values[~np.isnan(decomp.seasonal.values)])
        if decomp.resid is not None: resid_vals.extend(decomp.resid.values[~np.isnan(decomp.resid.values)])

# --- PASSO 2: Limites ---
def safe_minmax(arr):
    if len(arr) == 0: return -1.0, 1.0
    return np.nanmin(arr), np.nanmax(arr)

def with_margin(vmin, vmax, margin=0.1):
    span = vmax - vmin
    if span == 0: span = abs(vmax) if vmax != 0 else 1.0
    m = span * margin
    return vmin - m, vmax + m

dv_ylim = with_margin(*safe_minmax(dv_vals), 0.1)
trend_ylim = with_margin(*safe_minmax(trend_vals), 0.1)
season_ylim = with_margin(*safe_minmax(season_vals), 0.1)
resid_ylim = with_margin(*safe_minmax(resid_vals), 0.1)

# --- PASSO 3: Plotagem ---
for idx, cluster_id in enumerate(clusters_present):
    cluster_data, cluster_mean = cluster_mean_dict[cluster_id]
    decomp = decomp_dict[cluster_id]
    color = cluster_colors.get(cluster_id, 'black')

    # 1. Observed
    ax0 = axes[0, idx]
    if not cluster_data.empty:
        for cid in cluster_data.index:
            ax0.plot(cluster_data.columns, cluster_data.loc[cid].values, color='lightgray', alpha=0.5, linewidth=1.5)
        ax0.plot(cluster_mean.index, cluster_mean.values, color=color, linewidth=lw, label='Média')
    ax0.set_ylim(dv_ylim)
    #ax0.set_title(f'Cluster {cluster_id + 1}', fontsize=11)
    ax0.set_title(f'Seasonal Decompose - Cluster {cluster_id + 1}', fontsize=12)
    if idx == 0: ax0.set_ylabel('Observed', fontsize=10)
    ax0.grid(False)

    # 2. Trend
    ax1 = axes[1, idx]
    if decomp is not None and decomp.trend is not None:
        ax1.plot(decomp.trend.index, decomp.trend.values, color=color, linewidth=lw)
    ax1.set_ylim(trend_ylim)
    if idx == 0: ax1.set_ylabel('Trend', fontsize=10)
    ax1.grid(False)

    # 3. Seasonal
    ax2 = axes[2, idx]
    if decomp is not None and decomp.seasonal is not None:
        ax2.plot(decomp.seasonal.index, decomp.seasonal.values, color=color, linewidth=lw)
    ax2.set_ylim(season_ylim)
    if idx == 0: ax2.set_ylabel('Seasonal', fontsize=10)
    ax2.grid(False)

    # 4. Residual
    ax3 = axes[3, idx]
    if decomp is not None and decomp.resid is not None:
        ax3.scatter(decomp.resid.index, decomp.resid.values, color=color, s=8, alpha=0.7)
        ax3.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax3.set_ylim(resid_ylim)
    if idx == 0: ax3.set_ylabel('Residual', fontsize=10)
    
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax3.xaxis.set_major_locator(mdates.YearLocator())
    ax3.grid(False)

plt.tight_layout()
plt.show()

# DTW+hier

## Ward

In [ ]:
# Escolha aqui o que quer analisar: 'dV' ou 'dH'
TARGET_VAR = 'dV'  # ou 'dV'

# grelha
grid_size = 50

# Tipo de ligação
LINKAGE_METHOD = "ward"
# Opções possíveis:
# "complete", "average", "ward"

# ==============================================================================
# SCRIPT COMPLETO: DTW + HIERÁRQUICO (COM ESTILO VISUAL K-MEANS MANTIDO)
# ==============================================================================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
import matplotlib.dates as mdates
from shapely.strtree import STRtree
from statsmodels.tsa.seasonal import seasonal_decompose
from matplotlib.lines import Line2D

# Bibliotecas para DTW e Hierárquico
try:
    from dtaidistance import dtw
    from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
    from scipy.spatial.distance import squareform
    from sklearn.preprocessing import StandardScaler
    import seaborn as sns
    from matplotlib.colors import to_hex
except ImportError as e:
    print(f"Erro de Importação: {e}. Certifique-se de que instalou: dtaidistance, scipy, seaborn, scikit-learn")

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    #barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
    #barragem = cos[cos['COS23_n4_L'] == 'Superfícies silvopastoris de azinheira']
    #barragem = cos[cos['COS23_n4_L'] == 'Matos']
    
    # Podes adicionar ou remover categorias dentro dos parênteses retos []
    barragem = cos[cos['COS23_n4_L'].isin([
    #'Infraestruturas de produção de energia hídrica', 
    #'Albufeiras de barragens',
    #'Equipamentos culturais',
    'Florestas de azinheira',
    'Matos',
    'Pastagens melhoradas',
    #'Rede rodoviária',
    'Superfícies silvopastoris de azinheira'
])]
    
# - Albufeiras de barragens
# - Equipamentos culturais
# - Florestas de azinheira
# - Infraestruturas de produção de energia hídrica
# - Matos
# - Pastagens melhoradas
# - Rede rodoviária
# - Superfícies silvopastoris de azinheira
    
    
except Exception as e:
    print(f"Aviso: COS placeholder. {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs e Filtro
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df): return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) & (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'], value_vars=disp_cols, var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(start=max(asc_long['date'].min(), desc_long['date'].min()), end=min(asc_long['date'].max(), desc_long['date'].max()), freq='MS')

# ==============================
# 3. Interpolação
# ==============================
def interpolate_ps(df, dates):
    dfs = []
    for (x, y), g in df.groupby(['easting','northing']):
        g = g.sort_values('date')
        interp = np.interp(pd.to_datetime(dates).astype(np.int64), g['date'].astype(np.int64), g['disp'])
        dfs.append(pd.DataFrame({'easting': x, 'northing': y, 'latitude': g['latitude'].iloc[0], 'longitude': g['longitude'].iloc[0], 'date': dates, 'disp': interp, 'incidence_angle': g['incidence_angle'].iloc[0], 'track_angle': g['track_angle'].iloc[0]}))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. IDW
# ==============================
def idw(source, target, radius=50, power=3):
    out = []
    for d, src in source.groupby('date'):
        tgt = target[target['date']==d].copy()
        if src.empty or tgt.empty: continue
        tree = cKDTree(list(zip(src['easting'], src['northing'])))
        dist, idx = tree.query(list(zip(tgt['easting'], tgt['northing'])), k=5, distance_upper_bound=radius)
        vals, thetas, alphas = [], [], []
        for d_i, i_i in zip(dist, idx):
            m = np.isfinite(d_i)
            if not np.any(m): vals.append(np.nan); thetas.append(np.nan); alphas.append(np.nan); continue
            w = 1/(d_i[m]**power)
            vals.append(np.sum(w*src.iloc[i_i[m]]['disp'])/np.sum(w))
            thetas.append(np.sum(w*src.iloc[i_i[m]]['incidence_angle'])/np.sum(w))
            alphas.append(np.sum(w*src.iloc[i_i[m]]['track_angle'])/np.sum(w))
        tgt['disp_idw'] = vals; tgt['theta_desc'] = thetas; tgt['alpha_desc'] = alphas
        out.append(tgt)
    return pd.concat(out, ignore_index=True)
asc_interp = idw(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw'])

# ==============================
# 5. dV (ou dH)
# ==============================
orb_inc = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orb_inc) * np.cos(np.deg2rad(asc_interp['latitude'])))

def get_comps(row):
    ta, td, beta = np.deg2rad(row['incidence_angle']), np.deg2rad(row['theta_desc']), row['beta']
    denom = (np.cos(ta)*np.sin(td)*np.cos(beta) + np.cos(td)*np.sin(ta)*np.cos(beta))
    if denom == 0: return np.nan, np.nan
    dV = (row['disp_idw']*np.sin(ta)*np.cos(beta) + row['disp']*np.sin(td)*np.cos(beta))/denom
    dH = (row['disp_idw']*np.cos(ta) - row['disp']*np.cos(td))/denom
    return dV, dH

asc_interp[['dV', 'dH']] = asc_interp.apply(lambda x: pd.Series(get_comps(x)), axis=1)
asc_interp = asc_interp.dropna(subset=['dV', 'dH'])

# ==============================
# 6. Grelha
# ==============================
#grid_size = 100
xe = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
ye = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
xs, ys = xe - grid_size/2, ye - grid_size/2
asc_interp['cx'] = pd.cut(asc_interp['easting'], bins=xs, labels=False)
asc_interp['cy'] = pd.cut(asc_interp['northing'], bins=ys, labels=False)
asc_interp = asc_interp.dropna(subset=['cx','cy'])
asc_interp['cell_id'] = asc_interp['cx'].astype(int).astype(str)+"_"+asc_interp['cy'].astype(int).astype(str)

grid_data = [{'cell_id': f"{ix}_{iy}", 'geometry': box(xs[ix], ys[iy], xs[ix+1], ys[iy+1])} for ix in range(len(xs)-1) for iy in range(len(ys)-1)]
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# Escolha aqui se usa dV ou dH para a agregação
#TARGET_VAR = 'dV' 
agg = asc_interp.groupby(['cell_id','date']).agg(dV=(TARGET_VAR,'mean')).reset_index() # Coluna final chama-se sempre 'dV' para manter compatibilidade com resto do script

# ==============================
# 7. Recorte e Filtro
# ==============================
points_gdf = gpd.GeoDataFrame(asc_interp, geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']), crs="EPSG:3035").to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

poly = grid_recort.geometry.values; ids = grid_recort['cell_id'].values; tree = STRtree(poly)
valid = set()
for pt in points_gdf.geometry:
    for idx in tree.query(pt):
        if poly[idx].contains(pt): valid.add(ids[idx])
grid_barragem = grid_recort[grid_recort['cell_id'].isin(valid)]
agg = agg[agg['cell_id'].isin(valid)]

# ==============================
# 8. CLUSTERING (DTW + HIERÁRQUICO) - SUBSTITUI K-MEANS
# ==============================
print(">>> A calcular Matriz de Distâncias DTW...")

# 1. Preparar Matriz (Pivot)
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

if n < 2:
    print("Aviso: Menos de 2 células para agrupar. Clustering ignorado.")
    cluster_labels = np.zeros(n, dtype=int)
    num_clusters = 1
    cut_distance = 0
    Z = None
else:
    # 2. Normalizar (Z-Score) para comparar formas
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X.T).T 

    # 3. Calcular DTW
    dist_matrix = np.zeros((n, n))
    window_dtw = int(0.1 * X_scaled.shape[1]) # Janela 10%
    
    # Esta parte pode demorar dependendo do nº de células
    for i in range(n):
        for j in range(i + 1, n):
            d = dtw.distance(X_scaled[i], X_scaled[j], window=window_dtw)
            dist_matrix[i, j] = d
            dist_matrix[j, i] = d

    # 4. Clustering Hierárquico
    condensed = squareform(dist_matrix)
    #Z = linkage(condensed, method='average')
    from scipy.cluster.hierarchy import linkage, fcluster
    Z = linkage(condensed, method=LINKAGE_METHOD)

    # 5. Corte Automático (Elbow)
    last = Z[-10:, 2] # Últimas distâncias de fusão
    acceleration = np.diff(last, 2)
    try:
        k_idx = np.argmax(acceleration) + 2
        cut_distance = (last[k_idx] + last[k_idx-1]) / 2
    except:
        cut_distance = last[len(last)//2]

    # Labels
    cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
    
    # Reajustar labels para começar em 0
    u_labels = np.unique(cluster_labels)
    map_l = {old: new for new, old in enumerate(u_labels)}
    cluster_labels = np.array([map_l[x] for x in cluster_labels])
    
    num_clusters = len(u_labels)
    print(f"DTW Concluído. Corte: {cut_distance:.2f}. Clusters: {num_clusters}")

# DataFrame Final
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})
# Cores Dinâmicas (para N clusters)
palette = sns.color_palette("Set2", num_clusters) if num_clusters <= 8 else sns.color_palette("tab20", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(range(num_clusters), palette)}

grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='inner')

# ==============================
# FIGURA EXTRA: DENDROGRAMA
# ==============================
if Z is not None:
    print("Gerando Dendrograma...")
    plt.figure(figsize=(12, 5))
    dendrogram(Z, leaf_rotation=90, leaf_font_size=8, color_threshold=cut_distance)
    plt.axhline(y=cut_distance, c='k', ls='--', lw=1, label='Corte Automático')
    plt.title('Dendrograma de Clustering Hierárquico (DTW)')
    plt.xlabel('Células'); plt.ylabel('Distância')
    plt.legend(); plt.tight_layout(); plt.show()


# ==============================
# 9. Dados Hidro (IGUAL)
# ==============================
date_range = agg_pivot.columns
win = 13

# Temp
try: df_t = pd.read_excel("data/alqueva_temp.xlsx"); df_t['data']=pd.to_datetime(df_t['data'])
except: df_t = pd.DataFrame({'data': date_range, 'med': 0})
ts = df_t.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': ts.index, 'med': ts.values})
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), win, 2)

# Nível
try: 
    df_n = pd.read_excel("data/alqueva_nivel.xlsx"); df_n['data']=pd.to_datetime(df_n['data'])
    df_n['nivel'] = pd.to_numeric(df_n['nivel'], errors='coerce'); df_n = df_n.dropna(subset=['nivel'])
except: df_n = pd.DataFrame({'data': date_range, 'nivel': 0})
ns = df_n.set_index('data')['nivel'].resample('MS').mean().reindex(date_range).interpolate(limit_direction='both').ffill().bfill()
df_nivel = pd.DataFrame({'data': ns.index, 'nivel': ns.values})
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], win, 2)

# Precipitação
try: df_p = pd.read_excel("data/prec.xlsx"); df_p['data'] = pd.to_datetime(df_p['data'])
except: np.random.seed(42); df_p = pd.DataFrame({'data': date_range, 'prec': 0})
ps = df_p.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': ps.index, 'prec': ps.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções Visuais (IGUAL)
# ==============================
# def scale_vis(d, vmin, vmax, s=0.20, o=0.30):
#     d = np.array(d); mn, mx = np.nanmin(d), np.nanmax(d)
#     span = vmax - vmin
#     norm = np.zeros_like(d) if mx==mn else (d-mn)/(mx-mn)
#     return norm * span * s + (vmax - span * o), mn, mx

# def create_ticks(rmin, rmax, vmin, vmax, s=0.20, o=0.30, n=5):
#     rt = np.linspace(rmin, rmax, n)
#     span = vmax - vmin
#     nt = np.linspace(0,1,n) if rmax==rmin else (rt-rmin)/(rmax-rmin)
#     vt = nt * span * s + (vmax - span * o)
#     return vt, rt


# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks


# ==============================
# FIGURA 1: MAPA (ESTILO ORIGINAL)
# ==============================
print("Gerando Figura 1...")
clusters_present = sorted(cluster_df['cluster'].unique())
k_plot = len(clusters_present)

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', lw=1)

for i in clusters_present:
    sub = grid_sel[grid_sel['cluster'] == i]
    sub.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    cents = sub.copy(); cents.geometry = cents.geometry.centroid
    cents.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

hdl = [plt.Rectangle((0,0),1,1, fc=cluster_colors[i], alpha=0.4) for i in clusters_present]
lbl = [f'Cluster {i+1} ({len(grid_sel[grid_sel["cluster"]==i])} cel)' for i in clusters_present]
hdl.append(Line2D([0],[0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None')); lbl.append('Centróides')
ax_map.legend(hdl, lbl, loc='upper left', fontsize=10, title=f"Clusters DTW (K={k_plot})")
ax_map.set_title(f"Mapa de Clusters ({TARGET_VAR})", fontsize=15)
plt.show()

# ==============================
# FIGURA 2: SÉRIES TEMPORAIS
# ==============================

print("Gerando Figura 2...")

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5

fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

dV_min = agg[TARGET_VAR].min()
dV_max = agg[TARGET_VAR].max()
dV_margin = (dV_max - dV_min) * 0.1
dV_ylim = (dV_min - dV_margin, dV_max + dV_margin)

# --- Séries visuais ---
temp_visual, _, _ = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, _, _ = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)

# Séries visuais
# tv, tmi, tmx = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
# nivel_visual, nivel_min, nivel_max = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
# prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)


for idx, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)


    # --- Função atualizada ---
    def plot_dV_comparison_series(
        row_idx, ext_data_visual, ext_data_df, ext_col_real,
        ext_label, ext_color,
        plot_as_bar=False, bar_color=None,
        plot_real_scale_line=False,
        combine_annual_prec=False,
        integer_ticks=False, show_background_bars=False):

        ax = fig_series.add_subplot(gs_series[row_idx, idx])

        # --- dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid],
                    color='lightgray', alpha=0.7, linewidth=1.0)

        ax.plot(cluster_data.columns, cluster_mean_dV,
                color=cluster_color, linewidth=1.0,
                label=f'Média Cluster {cluster_id+1}')

        # --- Eixo Y2 ---
        ax2 = ax.twinx()
        lines2 = []

        # -------------------------
        #   CASOS DE PRECIPITAÇÃO
        # -------------------------
        if combine_annual_prec:

            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)

            for ano in df_prec['ano_hidrologico'].unique():
                grupo = df_prec[df_prec['ano_hidrologico'] == ano]
                ax2.plot(grupo['data'], grupo['prec_acum_anual'],
                         color='teal', linewidth=1.0, alpha=0.9)

            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            ax2.set_ylabel("Prec. acumulada anual", color="teal")
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=1.0, label='Prec. acumulada anual (mm)')
            ]

        elif plot_as_bar:

            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15,
                    color=bar_color or ext_color, alpha=0.5)

            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            ax2.set_ylabel(ext_label, color='teal')
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)
            ]

        elif plot_real_scale_line:

            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15,
                        color='teal', alpha=0.3)

            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real],
                     color='teal', linewidth=1.0, alpha=0.85)

            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            ax2.set_ylabel(ext_label, color='teal')
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=1.0, label=ext_label)
            ]

        # -------------------------
        #      OUTRAS SÉRIES
        # -------------------------
        else:

            ax2.plot(ext_data_df['data'], ext_data_visual,
                     color=ext_color, linewidth=1.0, alpha=0.85)

            visual_ticks, real_ticks = create_visual_ticks(
                ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(),
                dV_min, dV_max, scale_factor=0.20, offset_factor=0.30
            )

            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(visual_ticks)

            if integer_ticks:
                ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else:
                ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])

            label_color = 'black' if 'Temperatura' in ext_label else ext_color
            ax2.set_ylabel(ext_label, color=label_color)
            ax2.tick_params(axis='y', colors=label_color)

            lines2 = [ax2.lines[-1]]

        # --- Y1 sempre dV ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)

        if idx == 0:
            ax.set_ylabel('dV (mm)')
        else:
            ax.set_yticklabels([])

        # Y2 somente no último cluster
        if idx != n_clusters - 1:
            ax2.set_yticklabels([])

        ax.tick_params(left=(idx==0))
        ax2.tick_params(right=(idx==n_clusters-1))

        # --- X-axis ---
        if row_idx == n_rows_series - 1:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        else:
            ax.set_xticklabels([])

        # --- Legenda somente no primeiro e no último cluster ---
        if idx == 0 or idx == n_clusters - 1:
            lines1, labels1 = ax.get_legend_handles_labels()
            handles2, labels2 = ax2.get_legend_handles_labels()

            ax.legend(
                lines1 + handles2,
                labels1 + labels2,
                fontsize=8,
                loc='best',
                framealpha=1.0,
                facecolor='white',
                edgecolor='lightgray'
            ).set_zorder(100)

        return ax


    # ---- CHAMADA DAS 5 SÉRIES ----
    plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth',
                              'Temperatura média (°C)', 'black', integer_ticks=True)

    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth',
                              'Nível da albufeira (m)', 'navy', integer_ticks=True)

    plot_dV_comparison_series(2, None, df_prec, 'prec',
                              'Precipitação mensal (mm)', 'teal',
                              plot_as_bar=True, bar_color='teal')

    plot_dV_comparison_series(3, None, df_prec, 'prec_acum',
                              'Prec. acumulada total (mm)', 'teal',
                              plot_real_scale_line=True, show_background_bars=True)

    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual',
                              'Prec. acumulada anual', 'teal',
                              combine_annual_prec=True)


plt.tight_layout()
plt.show()


# ==============================
# FIGURA 3: DECOMPOSIÇÃO SAZONAL
# ==============================

print("Gerando Figura 3...")

n_rows = 4  # Observed, Trend, Seasonal, Residual
fig_series_decomp, axes = plt.subplots(
    n_rows, n_clusters, figsize=(6 * max(1, n_clusters), 7), sharex=False
)

for col, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    mean_series = cluster_data.mean(axis=0)

    decomposition = seasonal_decompose(mean_series, model="additive", period=12)

    series_list = [
        ("Observed", decomposition.observed),
        ("Trend", decomposition.trend),
        ("Seasonal", decomposition.seasonal),
        ("Residual", decomposition.resid)
    ]

    cluster_color = cluster_colors.get(cluster_id, "black")

    for row, (label, series) in enumerate(series_list):

        ax = axes[row, col]

        # --------------------------
        #   PLOT (agora com cores)
        # --------------------------
        ax.plot(series.index, series.values,
                color=cluster_color, linewidth=1.0)

        # --------------------------
        #  LIMITE EXTERIOR (spines)
        # --------------------------
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.8)
            spine.set_color("black")

        # --------------------------
        #     LIMITES Y
        # --------------------------
        if label != "Observed":
            ymin = np.nanmin(series)
            ymax = np.nanmax(series)
            if np.isnan(ymin) or np.isnan(ymax):
                ymin, ymax = -1, 1
            margin = (ymax - ymin) * 0.10
            ax.set_ylim(ymin - margin, ymax + margin)

        # --------------------------
        #  TÍTULOS DOS SUBPLOTS
        # --------------------------
        if row == 0:
            ax.set_title(f"Seasonal Decompose – Cluster {cluster_id + 1}",
                         fontsize=12)

        if col == 0:
            ax.set_ylabel(label, fontsize=10)
        else:
            ax.set_yticks([])

        # --------------------------
        #   EIXO X — só no último row
        # --------------------------
        if row == n_rows - 1:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.xaxis.set_major_locator(mdates.YearLocator())
            ax.tick_params(axis='x', labelrotation=0, labelsize=10)
        else:
            ax.set_xticks([])
            ax.set_xticklabels([])

plt.tight_layout()
plt.show()


In [ ]:
# Calcular correlação para cada cluster
for i in sorted(cluster_df['cluster'].unique()):
    c_cells = cluster_df[cluster_df['cluster'] == i]['cell_id']
    c_mean = agg_pivot.loc[c_cells].mean()
    
    # Calculamos a correlação entre a média do cluster e a temperatura
    correlacao = c_mean.corr(df_temp.set_index('data')['med'])
    
    print(f"Correlação Cluster {i+1} com Temperatura: {correlacao:.2f}")

In [ ]:
import numpy as np

# Criar a figura com subplots para cada cluster
fig, axes = plt.subplots(1, num_clusters, figsize=(6 * num_clusters, 5), sharey=True)

# Se houver apenas 1 cluster, axes não é uma lista, transformamos em lista
if num_clusters == 1:
    axes = [axes]

for i in range(num_clusters):
    # 1. Extrair dados do cluster
    c_cells = cluster_df[cluster_df['cluster'] == i]['cell_id']
    c_mean = agg_pivot.loc[c_cells].mean()
    
    # 2. Alinhar datas com a temperatura
    temp_series = df_temp.set_index('data')['med'].reindex(c_mean.index)
    
    # 3. Plot dos pontos
    axes[i].scatter(temp_series, c_mean, alpha=0.5, color=cluster_colors[i], edgecolors='w', s=40)
    
    # 4. Cálculo da Linha de Tendência (Regressão Linear)
    # Removemos NaNs para o cálculo
    mask = ~np.isnan(temp_series) & ~np.isnan(c_mean)
    if np.any(mask):
        m, b = np.polyfit(temp_series[mask], c_mean[mask], 1)
        axes[i].plot(temp_series, m * temp_series + b, color='black', linestyle='--', lw=2, 
                     label=f'Tendência: {m:.2f} mm/°C')
        
        # Calcular R² para a legenda
        correlation = c_mean.corr(temp_series)
        axes[i].annotate(f'r = {correlation:.2f}', xy=(0.05, 0.9), xycoords='axes fraction', 
                         fontsize=12, fontweight='bold', bbox=dict(boxstyle="round", fc="white", ec="gray"))

    # 5. Formatação
    axes[i].set_title(f'Cluster {i+1} (Correlação: {correlation:.2f})', fontsize=13)
    axes[i].set_xlabel('Temperatura ($^\circ C$)', fontsize=11)
    if i == 0:
        axes[i].set_ylabel('$dV$ (mm)', fontsize=11)
    axes[i].grid(True, linestyle=':', alpha=0.6)
    axes[i].legend(loc='lower right', fontsize=9)

plt.tight_layout()
plt.show()

o código anterior mas a definir variáveis

In [ ]:
# ==============================================================================
# CONFIGURAÇÃO DE PARÂMETROS (ALTERAR AQUI)
# ==============================================================================

# 1. Variável de Análise e Grelha
TARGET_VAR = 'dV'       # 'dV' (Vertical) ou 'dH' (Horizontal/Leste-Oeste)
GRID_SIZE  = 50         # Tamanho da célula em metros

# 2. Parâmetros de Interpolação Espacial (IDW)
IDW_RADIUS = 15        # Raio de busca para vizinhos (metros)
IDW_POWER  = 2          # Peso da distância (Power 2 = inverso do quadrado)

# 3. Parâmetros de Clustering (DTW + Hierárquico)
LINKAGE_METHOD = "average"  # Opções: "ward", "complete", "average", "single"
DTW_WINDOW_PCT = 0.05       # Janela de flexibilidade temporal (0.10 = 10% da série)
USE_NORMALIZATION = False    # True: Agrupa por FORMA | False: Agrupa por MAGNITUDE (mm)

# 4. Parâmetros do Corte Automático (Dendrograma)
N_LAST_FUSIONS = 15         # Quantas fusões finais analisar para detetar o "salto"
FALLBACK_PCT   = 0.70       # Se falhar o cálculo automático, corta a 70% da distância máx

# 5. Visualização e Gráficos
SMOOTHING_WINDOW = 13       # Janela para Savitzky-Golay (deve ser número ímpar)
SHOW_DENDROGRAM  = True     # Mostrar o gráfico de árvore do clustering

# ==============================================================================
# SCRIPT COMPLETO: ANÁLISE INSAR (ESTRUTURA DETALHADA)
# ==============================================================================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
import matplotlib.dates as mdates
from shapely.strtree import STRtree
from statsmodels.tsa.seasonal import seasonal_decompose
from matplotlib.lines import Line2D
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform
from sklearn.preprocessing import StandardScaler
import seaborn as sns
from matplotlib.colors import to_hex

# ==============================
# 0. Ler COS e definir categorias
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'].isin([
        'Infraestruturas de produção de energia hídrica'#, 
        #'Albufeiras de barragens',
        #'Equipamentos culturais',
        #'Florestas de azinheira',
        #'Matos'
    ])]
except Exception as e:
    print(f"Aviso: COS placeholder. Erro: {e}")
    barragem = gpd.GeoDataFrame({'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs e Filtrar Área
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt para Long Format
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'], 
                      value_vars=disp_cols, var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(start=max(asc_long['date'].min(), desc_long['date'].min()), 
                            end=min(asc_long['date'].max(), desc_long['date'].max()), freq='MS')

# ==============================
# 3. Interpolação Temporal
# ==============================
def interpolate_ps(df, dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y, 'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0], 'date': dates, 'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0], 'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. IDW (Interpolação Espacial)
# ==============================
def idw_interpolation(source_df, target_df, radius, power):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty: continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])), k=5, distance_upper_bound=radius)
        interpolated_disp, interpolated_theta, interpolated_alpha = [], [], []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan); interpolated_theta.append(np.nan); interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation(desc_interp, asc_interp, radius=IDW_RADIUS, power=IDW_POWER)
asc_interp = asc_interp.dropna(subset=['disp_idw'])

# ==============================
# 5. Cálculo dV e dH Real
# ==============================
orb_inc = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orb_inc) * np.cos(np.deg2rad(asc_interp['latitude'])))

def compute_dV_dH(row):
    theta_a, theta_d, beta = np.deg2rad(row['incidence_angle']), np.deg2rad(row['theta_desc']), row['beta']
    denom = (np.cos(theta_a)*np.sin(theta_d)*np.cos(beta) + np.cos(theta_d)*np.sin(theta_a)*np.cos(beta))
    if denom == 0: return pd.Series({'dV': np.nan, 'dH': np.nan})
    dV = (row['disp_idw']*np.sin(theta_a)*np.cos(beta) + row['disp']*np.sin(theta_d)*np.cos(beta)) / denom
    dH = (row['disp_idw']*np.cos(theta_a) - row['disp']*np.cos(theta_d)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH, axis=1)
asc_interp = asc_interp.dropna(subset=['dV','dH'])

# ==============================
# 6. Grelha e Agregação
# ==============================
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+GRID_SIZE, GRID_SIZE)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+GRID_SIZE, GRID_SIZE)
x_shift, y_shift = x_edges - GRID_SIZE/2, y_edges - GRID_SIZE/2

asc_interp['cx'] = pd.cut(asc_interp['easting'], bins=x_shift, labels=False)
asc_interp['cy'] = pd.cut(asc_interp['northing'], bins=y_shift, labels=False)
asc_interp = asc_interp.dropna(subset=['cx','cy'])
asc_interp['cell_id'] = asc_interp['cx'].astype(int).astype(str) + "_" + asc_interp['cy'].astype(int).astype(str)

grid_data = [{'cell_id': f"{ix}_{iy}", 'geometry': box(x_shift[ix], y_shift[iy], x_shift[ix+1], y_shift[iy+1])} 
             for ix in range(len(x_shift)-1) for iy in range(len(y_shift)-1)]
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')
agg = asc_interp.groupby(['cell_id', 'date']).agg(val=(TARGET_VAR, 'mean')).reset_index()

# ==============================
# 7. Recorte pela COS
# ==============================
points_gdf = gpd.GeoDataFrame(asc_interp, geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']), crs="EPSG:3035").to_crs(epsg=3857)
grid_3857, barragem_3857 = grid.to_crs(epsg=3857), barragem.to_crs(epsg=3857)
grid_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

cell_polygons, cell_ids_arr = grid_recort.geometry.values, grid_recort['cell_id'].values
tree = STRtree(cell_polygons)
valid_cell_ids = set()
for pt in points_gdf.geometry:
    for idx in tree.query(pt):
        if cell_polygons[idx].contains(pt): valid_cell_ids.add(cell_ids_arr[idx])

grid_barragem = grid_recort[grid_recort['cell_id'].isin(valid_cell_ids)]
agg = agg[agg['cell_id'].isin(valid_cell_ids)]

# ==============================
# 8. CLUSTERING (DTW + HIERÁRQUICO)
# ==============================
print(f">>> A calcular DTW (Normalização: {USE_NORMALIZATION})...")
agg_pivot = agg.pivot(index='cell_id', columns='date', values='val').fillna(0)
cell_ids_list, X = agg_pivot.index.tolist(), agg_pivot.values.astype(float)
n_cells = X.shape[0]

if n_cells < 2:
    print("Erro: Células insuficientes.")
else:
    # Normalização Condicional
    X_scaled = StandardScaler().fit_transform(X.T).T if USE_NORMALIZATION else X
    
    # Matriz DTW rápida
    window_dtw = int(DTW_WINDOW_PCT * X_scaled.shape[1])
    dist_matrix = dtw.distance_matrix_fast(X_scaled, window=window_dtw)
    
    # Linkage
    Z = linkage(squareform(dist_matrix), method=LINKAGE_METHOD)

    # Corte Automático Robusto
    max_dist = Z[-1, 2]
    n_analise = min(n_cells - 1, N_LAST_FUSIONS)
    last_dists = Z[-n_analise:, 2]
    accel = np.diff(last_dists, 2)
    
    if len(accel) > 0:
        k_idx = np.argmax(accel) + 2
        cut_distance = (last_dists[k_idx] + last_dists[k_idx-1]) / 2
    else:
        cut_distance = max_dist * FALLBACK_PCT

    cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
    u_labels = np.unique(cluster_labels)
    map_l = {old: new for new, old in enumerate(u_labels)}
    cluster_labels = np.array([map_l[x] for x in cluster_labels])
    num_clusters = len(u_labels)

# DataFrame Final
cluster_df = pd.DataFrame({'cell_id': cell_ids_list, 'cluster': cluster_labels})
palette = sns.color_palette("Set2", num_clusters) if num_clusters <= 8 else sns.color_palette("tab20", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(range(num_clusters), palette)}
grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='inner')

if SHOW_DENDROGRAM:
    plt.figure(figsize=(12, 5)); dendrogram(Z, color_threshold=cut_distance)
    plt.axhline(y=cut_distance, c='k', ls='--', lw=1, label='Corte Automático')
    plt.title('Dendrograma de Clustering Hierárquico (DTW)'); plt.legend(); plt.show()

# ==============================
# 9. Dados Hidro-Climáticos
# ==============================
date_range = agg_pivot.columns

def load_hydro_data(path, col, is_prec=False):
    try:
        df = pd.read_excel(path); df['data'] = pd.to_datetime(df['data'])
        if is_prec:
            s = df.set_index('data')[col].reindex(date_range, fill_value=0)
            return pd.DataFrame({'data': date_range, 'val': s.values})
        s = df.set_index('data')[col].resample('MS').mean().reindex(date_range).interpolate().ffill().bfill()
        return pd.DataFrame({'data': date_range, 'val': s.values, 'smooth': savgol_filter(s.values, SMOOTHING_WINDOW, 2)})
    except:
        return pd.DataFrame({'data': date_range, 'val': 0, 'smooth': 0})

df_temp = load_hydro_data("data/alqueva_temp.xlsx", 'med')
df_nivel = load_hydro_data("data/alqueva_nivel.xlsx", 'nivel')
df_prec_raw = load_hydro_data("data/prec.xlsx", 'prec', is_prec=True)
df_prec_raw['prec_acum'] = df_prec_raw['val'].cumsum()
df_prec_raw['ano_hidro'] = df_prec_raw['data'].apply(lambda x: x.year if x.month>=10 else x.year-1)
df_prec_raw['acum_anual'] = df_prec_raw.groupby('ano_hidro')['val'].cumsum()

# ==============================
# 10. Funções de Escalonamento Visual (O SEGREDO DOS GRÁFICOS)
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data); dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks

# ==============================
# FIGURA 1: MAPA DE CLUSTERS
# ==============================
fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1.0)

for i in sorted(cluster_df['cluster'].unique()):
    subset = grid_sel[grid_sel['cluster'] == i]
    subset.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    gdf_cents = subset.copy(); gdf_cents.geometry = gdf_cents.geometry.centroid
    gdf_cents.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
legend_h = [plt.Rectangle((0,0),1,1, fc=cluster_colors[i], alpha=0.4) for i in sorted(cluster_colors.keys())]
legend_l = [f'Cluster {i+1} ({len(cluster_df[cluster_df["cluster"]==i])} cel)' for i in sorted(cluster_colors.keys())]
ax_map.legend(legend_h, legend_l, loc='upper left', title=f"Clusters {TARGET_VAR} (K={num_clusters})")
ax_map.set_axis_off(); plt.show()

# ==============================
# FIGURA 2: SÉRIES TEMPORAIS DETALHADAS
# ==============================
print("Gerando Figura 2 detalhada...")
n_rows_series = 5
fig_series = plt.figure(figsize=(5.5 * num_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, num_clusters)

dV_min, dV_max = agg['val'].min(), agg['val'].max()
dV_ylim = (dV_min - (dV_max-dV_min)*0.1, dV_max + (dV_max-dV_min)*0.1)

temp_v, _, _ = scale_series_visual(df_temp['smooth'], dV_min, dV_max)
nivel_v, _, _ = scale_series_visual(df_nivel['smooth'], dV_min, dV_max)
prec_acum_v, _, _ = scale_series_visual(df_prec_raw['prec_acum'], dV_min, dV_max)

def plot_complex_row(row_idx, col_idx, cluster_id, ext_visual, ext_df, ext_col, label, color, mode='line'):
    ax = fig_series.add_subplot(gs_series[row_idx, col_idx])
    c_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id']
    c_data = agg_pivot.loc[c_cells]
    
    for cid in c_cells: ax.plot(c_data.columns, c_data.loc[cid], color='lightgray', alpha=0.5, lw=0.5)
    ax.plot(c_data.columns, c_data.mean(), color=cluster_colors[cluster_id], lw=1.5, label=f'Média C{cluster_id+1}')
    
    ax2 = ax.twinx()
    if mode == 'bar':
        ax2.bar(ext_df['data'], ext_df[ext_col], width=15, color=color, alpha=0.4, label=label)
        ax2.set_ylim(0, ext_df[ext_col].max()*1.2)
    elif mode == 'real_line':
        ax2.plot(ext_df['data'], ext_df[ext_col], color=color, lw=1.5, label=label)
        ax2.set_ylim(0, ext_df[ext_col].max()*1.2)
    elif mode == 'annual':
        ax2.bar(ext_df['data'], ext_df['val'], width=15, color='teal', alpha=0.3)
        for y in ext_df['ano_hidro'].unique():
            g = ext_df[ext_df['ano_hidro']==y]
            ax2.plot(g['data'], g['acum_anual'], color='teal', lw=1.5)
        ax2.set_ylim(0, ext_df['acum_anual'].max()*1.1)
    else:
        ax2.plot(ext_df['data'], ext_visual, color=color, lw=1.5, alpha=0.8, label=label)
        vt, rt = create_visual_ticks(ext_df[ext_col].min(), ext_df[ext_col].max(), dV_min, dV_max)
        ax2.set_yticks(vt); ax2.set_yticklabels([f'{t:.1f}' for t in rt])
        ax2.set_ylim(dV_ylim)

    ax.set_ylim(dV_ylim)
    if col_idx == 0: ax.set_ylabel(f'{TARGET_VAR} (mm)')
    if col_idx == num_clusters - 1: ax2.set_ylabel(label, color=color)
    else: ax2.set_yticklabels([])
    if row_idx == 0: ax.set_title(f'Cluster {cluster_id+1}')
    if row_idx < 4: ax.set_xticklabels([])
    else: ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.legend(loc='upper left', fontsize=7)

for i in range(num_clusters):
    plot_complex_row(0, i, i, temp_v, df_temp, 'val', 'Temp (°C)', 'black')
    plot_complex_row(1, i, i, nivel_v, df_nivel, 'val', 'Nível (m)', 'navy')
    plot_complex_row(2, i, i, None, df_prec_raw, 'val', 'Prec (mm)', 'teal', mode='bar')
    plot_complex_row(3, i, i, None, df_prec_raw, 'prec_acum', 'Prec Acum (mm)', 'teal', mode='real_line')
    plot_complex_row(4, i, i, None, df_prec_raw, 'acum_anual', 'Anual (mm)', 'teal', mode='annual')

plt.tight_layout(); plt.show()

# ==============================
# FIGURA 3: DECOMPOSIÇÃO SAZONAL
# ==============================
fig_decomp, axes_d = plt.subplots(4, num_clusters, figsize=(6*num_clusters, 8))
for i in range(num_clusters):
    c_mean = agg_pivot.loc[cluster_df[cluster_df['cluster']==i]['cell_id']].mean()
    dec = seasonal_decompose(c_mean, model='additive', period=12)
    
    plots = [dec.observed, dec.trend, dec.seasonal, dec.resid]
    labels_d = ['Observed', 'Trend', 'Seasonal', 'Residual']
    
    for r in range(4):
        ax = axes_d[r, i]
        ax.plot(plots[r], color=cluster_colors[i])
        if i == 0: ax.set_ylabel(labels_d[r])
        if r == 0: ax.set_title(f'Decompose Cluster {i+1}')
        if r < 3: ax.set_xticks([])
        else: ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout(); plt.show()

In [ ]:
print("\n" + "="*40)
print("ANÁLISE DE CORRELAÇÃO POR CLUSTER")
print("="*40)

for i in sorted(cluster_df['cluster'].unique()):
    # 1. Extrair média do deslocamento (dV) para o cluster
    c_cells = cluster_df[cluster_df['cluster'] == i]['cell_id']
    c_mean = agg_pivot.loc[c_cells].mean()
    
    # 2. Alinhar as variáveis externas (Temp, Nível, Prec) com as datas do InSAR
    df_corr = pd.DataFrame({'dV': c_mean})
    df_corr['Temp'] = df_temp.set_index('data')['med'].reindex(c_mean.index)
    df_corr['Nivel'] = df_nivel.set_index('data')['nivel'].reindex(c_mean.index)
    df_corr['Prec_Acum'] = df_prec.set_index('data')['prec_acum'].reindex(c_mean.index)
    
    # 3. Calcular Matriz de Correlação
    matriz = df_corr.corr()
    
    print(f"\n--- Cluster {i+1} ({len(c_cells)} células) ---")
    print(f"  vs Temperatura:    {matriz.loc['dV', 'Temp']:.2f}")
    print(f"  vs Nível Albufeira: {matriz.loc['dV', 'Nivel']:.2f}")
    print(f"  vs Prec. Acumulada: {matriz.loc['dV', 'Prec_Acum']:.2f}")

In [ ]:
# Configuração dos dados para o gráfico
vars_estudo = [
    ('Temp', df_temp.set_index('data')['med'], 'Temperatura (°C)', 'black'),
    ('Nivel', df_nivel.set_index('data')['nivel'], 'Nível da Albufeira (m)', 'navy'),
    ('Prec_Acum', df_prec.set_index('data')['prec_acum'], 'Prec. Acumulada (mm)', 'teal')
]

fig_corr, axes_corr = plt.subplots(len(vars_estudo), n_clusters, 
                                  figsize=(4 * n_clusters, 10), sharey='row')

# Ajuste para caso de apenas 1 cluster
if n_clusters == 1: axes_corr = axes_corr.reshape(-1, 1)

for row, (var_key, var_data, var_label, var_color) in enumerate(vars_estudo):
    for col, cluster_id in enumerate(sorted(cluster_df['cluster'].unique())):
        ax = axes_corr[row, col]
        
        # Dados do cluster
        c_mean = agg_pivot.loc[cluster_df[cluster_df['cluster'] == cluster_id]['cell_id']].mean()
        x_vals = var_data.reindex(c_mean.index)
        y_vals = c_mean
        
        # Limpar NaNs
        mask = ~np.isnan(x_vals) & ~np.isnan(y_vals)
        
        # Plot
        ax.scatter(x_vals[mask], y_vals[mask], alpha=0.5, color=cluster_colors[cluster_id], s=20)
        
        # Linha de tendência e valor de r
        if np.any(mask) and len(x_vals[mask]) > 1:
            m, b = np.polyfit(x_vals[mask], y_vals[mask], 1)
            ax.plot(x_vals, m*x_vals + b, color='red', linestyle='--', alpha=0.8)
            r = y_vals.corr(x_vals)
            ax.text(0.05, 0.85, f'r = {r:.2f}', transform=ax.transAxes, 
                    fontweight='bold', bbox=dict(facecolor='white', alpha=0.7))

        if col == 0: ax.set_ylabel(var_label)
        if row == 0: ax.set_title(f'Cluster {cluster_id+1}')
        ax.grid(True, linestyle=':', alpha=0.4)

plt.tight_layout()
plt.show()

com correção do dtw

In [ ]:
# Escolha aqui o que quer analisar: 'dV' ou 'dH'
TARGET_VAR = 'dV'  # ou 'dV'

# grelha
grid_size = 25

DTW_WINDOW_PCT = 0.01  # Aqui podes mudar para 0.1, 0.05, 0.03, etc.

# Tipo de ligação
LINKAGE_METHOD = "ward"
# Opções possíveis:
# "complete", "average", "ward"

# ==============================================================================
# SCRIPT COMPLETO: DTW + HIERÁRQUICO (COM ESTILO VISUAL K-MEANS MANTIDO)
# ==============================================================================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
import matplotlib.dates as mdates
from shapely.strtree import STRtree
from statsmodels.tsa.seasonal import seasonal_decompose
from matplotlib.lines import Line2D

# Bibliotecas para DTW e Hierárquico
try:
    from dtaidistance import dtw
    from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
    from scipy.spatial.distance import squareform
    from sklearn.preprocessing import StandardScaler
    import seaborn as sns
    from matplotlib.colors import to_hex
except ImportError as e:
    print(f"Erro de Importação: {e}. Certifique-se de que instalou: dtaidistance, scipy, seaborn, scikit-learn")

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    #barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
    #barragem = cos[cos['COS23_n4_L'] == 'Superfícies silvopastoris de azinheira']
    #barragem = cos[cos['COS23_n4_L'] == 'Matos']
    
    # Podes adicionar ou remover categorias dentro dos parênteses retos []
    barragem = cos[cos['COS23_n4_L'].isin([
    'Infraestruturas de produção de energia hídrica', 
    #'Albufeiras de barragens',
    #'Equipamentos culturais',
    #'Florestas de azinheira',
    #'Matos',
    #'Pastagens melhoradas',
    #'Rede rodoviária',
    'Superfícies silvopastoris de azinheira'
])]
    
# - Albufeiras de barragens
# - Equipamentos culturais
# - Florestas de azinheira
# - Infraestruturas de produção de energia hídrica
# - Matos
# - Pastagens melhoradas
# - Rede rodoviária
# - Superfícies silvopastoris de azinheira
    
    
except Exception as e:
    print(f"Aviso: COS placeholder. {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs e Filtro
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df): return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) & (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'], value_vars=disp_cols, var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(start=max(asc_long['date'].min(), desc_long['date'].min()), end=min(asc_long['date'].max(), desc_long['date'].max()), freq='MS')

# ==============================
# 3. Interpolação
# ==============================
def interpolate_ps(df, dates):
    dfs = []
    for (x, y), g in df.groupby(['easting','northing']):
        g = g.sort_values('date')
        interp = np.interp(pd.to_datetime(dates).astype(np.int64), g['date'].astype(np.int64), g['disp'])
        dfs.append(pd.DataFrame({'easting': x, 'northing': y, 'latitude': g['latitude'].iloc[0], 'longitude': g['longitude'].iloc[0], 'date': dates, 'disp': interp, 'incidence_angle': g['incidence_angle'].iloc[0], 'track_angle': g['track_angle'].iloc[0]}))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. IDW
# ==============================
def idw(source, target, radius=25, power=3):
    out = []
    for d, src in source.groupby('date'):
        tgt = target[target['date']==d].copy()
        if src.empty or tgt.empty: continue
        tree = cKDTree(list(zip(src['easting'], src['northing'])))
        dist, idx = tree.query(list(zip(tgt['easting'], tgt['northing'])), k=5, distance_upper_bound=radius)
        vals, thetas, alphas = [], [], []
        for d_i, i_i in zip(dist, idx):
            m = np.isfinite(d_i)
            if not np.any(m): vals.append(np.nan); thetas.append(np.nan); alphas.append(np.nan); continue
            w = 1/(d_i[m]**power)
            vals.append(np.sum(w*src.iloc[i_i[m]]['disp'])/np.sum(w))
            thetas.append(np.sum(w*src.iloc[i_i[m]]['incidence_angle'])/np.sum(w))
            alphas.append(np.sum(w*src.iloc[i_i[m]]['track_angle'])/np.sum(w))
        tgt['disp_idw'] = vals; tgt['theta_desc'] = thetas; tgt['alpha_desc'] = alphas
        out.append(tgt)
    return pd.concat(out, ignore_index=True)
asc_interp = idw(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw'])

# ==============================
# 5. dV (ou dH)
# ==============================
orb_inc = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orb_inc) * np.cos(np.deg2rad(asc_interp['latitude'])))

def get_comps(row):
    ta, td, beta = np.deg2rad(row['incidence_angle']), np.deg2rad(row['theta_desc']), row['beta']
    denom = (np.cos(ta)*np.sin(td)*np.cos(beta) + np.cos(td)*np.sin(ta)*np.cos(beta))
    if denom == 0: return np.nan, np.nan
    dV = (row['disp_idw']*np.sin(ta)*np.cos(beta) + row['disp']*np.sin(td)*np.cos(beta))/denom
    dH = (row['disp_idw']*np.cos(ta) - row['disp']*np.cos(td))/denom
    return dV, dH

asc_interp[['dV', 'dH']] = asc_interp.apply(lambda x: pd.Series(get_comps(x)), axis=1)
asc_interp = asc_interp.dropna(subset=['dV', 'dH'])

# ==============================
# 6. Grelha
# ==============================
#grid_size = 100
xe = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
ye = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
xs, ys = xe - grid_size/2, ye - grid_size/2
asc_interp['cx'] = pd.cut(asc_interp['easting'], bins=xs, labels=False)
asc_interp['cy'] = pd.cut(asc_interp['northing'], bins=ys, labels=False)
asc_interp = asc_interp.dropna(subset=['cx','cy'])
asc_interp['cell_id'] = asc_interp['cx'].astype(int).astype(str)+"_"+asc_interp['cy'].astype(int).astype(str)

grid_data = [{'cell_id': f"{ix}_{iy}", 'geometry': box(xs[ix], ys[iy], xs[ix+1], ys[iy+1])} for ix in range(len(xs)-1) for iy in range(len(ys)-1)]
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# Escolha aqui se usa dV ou dH para a agregação
#TARGET_VAR = 'dV' 
agg = asc_interp.groupby(['cell_id','date']).agg(dV=(TARGET_VAR,'mean')).reset_index() # Coluna final chama-se sempre 'dV' para manter compatibilidade com resto do script

# ==============================
# 7. Recorte e Filtro
# ==============================
points_gdf = gpd.GeoDataFrame(asc_interp, geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']), crs="EPSG:3035").to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

poly = grid_recort.geometry.values; ids = grid_recort['cell_id'].values; tree = STRtree(poly)
valid = set()
for pt in points_gdf.geometry:
    for idx in tree.query(pt):
        if poly[idx].contains(pt): valid.add(ids[idx])
grid_barragem = grid_recort[grid_recort['cell_id'].isin(valid)]
agg = agg[agg['cell_id'].isin(valid)]

# ==============================
# 8. CLUSTERING (VERSÃO ROBUSTA CONTRA NaNs)
# ==============================
print(f">>> A calcular DTW (Flexibilidade: {DTW_WINDOW_PCT})...")

# 1. Preparar Matriz (Pivot)
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV')

# --- LIMPEZA DE NaNs (Obrigatório para o Linkage não dar erro) ---
# Interpola pequenos buracos e remove células que ainda tenham falhas críticas
agg_pivot = agg_pivot.interpolate(axis=1, limit_direction='both').dropna()

cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

if n < 2:
    print("Aviso: Células insuficientes após limpeza de dados.")
else:
    # 2. Normalizar
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X.T).T 

    # 3. Calcular DTW com a nova variável
    # max(1, ...) garante que a janela nunca é zero
    window_dtw = max(1, int(DTW_WINDOW_PCT * X_scaled.shape[1])) 
    
    # Cálculo da matriz
    dist_matrix = dtw.distance_matrix_fast(X_scaled, window=window_dtw)

    # --- TRATAMENTO PÓS-CÁLCULO ---
    # Se o DTW gerar algum NaN (acontece se a janela for muito pequena para certas séries)
    if not np.all(np.isfinite(dist_matrix)):
        # Substituímos NaNs pelo valor máximo da matriz (para dizer que são muito diferentes)
        mask_nan = np.isnan(dist_matrix)
        dist_matrix[mask_nan] = np.nanmax(dist_matrix) if not np.all(np.isnan(dist_matrix)) else 0

    # 4. Clustering Hierárquico
    condensed = squareform(dist_matrix)
    Z = linkage(condensed, method=LINKAGE_METHOD)

    # 5. Corte Automático (Elbow)
    last = Z[-10:, 2] # Últimas distâncias de fusão
    acceleration = np.diff(last, 2)
    try:
        k_idx = np.argmax(acceleration) + 2
        cut_distance = (last[k_idx] + last[k_idx-1]) / 2
    except:
        cut_distance = last[len(last)//2]

    # Labels
    cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
    
    # Reajustar labels para começar em 0
    u_labels = np.unique(cluster_labels)
    map_l = {old: new for new, old in enumerate(u_labels)}
    cluster_labels = np.array([map_l[x] for x in cluster_labels])
    
    num_clusters = len(u_labels)
    print(f"DTW Concluído. Corte: {cut_distance:.2f}. Clusters: {num_clusters}")

# DataFrame Final
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})
# Cores Dinâmicas (para N clusters)
palette = sns.color_palette("Set2", num_clusters) if num_clusters <= 8 else sns.color_palette("tab20", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(range(num_clusters), palette)}

grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='inner')

# ==============================
# FIGURA EXTRA: DENDROGRAMA
# ==============================
if Z is not None:
    print("Gerando Dendrograma...")
    plt.figure(figsize=(12, 5))
    dendrogram(Z, leaf_rotation=90, leaf_font_size=8, color_threshold=cut_distance)
    plt.axhline(y=cut_distance, c='k', ls='--', lw=1, label='Corte Automático')
    plt.title('Dendrograma de Clustering Hierárquico (DTW)')
    plt.xlabel('Células'); plt.ylabel('Distância')
    plt.legend(); plt.tight_layout(); plt.show()


# ==============================
# 9. Dados Hidro (IGUAL)
# ==============================
date_range = agg_pivot.columns
win = 13

# Temp
try: df_t = pd.read_excel("data/alqueva_temp.xlsx"); df_t['data']=pd.to_datetime(df_t['data'])
except: df_t = pd.DataFrame({'data': date_range, 'med': 0})
ts = df_t.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': ts.index, 'med': ts.values})
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), win, 2)

# Nível
try: 
    df_n = pd.read_excel("data/alqueva_nivel.xlsx"); df_n['data']=pd.to_datetime(df_n['data'])
    df_n['nivel'] = pd.to_numeric(df_n['nivel'], errors='coerce'); df_n = df_n.dropna(subset=['nivel'])
except: df_n = pd.DataFrame({'data': date_range, 'nivel': 0})
ns = df_n.set_index('data')['nivel'].resample('MS').mean().reindex(date_range).interpolate(limit_direction='both').ffill().bfill()
df_nivel = pd.DataFrame({'data': ns.index, 'nivel': ns.values})
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], win, 2)

# Precipitação
try: df_p = pd.read_excel("data/prec.xlsx"); df_p['data'] = pd.to_datetime(df_p['data'])
except: np.random.seed(42); df_p = pd.DataFrame({'data': date_range, 'prec': 0})
ps = df_p.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': ps.index, 'prec': ps.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções Visuais (IGUAL)
# ==============================
# def scale_vis(d, vmin, vmax, s=0.20, o=0.30):
#     d = np.array(d); mn, mx = np.nanmin(d), np.nanmax(d)
#     span = vmax - vmin
#     norm = np.zeros_like(d) if mx==mn else (d-mn)/(mx-mn)
#     return norm * span * s + (vmax - span * o), mn, mx

# def create_ticks(rmin, rmax, vmin, vmax, s=0.20, o=0.30, n=5):
#     rt = np.linspace(rmin, rmax, n)
#     span = vmax - vmin
#     nt = np.linspace(0,1,n) if rmax==rmin else (rt-rmin)/(rmax-rmin)
#     vt = nt * span * s + (vmax - span * o)
#     return vt, rt


# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks


# ==============================
# FIGURA 1: MAPA (ESTILO ORIGINAL)
# ==============================
print("Gerando Figura 1...")
clusters_present = sorted(cluster_df['cluster'].unique())
k_plot = len(clusters_present)

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', lw=1)

for i in clusters_present:
    sub = grid_sel[grid_sel['cluster'] == i]
    sub.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    cents = sub.copy(); cents.geometry = cents.geometry.centroid
    cents.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

hdl = [plt.Rectangle((0,0),1,1, fc=cluster_colors[i], alpha=0.4) for i in clusters_present]
lbl = [f'Cluster {i+1} ({len(grid_sel[grid_sel["cluster"]==i])} cel)' for i in clusters_present]
hdl.append(Line2D([0],[0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None')); lbl.append('Centróides')
ax_map.legend(hdl, lbl, loc='upper left', fontsize=10, title=f"Clusters DTW (K={k_plot})")
ax_map.set_title(f"Mapa de Clusters ({TARGET_VAR})", fontsize=15)
plt.show()

# ==============================
# FIGURA 2: SÉRIES TEMPORAIS
# ==============================

print("Gerando Figura 2...")

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5

fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

dV_min = agg[TARGET_VAR].min()
dV_max = agg[TARGET_VAR].max()
dV_margin = (dV_max - dV_min) * 0.1
dV_ylim = (dV_min - dV_margin, dV_max + dV_margin)

# --- Séries visuais ---
temp_visual, _, _ = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, _, _ = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)

# Séries visuais
# tv, tmi, tmx = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
# nivel_visual, nivel_min, nivel_max = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
# prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)


for idx, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)


    # --- Função atualizada ---
    def plot_dV_comparison_series(
        row_idx, ext_data_visual, ext_data_df, ext_col_real,
        ext_label, ext_color,
        plot_as_bar=False, bar_color=None,
        plot_real_scale_line=False,
        combine_annual_prec=False,
        integer_ticks=False, show_background_bars=False):

        ax = fig_series.add_subplot(gs_series[row_idx, idx])

        # --- dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid],
                    color='lightgray', alpha=0.7, linewidth=1.0)

        ax.plot(cluster_data.columns, cluster_mean_dV,
                color=cluster_color, linewidth=1.0,
                label=f'Média Cluster {cluster_id+1}')

        # --- Eixo Y2 ---
        ax2 = ax.twinx()
        lines2 = []

        # -------------------------
        #   CASOS DE PRECIPITAÇÃO
        # -------------------------
        if combine_annual_prec:

            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)

            for ano in df_prec['ano_hidrologico'].unique():
                grupo = df_prec[df_prec['ano_hidrologico'] == ano]
                ax2.plot(grupo['data'], grupo['prec_acum_anual'],
                         color='teal', linewidth=1.0, alpha=0.9)

            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            ax2.set_ylabel("Prec. acumulada anual", color="teal")
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=1.0, label='Prec. acumulada anual (mm)')
            ]

        elif plot_as_bar:

            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15,
                    color=bar_color or ext_color, alpha=0.5)

            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            ax2.set_ylabel(ext_label, color='teal')
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)
            ]

        elif plot_real_scale_line:

            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15,
                        color='teal', alpha=0.3)

            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real],
                     color='teal', linewidth=1.0, alpha=0.85)

            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            ax2.set_ylabel(ext_label, color='teal')
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=1.0, label=ext_label)
            ]

        # -------------------------
        #      OUTRAS SÉRIES
        # -------------------------
        else:

            ax2.plot(ext_data_df['data'], ext_data_visual,
                     color=ext_color, linewidth=1.0, alpha=0.85)

            visual_ticks, real_ticks = create_visual_ticks(
                ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(),
                dV_min, dV_max, scale_factor=0.20, offset_factor=0.30
            )

            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(visual_ticks)

            if integer_ticks:
                ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else:
                ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])

            label_color = 'black' if 'Temperatura' in ext_label else ext_color
            ax2.set_ylabel(ext_label, color=label_color)
            ax2.tick_params(axis='y', colors=label_color)

            lines2 = [ax2.lines[-1]]

        # --- Y1 sempre dV ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)

        if idx == 0:
            ax.set_ylabel('dV (mm)')
        else:
            ax.set_yticklabels([])

        # Y2 somente no último cluster
        if idx != n_clusters - 1:
            ax2.set_yticklabels([])

        ax.tick_params(left=(idx==0))
        ax2.tick_params(right=(idx==n_clusters-1))

        # --- X-axis ---
        if row_idx == n_rows_series - 1:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        else:
            ax.set_xticklabels([])

        # --- Legenda somente no primeiro e no último cluster ---
        if idx == 0 or idx == n_clusters - 1:
            lines1, labels1 = ax.get_legend_handles_labels()
            handles2, labels2 = ax2.get_legend_handles_labels()

            ax.legend(
                lines1 + handles2,
                labels1 + labels2,
                fontsize=8,
                loc='best',
                framealpha=1.0,
                facecolor='white',
                edgecolor='lightgray'
            ).set_zorder(100)

        return ax


    # ---- CHAMADA DAS 5 SÉRIES ----
    plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth',
                              'Temperatura média (°C)', 'black', integer_ticks=True)

    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth',
                              'Nível da albufeira (m)', 'navy', integer_ticks=True)

    plot_dV_comparison_series(2, None, df_prec, 'prec',
                              'Precipitação mensal (mm)', 'teal',
                              plot_as_bar=True, bar_color='teal')

    plot_dV_comparison_series(3, None, df_prec, 'prec_acum',
                              'Prec. acumulada total (mm)', 'teal',
                              plot_real_scale_line=True, show_background_bars=True)

    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual',
                              'Prec. acumulada anual', 'teal',
                              combine_annual_prec=True)


plt.tight_layout()
plt.show()


# ==============================
# FIGURA 3: DECOMPOSIÇÃO SAZONAL
# ==============================

print("Gerando Figura 3...")

n_rows = 4  # Observed, Trend, Seasonal, Residual
fig_series_decomp, axes = plt.subplots(
    n_rows, n_clusters, figsize=(6 * max(1, n_clusters), 7), sharex=False
)

for col, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    mean_series = cluster_data.mean(axis=0)

    decomposition = seasonal_decompose(mean_series, model="additive", period=12)

    series_list = [
        ("Observed", decomposition.observed),
        ("Trend", decomposition.trend),
        ("Seasonal", decomposition.seasonal),
        ("Residual", decomposition.resid)
    ]

    cluster_color = cluster_colors.get(cluster_id, "black")

    for row, (label, series) in enumerate(series_list):

        ax = axes[row, col]

        # --------------------------
        #   PLOT (agora com cores)
        # --------------------------
        ax.plot(series.index, series.values,
                color=cluster_color, linewidth=1.0)

        # --------------------------
        #  LIMITE EXTERIOR (spines)
        # --------------------------
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.8)
            spine.set_color("black")

        # --------------------------
        #     LIMITES Y
        # --------------------------
        if label != "Observed":
            ymin = np.nanmin(series)
            ymax = np.nanmax(series)
            if np.isnan(ymin) or np.isnan(ymax):
                ymin, ymax = -1, 1
            margin = (ymax - ymin) * 0.10
            ax.set_ylim(ymin - margin, ymax + margin)

        # --------------------------
        #  TÍTULOS DOS SUBPLOTS
        # --------------------------
        if row == 0:
            ax.set_title(f"Seasonal Decompose – Cluster {cluster_id + 1}",
                         fontsize=12)

        if col == 0:
            ax.set_ylabel(label, fontsize=10)
        else:
            ax.set_yticks([])

        # --------------------------
        #   EIXO X — só no último row
        # --------------------------
        if row == n_rows - 1:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.xaxis.set_major_locator(mdates.YearLocator())
            ax.tick_params(axis='x', labelrotation=0, labelsize=10)
        else:
            ax.set_xticks([])
            ax.set_xticklabels([])

plt.tight_layout()
plt.show()


## complete

In [ ]:
# Escolha aqui o que quer analisar: 'dV' ou 'dH'
TARGET_VAR = 'dV'  # ou 'dV'

# grelha
grid_size = 50

# Tipo de ligação
LINKAGE_METHOD = "complete"
# Opções possíveis:
# "complete", "average", "ward"

# ==============================================================================
# SCRIPT COMPLETO: DTW + HIERÁRQUICO (COM ESTILO VISUAL K-MEANS MANTIDO)
# ==============================================================================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
import matplotlib.dates as mdates
from shapely.strtree import STRtree
from statsmodels.tsa.seasonal import seasonal_decompose
from matplotlib.lines import Line2D

# Bibliotecas para DTW e Hierárquico
try:
    from dtaidistance import dtw
    from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
    from scipy.spatial.distance import squareform
    from sklearn.preprocessing import StandardScaler
    import seaborn as sns
    from matplotlib.colors import to_hex
except ImportError as e:
    print(f"Erro de Importação: {e}. Certifique-se de que instalou: dtaidistance, scipy, seaborn, scikit-learn")

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
    #barragem = cos[cos['COS23_n4_L'] == 'Superfícies silvopastoris de azinheira']
    #barragem = cos[cos['COS23_n4_L'] == 'Matos']
except Exception as e:
    print(f"Aviso: COS placeholder. {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs e Filtro
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df): return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) & (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'], value_vars=disp_cols, var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(start=max(asc_long['date'].min(), desc_long['date'].min()), end=min(asc_long['date'].max(), desc_long['date'].max()), freq='MS')

# ==============================
# 3. Interpolação
# ==============================
def interpolate_ps(df, dates):
    dfs = []
    for (x, y), g in df.groupby(['easting','northing']):
        g = g.sort_values('date')
        interp = np.interp(pd.to_datetime(dates).astype(np.int64), g['date'].astype(np.int64), g['disp'])
        dfs.append(pd.DataFrame({'easting': x, 'northing': y, 'latitude': g['latitude'].iloc[0], 'longitude': g['longitude'].iloc[0], 'date': dates, 'disp': interp, 'incidence_angle': g['incidence_angle'].iloc[0], 'track_angle': g['track_angle'].iloc[0]}))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. IDW
# ==============================
def idw(source, target, radius=150, power=2):
    out = []
    for d, src in source.groupby('date'):
        tgt = target[target['date']==d].copy()
        if src.empty or tgt.empty: continue
        tree = cKDTree(list(zip(src['easting'], src['northing'])))
        dist, idx = tree.query(list(zip(tgt['easting'], tgt['northing'])), k=5, distance_upper_bound=radius)
        vals, thetas, alphas = [], [], []
        for d_i, i_i in zip(dist, idx):
            m = np.isfinite(d_i)
            if not np.any(m): vals.append(np.nan); thetas.append(np.nan); alphas.append(np.nan); continue
            w = 1/(d_i[m]**power)
            vals.append(np.sum(w*src.iloc[i_i[m]]['disp'])/np.sum(w))
            thetas.append(np.sum(w*src.iloc[i_i[m]]['incidence_angle'])/np.sum(w))
            alphas.append(np.sum(w*src.iloc[i_i[m]]['track_angle'])/np.sum(w))
        tgt['disp_idw'] = vals; tgt['theta_desc'] = thetas; tgt['alpha_desc'] = alphas
        out.append(tgt)
    return pd.concat(out, ignore_index=True)
asc_interp = idw(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw'])

# ==============================
# 5. dV (ou dH)
# ==============================
orb_inc = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orb_inc) * np.cos(np.deg2rad(asc_interp['latitude'])))

def get_comps(row):
    ta, td, beta = np.deg2rad(row['incidence_angle']), np.deg2rad(row['theta_desc']), row['beta']
    denom = (np.cos(ta)*np.sin(td)*np.cos(beta) + np.cos(td)*np.sin(ta)*np.cos(beta))
    if denom == 0: return np.nan, np.nan
    dV = (row['disp_idw']*np.sin(ta)*np.cos(beta) + row['disp']*np.sin(td)*np.cos(beta))/denom
    dH = (row['disp_idw']*np.cos(ta) - row['disp']*np.cos(td))/denom
    return dV, dH

asc_interp[['dV', 'dH']] = asc_interp.apply(lambda x: pd.Series(get_comps(x)), axis=1)
asc_interp = asc_interp.dropna(subset=['dV', 'dH'])

# ==============================
# 6. Grelha
# ==============================
#grid_size = 100
xe = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
ye = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
xs, ys = xe - grid_size/2, ye - grid_size/2
asc_interp['cx'] = pd.cut(asc_interp['easting'], bins=xs, labels=False)
asc_interp['cy'] = pd.cut(asc_interp['northing'], bins=ys, labels=False)
asc_interp = asc_interp.dropna(subset=['cx','cy'])
asc_interp['cell_id'] = asc_interp['cx'].astype(int).astype(str)+"_"+asc_interp['cy'].astype(int).astype(str)

grid_data = [{'cell_id': f"{ix}_{iy}", 'geometry': box(xs[ix], ys[iy], xs[ix+1], ys[iy+1])} for ix in range(len(xs)-1) for iy in range(len(ys)-1)]
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# Escolha aqui se usa dV ou dH para a agregação
#TARGET_VAR = 'dV' 
agg = asc_interp.groupby(['cell_id','date']).agg(dV=(TARGET_VAR,'mean')).reset_index() # Coluna final chama-se sempre 'dV' para manter compatibilidade com resto do script

# ==============================
# 7. Recorte e Filtro
# ==============================
points_gdf = gpd.GeoDataFrame(asc_interp, geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']), crs="EPSG:3035").to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

poly = grid_recort.geometry.values; ids = grid_recort['cell_id'].values; tree = STRtree(poly)
valid = set()
for pt in points_gdf.geometry:
    for idx in tree.query(pt):
        if poly[idx].contains(pt): valid.add(ids[idx])
grid_barragem = grid_recort[grid_recort['cell_id'].isin(valid)]
agg = agg[agg['cell_id'].isin(valid)]

# ==============================
# 8. CLUSTERING (DTW + HIERÁRQUICO) - SUBSTITUI K-MEANS
# ==============================
print(">>> A calcular Matriz de Distâncias DTW...")

# 1. Preparar Matriz (Pivot)
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

if n < 2:
    print("Aviso: Menos de 2 células para agrupar. Clustering ignorado.")
    cluster_labels = np.zeros(n, dtype=int)
    num_clusters = 1
    cut_distance = 0
    Z = None
else:
    # 2. Normalizar (Z-Score) para comparar formas
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X.T).T 

    # 3. Calcular DTW
    dist_matrix = np.zeros((n, n))
    window_dtw = int(0.1 * X_scaled.shape[1]) # Janela 10%
    
    # Esta parte pode demorar dependendo do nº de células
    for i in range(n):
        for j in range(i + 1, n):
            d = dtw.distance(X_scaled[i], X_scaled[j], window=window_dtw)
            dist_matrix[i, j] = d
            dist_matrix[j, i] = d

    # 4. Clustering Hierárquico
    condensed = squareform(dist_matrix)
    #Z = linkage(condensed, method='average')
    from scipy.cluster.hierarchy import linkage, fcluster
    Z = linkage(condensed, method=LINKAGE_METHOD)

    # 5. Corte Automático (Elbow)
    last = Z[-10:, 2] # Últimas distâncias de fusão
    acceleration = np.diff(last, 2)
    try:
        k_idx = np.argmax(acceleration) + 2
        cut_distance = (last[k_idx] + last[k_idx-1]) / 2
    except:
        cut_distance = last[len(last)//2]

    # Labels
    cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
    
    # Reajustar labels para começar em 0
    u_labels = np.unique(cluster_labels)
    map_l = {old: new for new, old in enumerate(u_labels)}
    cluster_labels = np.array([map_l[x] for x in cluster_labels])
    
    num_clusters = len(u_labels)
    print(f"DTW Concluído. Corte: {cut_distance:.2f}. Clusters: {num_clusters}")

# DataFrame Final
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})
# Cores Dinâmicas (para N clusters)
palette = sns.color_palette("Set2", num_clusters) if num_clusters <= 8 else sns.color_palette("tab20", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(range(num_clusters), palette)}

grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='inner')

# ==============================
# FIGURA EXTRA: DENDROGRAMA
# ==============================
if Z is not None:
    print("Gerando Dendrograma...")
    plt.figure(figsize=(12, 5))
    dendrogram(Z, leaf_rotation=90, leaf_font_size=8, color_threshold=cut_distance)
    plt.axhline(y=cut_distance, c='k', ls='--', lw=1, label='Corte Automático')
    plt.title('Dendrograma de Clustering Hierárquico (DTW)')
    plt.xlabel('Células'); plt.ylabel('Distância')
    plt.legend(); plt.tight_layout(); plt.show()


# ==============================
# 9. Dados Hidro (IGUAL)
# ==============================
date_range = agg_pivot.columns
win = 13

# Temp
try: df_t = pd.read_excel("data/alqueva_temp.xlsx"); df_t['data']=pd.to_datetime(df_t['data'])
except: df_t = pd.DataFrame({'data': date_range, 'med': 0})
ts = df_t.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': ts.index, 'med': ts.values})
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), win, 2)

# Nível
try: 
    df_n = pd.read_excel("data/alqueva_nivel.xlsx"); df_n['data']=pd.to_datetime(df_n['data'])
    df_n['nivel'] = pd.to_numeric(df_n['nivel'], errors='coerce'); df_n = df_n.dropna(subset=['nivel'])
except: df_n = pd.DataFrame({'data': date_range, 'nivel': 0})
ns = df_n.set_index('data')['nivel'].resample('MS').mean().reindex(date_range).interpolate(limit_direction='both').ffill().bfill()
df_nivel = pd.DataFrame({'data': ns.index, 'nivel': ns.values})
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], win, 2)

# Precipitação
try: df_p = pd.read_excel("data/prec.xlsx"); df_p['data'] = pd.to_datetime(df_p['data'])
except: np.random.seed(42); df_p = pd.DataFrame({'data': date_range, 'prec': 0})
ps = df_p.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': ps.index, 'prec': ps.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções Visuais (IGUAL)
# ==============================
# def scale_vis(d, vmin, vmax, s=0.20, o=0.30):
#     d = np.array(d); mn, mx = np.nanmin(d), np.nanmax(d)
#     span = vmax - vmin
#     norm = np.zeros_like(d) if mx==mn else (d-mn)/(mx-mn)
#     return norm * span * s + (vmax - span * o), mn, mx

# def create_ticks(rmin, rmax, vmin, vmax, s=0.20, o=0.30, n=5):
#     rt = np.linspace(rmin, rmax, n)
#     span = vmax - vmin
#     nt = np.linspace(0,1,n) if rmax==rmin else (rt-rmin)/(rmax-rmin)
#     vt = nt * span * s + (vmax - span * o)
#     return vt, rt

# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks


# ==============================
# FIGURA 1: MAPA (ESTILO ORIGINAL)
# ==============================
print("Gerando Figura 1...")
clusters_present = sorted(cluster_df['cluster'].unique())
k_plot = len(clusters_present)

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', lw=1)

for i in clusters_present:
    sub = grid_sel[grid_sel['cluster'] == i]
    sub.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    cents = sub.copy(); cents.geometry = cents.geometry.centroid
    cents.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

hdl = [plt.Rectangle((0,0),1,1, fc=cluster_colors[i], alpha=0.4) for i in clusters_present]
lbl = [f'Cluster {i+1} ({len(grid_sel[grid_sel["cluster"]==i])} cel)' for i in clusters_present]
hdl.append(Line2D([0],[0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None')); lbl.append('Centróides')
ax_map.legend(hdl, lbl, loc='upper left', fontsize=10, title=f"Clusters DTW (K={k_plot})")
ax_map.set_title(f"Mapa de Clusters ({TARGET_VAR})", fontsize=15)
plt.show()

# ==============================
# FIGURA 2: SÉRIES TEMPORAIS
# ==============================

print("Gerando Figura 2...")

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5

fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

dV_min = agg[TARGET_VAR].min()
dV_max = agg[TARGET_VAR].max()
dV_margin = (dV_max - dV_min) * 0.1
dV_ylim = (dV_min - dV_margin, dV_max + dV_margin)

# --- Séries visuais ---
temp_visual, _, _ = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, _, _ = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)


for idx, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)


    # --- Função atualizada ---
    def plot_dV_comparison_series(
        row_idx, ext_data_visual, ext_data_df, ext_col_real,
        ext_label, ext_color,
        plot_as_bar=False, bar_color=None,
        plot_real_scale_line=False,
        combine_annual_prec=False,
        integer_ticks=False, show_background_bars=False):

        ax = fig_series.add_subplot(gs_series[row_idx, idx])

        # --- dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid],
                    color='lightgray', alpha=0.7, linewidth=1.0)

        ax.plot(cluster_data.columns, cluster_mean_dV,
                color=cluster_color, linewidth=1.0,
                label=f'Média Cluster {cluster_id+1}')

        # --- Eixo Y2 ---
        ax2 = ax.twinx()
        lines2 = []

        # -------------------------
        #   CASOS DE PRECIPITAÇÃO
        # -------------------------
        if combine_annual_prec:

            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)

            for ano in df_prec['ano_hidrologico'].unique():
                grupo = df_prec[df_prec['ano_hidrologico'] == ano]
                ax2.plot(grupo['data'], grupo['prec_acum_anual'],
                         color='teal', linewidth=1.0, alpha=0.9)

            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            ax2.set_ylabel("Prec. acumulada anual", color="teal")
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=1.0, label='Prec. acumulada anual (mm)')
            ]

        elif plot_as_bar:

            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15,
                    color=bar_color or ext_color, alpha=0.5)

            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            ax2.set_ylabel(ext_label, color='teal')
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)
            ]

        elif plot_real_scale_line:

            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15,
                        color='teal', alpha=0.3)

            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real],
                     color='teal', linewidth=1.0, alpha=0.85)

            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            ax2.set_ylabel(ext_label, color='teal')
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=1.0, label=ext_label)
            ]

        # -------------------------
        #      OUTRAS SÉRIES
        # -------------------------
        else:

            ax2.plot(ext_data_df['data'], ext_data_visual,
                     color=ext_color, linewidth=1.0, alpha=0.85)

            visual_ticks, real_ticks = create_visual_ticks(
                ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(),
                dV_min, dV_max, scale_factor=0.20, offset_factor=0.30
            )

            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(visual_ticks)

            if integer_ticks:
                ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else:
                ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])

            label_color = 'black' if 'Temperatura' in ext_label else ext_color
            ax2.set_ylabel(ext_label, color=label_color)
            ax2.tick_params(axis='y', colors=label_color)

            lines2 = [ax2.lines[-1]]

        # --- Y1 sempre dV ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)

        if idx == 0:
            ax.set_ylabel('dV (mm)')
        else:
            ax.set_yticklabels([])

        # Y2 somente no último cluster
        if idx != n_clusters - 1:
            ax2.set_yticklabels([])

        ax.tick_params(left=(idx==0))
        ax2.tick_params(right=(idx==n_clusters-1))

        # --- X-axis ---
        if row_idx == n_rows_series - 1:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        else:
            ax.set_xticklabels([])

        # --- Legenda somente no primeiro e no último cluster ---
        if idx == 0 or idx == n_clusters - 1:
            lines1, labels1 = ax.get_legend_handles_labels()
            handles2, labels2 = ax2.get_legend_handles_labels()

            ax.legend(
                lines1 + handles2,
                labels1 + labels2,
                fontsize=8,
                loc='best',
                framealpha=1.0,
                facecolor='white',
                edgecolor='lightgray'
            ).set_zorder(100)

        return ax


    # ---- CHAMADA DAS 5 SÉRIES ----
    plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth',
                              'Temperatura média (°C)', 'black', integer_ticks=True)

    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth',
                              'Nível da albufeira (m)', 'navy', integer_ticks=True)

    plot_dV_comparison_series(2, None, df_prec, 'prec',
                              'Precipitação mensal (mm)', 'teal',
                              plot_as_bar=True, bar_color='teal')

    plot_dV_comparison_series(3, None, df_prec, 'prec_acum',
                              'Prec. acumulada total (mm)', 'teal',
                              plot_real_scale_line=True, show_background_bars=True)

    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual',
                              'Prec. acumulada anual', 'teal',
                              combine_annual_prec=True)


plt.tight_layout()
plt.show()


# ==============================
# FIGURA 3: DECOMPOSIÇÃO SAZONAL
# ==============================

print("Gerando Figura 3...")

n_rows = 4  # Observed, Trend, Seasonal, Residual
fig_series_decomp, axes = plt.subplots(
    n_rows, n_clusters, figsize=(6 * max(1, n_clusters), 7), sharex=False
)

for col, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    mean_series = cluster_data.mean(axis=0)

    decomposition = seasonal_decompose(mean_series, model="additive", period=12)

    series_list = [
        ("Observed", decomposition.observed),
        ("Trend", decomposition.trend),
        ("Seasonal", decomposition.seasonal),
        ("Residual", decomposition.resid)
    ]

    cluster_color = cluster_colors.get(cluster_id, "black")

    for row, (label, series) in enumerate(series_list):

        ax = axes[row, col]

        # --------------------------
        #   PLOT (agora com cores)
        # --------------------------
        ax.plot(series.index, series.values,
                color=cluster_color, linewidth=1.0)

        # --------------------------
        #  LIMITE EXTERIOR (spines)
        # --------------------------
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.8)
            spine.set_color("black")

        # --------------------------
        #     LIMITES Y
        # --------------------------
        if label != "Observed":
            ymin = np.nanmin(series)
            ymax = np.nanmax(series)
            if np.isnan(ymin) or np.isnan(ymax):
                ymin, ymax = -1, 1
            margin = (ymax - ymin) * 0.10
            ax.set_ylim(ymin - margin, ymax + margin)

        # --------------------------
        #  TÍTULOS DOS SUBPLOTS
        # --------------------------
        if row == 0:
            ax.set_title(f"Seasonal Decompose – Cluster {cluster_id + 1}",
                         fontsize=12)

        if col == 0:
            ax.set_ylabel(label, fontsize=10)
        else:
            ax.set_yticks([])

        # --------------------------
        #   EIXO X — só no último row
        # --------------------------
        if row == n_rows - 1:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.xaxis.set_major_locator(mdates.YearLocator())
            ax.tick_params(axis='x', labelrotation=0, labelsize=10)
        else:
            ax.set_xticks([])
            ax.set_xticklabels([])

plt.tight_layout()
plt.show()


com mudança no dtw e hierárquico, o número de clusters pode variar. O código foi adaptado para lidar com isso, mantendo a estrutura visual e adicionando um dendrograma para análise. As funções de plotagem foram atualizadas para incluir casos específicos de precipitação, mantendo a flexibilidade para outros tipos de dados.

In [ ]:
# Escolha aqui o que quer analisar: 'dV' ou 'dH'
TARGET_VAR = 'dV'  # ou 'dV'

# grelha
grid_size = 25

DTW_WINDOW_PCT = 0.01  # Aqui podes mudar para 0.1, 0.05, 0.03, etc.

# Tipo de ligação
LINKAGE_METHOD = "complete"
# Opções possíveis:
# "complete", "average", "ward"

# ==============================================================================
# SCRIPT COMPLETO: DTW + HIERÁRQUICO (COM ESTILO VISUAL K-MEANS MANTIDO)
# ==============================================================================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
import matplotlib.dates as mdates
from shapely.strtree import STRtree
from statsmodels.tsa.seasonal import seasonal_decompose
from matplotlib.lines import Line2D

# Bibliotecas para DTW e Hierárquico
try:
    from dtaidistance import dtw
    from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
    from scipy.spatial.distance import squareform
    from sklearn.preprocessing import StandardScaler
    import seaborn as sns
    from matplotlib.colors import to_hex
except ImportError as e:
    print(f"Erro de Importação: {e}. Certifique-se de que instalou: dtaidistance, scipy, seaborn, scikit-learn")

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    #barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
    #barragem = cos[cos['COS23_n4_L'] == 'Superfícies silvopastoris de azinheira']
    #barragem = cos[cos['COS23_n4_L'] == 'Matos']
    
    # Podes adicionar ou remover categorias dentro dos parênteses retos []
    barragem = cos[cos['COS23_n4_L'].isin([
    'Infraestruturas de produção de energia hídrica', 
    #'Albufeiras de barragens',
    #'Equipamentos culturais',
    #'Florestas de azinheira',
    #'Matos',
    #'Pastagens melhoradas',
    #'Rede rodoviária',
    'Superfícies silvopastoris de azinheira'
])]
    
# - Albufeiras de barragens
# - Equipamentos culturais
# - Florestas de azinheira
# - Infraestruturas de produção de energia hídrica
# - Matos
# - Pastagens melhoradas
# - Rede rodoviária
# - Superfícies silvopastoris de azinheira
    
    
except Exception as e:
    print(f"Aviso: COS placeholder. {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs e Filtro
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df): return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) & (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'], value_vars=disp_cols, var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(start=max(asc_long['date'].min(), desc_long['date'].min()), end=min(asc_long['date'].max(), desc_long['date'].max()), freq='MS')

# ==============================
# 3. Interpolação
# ==============================
def interpolate_ps(df, dates):
    dfs = []
    for (x, y), g in df.groupby(['easting','northing']):
        g = g.sort_values('date')
        interp = np.interp(pd.to_datetime(dates).astype(np.int64), g['date'].astype(np.int64), g['disp'])
        dfs.append(pd.DataFrame({'easting': x, 'northing': y, 'latitude': g['latitude'].iloc[0], 'longitude': g['longitude'].iloc[0], 'date': dates, 'disp': interp, 'incidence_angle': g['incidence_angle'].iloc[0], 'track_angle': g['track_angle'].iloc[0]}))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. IDW
# ==============================
def idw(source, target, radius=25, power=3):
    out = []
    for d, src in source.groupby('date'):
        tgt = target[target['date']==d].copy()
        if src.empty or tgt.empty: continue
        tree = cKDTree(list(zip(src['easting'], src['northing'])))
        dist, idx = tree.query(list(zip(tgt['easting'], tgt['northing'])), k=5, distance_upper_bound=radius)
        vals, thetas, alphas = [], [], []
        for d_i, i_i in zip(dist, idx):
            m = np.isfinite(d_i)
            if not np.any(m): vals.append(np.nan); thetas.append(np.nan); alphas.append(np.nan); continue
            w = 1/(d_i[m]**power)
            vals.append(np.sum(w*src.iloc[i_i[m]]['disp'])/np.sum(w))
            thetas.append(np.sum(w*src.iloc[i_i[m]]['incidence_angle'])/np.sum(w))
            alphas.append(np.sum(w*src.iloc[i_i[m]]['track_angle'])/np.sum(w))
        tgt['disp_idw'] = vals; tgt['theta_desc'] = thetas; tgt['alpha_desc'] = alphas
        out.append(tgt)
    return pd.concat(out, ignore_index=True)
asc_interp = idw(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw'])

# ==============================
# 5. dV (ou dH)
# ==============================
orb_inc = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orb_inc) * np.cos(np.deg2rad(asc_interp['latitude'])))

def get_comps(row):
    ta, td, beta = np.deg2rad(row['incidence_angle']), np.deg2rad(row['theta_desc']), row['beta']
    denom = (np.cos(ta)*np.sin(td)*np.cos(beta) + np.cos(td)*np.sin(ta)*np.cos(beta))
    if denom == 0: return np.nan, np.nan
    dV = (row['disp_idw']*np.sin(ta)*np.cos(beta) + row['disp']*np.sin(td)*np.cos(beta))/denom
    dH = (row['disp_idw']*np.cos(ta) - row['disp']*np.cos(td))/denom
    return dV, dH

asc_interp[['dV', 'dH']] = asc_interp.apply(lambda x: pd.Series(get_comps(x)), axis=1)
asc_interp = asc_interp.dropna(subset=['dV', 'dH'])

# ==============================
# 6. Grelha
# ==============================
#grid_size = 100
xe = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
ye = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
xs, ys = xe - grid_size/2, ye - grid_size/2
asc_interp['cx'] = pd.cut(asc_interp['easting'], bins=xs, labels=False)
asc_interp['cy'] = pd.cut(asc_interp['northing'], bins=ys, labels=False)
asc_interp = asc_interp.dropna(subset=['cx','cy'])
asc_interp['cell_id'] = asc_interp['cx'].astype(int).astype(str)+"_"+asc_interp['cy'].astype(int).astype(str)

grid_data = [{'cell_id': f"{ix}_{iy}", 'geometry': box(xs[ix], ys[iy], xs[ix+1], ys[iy+1])} for ix in range(len(xs)-1) for iy in range(len(ys)-1)]
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# Escolha aqui se usa dV ou dH para a agregação
#TARGET_VAR = 'dV' 
agg = asc_interp.groupby(['cell_id','date']).agg(dV=(TARGET_VAR,'mean')).reset_index() # Coluna final chama-se sempre 'dV' para manter compatibilidade com resto do script

# ==============================
# 7. Recorte e Filtro
# ==============================
points_gdf = gpd.GeoDataFrame(asc_interp, geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']), crs="EPSG:3035").to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

poly = grid_recort.geometry.values; ids = grid_recort['cell_id'].values; tree = STRtree(poly)
valid = set()
for pt in points_gdf.geometry:
    for idx in tree.query(pt):
        if poly[idx].contains(pt): valid.add(ids[idx])
grid_barragem = grid_recort[grid_recort['cell_id'].isin(valid)]
agg = agg[agg['cell_id'].isin(valid)]

# ==============================
# 8. CLUSTERING (VERSÃO ROBUSTA CONTRA NaNs)
# ==============================
print(f">>> A calcular DTW (Flexibilidade: {DTW_WINDOW_PCT})...")

# 1. Preparar Matriz (Pivot)
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV')

# --- LIMPEZA DE NaNs (Obrigatório para o Linkage não dar erro) ---
# Interpola pequenos buracos e remove células que ainda tenham falhas críticas
agg_pivot = agg_pivot.interpolate(axis=1, limit_direction='both').dropna()

cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

if n < 2:
    print("Aviso: Células insuficientes após limpeza de dados.")
else:
    # 2. Normalizar
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X.T).T 

    # 3. Calcular DTW com a nova variável
    # max(1, ...) garante que a janela nunca é zero
    window_dtw = max(1, int(DTW_WINDOW_PCT * X_scaled.shape[1])) 
    
    # Cálculo da matriz
    dist_matrix = dtw.distance_matrix_fast(X_scaled, window=window_dtw)

    # --- TRATAMENTO PÓS-CÁLCULO ---
    # Se o DTW gerar algum NaN (acontece se a janela for muito pequena para certas séries)
    if not np.all(np.isfinite(dist_matrix)):
        # Substituímos NaNs pelo valor máximo da matriz (para dizer que são muito diferentes)
        mask_nan = np.isnan(dist_matrix)
        dist_matrix[mask_nan] = np.nanmax(dist_matrix) if not np.all(np.isnan(dist_matrix)) else 0

    # 4. Clustering Hierárquico
    condensed = squareform(dist_matrix)
    Z = linkage(condensed, method=LINKAGE_METHOD)

    # 5. Corte Automático (Elbow)
    last = Z[-10:, 2] # Últimas distâncias de fusão
    acceleration = np.diff(last, 2)
    try:
        k_idx = np.argmax(acceleration) + 2
        cut_distance = (last[k_idx] + last[k_idx-1]) / 2
    except:
        cut_distance = last[len(last)//2]

    # Labels
    cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
    
    # Reajustar labels para começar em 0
    u_labels = np.unique(cluster_labels)
    map_l = {old: new for new, old in enumerate(u_labels)}
    cluster_labels = np.array([map_l[x] for x in cluster_labels])
    
    num_clusters = len(u_labels)
    print(f"DTW Concluído. Corte: {cut_distance:.2f}. Clusters: {num_clusters}")

# DataFrame Final
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})
# Cores Dinâmicas (para N clusters)
palette = sns.color_palette("Set2", num_clusters) if num_clusters <= 8 else sns.color_palette("tab20", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(range(num_clusters), palette)}

grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='inner')

# ==============================
# FIGURA EXTRA: DENDROGRAMA
# ==============================
if Z is not None:
    print("Gerando Dendrograma...")
    plt.figure(figsize=(12, 5))
    dendrogram(Z, leaf_rotation=90, leaf_font_size=8, color_threshold=cut_distance)
    plt.axhline(y=cut_distance, c='k', ls='--', lw=1, label='Corte Automático')
    plt.title('Dendrograma de Clustering Hierárquico (DTW)')
    plt.xlabel('Células'); plt.ylabel('Distância')
    plt.legend(); plt.tight_layout(); plt.show()


# ==============================
# 9. Dados Hidro (IGUAL)
# ==============================
date_range = agg_pivot.columns
win = 13

# Temp
try: df_t = pd.read_excel("data/alqueva_temp.xlsx"); df_t['data']=pd.to_datetime(df_t['data'])
except: df_t = pd.DataFrame({'data': date_range, 'med': 0})
ts = df_t.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': ts.index, 'med': ts.values})
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), win, 2)

# Nível
try: 
    df_n = pd.read_excel("data/alqueva_nivel.xlsx"); df_n['data']=pd.to_datetime(df_n['data'])
    df_n['nivel'] = pd.to_numeric(df_n['nivel'], errors='coerce'); df_n = df_n.dropna(subset=['nivel'])
except: df_n = pd.DataFrame({'data': date_range, 'nivel': 0})
ns = df_n.set_index('data')['nivel'].resample('MS').mean().reindex(date_range).interpolate(limit_direction='both').ffill().bfill()
df_nivel = pd.DataFrame({'data': ns.index, 'nivel': ns.values})
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], win, 2)

# Precipitação
try: df_p = pd.read_excel("data/prec.xlsx"); df_p['data'] = pd.to_datetime(df_p['data'])
except: np.random.seed(42); df_p = pd.DataFrame({'data': date_range, 'prec': 0})
ps = df_p.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': ps.index, 'prec': ps.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções Visuais (IGUAL)
# ==============================
# def scale_vis(d, vmin, vmax, s=0.20, o=0.30):
#     d = np.array(d); mn, mx = np.nanmin(d), np.nanmax(d)
#     span = vmax - vmin
#     norm = np.zeros_like(d) if mx==mn else (d-mn)/(mx-mn)
#     return norm * span * s + (vmax - span * o), mn, mx

# def create_ticks(rmin, rmax, vmin, vmax, s=0.20, o=0.30, n=5):
#     rt = np.linspace(rmin, rmax, n)
#     span = vmax - vmin
#     nt = np.linspace(0,1,n) if rmax==rmin else (rt-rmin)/(rmax-rmin)
#     vt = nt * span * s + (vmax - span * o)
#     return vt, rt


# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks


# ==============================
# FIGURA 1: MAPA (ESTILO ORIGINAL)
# ==============================
print("Gerando Figura 1...")
clusters_present = sorted(cluster_df['cluster'].unique())
k_plot = len(clusters_present)

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', lw=1)

for i in clusters_present:
    sub = grid_sel[grid_sel['cluster'] == i]
    sub.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    cents = sub.copy(); cents.geometry = cents.geometry.centroid
    cents.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

hdl = [plt.Rectangle((0,0),1,1, fc=cluster_colors[i], alpha=0.4) for i in clusters_present]
lbl = [f'Cluster {i+1} ({len(grid_sel[grid_sel["cluster"]==i])} cel)' for i in clusters_present]
hdl.append(Line2D([0],[0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None')); lbl.append('Centróides')
ax_map.legend(hdl, lbl, loc='upper left', fontsize=10, title=f"Clusters DTW (K={k_plot})")
ax_map.set_title(f"Mapa de Clusters ({TARGET_VAR})", fontsize=15)
plt.show()

# ==============================
# FIGURA 2: SÉRIES TEMPORAIS
# ==============================

print("Gerando Figura 2...")

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5

fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

dV_min = agg[TARGET_VAR].min()
dV_max = agg[TARGET_VAR].max()
dV_margin = (dV_max - dV_min) * 0.1
dV_ylim = (dV_min - dV_margin, dV_max + dV_margin)

# --- Séries visuais ---
temp_visual, _, _ = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, _, _ = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)

# Séries visuais
# tv, tmi, tmx = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
# nivel_visual, nivel_min, nivel_max = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
# prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)


for idx, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)


    # --- Função atualizada ---
    def plot_dV_comparison_series(
        row_idx, ext_data_visual, ext_data_df, ext_col_real,
        ext_label, ext_color,
        plot_as_bar=False, bar_color=None,
        plot_real_scale_line=False,
        combine_annual_prec=False,
        integer_ticks=False, show_background_bars=False):

        ax = fig_series.add_subplot(gs_series[row_idx, idx])

        # --- dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid],
                    color='lightgray', alpha=0.7, linewidth=1.0)

        ax.plot(cluster_data.columns, cluster_mean_dV,
                color=cluster_color, linewidth=1.0,
                label=f'Média Cluster {cluster_id+1}')

        # --- Eixo Y2 ---
        ax2 = ax.twinx()
        lines2 = []

        # -------------------------
        #   CASOS DE PRECIPITAÇÃO
        # -------------------------
        if combine_annual_prec:

            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)

            for ano in df_prec['ano_hidrologico'].unique():
                grupo = df_prec[df_prec['ano_hidrologico'] == ano]
                ax2.plot(grupo['data'], grupo['prec_acum_anual'],
                         color='teal', linewidth=1.0, alpha=0.9)

            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            ax2.set_ylabel("Prec. acumulada anual", color="teal")
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=1.0, label='Prec. acumulada anual (mm)')
            ]

        elif plot_as_bar:

            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15,
                    color=bar_color or ext_color, alpha=0.5)

            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            ax2.set_ylabel(ext_label, color='teal')
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)
            ]

        elif plot_real_scale_line:

            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15,
                        color='teal', alpha=0.3)

            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real],
                     color='teal', linewidth=1.0, alpha=0.85)

            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            ax2.set_ylabel(ext_label, color='teal')
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=1.0, label=ext_label)
            ]

        # -------------------------
        #      OUTRAS SÉRIES
        # -------------------------
        else:

            ax2.plot(ext_data_df['data'], ext_data_visual,
                     color=ext_color, linewidth=1.0, alpha=0.85)

            visual_ticks, real_ticks = create_visual_ticks(
                ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(),
                dV_min, dV_max, scale_factor=0.20, offset_factor=0.30
            )

            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(visual_ticks)

            if integer_ticks:
                ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else:
                ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])

            label_color = 'black' if 'Temperatura' in ext_label else ext_color
            ax2.set_ylabel(ext_label, color=label_color)
            ax2.tick_params(axis='y', colors=label_color)

            lines2 = [ax2.lines[-1]]

        # --- Y1 sempre dV ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)

        if idx == 0:
            ax.set_ylabel('dV (mm)')
        else:
            ax.set_yticklabels([])

        # Y2 somente no último cluster
        if idx != n_clusters - 1:
            ax2.set_yticklabels([])

        ax.tick_params(left=(idx==0))
        ax2.tick_params(right=(idx==n_clusters-1))

        # --- X-axis ---
        if row_idx == n_rows_series - 1:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        else:
            ax.set_xticklabels([])

        # --- Legenda somente no primeiro e no último cluster ---
        if idx == 0 or idx == n_clusters - 1:
            lines1, labels1 = ax.get_legend_handles_labels()
            handles2, labels2 = ax2.get_legend_handles_labels()

            ax.legend(
                lines1 + handles2,
                labels1 + labels2,
                fontsize=8,
                loc='best',
                framealpha=1.0,
                facecolor='white',
                edgecolor='lightgray'
            ).set_zorder(100)

        return ax


    # ---- CHAMADA DAS 5 SÉRIES ----
    plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth',
                              'Temperatura média (°C)', 'black', integer_ticks=True)

    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth',
                              'Nível da albufeira (m)', 'navy', integer_ticks=True)

    plot_dV_comparison_series(2, None, df_prec, 'prec',
                              'Precipitação mensal (mm)', 'teal',
                              plot_as_bar=True, bar_color='teal')

    plot_dV_comparison_series(3, None, df_prec, 'prec_acum',
                              'Prec. acumulada total (mm)', 'teal',
                              plot_real_scale_line=True, show_background_bars=True)

    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual',
                              'Prec. acumulada anual', 'teal',
                              combine_annual_prec=True)


plt.tight_layout()
plt.show()


# ==============================
# FIGURA 3: DECOMPOSIÇÃO SAZONAL
# ==============================

print("Gerando Figura 3...")

n_rows = 4  # Observed, Trend, Seasonal, Residual
fig_series_decomp, axes = plt.subplots(
    n_rows, n_clusters, figsize=(6 * max(1, n_clusters), 7), sharex=False
)

for col, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    mean_series = cluster_data.mean(axis=0)

    decomposition = seasonal_decompose(mean_series, model="additive", period=12)

    series_list = [
        ("Observed", decomposition.observed),
        ("Trend", decomposition.trend),
        ("Seasonal", decomposition.seasonal),
        ("Residual", decomposition.resid)
    ]

    cluster_color = cluster_colors.get(cluster_id, "black")

    for row, (label, series) in enumerate(series_list):

        ax = axes[row, col]

        # --------------------------
        #   PLOT (agora com cores)
        # --------------------------
        ax.plot(series.index, series.values,
                color=cluster_color, linewidth=1.0)

        # --------------------------
        #  LIMITE EXTERIOR (spines)
        # --------------------------
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.8)
            spine.set_color("black")

        # --------------------------
        #     LIMITES Y
        # --------------------------
        if label != "Observed":
            ymin = np.nanmin(series)
            ymax = np.nanmax(series)
            if np.isnan(ymin) or np.isnan(ymax):
                ymin, ymax = -1, 1
            margin = (ymax - ymin) * 0.10
            ax.set_ylim(ymin - margin, ymax + margin)

        # --------------------------
        #  TÍTULOS DOS SUBPLOTS
        # --------------------------
        if row == 0:
            ax.set_title(f"Seasonal Decompose – Cluster {cluster_id + 1}",
                         fontsize=12)

        if col == 0:
            ax.set_ylabel(label, fontsize=10)
        else:
            ax.set_yticks([])

        # --------------------------
        #   EIXO X — só no último row
        # --------------------------
        if row == n_rows - 1:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.xaxis.set_major_locator(mdates.YearLocator())
            ax.tick_params(axis='x', labelrotation=0, labelsize=10)
        else:
            ax.set_xticks([])
            ax.set_xticklabels([])

plt.tight_layout()
plt.show()


## Average

In [ ]:
# Escolha aqui o que quer analisar: 'dV' ou 'dH'
TARGET_VAR = 'dV'  # ou 'dV'

# grelha
grid_size = 50

# Tipo de ligação
LINKAGE_METHOD = "average"
# Opções possíveis:
# "complete", "average", "ward"

# ==============================================================================
# SCRIPT COMPLETO: DTW + HIERÁRQUICO (COM ESTILO VISUAL K-MEANS MANTIDO)
# ==============================================================================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
import matplotlib.dates as mdates
from shapely.strtree import STRtree
from statsmodels.tsa.seasonal import seasonal_decompose
from matplotlib.lines import Line2D

# Bibliotecas para DTW e Hierárquico
try:
    from dtaidistance import dtw
    from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
    from scipy.spatial.distance import squareform
    from sklearn.preprocessing import StandardScaler
    import seaborn as sns
    from matplotlib.colors import to_hex
except ImportError as e:
    print(f"Erro de Importação: {e}. Certifique-se de que instalou: dtaidistance, scipy, seaborn, scikit-learn")

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
    #barragem = cos[cos['COS23_n4_L'] == 'Superfícies silvopastoris de azinheira']
    #barragem = cos[cos['COS23_n4_L'] == 'Matos']
except Exception as e:
    print(f"Aviso: COS placeholder. {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs e Filtro
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df): return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) & (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'], value_vars=disp_cols, var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(start=max(asc_long['date'].min(), desc_long['date'].min()), end=min(asc_long['date'].max(), desc_long['date'].max()), freq='MS')

# ==============================
# 3. Interpolação
# ==============================
def interpolate_ps(df, dates):
    dfs = []
    for (x, y), g in df.groupby(['easting','northing']):
        g = g.sort_values('date')
        interp = np.interp(pd.to_datetime(dates).astype(np.int64), g['date'].astype(np.int64), g['disp'])
        dfs.append(pd.DataFrame({'easting': x, 'northing': y, 'latitude': g['latitude'].iloc[0], 'longitude': g['longitude'].iloc[0], 'date': dates, 'disp': interp, 'incidence_angle': g['incidence_angle'].iloc[0], 'track_angle': g['track_angle'].iloc[0]}))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. IDW
# ==============================
def idw(source, target, radius=150, power=2):
    out = []
    for d, src in source.groupby('date'):
        tgt = target[target['date']==d].copy()
        if src.empty or tgt.empty: continue
        tree = cKDTree(list(zip(src['easting'], src['northing'])))
        dist, idx = tree.query(list(zip(tgt['easting'], tgt['northing'])), k=5, distance_upper_bound=radius)
        vals, thetas, alphas = [], [], []
        for d_i, i_i in zip(dist, idx):
            m = np.isfinite(d_i)
            if not np.any(m): vals.append(np.nan); thetas.append(np.nan); alphas.append(np.nan); continue
            w = 1/(d_i[m]**power)
            vals.append(np.sum(w*src.iloc[i_i[m]]['disp'])/np.sum(w))
            thetas.append(np.sum(w*src.iloc[i_i[m]]['incidence_angle'])/np.sum(w))
            alphas.append(np.sum(w*src.iloc[i_i[m]]['track_angle'])/np.sum(w))
        tgt['disp_idw'] = vals; tgt['theta_desc'] = thetas; tgt['alpha_desc'] = alphas
        out.append(tgt)
    return pd.concat(out, ignore_index=True)
asc_interp = idw(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw'])

# ==============================
# 5. dV (ou dH)
# ==============================
orb_inc = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orb_inc) * np.cos(np.deg2rad(asc_interp['latitude'])))

def get_comps(row):
    ta, td, beta = np.deg2rad(row['incidence_angle']), np.deg2rad(row['theta_desc']), row['beta']
    denom = (np.cos(ta)*np.sin(td)*np.cos(beta) + np.cos(td)*np.sin(ta)*np.cos(beta))
    if denom == 0: return np.nan, np.nan
    dV = (row['disp_idw']*np.sin(ta)*np.cos(beta) + row['disp']*np.sin(td)*np.cos(beta))/denom
    dH = (row['disp_idw']*np.cos(ta) - row['disp']*np.cos(td))/denom
    return dV, dH

asc_interp[['dV', 'dH']] = asc_interp.apply(lambda x: pd.Series(get_comps(x)), axis=1)
asc_interp = asc_interp.dropna(subset=['dV', 'dH'])

# ==============================
# 6. Grelha
# ==============================
#grid_size = 100
xe = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
ye = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
xs, ys = xe - grid_size/2, ye - grid_size/2
asc_interp['cx'] = pd.cut(asc_interp['easting'], bins=xs, labels=False)
asc_interp['cy'] = pd.cut(asc_interp['northing'], bins=ys, labels=False)
asc_interp = asc_interp.dropna(subset=['cx','cy'])
asc_interp['cell_id'] = asc_interp['cx'].astype(int).astype(str)+"_"+asc_interp['cy'].astype(int).astype(str)

grid_data = [{'cell_id': f"{ix}_{iy}", 'geometry': box(xs[ix], ys[iy], xs[ix+1], ys[iy+1])} for ix in range(len(xs)-1) for iy in range(len(ys)-1)]
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# Escolha aqui se usa dV ou dH para a agregação
#TARGET_VAR = 'dV' 
agg = asc_interp.groupby(['cell_id','date']).agg(dV=(TARGET_VAR,'mean')).reset_index() # Coluna final chama-se sempre 'dV' para manter compatibilidade com resto do script

# ==============================
# 7. Recorte e Filtro
# ==============================
points_gdf = gpd.GeoDataFrame(asc_interp, geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']), crs="EPSG:3035").to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

poly = grid_recort.geometry.values; ids = grid_recort['cell_id'].values; tree = STRtree(poly)
valid = set()
for pt in points_gdf.geometry:
    for idx in tree.query(pt):
        if poly[idx].contains(pt): valid.add(ids[idx])
grid_barragem = grid_recort[grid_recort['cell_id'].isin(valid)]
agg = agg[agg['cell_id'].isin(valid)]

# ==============================
# 8. CLUSTERING (DTW + HIERÁRQUICO) - SUBSTITUI K-MEANS
# ==============================
print(">>> A calcular Matriz de Distâncias DTW...")

# 1. Preparar Matriz (Pivot)
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

if n < 2:
    print("Aviso: Menos de 2 células para agrupar. Clustering ignorado.")
    cluster_labels = np.zeros(n, dtype=int)
    num_clusters = 1
    cut_distance = 0
    Z = None
else:
    # 2. Normalizar (Z-Score) para comparar formas
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X.T).T 

    # 3. Calcular DTW
    dist_matrix = np.zeros((n, n))
    window_dtw = int(0.1 * X_scaled.shape[1]) # Janela 10%
    
    # Esta parte pode demorar dependendo do nº de células
    for i in range(n):
        for j in range(i + 1, n):
            d = dtw.distance(X_scaled[i], X_scaled[j], window=window_dtw)
            dist_matrix[i, j] = d
            dist_matrix[j, i] = d

    # 4. Clustering Hierárquico
    condensed = squareform(dist_matrix)
    #Z = linkage(condensed, method='average')
    from scipy.cluster.hierarchy import linkage, fcluster
    Z = linkage(condensed, method=LINKAGE_METHOD)

    # 5. Corte Automático (Elbow)
    last = Z[-10:, 2] # Últimas distâncias de fusão
    acceleration = np.diff(last, 2)
    try:
        k_idx = np.argmax(acceleration) + 2
        cut_distance = (last[k_idx] + last[k_idx-1]) / 2
    except:
        cut_distance = last[len(last)//2]

    # Labels
    cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
    
    # Reajustar labels para começar em 0
    u_labels = np.unique(cluster_labels)
    map_l = {old: new for new, old in enumerate(u_labels)}
    cluster_labels = np.array([map_l[x] for x in cluster_labels])
    
    num_clusters = len(u_labels)
    print(f"DTW Concluído. Corte: {cut_distance:.2f}. Clusters: {num_clusters}")

# DataFrame Final
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})
# Cores Dinâmicas (para N clusters)
palette = sns.color_palette("Set2", num_clusters) if num_clusters <= 8 else sns.color_palette("tab20", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(range(num_clusters), palette)}

grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='inner')

# ==============================
# FIGURA EXTRA: DENDROGRAMA
# ==============================
if Z is not None:
    print("Gerando Dendrograma...")
    plt.figure(figsize=(12, 5))
    dendrogram(Z, leaf_rotation=90, leaf_font_size=8, color_threshold=cut_distance)
    plt.axhline(y=cut_distance, c='k', ls='--', lw=1, label='Corte Automático')
    plt.title('Dendrograma de Clustering Hierárquico (DTW)')
    plt.xlabel('Células'); plt.ylabel('Distância')
    plt.legend(); plt.tight_layout(); plt.show()


# ==============================
# 9. Dados Hidro (IGUAL)
# ==============================
date_range = agg_pivot.columns
win = 13

# Temp
try: df_t = pd.read_excel("data/alqueva_temp.xlsx"); df_t['data']=pd.to_datetime(df_t['data'])
except: df_t = pd.DataFrame({'data': date_range, 'med': 0})
ts = df_t.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': ts.index, 'med': ts.values})
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), win, 2)

# Nível
try: 
    df_n = pd.read_excel("data/alqueva_nivel.xlsx"); df_n['data']=pd.to_datetime(df_n['data'])
    df_n['nivel'] = pd.to_numeric(df_n['nivel'], errors='coerce'); df_n = df_n.dropna(subset=['nivel'])
except: df_n = pd.DataFrame({'data': date_range, 'nivel': 0})
ns = df_n.set_index('data')['nivel'].resample('MS').mean().reindex(date_range).interpolate(limit_direction='both').ffill().bfill()
df_nivel = pd.DataFrame({'data': ns.index, 'nivel': ns.values})
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], win, 2)

# Precipitação
try: df_p = pd.read_excel("data/prec.xlsx"); df_p['data'] = pd.to_datetime(df_p['data'])
except: np.random.seed(42); df_p = pd.DataFrame({'data': date_range, 'prec': 0})
ps = df_p.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': ps.index, 'prec': ps.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# # ==============================
# # 10. Funções Visuais (IGUAL)
# # ==============================
# def scale_vis(d, vmin, vmax, s=0.20, o=0.30):
#     d = np.array(d); mn, mx = np.nanmin(d), np.nanmax(d)
#     span = vmax - vmin
#     norm = np.zeros_like(d) if mx==mn else (d-mn)/(mx-mn)
#     return norm * span * s + (vmax - span * o), mn, mx

# def create_ticks(rmin, rmax, vmin, vmax, s=0.20, o=0.30, n=5):
#     rt = np.linspace(rmin, rmax, n)
#     span = vmax - vmin
#     nt = np.linspace(0,1,n) if rmax==rmin else (rt-rmin)/(rmax-rmin)
#     vt = nt * span * s + (vmax - span * o)
#     return vt, rt

# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks


# ==============================
# FIGURA 1: MAPA (ESTILO ORIGINAL)
# ==============================
print("Gerando Figura 1...")
clusters_present = sorted(cluster_df['cluster'].unique())
k_plot = len(clusters_present)

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', lw=1)

for i in clusters_present:
    sub = grid_sel[grid_sel['cluster'] == i]
    sub.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    cents = sub.copy(); cents.geometry = cents.geometry.centroid
    cents.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

hdl = [plt.Rectangle((0,0),1,1, fc=cluster_colors[i], alpha=0.4) for i in clusters_present]
lbl = [f'Cluster {i+1} ({len(grid_sel[grid_sel["cluster"]==i])} cel)' for i in clusters_present]
hdl.append(Line2D([0],[0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None')); lbl.append('Centróides')
ax_map.legend(hdl, lbl, loc='upper left', fontsize=10, title=f"Clusters DTW (K={k_plot})")
ax_map.set_title(f"Mapa de Clusters ({TARGET_VAR})", fontsize=15)
plt.show()

# ==============================
# FIGURA 2: SÉRIES TEMPORAIS
# ==============================

print("Gerando Figura 2...")

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5

fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

dV_min = agg[TARGET_VAR].min()
dV_max = agg[TARGET_VAR].max()
dV_margin = (dV_max - dV_min) * 0.1
dV_ylim = (dV_min - dV_margin, dV_max + dV_margin)

# --- Séries visuais ---
temp_visual, _, _ = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, _, _ = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)


for idx, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)


    # --- Função atualizada ---
    def plot_dV_comparison_series(
        row_idx, ext_data_visual, ext_data_df, ext_col_real,
        ext_label, ext_color,
        plot_as_bar=False, bar_color=None,
        plot_real_scale_line=False,
        combine_annual_prec=False,
        integer_ticks=False, show_background_bars=False):

        ax = fig_series.add_subplot(gs_series[row_idx, idx])

        # --- dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid],
                    color='lightgray', alpha=0.7, linewidth=1.0)

        ax.plot(cluster_data.columns, cluster_mean_dV,
                color=cluster_color, linewidth=1.0,
                label=f'Média Cluster {cluster_id+1}')

        # --- Eixo Y2 ---
        ax2 = ax.twinx()
        lines2 = []

        # -------------------------
        #   CASOS DE PRECIPITAÇÃO
        # -------------------------
        if combine_annual_prec:

            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)

            for ano in df_prec['ano_hidrologico'].unique():
                grupo = df_prec[df_prec['ano_hidrologico'] == ano]
                ax2.plot(grupo['data'], grupo['prec_acum_anual'],
                         color='teal', linewidth=1.0, alpha=0.9)

            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            ax2.set_ylabel("Prec. acumulada anual", color="teal")
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=1.0, label='Prec. acumulada anual (mm)')
            ]

        elif plot_as_bar:

            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15,
                    color=bar_color or ext_color, alpha=0.5)

            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            ax2.set_ylabel(ext_label, color='teal')
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)
            ]

        elif plot_real_scale_line:

            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15,
                        color='teal', alpha=0.3)

            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real],
                     color='teal', linewidth=1.0, alpha=0.85)

            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            ax2.set_ylabel(ext_label, color='teal')
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=1.0, label=ext_label)
            ]

        # -------------------------
        #      OUTRAS SÉRIES
        # -------------------------
        else:

            ax2.plot(ext_data_df['data'], ext_data_visual,
                     color=ext_color, linewidth=1.0, alpha=0.85)

            visual_ticks, real_ticks = create_visual_ticks(
                ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(),
                dV_min, dV_max, scale_factor=0.20, offset_factor=0.30
            )

            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(visual_ticks)

            if integer_ticks:
                ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else:
                ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])

            label_color = 'black' if 'Temperatura' in ext_label else ext_color
            ax2.set_ylabel(ext_label, color=label_color)
            ax2.tick_params(axis='y', colors=label_color)

            lines2 = [ax2.lines[-1]]

        # --- Y1 sempre dV ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)

        if idx == 0:
            ax.set_ylabel('dV (mm)')
        else:
            ax.set_yticklabels([])

        # Y2 somente no último cluster
        if idx != n_clusters - 1:
            ax2.set_yticklabels([])

        ax.tick_params(left=(idx==0))
        ax2.tick_params(right=(idx==n_clusters-1))

        # --- X-axis ---
        if row_idx == n_rows_series - 1:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        else:
            ax.set_xticklabels([])

        # --- Legenda somente no primeiro e no último cluster ---
        if idx == 0 or idx == n_clusters - 1:
            lines1, labels1 = ax.get_legend_handles_labels()
            handles2, labels2 = ax2.get_legend_handles_labels()

            ax.legend(
                lines1 + handles2,
                labels1 + labels2,
                fontsize=8,
                loc='best',
                framealpha=1.0,
                facecolor='white',
                edgecolor='lightgray'
            ).set_zorder(100)

        return ax


    # ---- CHAMADA DAS 5 SÉRIES ----
    plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth',
                              'Temperatura média (°C)', 'black', integer_ticks=True)

    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth',
                              'Nível da albufeira (m)', 'navy', integer_ticks=True)

    plot_dV_comparison_series(2, None, df_prec, 'prec',
                              'Precipitação mensal (mm)', 'teal',
                              plot_as_bar=True, bar_color='teal')

    plot_dV_comparison_series(3, None, df_prec, 'prec_acum',
                              'Prec. acumulada total (mm)', 'teal',
                              plot_real_scale_line=True, show_background_bars=True)

    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual',
                              'Prec. acumulada anual', 'teal',
                              combine_annual_prec=True)


plt.tight_layout()
plt.show()


# ==============================
# FIGURA 3: DECOMPOSIÇÃO SAZONAL
# ==============================

print("Gerando Figura 3...")

n_rows = 4  # Observed, Trend, Seasonal, Residual
fig_series_decomp, axes = plt.subplots(
    n_rows, n_clusters, figsize=(6 * max(1, n_clusters), 7), sharex=False
)

for col, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    mean_series = cluster_data.mean(axis=0)

    decomposition = seasonal_decompose(mean_series, model="additive", period=12)

    series_list = [
        ("Observed", decomposition.observed),
        ("Trend", decomposition.trend),
        ("Seasonal", decomposition.seasonal),
        ("Residual", decomposition.resid)
    ]

    cluster_color = cluster_colors.get(cluster_id, "black")

    for row, (label, series) in enumerate(series_list):

        ax = axes[row, col]

        # --------------------------
        #   PLOT (agora com cores)
        # --------------------------
        ax.plot(series.index, series.values,
                color=cluster_color, linewidth=1.0)

        # --------------------------
        #  LIMITE EXTERIOR (spines)
        # --------------------------
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.8)
            spine.set_color("black")

        # --------------------------
        #     LIMITES Y
        # --------------------------
        if label != "Observed":
            ymin = np.nanmin(series)
            ymax = np.nanmax(series)
            if np.isnan(ymin) or np.isnan(ymax):
                ymin, ymax = -1, 1
            margin = (ymax - ymin) * 0.10
            ax.set_ylim(ymin - margin, ymax + margin)

        # --------------------------
        #  TÍTULOS DOS SUBPLOTS
        # --------------------------
        if row == 0:
            ax.set_title(f"Seasonal Decompose – Cluster {cluster_id + 1}",
                         fontsize=12)

        if col == 0:
            ax.set_ylabel(label, fontsize=10)
        else:
            ax.set_yticks([])

        # --------------------------
        #   EIXO X — só no último row
        # --------------------------
        if row == n_rows - 1:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.xaxis.set_major_locator(mdates.YearLocator())
            ax.tick_params(axis='x', labelrotation=0, labelsize=10)
        else:
            ax.set_xticks([])
            ax.set_xticklabels([])

plt.tight_layout()
plt.show()


In [ ]:
# ==============================
# FIGURA 5: MAPAS DE CORRELAÇÃO DE PEARSON POR CÉLULA
# (linhas = variáveis, colunas = clusters)
# ==============================

print("Gerando Figura 5 (Mapas de Correlação por Célula)...")

from scipy.stats import pearsonr
import matplotlib.colors as mcolors

# --- Variáveis externas (5) ---
external_vars_map = {
    'Temperatura (°C)':         df_temp.set_index('data')['med_smooth'],
    'Nível albufeira (m)':      df_nivel.set_index('data')['nivel_smooth'],
    'Precipitação mensal (mm)': df_prec.set_index('data')['prec'],
    'Prec. acumulada total':    df_prec.set_index('data')['prec_acum'],
    'Prec. acumulada anual':    df_prec.set_index('data')['prec_acum_anual'],
}

var_list   = list(external_vars_map.keys())
n_vars     = len(var_list)       # 5 linhas
n_cols_fig = n_clusters          # colunas = clusters

cmap_corr = plt.cm.RdBu_r
norm_corr = mcolors.Normalize(vmin=-1, vmax=1)

fig_map_corr, axes_map = plt.subplots(
    n_vars, n_cols_fig,
    figsize=(5.5 * n_cols_fig, 5.0 * n_vars)
)

# Garantir sempre array 2D
if n_vars == 1 and n_cols_fig == 1:
    axes_map = np.array([[axes_map]])
elif n_vars == 1:
    axes_map = axes_map[np.newaxis, :]
elif n_cols_fig == 1:
    axes_map = axes_map[:, np.newaxis]

# --- Extensão do mapa ---
grid_extent = grid_barragem.to_crs(epsg=3857).total_bounds
margin = 100
xlim = (grid_extent[0] - margin, grid_extent[2] + margin)
ylim = (grid_extent[1] - margin, grid_extent[3] + margin)

# --- Pré-calcular r para todas as combinações (célula × variável) ---
# Evita recalcular o mesmo cell_id múltiplas vezes
print("  A calcular correlações por célula...")

# dict: vname -> DataFrame com cell_id, r, p
corr_by_var = {}
for vname, vseries in external_vars_map.items():
    records = []
    for cell_id in valid:
        cell_ts  = agg[agg['cell_id'] == cell_id].set_index('date')['dV']
        combined = pd.DataFrame({'dv': cell_ts, 'ext': vseries}).dropna()
        if len(combined) > 5:
            r, p = pearsonr(combined['ext'], combined['dv'])
        else:
            r, p = np.nan, np.nan
        records.append({'cell_id': cell_id, 'r': r, 'p': p})
    corr_by_var[vname] = pd.DataFrame(records)

# --- Plot ---
for vi, vname in enumerate(var_list):
    corr_df = corr_by_var[vname]

    for ci, cluster_id in enumerate(clusters_present):

        ax = axes_map[vi, ci]

        # Células deste cluster
        cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
        corr_cluster  = corr_df[corr_df['cell_id'].isin(cluster_cells)]

        # Merge com geometria (só células do cluster)
        grid_cluster = (
            grid_barragem[grid_barragem['cell_id'].isin(cluster_cells)]
            .merge(corr_cluster, on='cell_id', how='left')
            .to_crs(epsg=3857)
        )

        # Limites antes do basemap
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)

        # Basemap
        ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery,
                        crs='EPSG:3857', reset_extent=False)

        # Todas as células do mapa (cinza claro) para contexto
        grid_barragem.to_crs(epsg=3857).plot(
            ax=ax, color='none', edgecolor='white',
            linewidth=0.3, alpha=0.4
        )

        # Células do cluster coloridas por r
        grid_valid = grid_cluster.dropna(subset=['r'])
        if not grid_valid.empty:
            grid_valid.plot(
                column='r', ax=ax,
                cmap=cmap_corr, norm=norm_corr,
                alpha=0.80, edgecolor='black', linewidth=0.5
            )

        # # Contorno dourado para |r| > 0.5
        # edge_color = 'gold' if abs(r_val) >= 0.5 else 'black'
        # line_width  = 2.0   if abs(r_val) >= 0.5 else 0.5
        # alpha_val   = 1.0   if p_val < 0.05 else 0.25  # máscara para não-significativo
        # font_sz     = 6 + abs(r_val) * 5               # texto proporcional

        # Anotar r dentro de cada célula
        for _, row in grid_valid.iterrows():
            r_val, p_val = row['r'], row['p']
            if np.isnan(r_val):
                continue
            sig       = '*' if (not np.isnan(p_val) and p_val < 0.05) else ''
            txt_color = 'white' if abs(r_val) > 0.5 else 'black'
            ax.text(
                row.geometry.centroid.x,
                row.geometry.centroid.y,
                f'{r_val:.2f}{sig}',
                ha='center', va='center',
                fontsize=6.5, color=txt_color, fontweight='bold'
            )

        # Repor limites (geopandas reseta)
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_axis_off()

        # Títulos: coluna no topo, linha à esquerda
        if vi == 0:
            cluster_color = cluster_colors.get(cluster_id, 'black')
            ax.set_title(f'Cluster {cluster_id + 1}', fontsize=11,
                         color=cluster_color, pad=6)
        if ci == 0:
            ax.set_ylabel(vname, fontsize=9, labelpad=6)
            # ylabel não aparece com set_axis_off — usar texto
            ax.text(-0.04, 0.5, vname,
                    transform=ax.transAxes,
                    fontsize=8.5, va='center', ha='right',
                    rotation=90, color='black')

# --- Colorbar partilhada ---
sm = plt.cm.ScalarMappable(cmap=cmap_corr, norm=norm_corr)
sm.set_array([])
cbar = fig_map_corr.colorbar(
    sm, ax=axes_map,
    fraction=0.012, pad=0.02,
    orientation='vertical'
)
cbar.set_label('r de Pearson', fontsize=11)
cbar.ax.tick_params(labelsize=9)
for ref in [-0.5, 0, 0.5]:
    cbar.ax.axhline((ref + 1) / 2, color='gray', linewidth=0.8, linestyle='--')

fig_map_corr.suptitle(
    f'Correlação de Pearson por célula — {TARGET_VAR} vs. variáveis externas\n(* p < 0.05)',
    fontsize=14
)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# CONFIGURAÇÃO DA FIGURA ÚNICA (FILTRO)
# ==============================================================================

# 1. Escolhe o cluster que queres isolar (ex: Cluster 4 é o id 3)
CLUSTER_ALVO = 3  # Lembra-te: ID 0 = Cluster 1, ID 3 = Cluster 4

# 2. Escolhe as variáveis que queres mostrar (podes deixar apenas uma ou duas)
VARS_INTERESSE = [
    'Temperatura (°C)', 
    'Nível albufeira (m)'
]

# --- Filtragem dos dados para a Figura Única ---
vars_filtradas = {k: v for k, v in external_vars_map.items() if k in VARS_INTERESSE}
var_list = list(vars_filtradas.keys())
n_vars = len(var_list)
n_cols_fig = 1  # Apenas uma coluna porque isolámos 1 cluster

fig_unica, axes_map = plt.subplots(
    n_vars, n_cols_fig, 
    figsize=(7, 6 * n_vars) # Ajuste de tamanho para uma coluna
)

# Garantir array 2D para o loop funcionar
if n_vars == 1: axes_map = np.array([axes_map])
axes_map = axes_map.reshape(n_vars, 1)

# --- Loop de Plot (Simplificado para o Alvo) ---
for vi, vname in enumerate(var_list):
    corr_df = corr_by_var[vname] # Usa o pré-cálculo que já fizeste
    ax = axes_map[vi, 0]

    # Filtrar apenas células do cluster alvo
    cluster_cells = cluster_df[cluster_df['cluster'] == CLUSTER_ALVO]['cell_id'].tolist()
    corr_cluster = corr_df[corr_df['cell_id'].isin(cluster_cells)]

    # Geometria
    grid_cluster = (
        grid_barragem[grid_barragem['cell_id'].isin(cluster_cells)]
        .merge(corr_cluster, on='cell_id', how='left')
        .to_crs(epsg=3857)
    )

    # Plot Basemap e Células (mesma lógica do teu código anterior)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, crs='EPSG:3857', reset_extent=False)
    
    grid_valid = grid_cluster.dropna(subset=['r'])
    if not grid_valid.empty:
        grid_valid.plot(column='r', ax=ax, cmap=cmap_corr, norm=norm_corr, 
                        alpha=0.85, edgecolor='black', linewidth=0.6)

        # Anotações de r dentro das células
        for _, row in grid_valid.iterrows():
            r_val, p_val = row['r'], row['p']
            sig = '*' if (p_val < 0.05) else ''
            txt_color = 'white' if abs(r_val) > 0.5 else 'black'
            ax.text(row.geometry.centroid.x, row.geometry.centroid.y,
                    f'{r_val:.2f}{sig}', ha='center', va='center',
                    fontsize=8, color=txt_color, fontweight='bold')

    ax.set_title(f'Cluster {CLUSTER_ALVO + 1} vs {vname}', fontsize=12)
    ax.set_axis_off()

# Adicionar a colorbar à direita
sm = plt.cm.ScalarMappable(cmap=cmap_corr, norm=norm_corr)
fig_unica.colorbar(sm, ax=axes_map, fraction=0.046, pad=0.04, label='r de Pearson')

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# FIGURA ÚNICA: CLUSTER ALVO - LADO A LADO
# ==============================================================================

# 1. Escolhe o cluster (ID 3 = Cluster 4)
CLUSTER_ALVO = 3 

# 2. Variáveis de interesse (máximo 2 para este layout lado a lado)
VARS_INTERESSE = ['Temperatura (°C)', 'Nível albufeira (m)']

vars_filtradas = {k: v for k, v in external_vars_map.items() if k in VARS_INTERESSE}
var_list = list(vars_filtradas.keys())
n_vars = len(var_list)

# Criamos 1 linha e 2 colunas. 
# Adicionamos 'gridspec_kw' para deixar um espaço reservado para a colorbar à direita
fig_unica, axes_map = plt.subplots(
    1, n_vars, 
    figsize=(14, 7), 
    gridspec_kw={'width_ratios': [1]*n_vars}
)

# Garantir que axes_map é iterável se houver apenas 1 variável
if n_vars == 1: axes_map = [axes_map]

for i, vname in enumerate(var_list):
    corr_df = corr_by_var[vname]
    ax = axes_map[i]

    # Filtrar células do cluster
    cluster_cells = cluster_df[cluster_df['cluster'] == CLUSTER_ALVO]['cell_id'].tolist()
    corr_cluster = corr_df[corr_df['cell_id'].isin(cluster_cells)]

    grid_cluster = (
        grid_barragem[grid_barragem['cell_id'].isin(cluster_cells)]
        .merge(corr_cluster, on='cell_id', how='left')
        .to_crs(epsg=3857)
    )

    # Configuração do mapa
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, crs='EPSG:3857', reset_extent=False)
    
    grid_valid = grid_cluster.dropna(subset=['r'])
    if not grid_valid.empty:
        grid_valid.plot(column='r', ax=ax, cmap=cmap_corr, norm=norm_corr, 
                        alpha=0.85, edgecolor='black', linewidth=0.6)

        for _, row in grid_valid.iterrows():
            r_val, p_val = row['r'], row['p']
            sig = '*' if (not np.isnan(p_val) and p_val < 0.05) else ''
            txt_color = 'white' if abs(r_val) > 0.5 else 'black'
            ax.text(row.geometry.centroid.x, row.geometry.centroid.y,
                    f'{r_val:.2f}{sig}', ha='center', va='center',
                    fontsize=9, color=txt_color, fontweight='bold')

    ax.set_title(f'{vname}\n(Cluster {CLUSTER_ALVO + 1})', fontsize=13, pad=10)
    ax.set_axis_off()

# --- Ajuste da Colorbar (Legenda) ---
# Criamos um eixo específico para a colorbar para não afetar o tamanho dos mapas
cbar_ax = fig_unica.add_axes([0.92, 0.25, 0.02, 0.5]) # [esquerda, baixo, largura, altura]
sm = plt.cm.ScalarMappable(cmap=cmap_corr, norm=norm_corr)
cb = fig_unica.colorbar(sm, cax=cbar_ax)
cb.set_label('r de Pearson', fontsize=12, fontweight='bold')
cb.outline.set_linewidth(1)

# Ajuste final para evitar sobreposição
plt.subplots_adjust(right=0.9, wspace=0.1)
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# 1. Estilo Profissional e Limpo
sns.set_context("paper")
sns.set_style("white") # Remove o fundo cinza e as grelhas

# 2. DEFINE O CLUSTER
CLUSTER_ALVO_ID = 3

# (O Teu bloco de preparação de dados mantém-se igual)
c_cells = cluster_df[cluster_df['cluster'] == CLUSTER_ALVO_ID]['cell_id']
dV_mean = agg_pivot.loc[c_cells].mean()

df_cluster_unico = pd.DataFrame({
    'dV_InSAR': dV_mean,
    'Temp':     df_temp.set_index('data')['med_smooth'].reindex(dV_mean.index),
    'Nivel':    df_nivel.set_index('data')['nivel_smooth'].reindex(dV_mean.index),
    'Prec':     df_prec.set_index('data')['prec'].reindex(dV_mean.index),
    'Prec_Acum': df_prec.set_index('data')['prec_acum'].reindex(dV_mean.index),
    'Prec_Anual': df_prec.set_index('data')['prec_acum_anual'].reindex(dV_mean.index)
}).dropna()

cols_num = ['dV_InSAR', 'Temp', 'Nivel', 'Prec', 'Prec_Acum', 'Prec_Anual']

# 3. Criar a Grid QUADRADA (height=aspect)
# Sem linhas auxiliares internas
g = sns.PairGrid(df_cluster_unico, vars=cols_num, diag_sharey=False, 
                 height=0.5,  # Altura total do quadrado
                 aspect=1.0)  # Aspecto 1.0 garante que cada sub-gráfico é quadrado

# --- DIAGONAL: Histogramas Cinzentos ---
g.map_diag(sns.histplot, kde=True, color="#404040", alpha=0.8, edgecolor='white')

# --- PARTE INFERIOR: Scatter Plots Pretos com Regressão Vermelha ---
g.map_lower(sns.regplot, 
            scatter_kws={'s': 10, 'alpha': 0.6, 'color': 'black'}, 
            line_kws={'color': '#d7191c', 'linewidth': 1.5}) # Vermelho sóbrio

# --- PARTE SUPERIOR: Cores de fundo em escala de cinza/vermelho ---
def cor_func_profissional(x, y, **kwargs):
    r, p = pearsonr(x, y)
    ax = plt.gca()
    
    # Usar escala de cinzas para fundo, ou vermelho muito suave se r for alto
    abs_r = abs(r)
    # Se r for positivo, azul suave; se negativo, vermelho suave (escala profissional)
    cmap = plt.cm.Greys # Ou podes usar RdGy_r para um toque de cor profissional
    facecolor = cmap(abs_r * 0.5) # Escurece conforme a força da correlação
    
    ax.set_facecolor(facecolor)
    
    # Texto a preto para contraste
    sig = "*" if p < 0.05 else ""
    ax.annotate(f"{r:.2f}{sig}", xy=(0.5, 0.5), xycoords=ax.transAxes,
                ha='center', va='center', fontsize=11,
                #fontweight='bold',
                color='black'
                )
    
    # Remover bordas internas de cada quadrado de texto
    for spine in ax.spines.values():
        spine.set_visible(False)

g.map_upper(cor_func_profissional)

# 4. Ajustes Finais de "Limpeza"
sns.despine() # Remove as bordas superiores e direitas da figura total

# Título e Labels
g.fig.suptitle(f'Structural Correlation Analysis: Cluster {CLUSTER_ALVO_ID + 1}', 
               y=1.03, fontsize=14,
               #fontweight='bold',
               color='black')

for ax in g.axes.flatten():
    ax.tick_params(colors='#333333', labelsize=8)
    ax.xaxis.label.set_color('black')
    ax.yaxis.label.set_color('black')
    # Garantir que não há grelha
    ax.grid(False)

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# 1. Configuração de Estilo Minimalista
sns.set_context("paper", font_scale=0.9)
sns.set_style("white")

CLUSTER_ALVO_ID = 3

# (Preparação de dados)
c_cells = cluster_df[cluster_df['cluster'] == CLUSTER_ALVO_ID]['cell_id']
dV_mean = agg_pivot.loc[c_cells].mean()

df_cluster_unico = pd.DataFrame({
    'dV_InSAR': dV_mean,
    'Temp':     df_temp.set_index('data')['med_smooth'].reindex(dV_mean.index),
    'Nivel':    df_nivel.set_index('data')['nivel_smooth'].reindex(dV_mean.index),
    'Prec':     df_prec.set_index('data')['prec'].reindex(dV_mean.index),
    'Prec_Acum': df_prec.set_index('data')['prec_acum'].reindex(dV_mean.index),
    'Prec_Anual': df_prec.set_index('data')['prec_acum_anual'].reindex(dV_mean.index)
}).dropna()

cols_num = ['dV_InSAR', 'Temp', 'Nivel', 'Prec', 'Prec_Acum', 'Prec_Anual']

# 2. Criar a Grid (Height ajustado para 2.0 para ser visível e quadrado)
g = sns.PairGrid(df_cluster_unico, vars=cols_num, diag_sharey=False, 
                 height=1.0, aspect=1.0)

# --- DIAGONAL: Histogramas ---
g.map_diag(sns.histplot, kde=True, color="#505050", alpha=0.2, 
           edgecolor='black', linewidth=0.5)

# --- PARTE INFERIOR: Scatter Plots (CORRIGIDO) ---
g.map_lower(sns.regplot, 
            # Removido 'lw' para evitar conflito com 'linewidths' interno do Seaborn
            scatter_kws={'s': 6, 'alpha': 0.5, 'color': 'black', 'linewidths': 0}, 
            line_kws={'color': '#d7191c', 'linewidth': 0.8})

# --- PARTE SUPERIOR: Correlações ---
def cor_func_elegante(x, y, **kwargs):
    r, p = pearsonr(x, y)
    ax = plt.gca()
    
    abs_r = abs(r)
    cmap = plt.cm.Greys
    facecolor = cmap(abs_r * 0.25) 
    ax.set_facecolor(facecolor)
    
    sig = "*" if p < 0.05 else ""
    ax.annotate(f"{r:.2f}{sig}", xy=(0.5, 0.5), xycoords=ax.transAxes,
                ha='center', va='center', fontsize=9, 
                fontweight='normal', color='#222222')
    
    for spine in ax.spines.values():
        spine.set_visible(False)

g.map_upper(cor_func_elegante)

# 3. Ajustes de Limpeza Final
sns.despine()

g.fig.suptitle(f'Structural Correlation Analysis | Cluster {CLUSTER_ALVO_ID + 1}', 
               y=1.02, fontsize=11, fontweight='normal', color='black')

for ax in g.axes.flatten():
    # Linhas dos eixos muito finas
    for spine in ax.spines.values():
        spine.set_linewidth(0.4)
    
    # Ticks finos
    ax.tick_params(colors='#333333', labelsize=7, width=0.4, length=2)
    
    ax.xaxis.label.set_fontsize(8)
    ax.yaxis.label.set_fontsize(8)
    ax.grid(False)

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import matplotlib.colors as mcolors

# 1. Estilo Base "Tabela" (fechado)
sns.set_context("paper", font_scale=0.9)
sns.set_style("white")

CLUSTER_ALVO_ID = 3

# (Preparação de dados igual)
c_cells = cluster_df[cluster_df['cluster'] == CLUSTER_ALVO_ID]['cell_id']
dV_mean = agg_pivot.loc[c_cells].mean()

df_cluster_unico = pd.DataFrame({
    'dV_InSAR': dV_mean,
    'Temp':     df_temp.set_index('data')['med_smooth'].reindex(dV_mean.index),
    'Nivel':    df_nivel.set_index('data')['nivel_smooth'].reindex(dV_mean.index),
    'Prec':     df_prec.set_index('data')['prec'].reindex(dV_mean.index),
    'Prec_Acum': df_prec.set_index('data')['prec_acum'].reindex(dV_mean.index),
    'Prec_Anual': df_prec.set_index('data')['prec_acum_anual'].reindex(dV_mean.index)
}).dropna()

cols_num = ['dV_InSAR', 'Temp', 'Nivel', 'Prec', 'Prec_Acum', 'Prec_Anual']

# 2. Criar a Grid (Ajustei height para 1.5 para melhor leitura)
g = sns.PairGrid(df_cluster_unico, vars=cols_num, diag_sharey=False, height=1.0, aspect=1.0)

# --- DIAGONAL: Histogramas com Riscas Vermelhas (igual à imagem) ---
g.map_diag(sns.histplot, color="white", edgecolor='red', linewidth=0.5, hatch='///', alpha=0)

# --- PARTE INFERIOR: Scatter Plots Pretos SEM Intervalo de Confiança ---
g.map_lower(sns.regplot, 
            ci=None, # <--- REMOVE O INTERVALO DE CONFIANÇA (SOMBRA)
            scatter_kws={'s': 5, 'alpha': 0.9, 'color': 'black', 'linewidths': 0}, 
            line_kws={'color': 'red', 'linewidth': 0.8})

# --- PARTE SUPERIOR: Mapa de Cores Azul/Vermelho (Estilo Imagem) ---
def cor_func_replica(x, y, **kwargs):
    r, p = pearsonr(x, y)
    ax = plt.gca()
    
    # RdBu_r: Vermelho para Negativos, Azul para Positivos
    cmap = plt.cm.RdBu_r
    # Normalização suave (transparência 0.6 para tons pastel)
    facecolor = cmap((r + 1) / 2)
    ax.set_facecolor((*facecolor[:3], 0.6)) 
    
    # Texto com precisão de 5 casas (igual à tua imagem)
    ax.annotate(f"{r:.3f}", xy=(0.5, 0.5), xycoords=ax.transAxes,
                ha='center', va='center', fontsize=8, color='black')
    
    # Moldura preta fina em cada quadrado
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(0.5)

g.map_upper(cor_func_replica)

# 3. Ajustes de Limpeza Final e "Grelha"
for ax in g.axes.flatten():
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(0.5)
    
    ax.tick_params(labelsize=7, width=0.5, direction='out', length=2)
    ax.grid(False)

# Adicionar nomes das variáveis no topo da diagonal
for i, var in enumerate(cols_num):
    g.axes[i, i].set_title(var, fontsize=8, pad=2)

# Colar os gráficos para parecer uma tabela única
plt.subplots_adjust(hspace=0.0, wspace=0.0) 

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import matplotlib.colors as mcolors

# 1. Configuração de Estilo Minimalista
sns.set_context("paper", font_scale=0.9)
sns.set_style("white")

CLUSTER_ALVO_ID = 3

# (Preparação de dados - mantida)
c_cells = cluster_df[cluster_df['cluster'] == CLUSTER_ALVO_ID]['cell_id']
dV_mean = agg_pivot.loc[c_cells].mean()

df_cluster_unico = pd.DataFrame({
    'dV': dV_mean,
    'Temp': df_temp.set_index('data')['med_smooth'].reindex(dV_mean.index),
    'Nivel': df_nivel.set_index('data')['nivel_smooth'].reindex(dV_mean.index),
    'Prec': df_prec.set_index('data')['prec'].reindex(dV_mean.index),
    'P_Acum': df_prec.set_index('data')['prec_acum'].reindex(dV_mean.index),
    'P_Anual': df_prec.set_index('data')['prec_acum_anual'].reindex(dV_mean.index)
}).dropna()

cols_num = df_cluster_unico.columns.tolist()

# 2. Criar a Grid
g = sns.PairGrid(df_cluster_unico, vars=cols_num, diag_sharey=False, height=1.0, aspect=1.0)

# --- DIAGONAL: Histogramas limpos ---
g.map_diag(sns.histplot, color="white", edgecolor='#d7191c', linewidth=0.6, hatch='///', alpha=0)

# --- PARTE INFERIOR: Scatter Plots (Ponto pequeno e traço fino) ---
g.map_lower(sns.regplot, ci=None, 
            scatter_kws={'s': 4, 'alpha': 0.8, 'color': 'black', 'linewidths': 0}, 
            line_kws={'color': '#d7191c', 'linewidth': 0.7})

# --- PARTE SUPERIOR: Cores de Correlação ---
def cor_func_clean(x, y, **kwargs):
    r, p = pearsonr(x, y)
    ax = plt.gca()
    cmap = plt.cm.RdBu_r
    facecolor = cmap((r + 1) / 2)
    ax.set_facecolor((*facecolor[:3], 0.5)) 
    
    # r com 2 casas decimais é geralmente mais clean que 5
    ax.annotate(f"{r:.2f}", xy=(0.5, 0.5), xycoords=ax.transAxes,
                ha='center', va='center', fontsize=8, color='black')
    
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.4)

g.map_upper(cor_func_clean)

# 3. LIMPEZA RADICAL DE EIXOS E LEGENDAS
for i, j in zip(*plt.np.indices(g.axes.shape).reshape(2, -1)):
    ax = g.axes[i, j]
    
    # Moldura fina em todos
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.4)
        spine.set_color('#333333')

    # Remover labels de eixos internos (Manter apenas na borda esquerda e inferior)
    if j != 0: # Não é a primeira coluna
        ax.set_ylabel('')
    if i != len(cols_num) - 1: # Não é a última linha
        ax.set_xlabel('')
    
    # Ajustar Ticks
    ax.tick_params(labelsize=6, width=0.4, length=2, pad=1, direction='in')
    
    # Se for a diagonal, colocar o nome da variável no centro ou topo
    if i == j:
        ax.set_title(cols_num[i], fontsize=8, fontweight='bold', pad=2)

# Colar os gráficos completamente
plt.subplots_adjust(hspace=0.0, wspace=0.0)

# Título Principal (opcional, para artigo às vezes retira-se e usa-se a legenda da figura)
# g.fig.suptitle(f'Correlation Matrix - Cluster {CLUSTER_ALVO_ID + 1}', y=1.02, fontsize=10)

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import matplotlib.colors as mcolors
import numpy as np

# 1. Configuração de Estilo Académico (Estilo Elsevier/Nature)
sns.set_context("paper", font_scale=0.9)
sns.set_style("white")

# --- DEFINE O CLUSTER ---
CLUSTER_ALVO_ID = 3 

# 2. Preparação de Dados (dV + Variáveis Externas)
c_cells = cluster_df[cluster_df['cluster'] == CLUSTER_ALVO_ID]['cell_id']
dV_mean = agg_pivot.loc[c_cells].mean()

df_cluster_unico = pd.DataFrame({
    'dV': dV_mean,
    'Temp': df_temp.set_index('data')['med_smooth'].reindex(dV_mean.index),
    'Nivel': df_nivel.set_index('data')['nivel_smooth'].reindex(dV_mean.index),
    'Prec': df_prec.set_index('data')['prec'].reindex(dV_mean.index),
    'P_Acum': df_prec.set_index('data')['prec_acum'].reindex(dV_mean.index),
    'P_Anual': df_prec.set_index('data')['prec_acum_anual'].reindex(dV_mean.index)
}).dropna()

cols_num = df_cluster_unico.columns.tolist()

# 3. Criar a Grid (Height 1.3 é o ideal para 6 variáveis em A4)
g = sns.PairGrid(df_cluster_unico, vars=cols_num, diag_sharey=False, height=0.8, aspect=1.0)

# --- DIAGONAL: Histogramas Académicos (Preenchimento Sólido + KDE) ---
def diag_hist_prof(x, **kwargs):
    sns.histplot(x, kde=True, color="#2c3e50", alpha=0.3, 
                 edgecolor='white', linewidth=0.3, 
                 line_kws={'linewidth': 1.0, 'color': '#d7191c'})
g.map_diag(diag_hist_prof)

# --- PARTE INFERIOR: Scatter Plots (Pontos Pretos Limpos + Regressão Fina) ---
g.map_lower(sns.regplot, ci=None, 
            scatter_kws={'s': 3.5, 'alpha': 0.7, 'color': 'black', 'linewidths': 0}, 
            line_kws={'color': '#d7191c', 'linewidth': 0.8})

# --- PARTE SUPERIOR: Matriz de Cores Pastel (RdBu_r) ---
def cor_func_academica(x, y, **kwargs):
    r, p = pearsonr(x, y)
    ax = plt.gca()
    cmap = plt.cm.RdBu_r
    # Normalização suave (transparência 0.4 para look académico)
    facecolor = cmap((r + 1) / 2)
    ax.set_facecolor((*facecolor[:3], 0.4)) 
    
    # Texto sem negrito, limpo
    sig = "*" if p < 0.05 else ""
    ax.annotate(f"{r:.2f}{sig}", xy=(0.5, 0.5), xycoords=ax.transAxes,
                ha='center', va='center', fontsize=8, color='black')
    
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.4)
        spine.set_color('#333333')

g.map_upper(cor_func_academica)

# 4. LIMPEZA FINAL DE EIXOS (REMOÇÃO DE REDUNDÂNCIA)
for i, j in zip(*np.indices(g.axes.shape).reshape(2, -1)):
    ax = g.axes[i, j]
    
    # Moldura fina em todos
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.4)
        spine.set_color('#333333')

    # Manter labels APENAS nas extremidades para aspeto clean
    if j != 0: 
        ax.set_yticklabels([])
        ax.set_ylabel('')
    if i != len(cols_num) - 1: 
        ax.set_xticklabels([])
        ax.set_xlabel('')
    
    # Ticks pequenos e para dentro (estilo profissional)
    ax.tick_params(labelsize=6, width=0.4, length=2, pad=1, direction='in')
    
    # Nomes das variáveis no topo da diagonal
    # if i == j:
    #     ax.set_title(cols_num[i], fontsize=8, fontweight='bold', pad=3)

# Eliminar espaços entre gráficos
plt.subplots_adjust(hspace=0.0, wspace=0.0)

# Guardar com alta resolução para o artigo
# plt.savefig(f"analise_academica_cluster_{CLUSTER_ALVO_ID+1}.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import numpy as np

# 1. Definições Globais de Estilo
LINE_WIDTH = 0.4
LINE_COLOR = '#333333' 
MY_CMAP = plt.cm.RdBu # RdBu padrão: Azul é Positivo (+1), Vermelho é Negativo (-1)

sns.set_context("paper", font_scale=0.9)
sns.set_style("white")
plt.rcParams['axes.linewidth'] = LINE_WIDTH

# --- Dados (Certifica-te que estas variáveis existem no teu ambiente) ---
CLUSTER_ALVO_ID = 3 
c_cells = cluster_df[cluster_df['cluster'] == CLUSTER_ALVO_ID]['cell_id']
dV_mean = agg_pivot.loc[c_cells].mean()
df_plot = pd.DataFrame({
    'dV': dV_mean,
    'Temp': df_temp.set_index('data')['med_smooth'].reindex(dV_mean.index),
    'Niv': df_nivel.set_index('data')['nivel_smooth'].reindex(dV_mean.index),
    'Prec': df_prec.set_index('data')['prec'].reindex(dV_mean.index),
    'P_Ac': df_prec.set_index('data')['prec_acum'].reindex(dV_mean.index),
    'P_An': df_prec.set_index('data')['prec_acum_anual'].reindex(dV_mean.index)
}).dropna()

cols = df_plot.columns.tolist()
n_vars = len(cols)

# 2. Criar o PairGrid
g = sns.PairGrid(df_plot, vars=cols, diag_sharey=False, height=1.0, aspect=1.0)

# --- Funções de Desenho ---
def cor_func(x, y, **kwargs):
    r, p = pearsonr(x, y)
    ax = plt.gca()
    # Mapeamento: r=1 -> Azul, r=-1 -> Vermelho
    color_val = (r + 1) / 2
    facecolor = MY_CMAP(color_val)
    ax.set_facecolor((*facecolor[:3], 0.5))
    ax.annotate(f"{r:.2f}", xy=(0.5, 0.5), xycoords=ax.transAxes, 
                ha='center', va='center', fontsize=8, fontweight='normal')

g.map_diag(sns.histplot, kde=True, color="#2c3e50", alpha=0.2, edgecolor='white', linewidth=0.3, line_kws={'linewidth': 0.7})
# --- PARTE INFERIOR: Scatter Plots ---
g.map_lower(sns.regplot, ci=None, 
            scatter_kws={
                's': 3,                # Tamanho do ponto
                'alpha': 1.0,          # Opacidade total (sem transparência)
                'color': 'black',      # Cor base
                'facecolor': 'black',  # Preenchimento preto sólido
                'edgecolor': 'black',  # Contorno preto sólido
                'linewidths': 0        # Remove qualquer largura de linha de bordo para evitar reflexos
            }, 
            line_kws={'color': 'red', 'linewidth': 0.7})
g.map_upper(cor_func)

# 3. Uniformização Total das Linhas e Padding
for i in range(n_vars):
    for j in range(n_vars):
        ax = g.axes[i, j]
        ax.set_xlabel(""); ax.set_ylabel("")
        
        # Ajuste de Padding interno (Respiro de 25%)
        if i != j:
            x_min, x_max = df_plot[cols[j]].min(), df_plot[cols[j]].max()
            y_min, y_max = df_plot[cols[i]].min(), df_plot[cols[i]].max()
            x_range, y_range = x_max - x_min, y_max - y_min
            ax.set_xlim(x_min - 0.3 * x_range, x_max + 0.3 * x_range)
            ax.set_ylim(y_min - 0.3 * y_range, y_max + 0.3 * y_range)

        # Labels apenas nas extremidades
        if j != 0: ax.set_yticklabels([])
        if i != n_vars - 1: ax.set_xticklabels([])
        
        ax.tick_params(labelsize=6, direction='in', pad=1, width=LINE_WIDTH, color=LINE_COLOR)
        
        # Forçar todas as linhas da grelha
        for edge in ['top', 'bottom', 'left', 'right']:
            ax.spines[edge].set_visible(True)
            ax.spines[edge].set_linewidth(LINE_WIDTH)
            ax.spines[edge].set_color(LINE_COLOR)
        
        # Identificação na Diagonal em Vermelho (Sem Bold)
        if i == j:
            ax.annotate(cols[i], xy=(0.05, 0.90), xycoords='axes fraction', 
                        ha='left', va='top', fontsize=8, fontweight='normal', color='red',
                        bbox=dict(facecolor='white', alpha=0.4, edgecolor='none', pad=0))

# 4. Ajuste de Layout para Colagem
plt.subplots_adjust(hspace=0, wspace=0, left=0.1, right=0.85, bottom=0.18, top=0.95)

# 5. Colorbar Corrigida (Azul=Positivo, Vermelho=Negativo)
cax = g.fig.add_axes([0.87, 0.18, 0.02, 0.77]) 
sm = plt.cm.ScalarMappable(cmap=MY_CMAP, norm=plt.Normalize(-1, 1))
cbar = g.fig.colorbar(sm, cax=cax)
cbar.set_ticks([-1, 0, 1])
cbar.outline.set_linewidth(LINE_WIDTH)
cbar.outline.set_edgecolor(LINE_COLOR)
cbar.set_label('Correlation coefficient', rotation=270, labelpad=15, fontsize=9)
cbar.ax.tick_params(labelsize=7, width=LINE_WIDTH, color=LINE_COLOR)

# 6. Legenda Inferior (Recuperada e Colada)
ax_table = g.fig.add_axes([0.1, 0.06, 0.75, 0.12]) 
ax_table.axis('off')

legend_data = [
    ["dV: Vertical Displacement", "Temp: Temperature", "Niv: Reservoir Level"],
    ["Prec: Daily Precipitation", "P_Ac: Accumulated Prec.", "P_An: Annual Acc. Prec."]
]

table = ax_table.table(cellText=legend_data, loc='upper center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.4) 

for (row, col), cell in table.get_celld().items():
    cell.set_linewidth(LINE_WIDTH)
    cell.set_edgecolor(LINE_COLOR)
    cell.get_text().set_fontweight('normal')

# 7. Finalização
plt.savefig("Matriz_Correlacao_Final_Final.png", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import numpy as np

# ==============================================================================
# CONFIGURAÇÃO MULTI-CLUSTER: TODAS AS VARIÁVEIS (2x2)
# ==============================================================================

CLUSTERS_PARA_MOSTRAR = [0, 1, 2, 3] 
N_COLS = 2 

def plot_cluster_matrix_full(cluster_id, target_ax, df_pivot, cluster_info, env_vars):
    # 1. Preparação de Dados com TODAS as variáveis
    c_cells = cluster_info[cluster_info['cluster'] == cluster_id]['cell_id']
    dV_mean = df_pivot.loc[c_cells].mean()

    df_plot = pd.DataFrame({
        'dV': dV_mean,
        'Temp': env_vars['temp'].reindex(dV_mean.index),
        'Nivel': env_vars['nivel'].reindex(dV_mean.index),
        'Prec': env_vars['prec'].reindex(dV_mean.index),
        'P_Acum': env_vars['p_acum'].reindex(dV_mean.index),
        'P_Anual': env_vars['p_anual'].reindex(dV_mean.index)
    }).dropna()

    cols = df_plot.columns.tolist()
    n_vars = len(cols)
    
    # Criar sub-grelha interna para este cluster
    inner_gs = target_ax.get_subplotspec().subgridspec(n_vars, n_vars, hspace=0, wspace=0)
    
    for i in range(n_vars):
        for j in range(n_vars):
            ax = fig.add_subplot(inner_gs[i, j])
            v1, v2 = cols[j], cols[i]
            
            # DIAGONAL: Histogramas
            if i == j:
                sns.histplot(df_plot[v1], kde=True, ax=ax, color="#2c3e50", alpha=0.3, 
                             edgecolor='white', linewidth=0.2, line_kws={'linewidth': 0.8})
                # Nome da variável pequeno no centro
                # ax.annotate(v1, xy=(0.5, 0.5), xycoords='axes fraction', 
                #             ha='center', fontsize=7, fontweight='bold', color='black')
                
                # Coloca o texto na parte inferior (y=0.1), centrado horizontalmente
                ax.annotate(v1, xy=(0.5, 0.1), xycoords='axes fraction', 
                            ha='center', va='bottom', 
                            fontsize=6, fontweight='normal', color='#333333')

            # INFERIOR: Regressão (Pontos pretos, linha vermelha)
            elif i > j:
                ax.scatter(df_plot[v1], df_plot[v2], s=1.5, alpha=0.5, color='black', linewidths=0)
                # Linha de regressão manual para ser mais leve que o regplot
                m, b = np.polyfit(df_plot[v1], df_plot[v2], 1)
                ax.plot(df_plot[v1], m*df_plot[v1] + b, color='#d7191c', linewidth=0.7)
            
            # SUPERIOR: Correlação (Cores pastel)
            else:
                r, p = pearsonr(df_plot[v1], df_plot[v2])
                cmap = plt.cm.RdBu_r
                ax.set_facecolor((*cmap((r + 1) / 2)[:3], 0.4))
                sig = "*" if p < 0.05 else ""
                ax.annotate(f"{r:.2f}{sig}", xy=(0.5, 0.5), xycoords='axes fraction', 
                            ha='center', va='center', fontsize=7)

            # Estilização Minimalista
            ax.set_xlabel(''); ax.set_ylabel('')
            ax.tick_params(labelsize=5, width=0.3, length=1.5, direction='in', pad=1)
            for spine in ax.spines.values(): 
                spine.set_linewidth(0.3)
                spine.set_color('#333333')
            
            # Labels apenas nas bordas de cada bloco de cluster
            if j != 0: ax.set_yticklabels([])
            if i != n_vars - 1: ax.set_xticklabels([])

# --- EXECUÇÃO ---

n_clusters = len(CLUSTERS_PARA_MOSTRAR)
n_rows = int(np.ceil(n_clusters / N_COLS))

fig = plt.figure(figsize=(10, 10)) # Tamanho grande para acomodar 144 gráficos
main_gs = fig.add_gridspec(n_rows, N_COLS, hspace=0.15, wspace=0.15)

env_vars = {
    'temp': df_temp.set_index('data')['med_smooth'],
    'nivel': df_nivel.set_index('data')['nivel_smooth'],
    'prec': df_prec.set_index('data')['prec'],
    'p_acum': df_prec.set_index('data')['prec_acum'],
    'p_anual': df_prec.set_index('data')['prec_acum_anual']
}

for idx, cluster_id in enumerate(CLUSTERS_PARA_MOSTRAR):
    r, c = divmod(idx, N_COLS)
    fake_ax = fig.add_subplot(main_gs[r, c])
    fake_ax.axis('off')
    fake_ax.set_title(f"C {cluster_id + 1}",
                      fontsize=10,
                      #fontweight='bold',
                      pad=5
                      )
    
    plot_cluster_matrix_full(cluster_id, fake_ax, agg_pivot, cluster_df, env_vars)

plt.show()

In [ ]:
# ==============================
# FIGURA 4: CORRELAÇÃO DE PEARSON (lag 0)
# ==============================

print("Gerando Figura 4 (Correlações Pearson)...")

from scipy.stats import pearsonr

external_vars = {
    'Temperatura (°C)':    (df_temp.set_index('data')['med_smooth'],        'black'),
    'Nível albufeira (m)': (df_nivel.set_index('data')['nivel_smooth'],     'navy'),
    'Precipitação (mm)':   (df_prec.set_index('data')['prec'],              'teal'),
}

fig_corr, axes_corr = plt.subplots(1, n_clusters, figsize=(4.5 * n_clusters, 5), sharey=True)
if n_clusters == 1:
    axes_corr = [axes_corr]

for col, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    mean_series   = agg_pivot.loc[cluster_cells].mean(axis=0)
    cluster_color = cluster_colors.get(cluster_id, 'gray')
    ax = axes_corr[col]

    var_names, r_vals, p_vals, bar_colors = [], [], [], []

    for vname, (vseries, vcolor) in external_vars.items():
        combined = pd.DataFrame({'dv': mean_series, 'ext': vseries}).dropna()
        if len(combined) > 5:
            r, p = pearsonr(combined['ext'], combined['dv'])
        else:
            r, p = np.nan, np.nan
        var_names.append(vname)
        r_vals.append(r)
        p_vals.append(p)
        bar_colors.append(vcolor)

    x_pos = np.arange(len(var_names))
    bars = ax.bar(x_pos, r_vals, color=bar_colors, alpha=0.7,
                  edgecolor='black', linewidth=0.6)

    # Anotar r e significância
    for i, (r, p) in enumerate(zip(r_vals, p_vals)):
        if np.isnan(r):
            continue
        sig = '*' if p < 0.05 else ''
        yoff = 0.03 if r >= 0 else -0.07
        ax.text(i, r + yoff, f'{r:.2f}{sig}',
                ha='center', va='bottom', fontsize=9)

    ax.axhline(0,    color='black', linewidth=0.8)
    ax.axhline( 0.3, color='gray',  linewidth=0.7, linestyle='--', alpha=0.5)
    ax.axhline(-0.3, color='gray',  linewidth=0.7, linestyle='--', alpha=0.5)
    ax.set_ylim(-1.15, 1.15)
    ax.set_xticks(x_pos)
    ax.set_xticklabels([v.split('(')[0].strip() for v in var_names],
                       fontsize=9, rotation=20, ha='right')
    ax.set_title(f'Cluster {cluster_id + 1}', fontsize=11, color=cluster_color)
    if col == 0:
        ax.set_ylabel('r de Pearson', fontsize=10)

fig_corr.suptitle(f'Correlação de Pearson: {TARGET_VAR} vs. variáveis externas',
                  fontsize=13)
plt.tight_layout()
plt.show()

# Tabela no terminal
print(f"\n{'='*55}")
print(f"{'Cluster':<10} {'Variável':<28} {'r':>6} {'p':>8}")
print(f"{'-'*55}")
for cluster_id in clusters_present:
    mean_series = agg_pivot.loc[
        cluster_df[cluster_df['cluster'] == cluster_id]['cell_id']
    ].mean(axis=0)
    for vname, (vseries, _) in external_vars.items():
        combined = pd.DataFrame({'dv': mean_series, 'ext': vseries}).dropna()
        if len(combined) > 5:
            r, p = pearsonr(combined['ext'], combined['dv'])
            sig = '*' if p < 0.05 else ' '
            print(f"{'Cluster '+str(cluster_id+1):<10} {vname:<28} {r:>6.3f} {p:>8.4f}{sig}")
print(f"{'='*55}\nLegenda: * p < 0.05")

In [ ]:
# Escolha aqui o que quer analisar: 'dV' ou 'dH'
TARGET_VAR = 'dH'  # ou 'dV'

# grelha
grid_size = 50

# Tipo de ligação
LINKAGE_METHOD = "ward"
# Opções possíveis:
# "complete", "average", "ward"

# ==============================================================================
# SCRIPT COMPLETO: DTW + HIERÁRQUICO (COM ESTILO VISUAL K-MEANS MANTIDO)
# ==============================================================================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
import matplotlib.dates as mdates
from shapely.strtree import STRtree
from statsmodels.tsa.seasonal import seasonal_decompose
from matplotlib.lines import Line2D

# Bibliotecas para DTW e Hierárquico
try:
    from dtaidistance import dtw
    from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
    from scipy.spatial.distance import squareform
    from sklearn.preprocessing import StandardScaler
    import seaborn as sns
    from matplotlib.colors import to_hex
except ImportError as e:
    print(f"Erro de Importação: {e}. Certifique-se de que instalou: dtaidistance, scipy, seaborn, scikit-learn")

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
    #barragem = cos[cos['COS23_n4_L'] == 'Superfícies silvopastoris de azinheira']
    #barragem = cos[cos['COS23_n4_L'] == 'Matos']
except Exception as e:
    print(f"Aviso: COS placeholder. {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs e Filtro
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df): return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) & (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'], value_vars=disp_cols, var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(start=max(asc_long['date'].min(), desc_long['date'].min()), end=min(asc_long['date'].max(), desc_long['date'].max()), freq='MS')

# ==============================
# 3. Interpolação
# ==============================
def interpolate_ps(df, dates):
    dfs = []
    for (x, y), g in df.groupby(['easting','northing']):
        g = g.sort_values('date')
        interp = np.interp(pd.to_datetime(dates).astype(np.int64), g['date'].astype(np.int64), g['disp'])
        dfs.append(pd.DataFrame({'easting': x, 'northing': y, 'latitude': g['latitude'].iloc[0], 'longitude': g['longitude'].iloc[0], 'date': dates, 'disp': interp, 'incidence_angle': g['incidence_angle'].iloc[0], 'track_angle': g['track_angle'].iloc[0]}))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. IDW
# ==============================
def idw(source, target, radius=150, power=2):
    out = []
    for d, src in source.groupby('date'):
        tgt = target[target['date']==d].copy()
        if src.empty or tgt.empty: continue
        tree = cKDTree(list(zip(src['easting'], src['northing'])))
        dist, idx = tree.query(list(zip(tgt['easting'], tgt['northing'])), k=5, distance_upper_bound=radius)
        vals, thetas, alphas = [], [], []
        for d_i, i_i in zip(dist, idx):
            m = np.isfinite(d_i)
            if not np.any(m): vals.append(np.nan); thetas.append(np.nan); alphas.append(np.nan); continue
            w = 1/(d_i[m]**power)
            vals.append(np.sum(w*src.iloc[i_i[m]]['disp'])/np.sum(w))
            thetas.append(np.sum(w*src.iloc[i_i[m]]['incidence_angle'])/np.sum(w))
            alphas.append(np.sum(w*src.iloc[i_i[m]]['track_angle'])/np.sum(w))
        tgt['disp_idw'] = vals; tgt['theta_desc'] = thetas; tgt['alpha_desc'] = alphas
        out.append(tgt)
    return pd.concat(out, ignore_index=True)
asc_interp = idw(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw'])

# ==============================
# 5. dV (ou dH)
# ==============================
orb_inc = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orb_inc) * np.cos(np.deg2rad(asc_interp['latitude'])))

def get_comps(row):
    ta, td, beta = np.deg2rad(row['incidence_angle']), np.deg2rad(row['theta_desc']), row['beta']
    denom = (np.cos(ta)*np.sin(td)*np.cos(beta) + np.cos(td)*np.sin(ta)*np.cos(beta))
    if denom == 0: return np.nan, np.nan
    dV = (row['disp_idw']*np.sin(ta)*np.cos(beta) + row['disp']*np.sin(td)*np.cos(beta))/denom
    dH = (row['disp_idw']*np.cos(ta) - row['disp']*np.cos(td))/denom
    return dV, dH

asc_interp[['dV', 'dH']] = asc_interp.apply(lambda x: pd.Series(get_comps(x)), axis=1)
asc_interp = asc_interp.dropna(subset=['dV', 'dH'])

# ==============================
# 6. Grelha
# ==============================
#grid_size = 100
xe = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
ye = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
xs, ys = xe - grid_size/2, ye - grid_size/2
asc_interp['cx'] = pd.cut(asc_interp['easting'], bins=xs, labels=False)
asc_interp['cy'] = pd.cut(asc_interp['northing'], bins=ys, labels=False)
asc_interp = asc_interp.dropna(subset=['cx','cy'])
asc_interp['cell_id'] = asc_interp['cx'].astype(int).astype(str)+"_"+asc_interp['cy'].astype(int).astype(str)

grid_data = [{'cell_id': f"{ix}_{iy}", 'geometry': box(xs[ix], ys[iy], xs[ix+1], ys[iy+1])} for ix in range(len(xs)-1) for iy in range(len(ys)-1)]
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# Escolha aqui se usa dV ou dH para a agregação
#TARGET_VAR = 'dV' 
#agg = asc_interp.groupby(['cell_id','date']).agg(dV=(TARGET_VAR,'mean')).reset_index() # Coluna final chama-se sempre 'dV' para manter compatibilidade com resto do script
agg = asc_interp.groupby(['cell_id','date']).agg(**{TARGET_VAR: (TARGET_VAR, 'mean')}).reset_index()


# ==============================
# 7. Recorte e Filtro
# ==============================
points_gdf = gpd.GeoDataFrame(asc_interp, geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']), crs="EPSG:3035").to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

poly = grid_recort.geometry.values; ids = grid_recort['cell_id'].values; tree = STRtree(poly)
valid = set()
for pt in points_gdf.geometry:
    for idx in tree.query(pt):
        if poly[idx].contains(pt): valid.add(ids[idx])
grid_barragem = grid_recort[grid_recort['cell_id'].isin(valid)]
agg = agg[agg['cell_id'].isin(valid)]

# ==============================
# 8. CLUSTERING (DTW + HIERÁRQUICO) - SUBSTITUI K-MEANS
# ==============================
print(">>> A calcular Matriz de Distâncias DTW...")

# 1. Preparar Matriz (Pivot)
#agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
agg_pivot = agg.pivot(index='cell_id', columns='date', values=TARGET_VAR).fillna(0)

cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

if n < 2:
    print("Aviso: Menos de 2 células para agrupar. Clustering ignorado.")
    cluster_labels = np.zeros(n, dtype=int)
    num_clusters = 1
    cut_distance = 0
    Z = None
else:
    # 2. Normalizar (Z-Score) para comparar formas
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X.T).T 

    # 3. Calcular DTW
    dist_matrix = np.zeros((n, n))
    window_dtw = int(0.1 * X_scaled.shape[1]) # Janela 10%
    
    # Esta parte pode demorar dependendo do nº de células
    for i in range(n):
        for j in range(i + 1, n):
            d = dtw.distance(X_scaled[i], X_scaled[j], window=window_dtw)
            dist_matrix[i, j] = d
            dist_matrix[j, i] = d

    # 4. Clustering Hierárquico
    condensed = squareform(dist_matrix)
    #Z = linkage(condensed, method='average')
    from scipy.cluster.hierarchy import linkage, fcluster
    Z = linkage(condensed, method=LINKAGE_METHOD)

    # 5. Corte Automático (Elbow)
    last = Z[-10:, 2] # Últimas distâncias de fusão
    acceleration = np.diff(last, 2)
    try:
        k_idx = np.argmax(acceleration) + 2
        cut_distance = (last[k_idx] + last[k_idx-1]) / 2
    except:
        cut_distance = last[len(last)//2]

    # Labels
    cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
    
    # Reajustar labels para começar em 0
    u_labels = np.unique(cluster_labels)
    map_l = {old: new for new, old in enumerate(u_labels)}
    cluster_labels = np.array([map_l[x] for x in cluster_labels])
    
    num_clusters = len(u_labels)
    print(f"DTW Concluído. Corte: {cut_distance:.2f}. Clusters: {num_clusters}")

# DataFrame Final
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})
# Cores Dinâmicas (para N clusters)
palette = sns.color_palette("Set2", num_clusters) if num_clusters <= 8 else sns.color_palette("tab20", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(range(num_clusters), palette)}

grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='inner')

# ==============================
# FIGURA EXTRA: DENDROGRAMA
# ==============================
if Z is not None:
    print("Gerando Dendrograma...")
    plt.figure(figsize=(12, 5))
    dendrogram(Z, leaf_rotation=90, leaf_font_size=8, color_threshold=cut_distance)
    plt.axhline(y=cut_distance, c='k', ls='--', lw=1, label='Corte Automático')
    plt.title('Dendrograma de Clustering Hierárquico (DTW)')
    plt.xlabel('Células'); plt.ylabel('Distância')
    plt.legend(); plt.tight_layout(); plt.show()


# ==============================
# 9. Dados Hidro (IGUAL)
# ==============================
date_range = agg_pivot.columns
win = 13

# Temp
try: df_t = pd.read_excel("data/alqueva_temp.xlsx"); df_t['data']=pd.to_datetime(df_t['data'])
except: df_t = pd.DataFrame({'data': date_range, 'med': 0})
ts = df_t.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': ts.index, 'med': ts.values})
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), win, 2)

# Nível
try: 
    df_n = pd.read_excel("data/alqueva_nivel.xlsx"); df_n['data']=pd.to_datetime(df_n['data'])
    df_n['nivel'] = pd.to_numeric(df_n['nivel'], errors='coerce'); df_n = df_n.dropna(subset=['nivel'])
except: df_n = pd.DataFrame({'data': date_range, 'nivel': 0})
ns = df_n.set_index('data')['nivel'].resample('MS').mean().reindex(date_range).interpolate(limit_direction='both').ffill().bfill()
df_nivel = pd.DataFrame({'data': ns.index, 'nivel': ns.values})
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], win, 2)

# Precipitação
try: df_p = pd.read_excel("data/prec.xlsx"); df_p['data'] = pd.to_datetime(df_p['data'])
except: np.random.seed(42); df_p = pd.DataFrame({'data': date_range, 'prec': 0})
ps = df_p.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': ps.index, 'prec': ps.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções Visuais (IGUAL)
# ==============================
def scale_vis(d, vmin, vmax, s=0.20, o=0.30):
    d = np.array(d); mn, mx = np.nanmin(d), np.nanmax(d)
    span = vmax - vmin
    norm = np.zeros_like(d) if mx==mn else (d-mn)/(mx-mn)
    return norm * span * s + (vmax - span * o), mn, mx

def create_ticks(rmin, rmax, vmin, vmax, s=0.20, o=0.30, n=5):
    rt = np.linspace(rmin, rmax, n)
    span = vmax - vmin
    nt = np.linspace(0,1,n) if rmax==rmin else (rt-rmin)/(rmax-rmin)
    vt = nt * span * s + (vmax - span * o)
    return vt, rt

# ==============================
# FIGURA 1: MAPA (ESTILO ORIGINAL)
# ==============================
print("Gerando Figura 1...")
clusters_present = sorted(cluster_df['cluster'].unique())
k_plot = len(clusters_present)

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', lw=1)

for i in clusters_present:
    sub = grid_sel[grid_sel['cluster'] == i]
    sub.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    cents = sub.copy(); cents.geometry = cents.geometry.centroid
    cents.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

hdl = [plt.Rectangle((0,0),1,1, fc=cluster_colors[i], alpha=0.4) for i in clusters_present]
lbl = [f'Cluster {i+1} ({len(grid_sel[grid_sel["cluster"]==i])} cel)' for i in clusters_present]
hdl.append(Line2D([0],[0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None')); lbl.append('Centróides')
ax_map.legend(hdl, lbl, loc='upper left', fontsize=10, title=f"Clusters DTW (K={k_plot})")
ax_map.set_title(f"Mapa de Clusters ({TARGET_VAR})", fontsize=15)
plt.show()

# ==============================
# FIGURA 2: SÉRIES TEMPORAIS
# ==============================

print("Gerando Figura 2...")

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5

fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

dV_min = agg[TARGET_VAR].min()
dV_max = agg[TARGET_VAR].max()
dV_margin = (dV_max - dV_min) * 0.1
dV_ylim = (dV_min - dV_margin, dV_max + dV_margin)

# --- Séries visuais ---
temp_visual, _, _ = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, _, _ = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)


for idx, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)


    # --- Função atualizada ---
    def plot_dV_comparison_series(
        row_idx, ext_data_visual, ext_data_df, ext_col_real,
        ext_label, ext_color,
        plot_as_bar=False, bar_color=None,
        plot_real_scale_line=False,
        combine_annual_prec=False,
        integer_ticks=False, show_background_bars=False):

        ax = fig_series.add_subplot(gs_series[row_idx, idx])

        # --- dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid],
                    color='lightgray', alpha=0.7, linewidth=1.0)

        ax.plot(cluster_data.columns, cluster_mean_dV,
                color=cluster_color, linewidth=1.0,
                label=f'Média Cluster {cluster_id+1}')

        # --- Eixo Y2 ---
        ax2 = ax.twinx()
        lines2 = []

        # -------------------------
        #   CASOS DE PRECIPITAÇÃO
        # -------------------------
        if combine_annual_prec:

            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)

            for ano in df_prec['ano_hidrologico'].unique():
                grupo = df_prec[df_prec['ano_hidrologico'] == ano]
                ax2.plot(grupo['data'], grupo['prec_acum_anual'],
                         color='teal', linewidth=1.0, alpha=0.9)

            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            ax2.set_ylabel("Prec. acumulada anual", color="teal")
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=1.0, label='Prec. acumulada anual (mm)')
            ]

        elif plot_as_bar:

            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15,
                    color=bar_color or ext_color, alpha=0.5)

            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            ax2.set_ylabel(ext_label, color='teal')
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)
            ]

        elif plot_real_scale_line:

            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15,
                        color='teal', alpha=0.3)

            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real],
                     color='teal', linewidth=1.0, alpha=0.85)

            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            ax2.set_ylabel(ext_label, color='teal')
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=1.0, label=ext_label)
            ]

        # -------------------------
        #      OUTRAS SÉRIES
        # -------------------------
        else:

            ax2.plot(ext_data_df['data'], ext_data_visual,
                     color=ext_color, linewidth=1.0, alpha=0.85)

            visual_ticks, real_ticks = create_visual_ticks(
                ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(),
                dV_min, dV_max, scale_factor=0.20, offset_factor=0.30
            )

            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(visual_ticks)

            if integer_ticks:
                ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else:
                ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])

            label_color = 'black' if 'Temperatura' in ext_label else ext_color
            ax2.set_ylabel(ext_label, color=label_color)
            ax2.tick_params(axis='y', colors=label_color)

            lines2 = [ax2.lines[-1]]

        # --- Y1 sempre dV ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)

        if idx == 0:
            ax.set_ylabel('dV (mm)')
        else:
            ax.set_yticklabels([])

        # Y2 somente no último cluster
        if idx != n_clusters - 1:
            ax2.set_yticklabels([])

        ax.tick_params(left=(idx==0))
        ax2.tick_params(right=(idx==n_clusters-1))

        # --- X-axis ---
        if row_idx == n_rows_series - 1:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        else:
            ax.set_xticklabels([])

        # --- Legenda somente no primeiro e no último cluster ---
        if idx == 0 or idx == n_clusters - 1:
            lines1, labels1 = ax.get_legend_handles_labels()
            handles2, labels2 = ax2.get_legend_handles_labels()

            ax.legend(
                lines1 + handles2,
                labels1 + labels2,
                fontsize=8,
                loc='best',
                framealpha=1.0,
                facecolor='white',
                edgecolor='lightgray'
            ).set_zorder(100)

        return ax


    # ---- CHAMADA DAS 5 SÉRIES ----
    plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth',
                              'Temperatura média (°C)', 'black', integer_ticks=True)

    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth',
                              'Nível da albufeira (m)', 'navy', integer_ticks=True)

    plot_dV_comparison_series(2, None, df_prec, 'prec',
                              'Precipitação mensal (mm)', 'teal',
                              plot_as_bar=True, bar_color='teal')

    plot_dV_comparison_series(3, None, df_prec, 'prec_acum',
                              'Prec. acumulada total (mm)', 'teal',
                              plot_real_scale_line=True, show_background_bars=True)

    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual',
                              'Prec. acumulada anual', 'teal',
                              combine_annual_prec=True)


plt.tight_layout()
plt.show()


# ==============================
# FIGURA 3: DECOMPOSIÇÃO SAZONAL
# ==============================

print("Gerando Figura 3...")

n_rows = 4  # Observed, Trend, Seasonal, Residual
fig_series_decomp, axes = plt.subplots(
    n_rows, n_clusters, figsize=(6 * max(1, n_clusters), 7), sharex=False
)

for col, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    mean_series = cluster_data.mean(axis=0)

    decomposition = seasonal_decompose(mean_series, model="additive", period=12)

    series_list = [
        ("Observed", decomposition.observed),
        ("Trend", decomposition.trend),
        ("Seasonal", decomposition.seasonal),
        ("Residual", decomposition.resid)
    ]

    cluster_color = cluster_colors.get(cluster_id, "black")

    for row, (label, series) in enumerate(series_list):

        ax = axes[row, col]

        # --------------------------
        #   PLOT (agora com cores)
        # --------------------------
        ax.plot(series.index, series.values,
                color=cluster_color, linewidth=1.0)

        # --------------------------
        #  LIMITE EXTERIOR (spines)
        # --------------------------
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.8)
            spine.set_color("black")

        # --------------------------
        #     LIMITES Y
        # --------------------------
        if label != "Observed":
            ymin = np.nanmin(series)
            ymax = np.nanmax(series)
            if np.isnan(ymin) or np.isnan(ymax):
                ymin, ymax = -1, 1
            margin = (ymax - ymin) * 0.10
            ax.set_ylim(ymin - margin, ymax + margin)

        # --------------------------
        #  TÍTULOS DOS SUBPLOTS
        # --------------------------
        if row == 0:
            ax.set_title(f"Seasonal Decompose – Cluster {cluster_id + 1}",
                         fontsize=12)

        if col == 0:
            ax.set_ylabel(label, fontsize=10)
        else:
            ax.set_yticks([])

        # --------------------------
        #   EIXO X — só no último row
        # --------------------------
        if row == n_rows - 1:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.xaxis.set_major_locator(mdates.YearLocator())
            ax.tick_params(axis='x', labelrotation=0, labelsize=10)
        else:
            ax.set_xticks([])
            ax.set_xticklabels([])

plt.tight_layout()
plt.show()


In [ ]:
# Escolha aqui o que quer analisar: 'dV' ou 'dH'
TARGET_VAR = 'dV'  # ou 'dV'

# grelha
grid_size = 50

DTW_WINDOW_PCT = 0.03  # Aqui podes mudar para 0.1, 0.05, 0.03, etc.

# Tipo de ligação
LINKAGE_METHOD = "average"
# Opções possíveis:
# "complete", "average", "ward"

# ==============================================================================
# SCRIPT COMPLETO: DTW + HIERÁRQUICO (COM ESTILO VISUAL K-MEANS MANTIDO)
# ==============================================================================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
import matplotlib.dates as mdates
from shapely.strtree import STRtree
from statsmodels.tsa.seasonal import seasonal_decompose
from matplotlib.lines import Line2D

# Bibliotecas para DTW e Hierárquico
try:
    from dtaidistance import dtw
    from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
    from scipy.spatial.distance import squareform
    from sklearn.preprocessing import StandardScaler
    import seaborn as sns
    from matplotlib.colors import to_hex
except ImportError as e:
    print(f"Erro de Importação: {e}. Certifique-se de que instalou: dtaidistance, scipy, seaborn, scikit-learn")

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    #barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
    #barragem = cos[cos['COS23_n4_L'] == 'Superfícies silvopastoris de azinheira']
    #barragem = cos[cos['COS23_n4_L'] == 'Matos']
    
    # Podes adicionar ou remover categorias dentro dos parênteses retos []
    barragem = cos[cos['COS23_n4_L'].isin([
    'Infraestruturas de produção de energia hídrica', 
    'Albufeiras de barragens',
    'Equipamentos culturais',
    'Florestas de azinheira',
    'Matos',
    'Pastagens melhoradas',
    'Rede rodoviária',
    'Superfícies silvopastoris de azinheira'
])]
    
# - Albufeiras de barragens
# - Equipamentos culturais
# - Florestas de azinheira
# - Infraestruturas de produção de energia hídrica
# - Matos
# - Pastagens melhoradas
# - Rede rodoviária
# - Superfícies silvopastoris de azinheira
    
    
except Exception as e:
    print(f"Aviso: COS placeholder. {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs e Filtro
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df): return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) & (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'], value_vars=disp_cols, var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(start=max(asc_long['date'].min(), desc_long['date'].min()), end=min(asc_long['date'].max(), desc_long['date'].max()), freq='MS')

# ==============================
# 3. Interpolação
# ==============================
def interpolate_ps(df, dates):
    dfs = []
    for (x, y), g in df.groupby(['easting','northing']):
        g = g.sort_values('date')
        interp = np.interp(pd.to_datetime(dates).astype(np.int64), g['date'].astype(np.int64), g['disp'])
        dfs.append(pd.DataFrame({'easting': x, 'northing': y, 'latitude': g['latitude'].iloc[0], 'longitude': g['longitude'].iloc[0], 'date': dates, 'disp': interp, 'incidence_angle': g['incidence_angle'].iloc[0], 'track_angle': g['track_angle'].iloc[0]}))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. IDW
# ==============================
def idw(source, target, radius=25, power=2):
    out = []
    for d, src in source.groupby('date'):
        tgt = target[target['date']==d].copy()
        if src.empty or tgt.empty: continue
        tree = cKDTree(list(zip(src['easting'], src['northing'])))
        dist, idx = tree.query(list(zip(tgt['easting'], tgt['northing'])), k=5, distance_upper_bound=radius)
        vals, thetas, alphas = [], [], []
        for d_i, i_i in zip(dist, idx):
            m = np.isfinite(d_i)
            if not np.any(m): vals.append(np.nan); thetas.append(np.nan); alphas.append(np.nan); continue
            w = 1/(d_i[m]**power)
            vals.append(np.sum(w*src.iloc[i_i[m]]['disp'])/np.sum(w))
            thetas.append(np.sum(w*src.iloc[i_i[m]]['incidence_angle'])/np.sum(w))
            alphas.append(np.sum(w*src.iloc[i_i[m]]['track_angle'])/np.sum(w))
        tgt['disp_idw'] = vals; tgt['theta_desc'] = thetas; tgt['alpha_desc'] = alphas
        out.append(tgt)
    return pd.concat(out, ignore_index=True)
asc_interp = idw(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw'])

# ==============================
# 5. dV (ou dH)
# ==============================
orb_inc = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orb_inc) * np.cos(np.deg2rad(asc_interp['latitude'])))

def get_comps(row):
    ta, td, beta = np.deg2rad(row['incidence_angle']), np.deg2rad(row['theta_desc']), row['beta']
    denom = (np.cos(ta)*np.sin(td)*np.cos(beta) + np.cos(td)*np.sin(ta)*np.cos(beta))
    if denom == 0: return np.nan, np.nan
    dV = (row['disp_idw']*np.sin(ta)*np.cos(beta) + row['disp']*np.sin(td)*np.cos(beta))/denom
    dH = (row['disp_idw']*np.cos(ta) - row['disp']*np.cos(td))/denom
    return dV, dH

asc_interp[['dV', 'dH']] = asc_interp.apply(lambda x: pd.Series(get_comps(x)), axis=1)
asc_interp = asc_interp.dropna(subset=['dV', 'dH'])

# ==============================
# 6. Grelha
# ==============================
#grid_size = 100
xe = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
ye = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
xs, ys = xe - grid_size/2, ye - grid_size/2
asc_interp['cx'] = pd.cut(asc_interp['easting'], bins=xs, labels=False)
asc_interp['cy'] = pd.cut(asc_interp['northing'], bins=ys, labels=False)
asc_interp = asc_interp.dropna(subset=['cx','cy'])
asc_interp['cell_id'] = asc_interp['cx'].astype(int).astype(str)+"_"+asc_interp['cy'].astype(int).astype(str)

grid_data = [{'cell_id': f"{ix}_{iy}", 'geometry': box(xs[ix], ys[iy], xs[ix+1], ys[iy+1])} for ix in range(len(xs)-1) for iy in range(len(ys)-1)]
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# Escolha aqui se usa dV ou dH para a agregação
#TARGET_VAR = 'dV' 
agg = asc_interp.groupby(['cell_id','date']).agg(dV=(TARGET_VAR,'mean')).reset_index() # Coluna final chama-se sempre 'dV' para manter compatibilidade com resto do script

# ==============================
# 7. Recorte e Filtro
# ==============================
points_gdf = gpd.GeoDataFrame(asc_interp, geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']), crs="EPSG:3035").to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

poly = grid_recort.geometry.values; ids = grid_recort['cell_id'].values; tree = STRtree(poly)
valid = set()
for pt in points_gdf.geometry:
    for idx in tree.query(pt):
        if poly[idx].contains(pt): valid.add(ids[idx])
grid_barragem = grid_recort[grid_recort['cell_id'].isin(valid)]
agg = agg[agg['cell_id'].isin(valid)]

# ==============================
# 8. CLUSTERING (VERSÃO ROBUSTA CONTRA NaNs)
# ==============================
print(f">>> A calcular DTW (Flexibilidade: {DTW_WINDOW_PCT})...")

# 1. Preparar Matriz (Pivot)
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV')

# --- LIMPEZA DE NaNs (Obrigatório para o Linkage não dar erro) ---
# Interpola pequenos buracos e remove células que ainda tenham falhas críticas
agg_pivot = agg_pivot.interpolate(axis=1, limit_direction='both').dropna()

cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

if n < 2:
    print("Aviso: Células insuficientes após limpeza de dados.")
else:
    # 2. Normalizar
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X.T).T 

    # 3. Calcular DTW com a nova variável
    # max(1, ...) garante que a janela nunca é zero
    window_dtw = max(1, int(DTW_WINDOW_PCT * X_scaled.shape[1])) 
    
    # Cálculo da matriz
    dist_matrix = dtw.distance_matrix_fast(X_scaled, window=window_dtw)

    # --- TRATAMENTO PÓS-CÁLCULO ---
    # Se o DTW gerar algum NaN (acontece se a janela for muito pequena para certas séries)
    if not np.all(np.isfinite(dist_matrix)):
        # Substituímos NaNs pelo valor máximo da matriz (para dizer que são muito diferentes)
        mask_nan = np.isnan(dist_matrix)
        dist_matrix[mask_nan] = np.nanmax(dist_matrix) if not np.all(np.isnan(dist_matrix)) else 0

    # 4. Clustering Hierárquico
    condensed = squareform(dist_matrix)
    Z = linkage(condensed, method=LINKAGE_METHOD)

    # 5. Corte Automático (Elbow)
    last = Z[-10:, 2] # Últimas distâncias de fusão
    acceleration = np.diff(last, 2)
    try:
        k_idx = np.argmax(acceleration) + 2
        cut_distance = (last[k_idx] + last[k_idx-1]) / 2
    except:
        cut_distance = last[len(last)//2]

    # Labels
    cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
    
    # Reajustar labels para começar em 0
    u_labels = np.unique(cluster_labels)
    map_l = {old: new for new, old in enumerate(u_labels)}
    cluster_labels = np.array([map_l[x] for x in cluster_labels])
    
    num_clusters = len(u_labels)
    print(f"DTW Concluído. Corte: {cut_distance:.2f}. Clusters: {num_clusters}")

# DataFrame Final
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})
# Cores Dinâmicas (para N clusters)
palette = sns.color_palette("Set2", num_clusters) if num_clusters <= 8 else sns.color_palette("tab20", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(range(num_clusters), palette)}

grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='inner')

# ==============================
# FIGURA EXTRA: DENDROGRAMA
# ==============================
if Z is not None:
    print("Gerando Dendrograma...")
    plt.figure(figsize=(12, 5))
    dendrogram(Z, leaf_rotation=90, leaf_font_size=8, color_threshold=cut_distance)
    plt.axhline(y=cut_distance, c='k', ls='--', lw=1, label='Corte Automático')
    plt.title('Dendrograma de Clustering Hierárquico (DTW)')
    plt.xlabel('Células'); plt.ylabel('Distância')
    plt.legend(); plt.tight_layout(); plt.show()


# ==============================
# 9. Dados Hidro (IGUAL)
# ==============================
date_range = agg_pivot.columns
win = 13

# Temp
try: df_t = pd.read_excel("data/alqueva_temp.xlsx"); df_t['data']=pd.to_datetime(df_t['data'])
except: df_t = pd.DataFrame({'data': date_range, 'med': 0})
ts = df_t.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': ts.index, 'med': ts.values})
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), win, 2)

# Nível
try: 
    df_n = pd.read_excel("data/alqueva_nivel.xlsx"); df_n['data']=pd.to_datetime(df_n['data'])
    df_n['nivel'] = pd.to_numeric(df_n['nivel'], errors='coerce'); df_n = df_n.dropna(subset=['nivel'])
except: df_n = pd.DataFrame({'data': date_range, 'nivel': 0})
ns = df_n.set_index('data')['nivel'].resample('MS').mean().reindex(date_range).interpolate(limit_direction='both').ffill().bfill()
df_nivel = pd.DataFrame({'data': ns.index, 'nivel': ns.values})
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], win, 2)

# Precipitação
try: df_p = pd.read_excel("data/prec.xlsx"); df_p['data'] = pd.to_datetime(df_p['data'])
except: np.random.seed(42); df_p = pd.DataFrame({'data': date_range, 'prec': 0})
ps = df_p.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': ps.index, 'prec': ps.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções Visuais (IGUAL)
# ==============================
# def scale_vis(d, vmin, vmax, s=0.20, o=0.30):
#     d = np.array(d); mn, mx = np.nanmin(d), np.nanmax(d)
#     span = vmax - vmin
#     norm = np.zeros_like(d) if mx==mn else (d-mn)/(mx-mn)
#     return norm * span * s + (vmax - span * o), mn, mx

# def create_ticks(rmin, rmax, vmin, vmax, s=0.20, o=0.30, n=5):
#     rt = np.linspace(rmin, rmax, n)
#     span = vmax - vmin
#     nt = np.linspace(0,1,n) if rmax==rmin else (rt-rmin)/(rmax-rmin)
#     vt = nt * span * s + (vmax - span * o)
#     return vt, rt


# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks


# ==============================
# FIGURA 1: MAPA (ESTILO ORIGINAL)
# ==============================
print("Gerando Figura 1...")
clusters_present = sorted(cluster_df['cluster'].unique())
k_plot = len(clusters_present)

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', lw=1)

for i in clusters_present:
    sub = grid_sel[grid_sel['cluster'] == i]
    sub.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    cents = sub.copy(); cents.geometry = cents.geometry.centroid
    cents.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

hdl = [plt.Rectangle((0,0),1,1, fc=cluster_colors[i], alpha=0.4) for i in clusters_present]
lbl = [f'Cluster {i+1} ({len(grid_sel[grid_sel["cluster"]==i])} cel)' for i in clusters_present]
hdl.append(Line2D([0],[0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None')); lbl.append('Centróides')
ax_map.legend(hdl, lbl, loc='upper left', fontsize=10, title=f"Clusters DTW (K={k_plot})")
ax_map.set_title(f"Mapa de Clusters ({TARGET_VAR})", fontsize=15)
plt.show()

# ==============================
# FIGURA 2: SÉRIES TEMPORAIS
# ==============================

print("Gerando Figura 2...")

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5

fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

dV_min = agg[TARGET_VAR].min()
dV_max = agg[TARGET_VAR].max()
dV_margin = (dV_max - dV_min) * 0.1
dV_ylim = (dV_min - dV_margin, dV_max + dV_margin)

# --- Séries visuais ---
temp_visual, _, _ = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, _, _ = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)

# Séries visuais
# tv, tmi, tmx = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
# nivel_visual, nivel_min, nivel_max = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
# prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)


for idx, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)


    # --- Função atualizada ---
    def plot_dV_comparison_series(
        row_idx, ext_data_visual, ext_data_df, ext_col_real,
        ext_label, ext_color,
        plot_as_bar=False, bar_color=None,
        plot_real_scale_line=False,
        combine_annual_prec=False,
        integer_ticks=False, show_background_bars=False):

        ax = fig_series.add_subplot(gs_series[row_idx, idx])

        # --- dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid],
                    color='lightgray', alpha=0.7, linewidth=1.0)

        ax.plot(cluster_data.columns, cluster_mean_dV,
                color=cluster_color, linewidth=1.0,
                label=f'Média Cluster {cluster_id+1}')

        # --- Eixo Y2 ---
        ax2 = ax.twinx()
        lines2 = []

        # -------------------------
        #   CASOS DE PRECIPITAÇÃO
        # -------------------------
        if combine_annual_prec:

            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)

            for ano in df_prec['ano_hidrologico'].unique():
                grupo = df_prec[df_prec['ano_hidrologico'] == ano]
                ax2.plot(grupo['data'], grupo['prec_acum_anual'],
                         color='teal', linewidth=1.0, alpha=0.9)

            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            ax2.set_ylabel("Prec. acumulada anual", color="teal")
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=1.0, label='Prec. acumulada anual (mm)')
            ]

        elif plot_as_bar:

            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15,
                    color=bar_color or ext_color, alpha=0.5)

            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            ax2.set_ylabel(ext_label, color='teal')
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)
            ]

        elif plot_real_scale_line:

            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15,
                        color='teal', alpha=0.3)

            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real],
                     color='teal', linewidth=1.0, alpha=0.85)

            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            ax2.set_ylabel(ext_label, color='teal')
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=1.0, label=ext_label)
            ]

        # -------------------------
        #      OUTRAS SÉRIES
        # -------------------------
        else:

            ax2.plot(ext_data_df['data'], ext_data_visual,
                     color=ext_color, linewidth=1.0, alpha=0.85)

            visual_ticks, real_ticks = create_visual_ticks(
                ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(),
                dV_min, dV_max, scale_factor=0.20, offset_factor=0.30
            )

            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(visual_ticks)

            if integer_ticks:
                ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else:
                ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])

            label_color = 'black' if 'Temperatura' in ext_label else ext_color
            ax2.set_ylabel(ext_label, color=label_color)
            ax2.tick_params(axis='y', colors=label_color)

            lines2 = [ax2.lines[-1]]

        # --- Y1 sempre dV ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)

        if idx == 0:
            ax.set_ylabel('dV (mm)')
        else:
            ax.set_yticklabels([])

        # Y2 somente no último cluster
        if idx != n_clusters - 1:
            ax2.set_yticklabels([])

        ax.tick_params(left=(idx==0))
        ax2.tick_params(right=(idx==n_clusters-1))

        # --- X-axis ---
        if row_idx == n_rows_series - 1:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        else:
            ax.set_xticklabels([])

        # --- Legenda somente no primeiro e no último cluster ---
        if idx == 0 or idx == n_clusters - 1:
            lines1, labels1 = ax.get_legend_handles_labels()
            handles2, labels2 = ax2.get_legend_handles_labels()

            ax.legend(
                lines1 + handles2,
                labels1 + labels2,
                fontsize=8,
                loc='best',
                framealpha=1.0,
                facecolor='white',
                edgecolor='lightgray'
            ).set_zorder(100)

        return ax


    # ---- CHAMADA DAS 5 SÉRIES ----
    plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth',
                              'Temperatura média (°C)', 'black', integer_ticks=True)

    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth',
                              'Nível da albufeira (m)', 'navy', integer_ticks=True)

    plot_dV_comparison_series(2, None, df_prec, 'prec',
                              'Precipitação mensal (mm)', 'teal',
                              plot_as_bar=True, bar_color='teal')

    plot_dV_comparison_series(3, None, df_prec, 'prec_acum',
                              'Prec. acumulada total (mm)', 'teal',
                              plot_real_scale_line=True, show_background_bars=True)

    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual',
                              'Prec. acumulada anual', 'teal',
                              combine_annual_prec=True)


plt.tight_layout()
plt.show()


# ==============================
# FIGURA 3: DECOMPOSIÇÃO SAZONAL
# ==============================

print("Gerando Figura 3...")

n_rows = 4  # Observed, Trend, Seasonal, Residual
fig_series_decomp, axes = plt.subplots(
    n_rows, n_clusters, figsize=(6 * max(1, n_clusters), 7), sharex=False
)

for col, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    mean_series = cluster_data.mean(axis=0)

    decomposition = seasonal_decompose(mean_series, model="additive", period=12)

    series_list = [
        ("Observed", decomposition.observed),
        ("Trend", decomposition.trend),
        ("Seasonal", decomposition.seasonal),
        ("Residual", decomposition.resid)
    ]

    cluster_color = cluster_colors.get(cluster_id, "black")

    for row, (label, series) in enumerate(series_list):

        ax = axes[row, col]

        # --------------------------
        #   PLOT (agora com cores)
        # --------------------------
        ax.plot(series.index, series.values,
                color=cluster_color, linewidth=1.0)

        # --------------------------
        #  LIMITE EXTERIOR (spines)
        # --------------------------
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.8)
            spine.set_color("black")

        # --------------------------
        #     LIMITES Y
        # --------------------------
        if label != "Observed":
            ymin = np.nanmin(series)
            ymax = np.nanmax(series)
            if np.isnan(ymin) or np.isnan(ymax):
                ymin, ymax = -1, 1
            margin = (ymax - ymin) * 0.10
            ax.set_ylim(ymin - margin, ymax + margin)

        # --------------------------
        #  TÍTULOS DOS SUBPLOTS
        # --------------------------
        if row == 0:
            ax.set_title(f"Seasonal Decompose – Cluster {cluster_id + 1}",
                         fontsize=12)

        if col == 0:
            ax.set_ylabel(label, fontsize=10)
        else:
            ax.set_yticks([])

        # --------------------------
        #   EIXO X — só no último row
        # --------------------------
        if row == n_rows - 1:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.xaxis.set_major_locator(mdates.YearLocator())
            ax.tick_params(axis='x', labelrotation=0, labelsize=10)
        else:
            ax.set_xticks([])
            ax.set_xticklabels([])

plt.tight_layout()
plt.show()


In [ ]:
# ==============================
# FIGURA 4: CORRELAÇÃO DE PEARSON (lag 0)
# ==============================

print("Gerando Figura 4 (Correlações Pearson)...")

from scipy.stats import pearsonr

external_vars = {
    'Temperatura (°C)':    (df_temp.set_index('data')['med_smooth'],        'black'),
    'Nível albufeira (m)': (df_nivel.set_index('data')['nivel_smooth'],     'navy'),
    'Precipitação (mm)':   (df_prec.set_index('data')['prec'],              'teal'),
}

fig_corr, axes_corr = plt.subplots(1, n_clusters, figsize=(4.5 * n_clusters, 5), sharey=True)
if n_clusters == 1:
    axes_corr = [axes_corr]

for col, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    mean_series   = agg_pivot.loc[cluster_cells].mean(axis=0)
    cluster_color = cluster_colors.get(cluster_id, 'gray')
    ax = axes_corr[col]

    var_names, r_vals, p_vals, bar_colors = [], [], [], []

    for vname, (vseries, vcolor) in external_vars.items():
        combined = pd.DataFrame({'dv': mean_series, 'ext': vseries}).dropna()
        if len(combined) > 5:
            r, p = pearsonr(combined['ext'], combined['dv'])
        else:
            r, p = np.nan, np.nan
        var_names.append(vname)
        r_vals.append(r)
        p_vals.append(p)
        bar_colors.append(vcolor)

    x_pos = np.arange(len(var_names))
    bars = ax.bar(x_pos, r_vals, color=bar_colors, alpha=0.7,
                  edgecolor='black', linewidth=0.6)

    # Anotar r e significância
    for i, (r, p) in enumerate(zip(r_vals, p_vals)):
        if np.isnan(r):
            continue
        sig = '*' if p < 0.05 else ''
        yoff = 0.03 if r >= 0 else -0.07
        ax.text(i, r + yoff, f'{r:.2f}{sig}',
                ha='center', va='bottom', fontsize=9)

    ax.axhline(0,    color='black', linewidth=0.8)
    ax.axhline( 0.3, color='gray',  linewidth=0.7, linestyle='--', alpha=0.5)
    ax.axhline(-0.3, color='gray',  linewidth=0.7, linestyle='--', alpha=0.5)
    ax.set_ylim(-1.15, 1.15)
    ax.set_xticks(x_pos)
    ax.set_xticklabels([v.split('(')[0].strip() for v in var_names],
                       fontsize=9, rotation=20, ha='right')
    ax.set_title(f'Cluster {cluster_id + 1}', fontsize=11, color=cluster_color)
    if col == 0:
        ax.set_ylabel('r de Pearson', fontsize=10)

fig_corr.suptitle(f'Correlação de Pearson: {TARGET_VAR} vs. variáveis externas',
                  fontsize=13)
plt.tight_layout()
plt.show()

# Tabela no terminal
print(f"\n{'='*55}")
print(f"{'Cluster':<10} {'Variável':<28} {'r':>6} {'p':>8}")
print(f"{'-'*55}")
for cluster_id in clusters_present:
    mean_series = agg_pivot.loc[
        cluster_df[cluster_df['cluster'] == cluster_id]['cell_id']
    ].mean(axis=0)
    for vname, (vseries, _) in external_vars.items():
        combined = pd.DataFrame({'dv': mean_series, 'ext': vseries}).dropna()
        if len(combined) > 5:
            r, p = pearsonr(combined['ext'], combined['dv'])
            sig = '*' if p < 0.05 else ' '
            print(f"{'Cluster '+str(cluster_id+1):<10} {vname:<28} {r:>6.3f} {p:>8.4f}{sig}")
print(f"{'='*55}\nLegenda: * p < 0.05")

In [ ]:
import pandas as pd
print(pd.__version__)

In [ ]:
# Escolha aqui o que quer analisar: 'dV' ou 'dH'
TARGET_VAR = 'dV'  # ou 'dV'

# grelha
grid_size = 50

N_CLUSTERS_FIXO = 3

DTW_WINDOW_PCT = 0.01  # Aqui podes mudar para 0.1, 0.05, 0.03, etc.

# Tipo de ligação
LINKAGE_METHOD = "average"
# Opções possíveis:
# "complete", "average", "ward"

# ==============================================================================
# SCRIPT COMPLETO: DTW + HIERÁRQUICO (COM ESTILO VISUAL K-MEANS MANTIDO)
# ==============================================================================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
import matplotlib.dates as mdates
from shapely.strtree import STRtree
from statsmodels.tsa.seasonal import seasonal_decompose
from matplotlib.lines import Line2D

# Bibliotecas para DTW e Hierárquico
try:
    from dtaidistance import dtw
    from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
    from scipy.spatial.distance import squareform
    from sklearn.preprocessing import StandardScaler
    import seaborn as sns
    from matplotlib.colors import to_hex
except ImportError as e:
    print(f"Erro de Importação: {e}. Certifique-se de que instalou: dtaidistance, scipy, seaborn, scikit-learn")

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    #barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
    #barragem = cos[cos['COS23_n4_L'] == 'Superfícies silvopastoris de azinheira']
    #barragem = cos[cos['COS23_n4_L'] == 'Matos']
    
    # Podes adicionar ou remover categorias dentro dos parênteses retos []
    barragem = cos[cos['COS23_n4_L'].isin([
    'Infraestruturas de produção de energia hídrica', 
    #'Albufeiras de barragens',
    'Equipamentos culturais',
    'Florestas de azinheira',
    'Matos',
    'Pastagens melhoradas',
    'Rede rodoviária',
    'Superfícies silvopastoris de azinheira'
])]
    
# - Albufeiras de barragens
# - Equipamentos culturais
# - Florestas de azinheira
# - Infraestruturas de produção de energia hídrica
# - Matos
# - Pastagens melhoradas
# - Rede rodoviária
# - Superfícies silvopastoris de azinheira
    
    
except Exception as e:
    print(f"Aviso: COS placeholder. {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs e Filtro
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df): return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) & (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'], value_vars=disp_cols, var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(start=max(asc_long['date'].min(), desc_long['date'].min()), end=min(asc_long['date'].max(), desc_long['date'].max()), freq='MS')

# ==============================
# 3. Interpolação
# ==============================
def interpolate_ps(df, dates):
    dfs = []
    for (x, y), g in df.groupby(['easting','northing']):
        g = g.sort_values('date')
        interp = np.interp(pd.to_datetime(dates).astype(np.int64), g['date'].astype(np.int64), g['disp'])
        dfs.append(pd.DataFrame({'easting': x, 'northing': y, 'latitude': g['latitude'].iloc[0], 'longitude': g['longitude'].iloc[0], 'date': dates, 'disp': interp, 'incidence_angle': g['incidence_angle'].iloc[0], 'track_angle': g['track_angle'].iloc[0]}))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. IDW
# ==============================
def idw(source, target, radius=25, power=2):
    out = []
    for d, src in source.groupby('date'):
        tgt = target[target['date']==d].copy()
        if src.empty or tgt.empty: continue
        tree = cKDTree(list(zip(src['easting'], src['northing'])))
        dist, idx = tree.query(list(zip(tgt['easting'], tgt['northing'])), k=5, distance_upper_bound=radius)
        vals, thetas, alphas = [], [], []
        for d_i, i_i in zip(dist, idx):
            m = np.isfinite(d_i)
            if not np.any(m): vals.append(np.nan); thetas.append(np.nan); alphas.append(np.nan); continue
            w = 1/(d_i[m]**power)
            vals.append(np.sum(w*src.iloc[i_i[m]]['disp'])/np.sum(w))
            thetas.append(np.sum(w*src.iloc[i_i[m]]['incidence_angle'])/np.sum(w))
            alphas.append(np.sum(w*src.iloc[i_i[m]]['track_angle'])/np.sum(w))
        tgt['disp_idw'] = vals; tgt['theta_desc'] = thetas; tgt['alpha_desc'] = alphas
        out.append(tgt)
    return pd.concat(out, ignore_index=True)
asc_interp = idw(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw'])

# ==============================
# 5. dV (ou dH)
# ==============================
orb_inc = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orb_inc) * np.cos(np.deg2rad(asc_interp['latitude'])))

def get_comps(row):
    ta, td, beta = np.deg2rad(row['incidence_angle']), np.deg2rad(row['theta_desc']), row['beta']
    denom = (np.cos(ta)*np.sin(td)*np.cos(beta) + np.cos(td)*np.sin(ta)*np.cos(beta))
    if denom == 0: return np.nan, np.nan
    dV = (row['disp_idw']*np.sin(ta)*np.cos(beta) + row['disp']*np.sin(td)*np.cos(beta))/denom
    dH = (row['disp_idw']*np.cos(ta) - row['disp']*np.cos(td))/denom
    return dV, dH

asc_interp[['dV', 'dH']] = asc_interp.apply(lambda x: pd.Series(get_comps(x)), axis=1)
asc_interp = asc_interp.dropna(subset=['dV', 'dH'])

# ==============================
# 6. Grelha
# ==============================
#grid_size = 100
xe = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
ye = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
xs, ys = xe - grid_size/2, ye - grid_size/2
asc_interp['cx'] = pd.cut(asc_interp['easting'], bins=xs, labels=False)
asc_interp['cy'] = pd.cut(asc_interp['northing'], bins=ys, labels=False)
asc_interp = asc_interp.dropna(subset=['cx','cy'])
asc_interp['cell_id'] = asc_interp['cx'].astype(int).astype(str)+"_"+asc_interp['cy'].astype(int).astype(str)

grid_data = [{'cell_id': f"{ix}_{iy}", 'geometry': box(xs[ix], ys[iy], xs[ix+1], ys[iy+1])} for ix in range(len(xs)-1) for iy in range(len(ys)-1)]
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# Escolha aqui se usa dV ou dH para a agregação
#TARGET_VAR = 'dV' 
agg = asc_interp.groupby(['cell_id','date']).agg(dV=(TARGET_VAR,'mean')).reset_index() # Coluna final chama-se sempre 'dV' para manter compatibilidade com resto do script

# ==============================
# 7. Recorte e Filtro
# ==============================
points_gdf = gpd.GeoDataFrame(asc_interp, geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']), crs="EPSG:3035").to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

poly = grid_recort.geometry.values; ids = grid_recort['cell_id'].values; tree = STRtree(poly)
valid = set()
for pt in points_gdf.geometry:
    for idx in tree.query(pt):
        if poly[idx].contains(pt): valid.add(ids[idx])
grid_barragem = grid_recort[grid_recort['cell_id'].isin(valid)]
agg = agg[agg['cell_id'].isin(valid)]

# ==============================
# 8. CLUSTERING (VERSÃO ROBUSTA CONTRA NaNs)
# ==============================
# print(f">>> A calcular DTW (Flexibilidade: {DTW_WINDOW_PCT})...")

# # 1. Preparar Matriz (Pivot)
# agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV')

# # --- LIMPEZA DE NaNs (Obrigatório para o Linkage não dar erro) ---
# # Interpola pequenos buracos e remove células que ainda tenham falhas críticas
# agg_pivot = agg_pivot.interpolate(axis=1, limit_direction='both').dropna()

# cell_ids = agg_pivot.index.tolist()
# X = agg_pivot.values.astype(float)
# n = X.shape[0]

# if n < 2:
#     print("Aviso: Células insuficientes após limpeza de dados.")
# else:
#     # 2. Normalizar
#     scaler = StandardScaler()
#     X_scaled = scaler.fit_transform(X.T).T 

#     # 3. Calcular DTW com a nova variável
#     # max(1, ...) garante que a janela nunca é zero
#     window_dtw = max(1, int(DTW_WINDOW_PCT * X_scaled.shape[1])) 
    
#     # Cálculo da matriz
#     dist_matrix = dtw.distance_matrix_fast(X_scaled, window=window_dtw)

#     # --- TRATAMENTO PÓS-CÁLCULO ---
#     # Se o DTW gerar algum NaN (acontece se a janela for muito pequena para certas séries)
#     if not np.all(np.isfinite(dist_matrix)):
#         # Substituímos NaNs pelo valor máximo da matriz (para dizer que são muito diferentes)
#         mask_nan = np.isnan(dist_matrix)
#         dist_matrix[mask_nan] = np.nanmax(dist_matrix) if not np.all(np.isnan(dist_matrix)) else 0

#     # 4. Clustering Hierárquico
#     condensed = squareform(dist_matrix)
#     Z = linkage(condensed, method=LINKAGE_METHOD)

#     # 5. Corte Automático (Elbow)
#     last = Z[-10:, 2] # Últimas distâncias de fusão
#     acceleration = np.diff(last, 2)
#     try:
#         k_idx = np.argmax(acceleration) + 2
#         cut_distance = (last[k_idx] + last[k_idx-1]) / 2
#     except:
#         cut_distance = last[len(last)//2]

#     # Labels
#     cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
    
#     # Reajustar labels para começar em 0
#     u_labels = np.unique(cluster_labels)
#     map_l = {old: new for new, old in enumerate(u_labels)}
#     cluster_labels = np.array([map_l[x] for x in cluster_labels])
    
#     num_clusters = len(u_labels)
#     print(f"DTW Concluído. Corte: {cut_distance:.2f}. Clusters: {num_clusters}")


# ==============================
# 8. CLUSTERING (VERSÃO ROBUSTA CONTRA NaNs)
# ==============================
print(f">>> A calcular DTW (Flexibilidade: {DTW_WINDOW_PCT})...")

# 1. Preparar Matriz (Pivot)
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV')

# --- LIMPEZA DE NaNs (Obrigatório para o Linkage não dar erro) ---
# Interpola pequenos buracos e remove células que ainda tenham falhas críticas
agg_pivot = agg_pivot.interpolate(axis=1, limit_direction='both').dropna()

cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

if n < 2:
    print("Aviso: Células insuficientes após limpeza de dados.")
else:
    # 2. ESCOLHA DE NORMALIZAÇÃO
    # Se queres que a magnitude (quanto desce) seja o fator principal, 
    # COMENTA as linhas do scaler abaixo.
    
    # scaler = StandardScaler()
    # X_scaled = scaler.fit_transform(X.T).T 
    
    # Alternativa: Se não normalizares, usa os dados brutos (X)
    X_scaled = X 

    # 3. Calcular DTW
    window_dtw = max(1, int(DTW_WINDOW_PCT * X_scaled.shape[1])) 
    dist_matrix = dtw.distance_matrix_fast(X_scaled, window=window_dtw)

    # ... (Tratamento de NaNs igual) ...

    # 4. Clustering Hierárquico - MUDA O MÉTODO PARA 'WARD'
    # O método 'ward' é muito melhor para criar clusters compactos baseados em magnitude
    condensed = squareform(dist_matrix)
    Z = linkage(condensed, method='ward') 

    # 5. Forçar o número de Clusters
    # O corte automático (Elbow) muitas vezes é conservador demais e dá apenas 2 clusters.
    # Vamos definir um número fixo (ex: 4) para ver se ele isola a tendência descendente.
    # N_CLUSTERS_FIXO = 4
    cluster_labels = fcluster(Z, t=N_CLUSTERS_FIXO, criterion='maxclust')
    
    # Reajustar labels para começar em 0
    u_labels = np.unique(cluster_labels)
    map_l = {old: new for new, old in enumerate(u_labels)}
    cluster_labels = np.array([map_l[x] for x in cluster_labels])
    
    num_clusters = len(u_labels)
    print(f"DTW Concluído. Clusters: {num_clusters} (Método Ward, Sem Normalização)")



# DataFrame Final
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})
# Cores Dinâmicas (para N clusters)
palette = sns.color_palette("Set2", num_clusters) if num_clusters <= 8 else sns.color_palette("tab20", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(range(num_clusters), palette)}

grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='inner')

# ==============================
# FIGURA EXTRA: DENDROGRAMA
# ==============================
# if Z is not None:
#     print("Gerando Dendrograma...")
#     plt.figure(figsize=(12, 5))
#     dendrogram(Z, leaf_rotation=90, leaf_font_size=8,
#                color_threshold=cut_distance
#                )
#     plt.axhline(y=cut_distance, c='k', ls='--', lw=1, label='Corte Automático')
#     plt.title('Dendrograma de Clustering Hierárquico (DTW)')
#     plt.xlabel('Células'); plt.ylabel('Distância')
#     plt.legend(); plt.tight_layout();
    
#     plt.savefig("dendrograma_final.pdf", format='pdf')
    
#     plt.show()

if Z is not None:
    print("Gerando Dendrograma...")
    plt.figure(figsize=(12, 5))
    
    # 1. Recuperar a distância de corte para o número de clusters desejado
    # Se queres N_CLUSTERS_FIXO, o corte acontece entre a fusão N e a fusão N-1
    # Z[-N_CLUSTERS_FIXO + 1, 2] dá a distância exata onde o corte ocorre
    try:
        distancia_visual = Z[-num_clusters + 1, 2]
    except:
        distancia_visual = 0

    # 2. Plot do Dendrograma
    # Usamos color_threshold para pintar os ramos de acordo com os clusters
    dendrogram(Z, 
               leaf_rotation=90, 
               leaf_font_size=8,
               color_threshold=distancia_visual, # Pinta os clusters
               no_labels=True # Se tiveres muitas células, labels ficam ilegíveis
               )
    
    # 3. Linha horizontal para indicar o corte
    plt.axhline(y=distancia_visual, c='k', ls='--', lw=1.2, 
                label=f'Corte para K={num_clusters}')
    
    plt.title(f'Dendrograma de Clustering Hierárquico (DTW) - K={num_clusters}')
    plt.xlabel('Células (Grid 50m)')
    plt.ylabel('Distância (Average)')
    plt.legend()
    plt.tight_layout()
    
    # 4. Guardar com alta qualidade
    # plt.savefig("dendrograma_final.pdf", format='pdf', bbox_inches='tight')
    plt.savefig("figuras/dendrograma_final.pdf", bbox_inches='tight', pad_inches=0.02)
    
    plt.show()



# ==============================
# 9. Dados Hidro (IGUAL)
# ==============================
date_range = agg_pivot.columns
win = 13

# Temp
try: df_t = pd.read_excel("data/alqueva_temp.xlsx"); df_t['data']=pd.to_datetime(df_t['data'])
except: df_t = pd.DataFrame({'data': date_range, 'med': 0})
ts = df_t.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': ts.index, 'med': ts.values})
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), win, 2)

# Nível
try: 
    df_n = pd.read_excel("data/alqueva_nivel.xlsx"); df_n['data']=pd.to_datetime(df_n['data'])
    df_n['nivel'] = pd.to_numeric(df_n['nivel'], errors='coerce'); df_n = df_n.dropna(subset=['nivel'])
except: df_n = pd.DataFrame({'data': date_range, 'nivel': 0})
ns = df_n.set_index('data')['nivel'].resample('MS').mean().reindex(date_range).interpolate(limit_direction='both').ffill().bfill()
df_nivel = pd.DataFrame({'data': ns.index, 'nivel': ns.values})
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], win, 2)

# Precipitação
try: df_p = pd.read_excel("data/prec.xlsx"); df_p['data'] = pd.to_datetime(df_p['data'])
except: np.random.seed(42); df_p = pd.DataFrame({'data': date_range, 'prec': 0})
ps = df_p.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': ps.index, 'prec': ps.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções Visuais (IGUAL)
# ==============================
# def scale_vis(d, vmin, vmax, s=0.20, o=0.30):
#     d = np.array(d); mn, mx = np.nanmin(d), np.nanmax(d)
#     span = vmax - vmin
#     norm = np.zeros_like(d) if mx==mn else (d-mn)/(mx-mn)
#     return norm * span * s + (vmax - span * o), mn, mx

# def create_ticks(rmin, rmax, vmin, vmax, s=0.20, o=0.30, n=5):
#     rt = np.linspace(rmin, rmax, n)
#     span = vmax - vmin
#     nt = np.linspace(0,1,n) if rmax==rmin else (rt-rmin)/(rmax-rmin)
#     vt = nt * span * s + (vmax - span * o)
#     return vt, rt


# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks


# ==============================
# FIGURA 1: MAPA (ESTILO ORIGINAL)
# ==============================
print("Gerando Figura 1...")
clusters_present = sorted(cluster_df['cluster'].unique())
k_plot = len(clusters_present)

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', lw=1)

for i in clusters_present:
    sub = grid_sel[grid_sel['cluster'] == i]
    sub.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    cents = sub.copy(); cents.geometry = cents.geometry.centroid
    cents.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

hdl = [plt.Rectangle((0,0),1,1, fc=cluster_colors[i], alpha=0.4) for i in clusters_present]
lbl = [f'Cluster {i+1} ({len(grid_sel[grid_sel["cluster"]==i])} cel)' for i in clusters_present]
hdl.append(Line2D([0],[0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None')); lbl.append('Centróides')
ax_map.legend(hdl, lbl, loc='upper left', fontsize=10, title=f"Clusters DTW (K={k_plot})")
ax_map.set_title(f"Mapa de Clusters ({TARGET_VAR})", fontsize=15)

# plt.savefig("mapa.pdf", format='pdf')
plt.savefig("figuras/mapa.pdf", bbox_inches='tight', pad_inches=0.02)

plt.show()

# ==============================
# FIGURA 2: SÉRIES TEMPORAIS
# ==============================

print("Gerando Figura 2...")

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5

fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

dV_min = agg[TARGET_VAR].min()
dV_max = agg[TARGET_VAR].max()
dV_margin = (dV_max - dV_min) * 0.1
dV_ylim = (dV_min - dV_margin, dV_max + dV_margin)

# --- Séries visuais ---
temp_visual, _, _ = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, _, _ = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)

# Séries visuais
# tv, tmi, tmx = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
# nivel_visual, nivel_min, nivel_max = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
# prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)


for idx, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)


    # --- Função atualizada ---
    def plot_dV_comparison_series(
        row_idx, ext_data_visual, ext_data_df, ext_col_real,
        ext_label, ext_color,
        plot_as_bar=False, bar_color=None,
        plot_real_scale_line=False,
        combine_annual_prec=False,
        integer_ticks=False, show_background_bars=False):

        ax = fig_series.add_subplot(gs_series[row_idx, idx])

        # --- dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid],
                    color='lightgray', alpha=0.7, linewidth=1.0)

        ax.plot(cluster_data.columns, cluster_mean_dV,
                color=cluster_color, linewidth=1.0,
                label=f'Média Cluster {cluster_id+1}')

        # --- Eixo Y2 ---
        ax2 = ax.twinx()
        lines2 = []

        # -------------------------
        #   CASOS DE PRECIPITAÇÃO
        # -------------------------
        if combine_annual_prec:

            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)

            for ano in df_prec['ano_hidrologico'].unique():
                grupo = df_prec[df_prec['ano_hidrologico'] == ano]
                ax2.plot(grupo['data'], grupo['prec_acum_anual'],
                         color='teal', linewidth=1.0, alpha=0.9)

            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            ax2.set_ylabel("Prec. acumulada anual", color="teal")
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=1.0, label='Prec. acumulada anual (mm)')
            ]

        elif plot_as_bar:

            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15,
                    color=bar_color or ext_color, alpha=0.5)

            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            ax2.set_ylabel(ext_label, color='teal')
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)
            ]

        elif plot_real_scale_line:

            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15,
                        color='teal', alpha=0.3)

            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real],
                     color='teal', linewidth=1.0, alpha=0.85)

            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            ax2.set_ylabel(ext_label, color='teal')
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=1.0, label=ext_label)
            ]

        # -------------------------
        #      OUTRAS SÉRIES
        # -------------------------
        else:

            ax2.plot(ext_data_df['data'], ext_data_visual,
                     color=ext_color, linewidth=1.0, alpha=0.85)

            visual_ticks, real_ticks = create_visual_ticks(
                ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(),
                dV_min, dV_max, scale_factor=0.20, offset_factor=0.30
            )

            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(visual_ticks)

            if integer_ticks:
                ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else:
                ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])

            label_color = 'black' if 'Temperatura' in ext_label else ext_color
            ax2.set_ylabel(ext_label, color=label_color)
            ax2.tick_params(axis='y', colors=label_color)

            lines2 = [ax2.lines[-1]]

        # --- Y1 sempre dV ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)

        if idx == 0:
            ax.set_ylabel('dV (mm)')
        else:
            ax.set_yticklabels([])

        # Y2 somente no último cluster
        if idx != n_clusters - 1:
            ax2.set_yticklabels([])

        ax.tick_params(left=(idx==0))
        ax2.tick_params(right=(idx==n_clusters-1))

        # --- X-axis ---
        if row_idx == n_rows_series - 1:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        else:
            ax.set_xticklabels([])

        # --- Legenda somente no primeiro e no último cluster ---
        if idx == 0 or idx == n_clusters - 1:
            lines1, labels1 = ax.get_legend_handles_labels()
            handles2, labels2 = ax2.get_legend_handles_labels()

            ax.legend(
                lines1 + handles2,
                labels1 + labels2,
                fontsize=8,
                loc='best',
                framealpha=1.0,
                facecolor='white',
                edgecolor='lightgray'
            ).set_zorder(100)

        return ax


    # ---- CHAMADA DAS 5 SÉRIES ----
    plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth',
                              'Temperatura média (°C)', 'black', integer_ticks=True)

    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth',
                              'Nível da albufeira (m)', 'navy', integer_ticks=True)

    plot_dV_comparison_series(2, None, df_prec, 'prec',
                              'Precipitação mensal (mm)', 'teal',
                              plot_as_bar=True, bar_color='teal')

    plot_dV_comparison_series(3, None, df_prec, 'prec_acum',
                              'Prec. acumulada total (mm)', 'teal',
                              plot_real_scale_line=True, show_background_bars=True)

    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual',
                              'Prec. acumulada anual', 'teal',
                              combine_annual_prec=True)


plt.tight_layout()

# plt.savefig("clusters.pdf", format='pdf')
plt.savefig("figuras/clusters.pdf", bbox_inches='tight', pad_inches=0.02)

plt.show()


# ==============================
# FIGURA 3: DECOMPOSIÇÃO SAZONAL
# ==============================

print("Gerando Figura 3...")

n_rows = 4  # Observed, Trend, Seasonal, Residual
fig_series_decomp, axes = plt.subplots(
    n_rows, n_clusters, figsize=(6 * max(1, n_clusters), 7), sharex=False
)

for col, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    mean_series = cluster_data.mean(axis=0)

    decomposition = seasonal_decompose(mean_series, model="additive", period=12)

    series_list = [
        ("Observed", decomposition.observed),
        ("Trend", decomposition.trend),
        ("Seasonal", decomposition.seasonal),
        ("Residual", decomposition.resid)
    ]

    cluster_color = cluster_colors.get(cluster_id, "black")

    for row, (label, series) in enumerate(series_list):

        ax = axes[row, col]

        # --------------------------
        #   PLOT (agora com cores)
        # --------------------------
        ax.plot(series.index, series.values,
                color=cluster_color, linewidth=1.0)

        # --------------------------
        #  LIMITE EXTERIOR (spines)
        # --------------------------
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.8)
            spine.set_color("black")

        # --------------------------
        #     LIMITES Y
        # --------------------------
        if label != "Observed":
            ymin = np.nanmin(series)
            ymax = np.nanmax(series)
            if np.isnan(ymin) or np.isnan(ymax):
                ymin, ymax = -1, 1
            margin = (ymax - ymin) * 0.10
            ax.set_ylim(ymin - margin, ymax + margin)

        # --------------------------
        #  TÍTULOS DOS SUBPLOTS
        # --------------------------
        if row == 0:
            ax.set_title(f"Seasonal Decompose – Cluster {cluster_id + 1}",
                         fontsize=12)

        if col == 0:
            ax.set_ylabel(label, fontsize=10)
        else:
            ax.set_yticks([])

        # --------------------------
        #   EIXO X — só no último row
        # --------------------------
        if row == n_rows - 1:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.xaxis.set_major_locator(mdates.YearLocator())
            ax.tick_params(axis='x', labelrotation=0, labelsize=10)
        else:
            ax.set_xticks([])
            ax.set_xticklabels([])

plt.tight_layout()

# plt.savefig("stl.pdf", format='pdf')
plt.savefig("figuras/stl.pdf", bbox_inches='tight', pad_inches=0.02)

plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import numpy as np

# 1. Definições Globais de Estilo
LINE_WIDTH = 0.4
LINE_COLOR = '#333333' 
MY_CMAP = plt.cm.RdBu # RdBu padrão: Azul é Positivo (+1), Vermelho é Negativo (-1)

sns.set_context("paper", font_scale=0.9)
sns.set_style("white")
plt.rcParams['axes.linewidth'] = LINE_WIDTH

# --- Dados (Certifica-te que estas variáveis existem no teu ambiente) ---
CLUSTER_ALVO_ID = 0 
c_cells = cluster_df[cluster_df['cluster'] == CLUSTER_ALVO_ID]['cell_id']
dV_mean = agg_pivot.loc[c_cells].mean()
df_plot = pd.DataFrame({
    'dV': dV_mean,
    'Temp': df_temp.set_index('data')['med_smooth'].reindex(dV_mean.index),
    'Niv': df_nivel.set_index('data')['nivel_smooth'].reindex(dV_mean.index),
    'Prec': df_prec.set_index('data')['prec'].reindex(dV_mean.index),
    'P_Ac': df_prec.set_index('data')['prec_acum'].reindex(dV_mean.index),
    'P_An': df_prec.set_index('data')['prec_acum_anual'].reindex(dV_mean.index)
}).dropna()

cols = df_plot.columns.tolist()
n_vars = len(cols)

# 2. Criar o PairGrid
g = sns.PairGrid(df_plot, vars=cols, diag_sharey=False, height=1.0, aspect=1.0)

# --- Funções de Desenho ---
def cor_func(x, y, **kwargs):
    r, p = pearsonr(x, y)
    ax = plt.gca()
    # Mapeamento: r=1 -> Azul, r=-1 -> Vermelho
    color_val = (r + 1) / 2
    facecolor = MY_CMAP(color_val)
    ax.set_facecolor((*facecolor[:3], 0.5))
    ax.annotate(f"{r:.2f}", xy=(0.5, 0.5), xycoords=ax.transAxes, 
                ha='center', va='center', fontsize=8, fontweight='normal')

g.map_diag(sns.histplot, kde=True, color="#2c3e50", alpha=0.2, edgecolor='white', linewidth=0.3, line_kws={'linewidth': 0.7})
# --- PARTE INFERIOR: Scatter Plots ---
g.map_lower(sns.regplot, ci=None, 
            scatter_kws={
                's': 3,                # Tamanho do ponto
                'alpha': 1.0,          # Opacidade total (sem transparência)
                'color': 'black',      # Cor base
                'facecolor': 'black',  # Preenchimento preto sólido
                'edgecolor': 'black',  # Contorno preto sólido
                'linewidths': 0        # Remove qualquer largura de linha de bordo para evitar reflexos
            }, 
            line_kws={'color': 'red', 'linewidth': 0.7})
g.map_upper(cor_func)

# 3. Uniformização Total das Linhas e Padding
for i in range(n_vars):
    for j in range(n_vars):
        ax = g.axes[i, j]
        ax.set_xlabel(""); ax.set_ylabel("")
        
        # Ajuste de Padding interno (Respiro de 25%)
        if i != j:
            x_min, x_max = df_plot[cols[j]].min(), df_plot[cols[j]].max()
            y_min, y_max = df_plot[cols[i]].min(), df_plot[cols[i]].max()
            x_range, y_range = x_max - x_min, y_max - y_min
            ax.set_xlim(x_min - 0.3 * x_range, x_max + 0.3 * x_range)
            ax.set_ylim(y_min - 0.3 * y_range, y_max + 0.3 * y_range)

        # Labels apenas nas extremidades
        if j != 0: ax.set_yticklabels([])
        if i != n_vars - 1: ax.set_xticklabels([])
        
        ax.tick_params(labelsize=6, direction='in', pad=1, width=LINE_WIDTH, color=LINE_COLOR)
        
        # Forçar todas as linhas da grelha
        for edge in ['top', 'bottom', 'left', 'right']:
            ax.spines[edge].set_visible(True)
            ax.spines[edge].set_linewidth(LINE_WIDTH)
            ax.spines[edge].set_color(LINE_COLOR)
        
        # Identificação na Diagonal em Vermelho (Sem Bold)
        if i == j:
            ax.annotate(cols[i], xy=(0.05, 0.90), xycoords='axes fraction', 
                        ha='left', va='top', fontsize=8, fontweight='normal', color='red',
                        bbox=dict(facecolor='white', alpha=0.4, edgecolor='none', pad=0))

# 4. Ajuste de Layout para Colagem
plt.subplots_adjust(hspace=0, wspace=0, left=0.1, right=0.85, bottom=0.18, top=0.95)

# 5. Colorbar Corrigida (Azul=Positivo, Vermelho=Negativo)
cax = g.fig.add_axes([0.87, 0.18, 0.02, 0.77]) 
sm = plt.cm.ScalarMappable(cmap=MY_CMAP, norm=plt.Normalize(-1, 1))
cbar = g.fig.colorbar(sm, cax=cax)
cbar.set_ticks([-1, 0, 1])
cbar.outline.set_linewidth(LINE_WIDTH)
cbar.outline.set_edgecolor(LINE_COLOR)
cbar.set_label('Correlation coefficient', rotation=270, labelpad=15, fontsize=9)
cbar.ax.tick_params(labelsize=7, width=LINE_WIDTH, color=LINE_COLOR)

# 6. Legenda Inferior (Recuperada e Colada)
ax_table = g.fig.add_axes([0.1, 0.06, 0.75, 0.12]) 
ax_table.axis('off')

legend_data = [
    ["dV: Vertical Displacement", "Temp: Temperature", "Niv: Reservoir Level"],
    ["Prec: Daily Precipitation", "P_Ac: Accumulated Prec.", "P_An: Annual Acc. Prec."]
]

table = ax_table.table(cellText=legend_data, loc='upper center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.4) 

for (row, col), cell in table.get_celld().items():
    cell.set_linewidth(LINE_WIDTH)
    cell.set_edgecolor(LINE_COLOR)
    cell.get_text().set_fontweight('normal')

# 7. Finalização
plt.savefig("Matriz_Correlacao_Final_Final.png", dpi=600, bbox_inches='tight')

# plt.savefig("corr_1.pdf", format='pdf')
plt.savefig("figuras/corr_1.pdf", bbox_inches='tight', pad_inches=0.02)

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import numpy as np

# 1. Definições Globais de Estilo
LINE_WIDTH = 0.4
LINE_COLOR = '#333333' 
MY_CMAP = plt.cm.RdBu # RdBu padrão: Azul é Positivo (+1), Vermelho é Negativo (-1)

sns.set_context("paper", font_scale=0.9)
sns.set_style("white")
plt.rcParams['axes.linewidth'] = LINE_WIDTH

# --- Dados (Certifica-te que estas variáveis existem no teu ambiente) ---
CLUSTER_ALVO_ID = 1 
c_cells = cluster_df[cluster_df['cluster'] == CLUSTER_ALVO_ID]['cell_id']
dV_mean = agg_pivot.loc[c_cells].mean()
df_plot = pd.DataFrame({
    'dV': dV_mean,
    'Temp': df_temp.set_index('data')['med_smooth'].reindex(dV_mean.index),
    'Niv': df_nivel.set_index('data')['nivel_smooth'].reindex(dV_mean.index),
    'Prec': df_prec.set_index('data')['prec'].reindex(dV_mean.index),
    'P_Ac': df_prec.set_index('data')['prec_acum'].reindex(dV_mean.index),
    'P_An': df_prec.set_index('data')['prec_acum_anual'].reindex(dV_mean.index)
}).dropna()

cols = df_plot.columns.tolist()
n_vars = len(cols)

# 2. Criar o PairGrid
g = sns.PairGrid(df_plot, vars=cols, diag_sharey=False, height=1.0, aspect=1.0)

# --- Funções de Desenho ---
def cor_func(x, y, **kwargs):
    r, p = pearsonr(x, y)
    ax = plt.gca()
    # Mapeamento: r=1 -> Azul, r=-1 -> Vermelho
    color_val = (r + 1) / 2
    facecolor = MY_CMAP(color_val)
    ax.set_facecolor((*facecolor[:3], 0.5))
    ax.annotate(f"{r:.2f}", xy=(0.5, 0.5), xycoords=ax.transAxes, 
                ha='center', va='center', fontsize=8, fontweight='normal')

g.map_diag(sns.histplot, kde=True, color="#2c3e50", alpha=0.2, edgecolor='white', linewidth=0.3, line_kws={'linewidth': 0.7})
# --- PARTE INFERIOR: Scatter Plots ---
g.map_lower(sns.regplot, ci=None, 
            scatter_kws={
                's': 3,                # Tamanho do ponto
                'alpha': 1.0,          # Opacidade total (sem transparência)
                'color': 'black',      # Cor base
                'facecolor': 'black',  # Preenchimento preto sólido
                'edgecolor': 'black',  # Contorno preto sólido
                'linewidths': 0        # Remove qualquer largura de linha de bordo para evitar reflexos
            }, 
            line_kws={'color': 'red', 'linewidth': 0.7})
g.map_upper(cor_func)

# 3. Uniformização Total das Linhas e Padding
for i in range(n_vars):
    for j in range(n_vars):
        ax = g.axes[i, j]
        ax.set_xlabel(""); ax.set_ylabel("")
        
        # Ajuste de Padding interno (Respiro de 25%)
        if i != j:
            x_min, x_max = df_plot[cols[j]].min(), df_plot[cols[j]].max()
            y_min, y_max = df_plot[cols[i]].min(), df_plot[cols[i]].max()
            x_range, y_range = x_max - x_min, y_max - y_min
            ax.set_xlim(x_min - 0.3 * x_range, x_max + 0.3 * x_range)
            ax.set_ylim(y_min - 0.3 * y_range, y_max + 0.3 * y_range)

        # Labels apenas nas extremidades
        if j != 0: ax.set_yticklabels([])
        if i != n_vars - 1: ax.set_xticklabels([])
        
        ax.tick_params(labelsize=6, direction='in', pad=1, width=LINE_WIDTH, color=LINE_COLOR)
        
        # Forçar todas as linhas da grelha
        for edge in ['top', 'bottom', 'left', 'right']:
            ax.spines[edge].set_visible(True)
            ax.spines[edge].set_linewidth(LINE_WIDTH)
            ax.spines[edge].set_color(LINE_COLOR)
        
        # Identificação na Diagonal em Vermelho (Sem Bold)
        if i == j:
            ax.annotate(cols[i], xy=(0.05, 0.90), xycoords='axes fraction', 
                        ha='left', va='top', fontsize=8, fontweight='normal', color='red',
                        bbox=dict(facecolor='white', alpha=0.4, edgecolor='none', pad=0))

# 4. Ajuste de Layout para Colagem
plt.subplots_adjust(hspace=0, wspace=0, left=0.1, right=0.85, bottom=0.18, top=0.95)

# 5. Colorbar Corrigida (Azul=Positivo, Vermelho=Negativo)
cax = g.fig.add_axes([0.87, 0.18, 0.02, 0.77]) 
sm = plt.cm.ScalarMappable(cmap=MY_CMAP, norm=plt.Normalize(-1, 1))
cbar = g.fig.colorbar(sm, cax=cax)
cbar.set_ticks([-1, 0, 1])
cbar.outline.set_linewidth(LINE_WIDTH)
cbar.outline.set_edgecolor(LINE_COLOR)
cbar.set_label('Correlation coefficient', rotation=270, labelpad=15, fontsize=9)
cbar.ax.tick_params(labelsize=7, width=LINE_WIDTH, color=LINE_COLOR)

# 6. Legenda Inferior (Recuperada e Colada)
ax_table = g.fig.add_axes([0.1, 0.06, 0.75, 0.12]) 
ax_table.axis('off')

legend_data = [
    ["dV: Vertical Displacement", "Temp: Temperature", "Niv: Reservoir Level"],
    ["Prec: Daily Precipitation", "P_Ac: Accumulated Prec.", "P_An: Annual Acc. Prec."]
]

table = ax_table.table(cellText=legend_data, loc='upper center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.4) 

for (row, col), cell in table.get_celld().items():
    cell.set_linewidth(LINE_WIDTH)
    cell.set_edgecolor(LINE_COLOR)
    cell.get_text().set_fontweight('normal')

# 7. Finalização
plt.savefig("figuras/corr_2.pdf", bbox_inches='tight', pad_inches=0.02)

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import numpy as np

# 1. Definições Globais de Estilo
LINE_WIDTH = 0.4
LINE_COLOR = '#333333' 
MY_CMAP = plt.cm.RdBu # RdBu padrão: Azul é Positivo (+1), Vermelho é Negativo (-1)

sns.set_context("paper", font_scale=0.9)
sns.set_style("white")
plt.rcParams['axes.linewidth'] = LINE_WIDTH

# --- Dados (Certifica-te que estas variáveis existem no teu ambiente) ---
CLUSTER_ALVO_ID = 2 
c_cells = cluster_df[cluster_df['cluster'] == CLUSTER_ALVO_ID]['cell_id']
dV_mean = agg_pivot.loc[c_cells].mean()
df_plot = pd.DataFrame({
    'dV': dV_mean,
    'Temp': df_temp.set_index('data')['med_smooth'].reindex(dV_mean.index),
    'Niv': df_nivel.set_index('data')['nivel_smooth'].reindex(dV_mean.index),
    'Prec': df_prec.set_index('data')['prec'].reindex(dV_mean.index),
    'P_Ac': df_prec.set_index('data')['prec_acum'].reindex(dV_mean.index),
    'P_An': df_prec.set_index('data')['prec_acum_anual'].reindex(dV_mean.index)
}).dropna()

cols = df_plot.columns.tolist()
n_vars = len(cols)

# 2. Criar o PairGrid
g = sns.PairGrid(df_plot, vars=cols, diag_sharey=False, height=1.0, aspect=1.0)

# --- Funções de Desenho ---
def cor_func(x, y, **kwargs):
    r, p = pearsonr(x, y)
    ax = plt.gca()
    # Mapeamento: r=1 -> Azul, r=-1 -> Vermelho
    color_val = (r + 1) / 2
    facecolor = MY_CMAP(color_val)
    ax.set_facecolor((*facecolor[:3], 0.5))
    ax.annotate(f"{r:.2f}", xy=(0.5, 0.5), xycoords=ax.transAxes, 
                ha='center', va='center', fontsize=8, fontweight='normal')

g.map_diag(sns.histplot, kde=True, color="#2c3e50", alpha=0.2, edgecolor='white', linewidth=0.3, line_kws={'linewidth': 0.7})
# --- PARTE INFERIOR: Scatter Plots ---
g.map_lower(sns.regplot, ci=None, 
            scatter_kws={
                's': 3,                # Tamanho do ponto
                'alpha': 1.0,          # Opacidade total (sem transparência)
                'color': 'black',      # Cor base
                'facecolor': 'black',  # Preenchimento preto sólido
                'edgecolor': 'black',  # Contorno preto sólido
                'linewidths': 0        # Remove qualquer largura de linha de bordo para evitar reflexos
            }, 
            line_kws={'color': 'red', 'linewidth': 0.7})
g.map_upper(cor_func)

# 3. Uniformização Total das Linhas e Padding
for i in range(n_vars):
    for j in range(n_vars):
        ax = g.axes[i, j]
        ax.set_xlabel(""); ax.set_ylabel("")
        
        # Ajuste de Padding interno (Respiro de 25%)
        if i != j:
            x_min, x_max = df_plot[cols[j]].min(), df_plot[cols[j]].max()
            y_min, y_max = df_plot[cols[i]].min(), df_plot[cols[i]].max()
            x_range, y_range = x_max - x_min, y_max - y_min
            ax.set_xlim(x_min - 0.3 * x_range, x_max + 0.3 * x_range)
            ax.set_ylim(y_min - 0.3 * y_range, y_max + 0.3 * y_range)

        # Labels apenas nas extremidades
        if j != 0: ax.set_yticklabels([])
        if i != n_vars - 1: ax.set_xticklabels([])
        
        ax.tick_params(labelsize=6, direction='in', pad=1, width=LINE_WIDTH, color=LINE_COLOR)
        
        # Forçar todas as linhas da grelha
        for edge in ['top', 'bottom', 'left', 'right']:
            ax.spines[edge].set_visible(True)
            ax.spines[edge].set_linewidth(LINE_WIDTH)
            ax.spines[edge].set_color(LINE_COLOR)
        
        # Identificação na Diagonal em Vermelho (Sem Bold)
        if i == j:
            ax.annotate(cols[i], xy=(0.05, 0.90), xycoords='axes fraction', 
                        ha='left', va='top', fontsize=8, fontweight='normal', color='red',
                        bbox=dict(facecolor='white', alpha=0.4, edgecolor='none', pad=0))

# 4. Ajuste de Layout para Colagem
plt.subplots_adjust(hspace=0, wspace=0, left=0.1, right=0.85, bottom=0.18, top=0.95)

# 5. Colorbar Corrigida (Azul=Positivo, Vermelho=Negativo)
cax = g.fig.add_axes([0.87, 0.18, 0.02, 0.77]) 
sm = plt.cm.ScalarMappable(cmap=MY_CMAP, norm=plt.Normalize(-1, 1))
cbar = g.fig.colorbar(sm, cax=cax)
cbar.set_ticks([-1, 0, 1])
cbar.outline.set_linewidth(LINE_WIDTH)
cbar.outline.set_edgecolor(LINE_COLOR)
cbar.set_label('Correlation coefficient', rotation=270, labelpad=15, fontsize=9)
cbar.ax.tick_params(labelsize=7, width=LINE_WIDTH, color=LINE_COLOR)

# 6. Legenda Inferior (Recuperada e Colada)
ax_table = g.fig.add_axes([0.1, 0.06, 0.75, 0.12]) 
ax_table.axis('off')

legend_data = [
    ["dV: Vertical Displacement", "Temp: Temperature", "Niv: Reservoir Level"],
    ["Prec: Daily Precipitation", "P_Ac: Accumulated Prec.", "P_An: Annual Acc. Prec."]
]

table = ax_table.table(cellText=legend_data, loc='upper center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.4) 

for (row, col), cell in table.get_celld().items():
    cell.set_linewidth(LINE_WIDTH)
    cell.set_edgecolor(LINE_COLOR)
    cell.get_text().set_fontweight('normal')

# 7. Finalização
plt.savefig("figuras/corr_3.pdf", bbox_inches='tight', pad_inches=0.02)

plt.show()

com alterações no method fill

In [ ]:
# Escolha aqui o que quer analisar: 'dV' ou 'dH'
TARGET_VAR = 'dV'  # ou 'dV'

# grelha
grid_size = 50

N_CLUSTERS_FIXO = 3

DTW_WINDOW_PCT = 0.01  # Aqui podes mudar para 0.1, 0.05, 0.03, etc.

# Tipo de ligação
LINKAGE_METHOD = "average"
# Opções possíveis:
# "complete", "average", "ward"

# ==============================================================================
# SCRIPT COMPLETO: DTW + HIERÁRQUICO (COM ESTILO VISUAL K-MEANS MANTIDO)
# ==============================================================================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
import matplotlib.dates as mdates
from shapely.strtree import STRtree
from statsmodels.tsa.seasonal import seasonal_decompose
from matplotlib.lines import Line2D

# Bibliotecas para DTW e Hierárquico
try:
    from dtaidistance import dtw
    from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
    from scipy.spatial.distance import squareform
    from sklearn.preprocessing import StandardScaler
    import seaborn as sns
    from matplotlib.colors import to_hex
except ImportError as e:
    print(f"Erro de Importação: {e}. Certifique-se de que instalou: dtaidistance, scipy, seaborn, scikit-learn")

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    #barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
    #barragem = cos[cos['COS23_n4_L'] == 'Superfícies silvopastoris de azinheira']
    #barragem = cos[cos['COS23_n4_L'] == 'Matos']
    
    # Podes adicionar ou remover categorias dentro dos parênteses retos []
    barragem = cos[cos['COS23_n4_L'].isin([
    'Infraestruturas de produção de energia hídrica', 
    #'Albufeiras de barragens',
    'Equipamentos culturais',
    'Florestas de azinheira',
    'Matos',
    'Pastagens melhoradas',
    'Rede rodoviária',
    'Superfícies silvopastoris de azinheira'
])]
    
# - Albufeiras de barragens
# - Equipamentos culturais
# - Florestas de azinheira
# - Infraestruturas de produção de energia hídrica
# - Matos
# - Pastagens melhoradas
# - Rede rodoviária
# - Superfícies silvopastoris de azinheira
    
    
except Exception as e:
    print(f"Aviso: COS placeholder. {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs e Filtro
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df): return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) & (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'], value_vars=disp_cols, var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(start=max(asc_long['date'].min(), desc_long['date'].min()), end=min(asc_long['date'].max(), desc_long['date'].max()), freq='MS')

# ==============================
# 3. Interpolação
# ==============================
def interpolate_ps(df, dates):
    dfs = []
    for (x, y), g in df.groupby(['easting','northing']):
        g = g.sort_values('date')
        interp = np.interp(pd.to_datetime(dates).astype(np.int64), g['date'].astype(np.int64), g['disp'])
        dfs.append(pd.DataFrame({'easting': x, 'northing': y, 'latitude': g['latitude'].iloc[0], 'longitude': g['longitude'].iloc[0], 'date': dates, 'disp': interp, 'incidence_angle': g['incidence_angle'].iloc[0], 'track_angle': g['track_angle'].iloc[0]}))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. IDW
# ==============================
def idw(source, target, radius=25, power=2):
    out = []
    for d, src in source.groupby('date'):
        tgt = target[target['date']==d].copy()
        if src.empty or tgt.empty: continue
        tree = cKDTree(list(zip(src['easting'], src['northing'])))
        dist, idx = tree.query(list(zip(tgt['easting'], tgt['northing'])), k=5, distance_upper_bound=radius)
        vals, thetas, alphas = [], [], []
        for d_i, i_i in zip(dist, idx):
            m = np.isfinite(d_i)
            if not np.any(m): vals.append(np.nan); thetas.append(np.nan); alphas.append(np.nan); continue
            w = 1/(d_i[m]**power)
            vals.append(np.sum(w*src.iloc[i_i[m]]['disp'])/np.sum(w))
            thetas.append(np.sum(w*src.iloc[i_i[m]]['incidence_angle'])/np.sum(w))
            alphas.append(np.sum(w*src.iloc[i_i[m]]['track_angle'])/np.sum(w))
        tgt['disp_idw'] = vals; tgt['theta_desc'] = thetas; tgt['alpha_desc'] = alphas
        out.append(tgt)
    return pd.concat(out, ignore_index=True)
asc_interp = idw(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw'])

# ==============================
# 5. dV (ou dH)
# ==============================
orb_inc = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orb_inc) * np.cos(np.deg2rad(asc_interp['latitude'])))

def get_comps(row):
    ta, td, beta = np.deg2rad(row['incidence_angle']), np.deg2rad(row['theta_desc']), row['beta']
    denom = (np.cos(ta)*np.sin(td)*np.cos(beta) + np.cos(td)*np.sin(ta)*np.cos(beta))
    if denom == 0: return np.nan, np.nan
    dV = (row['disp_idw']*np.sin(ta)*np.cos(beta) + row['disp']*np.sin(td)*np.cos(beta))/denom
    dH = (row['disp_idw']*np.cos(ta) - row['disp']*np.cos(td))/denom
    return dV, dH

asc_interp[['dV', 'dH']] = asc_interp.apply(lambda x: pd.Series(get_comps(x)), axis=1)
asc_interp = asc_interp.dropna(subset=['dV', 'dH'])

# ==============================
# 6. Grelha
# ==============================
#grid_size = 100
xe = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
ye = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
xs, ys = xe - grid_size/2, ye - grid_size/2
asc_interp['cx'] = pd.cut(asc_interp['easting'], bins=xs, labels=False)
asc_interp['cy'] = pd.cut(asc_interp['northing'], bins=ys, labels=False)
asc_interp = asc_interp.dropna(subset=['cx','cy'])
asc_interp['cell_id'] = asc_interp['cx'].astype(int).astype(str)+"_"+asc_interp['cy'].astype(int).astype(str)

grid_data = [{'cell_id': f"{ix}_{iy}", 'geometry': box(xs[ix], ys[iy], xs[ix+1], ys[iy+1])} for ix in range(len(xs)-1) for iy in range(len(ys)-1)]
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# Escolha aqui se usa dV ou dH para a agregação
#TARGET_VAR = 'dV' 
agg = asc_interp.groupby(['cell_id','date']).agg(dV=(TARGET_VAR,'mean')).reset_index() # Coluna final chama-se sempre 'dV' para manter compatibilidade com resto do script

# ==============================
# 7. Recorte e Filtro
# ==============================
points_gdf = gpd.GeoDataFrame(asc_interp, geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']), crs="EPSG:3035").to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

poly = grid_recort.geometry.values; ids = grid_recort['cell_id'].values; tree = STRtree(poly)
valid = set()
for pt in points_gdf.geometry:
    for idx in tree.query(pt):
        if poly[idx].contains(pt): valid.add(ids[idx])
grid_barragem = grid_recort[grid_recort['cell_id'].isin(valid)]
agg = agg[agg['cell_id'].isin(valid)]

# ==============================
# 8. CLUSTERING (VERSÃO ROBUSTA CONTRA NaNs)
# ==============================
# print(f">>> A calcular DTW (Flexibilidade: {DTW_WINDOW_PCT})...")

# # 1. Preparar Matriz (Pivot)
# agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV')

# # --- LIMPEZA DE NaNs (Obrigatório para o Linkage não dar erro) ---
# # Interpola pequenos buracos e remove células que ainda tenham falhas críticas
# agg_pivot = agg_pivot.interpolate(axis=1, limit_direction='both').dropna()

# cell_ids = agg_pivot.index.tolist()
# X = agg_pivot.values.astype(float)
# n = X.shape[0]

# if n < 2:
#     print("Aviso: Células insuficientes após limpeza de dados.")
# else:
#     # 2. Normalizar
#     scaler = StandardScaler()
#     X_scaled = scaler.fit_transform(X.T).T 

#     # 3. Calcular DTW com a nova variável
#     # max(1, ...) garante que a janela nunca é zero
#     window_dtw = max(1, int(DTW_WINDOW_PCT * X_scaled.shape[1])) 
    
#     # Cálculo da matriz
#     dist_matrix = dtw.distance_matrix_fast(X_scaled, window=window_dtw)

#     # --- TRATAMENTO PÓS-CÁLCULO ---
#     # Se o DTW gerar algum NaN (acontece se a janela for muito pequena para certas séries)
#     if not np.all(np.isfinite(dist_matrix)):
#         # Substituímos NaNs pelo valor máximo da matriz (para dizer que são muito diferentes)
#         mask_nan = np.isnan(dist_matrix)
#         dist_matrix[mask_nan] = np.nanmax(dist_matrix) if not np.all(np.isnan(dist_matrix)) else 0

#     # 4. Clustering Hierárquico
#     condensed = squareform(dist_matrix)
#     Z = linkage(condensed, method=LINKAGE_METHOD)

#     # 5. Corte Automático (Elbow)
#     last = Z[-10:, 2] # Últimas distâncias de fusão
#     acceleration = np.diff(last, 2)
#     try:
#         k_idx = np.argmax(acceleration) + 2
#         cut_distance = (last[k_idx] + last[k_idx-1]) / 2
#     except:
#         cut_distance = last[len(last)//2]

#     # Labels
#     cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
    
#     # Reajustar labels para começar em 0
#     u_labels = np.unique(cluster_labels)
#     map_l = {old: new for new, old in enumerate(u_labels)}
#     cluster_labels = np.array([map_l[x] for x in cluster_labels])
    
#     num_clusters = len(u_labels)
#     print(f"DTW Concluído. Corte: {cut_distance:.2f}. Clusters: {num_clusters}")


# ==============================
# 8. CLUSTERING (VERSÃO ROBUSTA CONTRA NaNs)
# ==============================
print(f">>> A calcular DTW (Flexibilidade: {DTW_WINDOW_PCT})...")

# 1. Preparar Matriz (Pivot)
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV')

# --- LIMPEZA DE NaNs (Obrigatório para o Linkage não dar erro) ---
# Interpola pequenos buracos e remove células que ainda tenham falhas críticas
agg_pivot = agg_pivot.interpolate(axis=1, limit_direction='both').dropna()

cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

if n < 2:
    print("Aviso: Células insuficientes após limpeza de dados.")
else:
    # 2. ESCOLHA DE NORMALIZAÇÃO
    # Se queres que a magnitude (quanto desce) seja o fator principal, 
    # COMENTA as linhas do scaler abaixo.
    
    # scaler = StandardScaler()
    # X_scaled = scaler.fit_transform(X.T).T 
    
    # Alternativa: Se não normalizares, usa os dados brutos (X)
    X_scaled = X 

    # 3. Calcular DTW
    window_dtw = max(1, int(DTW_WINDOW_PCT * X_scaled.shape[1])) 
    dist_matrix = dtw.distance_matrix_fast(X_scaled, window=window_dtw)

    # ... (Tratamento de NaNs igual) ...

    # 4. Clustering Hierárquico - MUDA O MÉTODO PARA 'WARD'
    # O método 'ward' é muito melhor para criar clusters compactos baseados em magnitude
    condensed = squareform(dist_matrix)
    Z = linkage(condensed, method='ward') 

    # 5. Forçar o número de Clusters
    # O corte automático (Elbow) muitas vezes é conservador demais e dá apenas 2 clusters.
    # Vamos definir um número fixo (ex: 4) para ver se ele isola a tendência descendente.
    # N_CLUSTERS_FIXO = 4
    cluster_labels = fcluster(Z, t=N_CLUSTERS_FIXO, criterion='maxclust')
    
    # Reajustar labels para começar em 0
    u_labels = np.unique(cluster_labels)
    map_l = {old: new for new, old in enumerate(u_labels)}
    cluster_labels = np.array([map_l[x] for x in cluster_labels])
    
    num_clusters = len(u_labels)
    print(f"DTW Concluído. Clusters: {num_clusters} (Método Ward, Sem Normalização)")



# DataFrame Final
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})
# Cores Dinâmicas (para N clusters)
palette = sns.color_palette("Set2", num_clusters) if num_clusters <= 8 else sns.color_palette("tab20", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(range(num_clusters), palette)}

grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='inner')

# ==============================
# FIGURA EXTRA: DENDROGRAMA
# ==============================
# if Z is not None:
#     print("Gerando Dendrograma...")
#     plt.figure(figsize=(12, 5))
#     dendrogram(Z, leaf_rotation=90, leaf_font_size=8,
#                color_threshold=cut_distance
#                )
#     plt.axhline(y=cut_distance, c='k', ls='--', lw=1, label='Corte Automático')
#     plt.title('Dendrograma de Clustering Hierárquico (DTW)')
#     plt.xlabel('Células'); plt.ylabel('Distância')
#     plt.legend(); plt.tight_layout();
    
#     plt.savefig("dendrograma_final.pdf", format='pdf')
    
#     plt.show()

if Z is not None:
    print("Gerando Dendrograma...")
    plt.figure(figsize=(12, 5))
    
    # 1. Recuperar a distância de corte para o número de clusters desejado
    # Se queres N_CLUSTERS_FIXO, o corte acontece entre a fusão N e a fusão N-1
    # Z[-N_CLUSTERS_FIXO + 1, 2] dá a distância exata onde o corte ocorre
    try:
        distancia_visual = Z[-num_clusters + 1, 2]
    except:
        distancia_visual = 0

    # 2. Plot do Dendrograma
    # Usamos color_threshold para pintar os ramos de acordo com os clusters
    dendrogram(Z, 
               leaf_rotation=90, 
               leaf_font_size=8,
               color_threshold=distancia_visual, # Pinta os clusters
               no_labels=True # Se tiveres muitas células, labels ficam ilegíveis
               )
    
    # 3. Linha horizontal para indicar o corte
    plt.axhline(y=distancia_visual, c='k', ls='--', lw=1.2, 
                label=f'Corte para K={num_clusters}')
    
    plt.title(f'Dendrograma de Clustering Hierárquico (DTW) - K={num_clusters}')
    plt.xlabel('Células (Grid 50m)')
    plt.ylabel('Distância (Ward)')
    plt.legend()
    plt.tight_layout()
    
    # 4. Guardar com alta qualidade
    plt.savefig("dendrograma_final.pdf", format='pdf', bbox_inches='tight')
    
    plt.show()



# ==============================
# 9. Dados Hidro (VERSÃO FINAL ANTI-ERRO)
# ==============================
# Garantir que as datas do InSAR estão limpas (apenas data, sem horas)
date_range = pd.to_datetime(agg_pivot.columns).normalize()
win = 13

def process_hidro_variable(df, col_name, is_precip=False):
    try:
        df['data'] = pd.to_datetime(df['data']).dt.normalize()
        df = df.set_index('data')
        
        # Agrupar por mês para garantir que temos um valor por ponto InSAR
        if is_precip:
            s = df[col_name].resample('MS').sum()
        else:
            s = df[col_name].resample('MS').mean()
            
        # O segredo: reindexar e preencher buracos IMEDIATAMENTE
        s = s.reindex(date_range)
        if is_precip:
            s = s.fillna(0)
        else:
            s = s.ffill().bfill().fillna(0) # Tripla segurança
        return s
    except Exception as e:
        print(f"Erro ao processar {col_name}: {e}")
        return pd.Series(0, index=date_range)

# Processar cada uma
ts_temp = process_hidro_variable(pd.read_excel("data/alqueva_temp.xlsx"), 'med')
ts_nivel = process_hidro_variable(pd.read_excel("data/alqueva_nivel.xlsx"), 'nivel')
ts_prec = process_hidro_variable(pd.read_excel("data/prec.xlsx"), 'prec', is_precip=True)

# Criar os DataFrames finais
df_temp = pd.DataFrame({'data': date_range, 'med': ts_temp.values})
df_temp['med_smooth'] = savgol_filter(df_temp['med'], win, 2)

df_nivel = pd.DataFrame({'data': date_range, 'nivel': ts_nivel.values})
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], win, 2)

df_prec = pd.DataFrame({'data': date_range, 'prec': ts_prec.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
df_prec['ano_hidrologico'] = df_prec['data'].apply(lambda x: x.year if x.month>=10 else x.year-1)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções Visuais (IGUAL)
# ==============================
# def scale_vis(d, vmin, vmax, s=0.20, o=0.30):
#     d = np.array(d); mn, mx = np.nanmin(d), np.nanmax(d)
#     span = vmax - vmin
#     norm = np.zeros_like(d) if mx==mn else (d-mn)/(mx-mn)
#     return norm * span * s + (vmax - span * o), mn, mx

# def create_ticks(rmin, rmax, vmin, vmax, s=0.20, o=0.30, n=5):
#     rt = np.linspace(rmin, rmax, n)
#     span = vmax - vmin
#     nt = np.linspace(0,1,n) if rmax==rmin else (rt-rmin)/(rmax-rmin)
#     vt = nt * span * s + (vmax - span * o)
#     return vt, rt


# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks


# ==============================
# FIGURA 1: MAPA (ESTILO ORIGINAL)
# ==============================
print("Gerando Figura 1...")
clusters_present = sorted(cluster_df['cluster'].unique())
k_plot = len(clusters_present)

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', lw=1)

for i in clusters_present:
    sub = grid_sel[grid_sel['cluster'] == i]
    sub.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    cents = sub.copy(); cents.geometry = cents.geometry.centroid
    cents.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

hdl = [plt.Rectangle((0,0),1,1, fc=cluster_colors[i], alpha=0.4) for i in clusters_present]
lbl = [f'Cluster {i+1} ({len(grid_sel[grid_sel["cluster"]==i])} cel)' for i in clusters_present]
hdl.append(Line2D([0],[0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None')); lbl.append('Centróides')
ax_map.legend(hdl, lbl, loc='upper left', fontsize=10, title=f"Clusters DTW (K={k_plot})")
ax_map.set_title(f"Mapa de Clusters ({TARGET_VAR})", fontsize=15)

plt.savefig("mapa.pdf", format='pdf')

plt.show()

# ==============================
# FIGURA 2: SÉRIES TEMPORAIS
# ==============================

print("Gerando Figura 2...")

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5

fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

dV_min = agg[TARGET_VAR].min()
dV_max = agg[TARGET_VAR].max()
dV_margin = (dV_max - dV_min) * 0.1
dV_ylim = (dV_min - dV_margin, dV_max + dV_margin)

# --- Séries visuais ---
temp_visual, _, _ = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, _, _ = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)

# Séries visuais
# tv, tmi, tmx = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
# nivel_visual, nivel_min, nivel_max = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
# prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)


for idx, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)


    # --- Função atualizada ---
    def plot_dV_comparison_series(
        row_idx, ext_data_visual, ext_data_df, ext_col_real,
        ext_label, ext_color,
        plot_as_bar=False, bar_color=None,
        plot_real_scale_line=False,
        combine_annual_prec=False,
        integer_ticks=False, show_background_bars=False):

        ax = fig_series.add_subplot(gs_series[row_idx, idx])

        # --- dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid],
                    color='lightgray', alpha=0.7, linewidth=1.0)

        ax.plot(cluster_data.columns, cluster_mean_dV,
                color=cluster_color, linewidth=1.0,
                label=f'Média Cluster {cluster_id+1}')

        # --- Eixo Y2 ---
        ax2 = ax.twinx()
        lines2 = []

        # -------------------------
        #   CASOS DE PRECIPITAÇÃO
        # -------------------------
        if combine_annual_prec:

            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)

            for ano in df_prec['ano_hidrologico'].unique():
                grupo = df_prec[df_prec['ano_hidrologico'] == ano]
                ax2.plot(grupo['data'], grupo['prec_acum_anual'],
                         color='teal', linewidth=1.0, alpha=0.9)

            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            ax2.set_ylabel("Prec. acumulada anual", color="teal")
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=1.0, label='Prec. acumulada anual (mm)')
            ]

        elif plot_as_bar:

            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15,
                    color=bar_color or ext_color, alpha=0.5)

            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            ax2.set_ylabel(ext_label, color='teal')
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)
            ]

        elif plot_real_scale_line:

            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15,
                        color='teal', alpha=0.3)

            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real],
                     color='teal', linewidth=1.0, alpha=0.85)

            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            ax2.set_ylabel(ext_label, color='teal')
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=1.0, label=ext_label)
            ]

        # -------------------------
        #      OUTRAS SÉRIES
        # -------------------------
        else:

            ax2.plot(ext_data_df['data'], ext_data_visual,
                     color=ext_color, linewidth=1.0, alpha=0.85)

            visual_ticks, real_ticks = create_visual_ticks(
                ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(),
                dV_min, dV_max, scale_factor=0.20, offset_factor=0.30
            )

            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(visual_ticks)

            if integer_ticks:
                ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else:
                ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])

            label_color = 'black' if 'Temperatura' in ext_label else ext_color
            ax2.set_ylabel(ext_label, color=label_color)
            ax2.tick_params(axis='y', colors=label_color)

            lines2 = [ax2.lines[-1]]

        # --- Y1 sempre dV ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)

        if idx == 0:
            ax.set_ylabel('dV (mm)')
        else:
            ax.set_yticklabels([])

        # Y2 somente no último cluster
        if idx != n_clusters - 1:
            ax2.set_yticklabels([])

        ax.tick_params(left=(idx==0))
        ax2.tick_params(right=(idx==n_clusters-1))

        # --- X-axis ---
        if row_idx == n_rows_series - 1:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        else:
            ax.set_xticklabels([])

        # --- Legenda somente no primeiro e no último cluster ---
        if idx == 0 or idx == n_clusters - 1:
            lines1, labels1 = ax.get_legend_handles_labels()
            handles2, labels2 = ax2.get_legend_handles_labels()

            ax.legend(
                lines1 + handles2,
                labels1 + labels2,
                fontsize=8,
                loc='best',
                framealpha=1.0,
                facecolor='white',
                edgecolor='lightgray'
            ).set_zorder(100)

        return ax


    # ---- CHAMADA DAS 5 SÉRIES ----
    plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth',
                              'Temperatura média (°C)', 'black', integer_ticks=True)

    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth',
                              'Nível da albufeira (m)', 'navy', integer_ticks=True)

    plot_dV_comparison_series(2, None, df_prec, 'prec',
                              'Precipitação mensal (mm)', 'teal',
                              plot_as_bar=True, bar_color='teal')

    plot_dV_comparison_series(3, None, df_prec, 'prec_acum',
                              'Prec. acumulada total (mm)', 'teal',
                              plot_real_scale_line=True, show_background_bars=True)

    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual',
                              'Prec. acumulada anual', 'teal',
                              combine_annual_prec=True)


plt.tight_layout()

plt.savefig("clusters.pdf", format='pdf')

plt.show()


# ==============================
# FIGURA 3: DECOMPOSIÇÃO SAZONAL
# ==============================

print("Gerando Figura 3...")

n_rows = 4  # Observed, Trend, Seasonal, Residual
fig_series_decomp, axes = plt.subplots(
    n_rows, n_clusters, figsize=(6 * max(1, n_clusters), 7), sharex=False
)

for col, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    mean_series = cluster_data.mean(axis=0)

    decomposition = seasonal_decompose(mean_series, model="additive", period=12)

    series_list = [
        ("Observed", decomposition.observed),
        ("Trend", decomposition.trend),
        ("Seasonal", decomposition.seasonal),
        ("Residual", decomposition.resid)
    ]

    cluster_color = cluster_colors.get(cluster_id, "black")

    for row, (label, series) in enumerate(series_list):

        ax = axes[row, col]

        # --------------------------
        #   PLOT (agora com cores)
        # --------------------------
        ax.plot(series.index, series.values,
                color=cluster_color, linewidth=1.0)

        # --------------------------
        #  LIMITE EXTERIOR (spines)
        # --------------------------
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.8)
            spine.set_color("black")

        # --------------------------
        #     LIMITES Y
        # --------------------------
        if label != "Observed":
            ymin = np.nanmin(series)
            ymax = np.nanmax(series)
            if np.isnan(ymin) or np.isnan(ymax):
                ymin, ymax = -1, 1
            margin = (ymax - ymin) * 0.10
            ax.set_ylim(ymin - margin, ymax + margin)

        # --------------------------
        #  TÍTULOS DOS SUBPLOTS
        # --------------------------
        if row == 0:
            ax.set_title(f"Seasonal Decompose – Cluster {cluster_id + 1}",
                         fontsize=12)

        if col == 0:
            ax.set_ylabel(label, fontsize=10)
        else:
            ax.set_yticks([])

        # --------------------------
        #   EIXO X — só no último row
        # --------------------------
        if row == n_rows - 1:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.xaxis.set_major_locator(mdates.YearLocator())
            ax.tick_params(axis='x', labelrotation=0, labelsize=10)
        else:
            ax.set_xticks([])
            ax.set_xticklabels([])

plt.tight_layout()

plt.savefig("stl.pdf", format='pdf')

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import numpy as np

# 1. Definições Globais de Estilo
LINE_WIDTH = 0.4
LINE_COLOR = '#333333' 
MY_CMAP = plt.cm.RdBu # RdBu padrão: Azul é Positivo (+1), Vermelho é Negativo (-1)

sns.set_context("paper", font_scale=0.9)
sns.set_style("white")
plt.rcParams['axes.linewidth'] = LINE_WIDTH

# --- Dados (Certifica-te que estas variáveis existem no teu ambiente) ---
CLUSTER_ALVO_ID = 0 
c_cells = cluster_df[cluster_df['cluster'] == CLUSTER_ALVO_ID]['cell_id']
dV_mean = agg_pivot.loc[c_cells].mean()
df_plot = pd.DataFrame({
    'dV': dV_mean,
    'Temp': df_temp.set_index('data')['med_smooth'].reindex(dV_mean.index),
    'Niv': df_nivel.set_index('data')['nivel_smooth'].reindex(dV_mean.index),
    'Prec': df_prec.set_index('data')['prec'].reindex(dV_mean.index),
    'P_Ac': df_prec.set_index('data')['prec_acum'].reindex(dV_mean.index),
    'P_An': df_prec.set_index('data')['prec_acum_anual'].reindex(dV_mean.index)
}).dropna()

cols = df_plot.columns.tolist()
n_vars = len(cols)

# 2. Criar o PairGrid
g = sns.PairGrid(df_plot, vars=cols, diag_sharey=False, height=1.0, aspect=1.0)

# --- Funções de Desenho ---
def cor_func(x, y, **kwargs):
    r, p = pearsonr(x, y)
    ax = plt.gca()
    # Mapeamento: r=1 -> Azul, r=-1 -> Vermelho
    color_val = (r + 1) / 2
    facecolor = MY_CMAP(color_val)
    ax.set_facecolor((*facecolor[:3], 0.5))
    ax.annotate(f"{r:.2f}", xy=(0.5, 0.5), xycoords=ax.transAxes, 
                ha='center', va='center', fontsize=8, fontweight='normal')

g.map_diag(sns.histplot, kde=True, color="#2c3e50", alpha=0.2, edgecolor='white', linewidth=0.3, line_kws={'linewidth': 0.7})
# --- PARTE INFERIOR: Scatter Plots ---
g.map_lower(sns.regplot, ci=None, 
            scatter_kws={
                's': 3,                # Tamanho do ponto
                'alpha': 1.0,          # Opacidade total (sem transparência)
                'color': 'black',      # Cor base
                'facecolor': 'black',  # Preenchimento preto sólido
                'edgecolor': 'black',  # Contorno preto sólido
                'linewidths': 0        # Remove qualquer largura de linha de bordo para evitar reflexos
            }, 
            line_kws={'color': 'red', 'linewidth': 0.7})
g.map_upper(cor_func)

# 3. Uniformização Total das Linhas e Padding
for i in range(n_vars):
    for j in range(n_vars):
        ax = g.axes[i, j]
        ax.set_xlabel(""); ax.set_ylabel("")
        
        # Ajuste de Padding interno (Respiro de 25%)
        if i != j:
            x_min, x_max = df_plot[cols[j]].min(), df_plot[cols[j]].max()
            y_min, y_max = df_plot[cols[i]].min(), df_plot[cols[i]].max()
            x_range, y_range = x_max - x_min, y_max - y_min
            ax.set_xlim(x_min - 0.3 * x_range, x_max + 0.3 * x_range)
            ax.set_ylim(y_min - 0.3 * y_range, y_max + 0.3 * y_range)

        # Labels apenas nas extremidades
        if j != 0: ax.set_yticklabels([])
        if i != n_vars - 1: ax.set_xticklabels([])
        
        ax.tick_params(labelsize=6, direction='in', pad=1, width=LINE_WIDTH, color=LINE_COLOR)
        
        # Forçar todas as linhas da grelha
        for edge in ['top', 'bottom', 'left', 'right']:
            ax.spines[edge].set_visible(True)
            ax.spines[edge].set_linewidth(LINE_WIDTH)
            ax.spines[edge].set_color(LINE_COLOR)
        
        # Identificação na Diagonal em Vermelho (Sem Bold)
        if i == j:
            ax.annotate(cols[i], xy=(0.05, 0.90), xycoords='axes fraction', 
                        ha='left', va='top', fontsize=8, fontweight='normal', color='red',
                        bbox=dict(facecolor='white', alpha=0.4, edgecolor='none', pad=0))

# 4. Ajuste de Layout para Colagem
plt.subplots_adjust(hspace=0, wspace=0, left=0.1, right=0.85, bottom=0.18, top=0.95)

# 5. Colorbar Corrigida (Azul=Positivo, Vermelho=Negativo)
cax = g.fig.add_axes([0.87, 0.18, 0.02, 0.77]) 
sm = plt.cm.ScalarMappable(cmap=MY_CMAP, norm=plt.Normalize(-1, 1))
cbar = g.fig.colorbar(sm, cax=cax)
cbar.set_ticks([-1, 0, 1])
cbar.outline.set_linewidth(LINE_WIDTH)
cbar.outline.set_edgecolor(LINE_COLOR)
cbar.set_label('Correlation coefficient', rotation=270, labelpad=15, fontsize=9)
cbar.ax.tick_params(labelsize=7, width=LINE_WIDTH, color=LINE_COLOR)

# 6. Legenda Inferior (Recuperada e Colada)
ax_table = g.fig.add_axes([0.1, 0.06, 0.75, 0.12]) 
ax_table.axis('off')

legend_data = [
    ["dV: Vertical Displacement", "Temp: Temperature", "Niv: Reservoir Level"],
    ["Prec: Daily Precipitation", "P_Ac: Accumulated Prec.", "P_An: Annual Acc. Prec."]
]

table = ax_table.table(cellText=legend_data, loc='upper center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.4) 

for (row, col), cell in table.get_celld().items():
    cell.set_linewidth(LINE_WIDTH)
    cell.set_edgecolor(LINE_COLOR)
    cell.get_text().set_fontweight('normal')

# 7. Finalização
plt.savefig("Matriz_Correlacao_Final_Final.png", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================
# FIGURA 4: CORRELAÇÕES COM LAG
# ==============================

print("Gerando Figura 4 (Correlações)...")

from scipy.stats import pearsonr

MAX_LAG = 6  # meses

# --- Variáveis externas ---
external_vars = {
    'Temperatura (°C)':    (df_temp.set_index('data')['med_smooth'], 'black'),
    'Nível albufeira (m)': (df_nivel.set_index('data')['nivel_smooth'], 'navy'),
    'Precipitação (mm)':   (df_prec.set_index('data')['prec'], 'teal'),
}

lags = list(range(0, MAX_LAG + 1))
date_index = agg_pivot.columns  # DatetimeIndex

# Figura: 1 coluna por cluster, 2 rows (heatmap + barras)
n_rows_corr = 2
fig_corr, axes_corr = plt.subplots(
    n_rows_corr, n_clusters,
    figsize=(5.5 * n_clusters, 8),
    gridspec_kw={'height_ratios': [1.2, 1]}
)
if n_clusters == 1:
    axes_corr = axes_corr.reshape(n_rows_corr, 1)

for col, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data  = agg_pivot.loc[cluster_cells]
    mean_series   = cluster_data.mean(axis=0)        # Series com DatetimeIndex
    cluster_color = cluster_colors.get(cluster_id, 'gray')

    # ---- Calcular matriz r[variavel, lag] ----
    var_names = list(external_vars.keys())
    r_matrix  = np.full((len(var_names), len(lags)), np.nan)
    p_matrix  = np.full((len(var_names), len(lags)), np.nan)

    for vi, (vname, (vseries, _)) in enumerate(external_vars.items()):
        # Alinhar ao mesmo índice
        combined = pd.DataFrame({
            'dv':  mean_series,
            'ext': vseries
        }).dropna()

        for li, lag in enumerate(lags):
            if lag == 0:
                x = combined['ext'].values
                y = combined['dv'].values
            else:
                # Lag positivo: variável externa adiantada lag meses
                x = combined['ext'].iloc[:-lag].values
                y = combined['dv'].iloc[lag:].values

            if len(x) > 5:
                r, p = pearsonr(x, y)
                r_matrix[vi, li] = r
                p_matrix[vi, li] = p

    # ---- ROW 0: Heatmap ----
    ax_heat = axes_corr[0, col]
    im = ax_heat.imshow(
        r_matrix, aspect='auto', cmap='RdBu_r',
        vmin=-1, vmax=1,
        interpolation='nearest'
    )
    ax_heat.set_xticks(range(len(lags)))
    ax_heat.set_xticklabels([f'lag {l}' for l in lags], fontsize=8, rotation=45, ha='right')
    ax_heat.set_yticks(range(len(var_names)))
    ax_heat.set_yticklabels(var_names if col == 0 else [], fontsize=8)
    ax_heat.set_title(f'Cluster {cluster_id + 1}', fontsize=11, color=cluster_color)

    # Anotar r e marcar p<0.05
    for vi in range(len(var_names)):
        for li in range(len(lags)):
            r_val = r_matrix[vi, li]
            p_val = p_matrix[vi, li]
            if not np.isnan(r_val):
                txt = f'{r_val:.2f}'
                txt_color = 'white' if abs(r_val) > 0.55 else 'black'
                ax_heat.text(li, vi, txt, ha='center', va='center',
                             fontsize=7.5, color=txt_color, fontweight='normal')
                if not np.isnan(p_val) and p_val < 0.05:
                    ax_heat.text(li + 0.35, vi - 0.35, '*', ha='center',
                                 va='center', fontsize=9, color=txt_color)

    plt.colorbar(im, ax=ax_heat, fraction=0.046, pad=0.04, label='r' if col == n_clusters - 1 else '')

    # ---- ROW 1: Barras no lag de máxima correlação ----
    ax_bar = axes_corr[1, col]

    best_r    = []
    best_lag  = []
    colors_bar = []

    for vi, (vname, (_, vcolor)) in enumerate(external_vars.items()):
        row = r_matrix[vi, :]
        if np.all(np.isnan(row)):
            best_r.append(0); best_lag.append(0); colors_bar.append(vcolor)
            continue
        best_idx = np.nanargmax(np.abs(row))
        best_r.append(row[best_idx])
        best_lag.append(lags[best_idx])
        colors_bar.append(vcolor)

    x_pos = np.arange(len(var_names))
    bars = ax_bar.bar(x_pos, best_r, color=colors_bar, alpha=0.7, edgecolor='black', linewidth=0.5)

    # Anotar lag
    for bar_i, (bar, lag_val, r_val) in enumerate(zip(bars, best_lag, best_r)):
        if not np.isnan(r_val):
            yoff = 0.03 if r_val >= 0 else -0.06
            ax_bar.text(bar.get_x() + bar.get_width()/2,
                        r_val + yoff,
                        f'lag {lag_val}',
                        ha='center', va='bottom', fontsize=7.5)

    ax_bar.axhline(0, color='black', linewidth=0.8)
    ax_bar.set_ylim(-1.1, 1.1)
    ax_bar.set_xticks(x_pos)
    ax_bar.set_xticklabels(
        [v.split('(')[0].strip() for v in var_names],
        fontsize=8, rotation=20, ha='right'
    )
    if col == 0:
        ax_bar.set_ylabel('r (lag ótimo)', fontsize=9)
    else:
        ax_bar.set_yticklabels([])
    ax_bar.set_title('Correlação máxima (|r|)', fontsize=9)

    # Linha de significância indicativa
    ax_bar.axhline( 0.3, color='gray', linewidth=0.7, linestyle='--', alpha=0.6)
    ax_bar.axhline(-0.3, color='gray', linewidth=0.7, linestyle='--', alpha=0.6)


fig_corr.suptitle(
    f'Correlações com lag: {TARGET_VAR} por cluster vs. variáveis externas',
    fontsize=13, y=1.01
)
plt.tight_layout()
plt.show()


# ---- Tabela resumo no terminal ----
print(f"\n{'='*60}")
print(f"RESUMO DE CORRELAÇÕES — {TARGET_VAR}")
print(f"{'='*60}")
print(f"{'Cluster':<10} {'Variável':<28} {'r':>6} {'p':>8} {'lag':>5}")
print(f"{'-'*60}")

for cluster_id in clusters_present:
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    mean_series = agg_pivot.loc[cluster_cells].mean(axis=0)

    for vname, (vseries, _) in external_vars.items():
        combined = pd.DataFrame({'dv': mean_series, 'ext': vseries}).dropna()
        best = {'r': np.nan, 'p': np.nan, 'lag': 0}

        for lag in lags:
            if lag == 0:
                x, y = combined['ext'].values, combined['dv'].values
            else:
                x = combined['ext'].iloc[:-lag].values
                y = combined['dv'].iloc[lag:].values
            if len(x) > 5:
                r, p = pearsonr(x, y)
                if np.isnan(best['r']) or abs(r) > abs(best['r']):
                    best = {'r': r, 'p': p, 'lag': lag}

        sig = '*' if (not np.isnan(best['p']) and best['p'] < 0.05) else ' '
        print(f"{f'Cluster {cluster_id+1}':<10} {vname:<28} {best['r']:>6.3f} {best['p']:>8.4f}{sig} {best['lag']:>4}")

print(f"{'='*60}")
print("* p < 0.05")

In [ ]:
# ==============================
# FIGURA 4: CORRELAÇÃO DE PEARSON (lag 0)
# ==============================

print("Gerando Figura 4 (Correlações Pearson)...")

from scipy.stats import pearsonr

external_vars = {
    'Temperatura (°C)':    (df_temp.set_index('data')['med_smooth'],        'black'),
    'Nível albufeira (m)': (df_nivel.set_index('data')['nivel_smooth'],     'navy'),
    'Precipitação (mm)':   (df_prec.set_index('data')['prec'],              'teal'),
}

fig_corr, axes_corr = plt.subplots(1, n_clusters, figsize=(4.5 * n_clusters, 5), sharey=True)
if n_clusters == 1:
    axes_corr = [axes_corr]

for col, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    mean_series   = agg_pivot.loc[cluster_cells].mean(axis=0)
    cluster_color = cluster_colors.get(cluster_id, 'gray')
    ax = axes_corr[col]

    var_names, r_vals, p_vals, bar_colors = [], [], [], []

    for vname, (vseries, vcolor) in external_vars.items():
        combined = pd.DataFrame({'dv': mean_series, 'ext': vseries}).dropna()
        if len(combined) > 5:
            r, p = pearsonr(combined['ext'], combined['dv'])
        else:
            r, p = np.nan, np.nan
        var_names.append(vname)
        r_vals.append(r)
        p_vals.append(p)
        bar_colors.append(vcolor)

    x_pos = np.arange(len(var_names))
    bars = ax.bar(x_pos, r_vals, color=bar_colors, alpha=0.7,
                  edgecolor='black', linewidth=0.6)

    # Anotar r e significância
    for i, (r, p) in enumerate(zip(r_vals, p_vals)):
        if np.isnan(r):
            continue
        sig = '*' if p < 0.05 else ''
        yoff = 0.03 if r >= 0 else -0.07
        ax.text(i, r + yoff, f'{r:.2f}{sig}',
                ha='center', va='bottom', fontsize=9)

    ax.axhline(0,    color='black', linewidth=0.8)
    ax.axhline( 0.3, color='gray',  linewidth=0.7, linestyle='--', alpha=0.5)
    ax.axhline(-0.3, color='gray',  linewidth=0.7, linestyle='--', alpha=0.5)
    ax.set_ylim(-1.15, 1.15)
    ax.set_xticks(x_pos)
    ax.set_xticklabels([v.split('(')[0].strip() for v in var_names],
                       fontsize=9, rotation=20, ha='right')
    ax.set_title(f'Cluster {cluster_id + 1}', fontsize=11, color=cluster_color)
    if col == 0:
        ax.set_ylabel('r de Pearson', fontsize=10)

fig_corr.suptitle(f'Correlação de Pearson: {TARGET_VAR} vs. variáveis externas',
                  fontsize=13)
plt.tight_layout()
plt.show()

# Tabela no terminal
print(f"\n{'='*55}")
print(f"{'Cluster':<10} {'Variável':<28} {'r':>6} {'p':>8}")
print(f"{'-'*55}")
for cluster_id in clusters_present:
    mean_series = agg_pivot.loc[
        cluster_df[cluster_df['cluster'] == cluster_id]['cell_id']
    ].mean(axis=0)
    for vname, (vseries, _) in external_vars.items():
        combined = pd.DataFrame({'dv': mean_series, 'ext': vseries}).dropna()
        if len(combined) > 5:
            r, p = pearsonr(combined['ext'], combined['dv'])
            sig = '*' if p < 0.05 else ' '
            print(f"{'Cluster '+str(cluster_id+1):<10} {vname:<28} {r:>6.3f} {p:>8.4f}{sig}")
print(f"{'='*55}\nLegenda: * p < 0.05")

In [ ]:
# ==============================
# FIGURA 5: MAPAS INDIVIDUAIS DE CORRELAÇÃO DE PEARSON
# ==============================

from scipy.stats import pearsonr
import matplotlib.colors as mcolors

# Definir as variáveis externas
external_vars_map = {
    'Temperatura (°C)':    df_temp.set_index('data')['med_smooth'],
    'Nível albufeira (m)': df_nivel.set_index('data')['nivel_smooth'],
    'Precipitação (mm)':   df_prec.set_index('data')['prec'],
}

# Configurações Visuais
# RdBu_r: Vermelho para valores negativos, Azul para positivos
cmap_corr = plt.cm.RdBu_r 
norm_corr = mcolors.Normalize(vmin=-1, vmax=1)

# Calcular extensão da área (EPSG:3857)
grid_extent = grid_barragem.to_crs(epsg=3857).total_bounds
margin = 50 
xlim = (grid_extent[0] - margin, grid_extent[2] + margin)
ylim = (grid_extent[1] - margin, grid_extent[3] + margin)

for vname, vseries in external_vars_map.items():
    print(f"Gerando mapa para: {vname}...")
    
    # Criar figura individual
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # --- 1. Calcular r por célula ---
    records = []
    for cell_id in valid:
        cell_ts = agg[agg['cell_id'] == cell_id].set_index('date')['dV']
        combined = pd.DataFrame({'dv': cell_ts, 'ext': vseries}).dropna()
        if len(combined) > 5:
            r, p = pearsonr(combined['ext'], combined['dv'])
        else:
            r, p = np.nan, np.nan
        records.append({'cell_id': cell_id, 'r': r, 'p': p})

    corr_df = pd.DataFrame(records)
    grid_corr = grid_barragem.merge(corr_df, on='cell_id', how='left').to_crs(epsg=3857)

    # --- 2. Preparar o Eixo ---
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, crs='EPSG:3857', reset_extent=False)

    # --- 3. Desenhar Células ---
    grid_corr.dropna(subset=['r']).plot(
        column='r', ax=ax,
        cmap=cmap_corr, norm=norm_corr,
        alpha=0.75, edgecolor='black', linewidth=0.5
    )

    # --- 4. Anotar Valores (r) ---
    for _, row in grid_corr.dropna(subset=['r']).iterrows():
        cx = row.geometry.centroid.x
        cy = row.geometry.centroid.y
        r_val, p_val = row['r'], row['p']
        sig = '*' if (not np.isnan(p_val) and p_val < 0.05) else ''
        
        # Cor do texto adaptativa para legibilidade
        txt_color = 'white' if abs(r_val) > 0.6 else 'black'
        ax.text(cx, cy, f'{r_val:.2f}{sig}',
                ha='center', va='center', fontsize=7,
                color=txt_color, fontweight='bold')

    # --- 5. Títulos e Estética ---
    ax.set_axis_off()
    ax.set_title(f'Pearson Correlation: {TARGET_VAR} vs {vname}\n(* p < 0.05)', 
                 fontsize=14, pad=15, fontweight='bold')

    # --- 6. Colorbar Lateral (Ajustada à altura da figura) ---
    sm = plt.cm.ScalarMappable(cmap=cmap_corr, norm=norm_corr)
    sm.set_array([])
    
    # fraction e pad controlam a posição; ticks=[-1, 1] limpa a escala
    cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04, ticks=[-1, 1])
    cbar.set_label('r (Pearson)', fontsize=12, labelpad=-15)
    cbar.outline.set_linewidth(0.4)
    
    # Ajustar labels da colorbar
    cbar.ax.set_yticklabels(['-1 (Negativo)', '1 (Positivo)'], fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()

In [ ]:
# ==============================
# FIGURA 5: MAPAS DE CORRELAÇÃO DE PEARSON POR CÉLULA
# ==============================

print("Gerando Figura 5 (Mapas de Correlação por Célula)...")

from scipy.stats import pearsonr
import matplotlib.colors as mcolors

external_vars_map = {
    'Temperatura (°C)':    df_temp.set_index('data')['med_smooth'],
    'Nível albufeira (m)': df_nivel.set_index('data')['nivel_smooth'],
    'Precipitação (mm)':   df_prec.set_index('data')['prec'],
}

var_list = list(external_vars_map.keys())
n_vars   = len(var_list)

fig_map_corr, axes_map = plt.subplots(1, n_vars, figsize=(7 * n_vars, 8))
if n_vars == 1:
    axes_map = [axes_map]

cmap_corr = plt.cm.RdBu_r
norm_corr  = mcolors.Normalize(vmin=-1, vmax=1)

# --- Calcular extensão da área uma vez ---
grid_extent = grid_barragem.to_crs(epsg=3857).total_bounds  # [minx, miny, maxx, maxy]
margin = 100  # metros em 3857
xlim = (grid_extent[0] - margin, grid_extent[2] + margin)
ylim = (grid_extent[1] - margin, grid_extent[3] + margin)

for vi, (vname, vseries) in enumerate(external_vars_map.items()):
    ax = axes_map[vi]

    # --- Calcular r por célula ---
    records = []
    for cell_id in valid:
        cell_ts = agg[agg['cell_id'] == cell_id].set_index('date')['dV']
        combined = pd.DataFrame({'dv': cell_ts, 'ext': vseries}).dropna()
        if len(combined) > 5:
            r, p = pearsonr(combined['ext'], combined['dv'])
        else:
            r, p = np.nan, np.nan
        records.append({'cell_id': cell_id, 'r': r, 'p': p})

    corr_df   = pd.DataFrame(records)
    grid_corr = grid_barragem.merge(corr_df, on='cell_id', how='left').to_crs(epsg=3857)

    # --- Definir limites ANTES do basemap ---
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    # --- Basemap ---
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery,
                    crs='EPSG:3857', reset_extent=False)

    # --- Pintar células ---
    grid_corr.dropna(subset=['r']).plot(
        column='r', ax=ax,
        cmap=cmap_corr, norm=norm_corr,
        alpha=0.75, edgecolor='black', linewidth=0.4
    )

    # --- Anotar r ---
    for _, row in grid_corr.dropna(subset=['r']).iterrows():
        cx_map = row.geometry.centroid.x
        cy_map = row.geometry.centroid.y
        r_val, p_val = row['r'], row['p']
        sig       = '*' if (not np.isnan(p_val) and p_val < 0.05) else ''
        txt_color = 'white' if abs(r_val) > 0.5 else 'black'
        ax.text(cx_map, cy_map, f'{r_val:.2f}{sig}',
                ha='center', va='center', fontsize=6.5,
                color=txt_color, fontweight='bold')

    ax.set_xlim(xlim)  # repor depois do plot (geopandas pode resetar)
    ax.set_ylim(ylim)
    ax.set_axis_off()
    ax.set_title(f'{TARGET_VAR} × {vname}', fontsize=12, pad=8)

# --- Colorbar ---
sm = plt.cm.ScalarMappable(cmap=cmap_corr, norm=norm_corr)
sm.set_array([])
cbar = fig_map_corr.colorbar(sm, ax=axes_map, fraction=0.015, pad=0.02)
cbar.set_label('r de Pearson', fontsize=11)
for ref in [-0.5, 0, 0.5]:
    cbar.ax.axhline((ref + 1) / 2, color='gray', linewidth=0.8, linestyle='--')

fig_map_corr.suptitle(
    f'Correlação de Pearson por célula — {TARGET_VAR} vs. variáveis externas\n(* p < 0.05)',
    fontsize=14, y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# FIGURA 5: MAPAS DE CORRELAÇÃO DE PEARSON POR CÉLULA
# (linhas = variáveis, colunas = clusters)
# ==============================

print("Gerando Figura 5 (Mapas de Correlação por Célula)...")

from scipy.stats import pearsonr
import matplotlib.colors as mcolors

# --- Variáveis externas (5) ---
external_vars_map = {
    'Temperatura (°C)':         df_temp.set_index('data')['med_smooth'],
    'Nível albufeira (m)':      df_nivel.set_index('data')['nivel_smooth'],
    'Precipitação mensal (mm)': df_prec.set_index('data')['prec'],
    'Prec. acumulada total':    df_prec.set_index('data')['prec_acum'],
    'Prec. acumulada anual':    df_prec.set_index('data')['prec_acum_anual'],
}

var_list   = list(external_vars_map.keys())
n_vars     = len(var_list)       # 5 linhas
n_cols_fig = n_clusters          # colunas = clusters

cmap_corr = plt.cm.RdBu_r
norm_corr = mcolors.Normalize(vmin=-1, vmax=1)

fig_map_corr, axes_map = plt.subplots(
    n_vars, n_cols_fig,
    figsize=(5.5 * n_cols_fig, 5.0 * n_vars)
)

# Garantir sempre array 2D
if n_vars == 1 and n_cols_fig == 1:
    axes_map = np.array([[axes_map]])
elif n_vars == 1:
    axes_map = axes_map[np.newaxis, :]
elif n_cols_fig == 1:
    axes_map = axes_map[:, np.newaxis]

# --- Extensão do mapa ---
grid_extent = grid_barragem.to_crs(epsg=3857).total_bounds
margin = 100
xlim = (grid_extent[0] - margin, grid_extent[2] + margin)
ylim = (grid_extent[1] - margin, grid_extent[3] + margin)

# --- Pré-calcular r para todas as combinações (célula × variável) ---
# Evita recalcular o mesmo cell_id múltiplas vezes
print("  A calcular correlações por célula...")

# dict: vname -> DataFrame com cell_id, r, p
corr_by_var = {}
for vname, vseries in external_vars_map.items():
    records = []
    for cell_id in valid:
        cell_ts  = agg[agg['cell_id'] == cell_id].set_index('date')['dV']
        combined = pd.DataFrame({'dv': cell_ts, 'ext': vseries}).dropna()
        if len(combined) > 5:
            r, p = pearsonr(combined['ext'], combined['dv'])
        else:
            r, p = np.nan, np.nan
        records.append({'cell_id': cell_id, 'r': r, 'p': p})
    corr_by_var[vname] = pd.DataFrame(records)

# --- Plot ---
for vi, vname in enumerate(var_list):
    corr_df = corr_by_var[vname]

    for ci, cluster_id in enumerate(clusters_present):

        ax = axes_map[vi, ci]

        # Células deste cluster
        cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
        corr_cluster  = corr_df[corr_df['cell_id'].isin(cluster_cells)]

        # Merge com geometria (só células do cluster)
        grid_cluster = (
            grid_barragem[grid_barragem['cell_id'].isin(cluster_cells)]
            .merge(corr_cluster, on='cell_id', how='left')
            .to_crs(epsg=3857)
        )

        # Limites antes do basemap
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)

        # Basemap
        ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery,
                        crs='EPSG:3857', reset_extent=False)

        # Todas as células do mapa (cinza claro) para contexto
        grid_barragem.to_crs(epsg=3857).plot(
            ax=ax, color='none', edgecolor='white',
            linewidth=0.3, alpha=0.4
        )

        # Células do cluster coloridas por r
        grid_valid = grid_cluster.dropna(subset=['r'])
        if not grid_valid.empty:
            grid_valid.plot(
                column='r', ax=ax,
                cmap=cmap_corr, norm=norm_corr,
                alpha=0.80, edgecolor='black', linewidth=0.5
            )

        # Contorno dourado para |r| > 0.5
        edge_color = 'gold' if abs(r_val) >= 0.5 else 'black'
        line_width  = 2.0   if abs(r_val) >= 0.5 else 0.5
        alpha_val   = 1.0   if p_val < 0.05 else 0.25  # máscara para não-significativo
        font_sz     = 6 + abs(r_val) * 5               # texto proporcional

        # Anotar r dentro de cada célula
        for _, row in grid_valid.iterrows():
            r_val, p_val = row['r'], row['p']
            if np.isnan(r_val):
                continue
            sig       = '*' if (not np.isnan(p_val) and p_val < 0.05) else ''
            txt_color = 'white' if abs(r_val) > 0.5 else 'black'
            ax.text(
                row.geometry.centroid.x,
                row.geometry.centroid.y,
                f'{r_val:.2f}{sig}',
                ha='center', va='center',
                fontsize=6.5, color=txt_color, fontweight='bold'
            )

        # Repor limites (geopandas reseta)
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_axis_off()

        # Títulos: coluna no topo, linha à esquerda
        if vi == 0:
            cluster_color = cluster_colors.get(cluster_id, 'black')
            ax.set_title(f'Cluster {cluster_id + 1}', fontsize=11,
                         color=cluster_color, pad=6)
        if ci == 0:
            ax.set_ylabel(vname, fontsize=9, labelpad=6)
            # ylabel não aparece com set_axis_off — usar texto
            ax.text(-0.04, 0.5, vname,
                    transform=ax.transAxes,
                    fontsize=8.5, va='center', ha='right',
                    rotation=90, color='black')

# --- Colorbar partilhada ---
sm = plt.cm.ScalarMappable(cmap=cmap_corr, norm=norm_corr)
sm.set_array([])
cbar = fig_map_corr.colorbar(
    sm, ax=axes_map,
    fraction=0.012, pad=0.02,
    orientation='vertical'
)
cbar.set_label('r de Pearson', fontsize=11)
cbar.ax.tick_params(labelsize=9)
for ref in [-0.5, 0, 0.5]:
    cbar.ax.axhline((ref + 1) / 2, color='gray', linewidth=0.8, linestyle='--')

fig_map_corr.suptitle(
    f'Correlação de Pearson por célula — {TARGET_VAR} vs. variáveis externas\n(* p < 0.05)',
    fontsize=14
)
plt.tight_layout()
plt.show()

In [ ]:
# Escolha aqui o que quer analisar: 'dV' ou 'dH'
TARGET_VAR = 'dV'  # ou 'dV'

# grelha
grid_size = 50

DTW_WINDOW_PCT = 0  # Aqui podes mudar para 0.1, 0.05, 0.03, etc.

# Tipo de ligação
LINKAGE_METHOD = "average"
# Opções possíveis:
# "complete", "average", "ward"

# ==============================================================================
# SCRIPT COMPLETO: DTW + HIERÁRQUICO (COM ESTILO VISUAL K-MEANS MANTIDO)
# ==============================================================================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
import matplotlib.dates as mdates
from shapely.strtree import STRtree
from statsmodels.tsa.seasonal import seasonal_decompose
from matplotlib.lines import Line2D

# Bibliotecas para DTW e Hierárquico
try:
    from dtaidistance import dtw
    from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
    from scipy.spatial.distance import squareform
    from sklearn.preprocessing import StandardScaler
    import seaborn as sns
    from matplotlib.colors import to_hex
except ImportError as e:
    print(f"Erro de Importação: {e}. Certifique-se de que instalou: dtaidistance, scipy, seaborn, scikit-learn")

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    #barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
    #barragem = cos[cos['COS23_n4_L'] == 'Superfícies silvopastoris de azinheira']
    #barragem = cos[cos['COS23_n4_L'] == 'Matos']
    
    # Podes adicionar ou remover categorias dentro dos parênteses retos []
    barragem = cos[cos['COS23_n4_L'].isin([
    'Infraestruturas de produção de energia hídrica', 
    #'Albufeiras de barragens',
    'Equipamentos culturais',
    'Florestas de azinheira',
    'Matos',
    'Pastagens melhoradas',
    'Rede rodoviária',
    'Superfícies silvopastoris de azinheira'
])]
    
# - Albufeiras de barragens
# - Equipamentos culturais
# - Florestas de azinheira
# - Infraestruturas de produção de energia hídrica
# - Matos
# - Pastagens melhoradas
# - Rede rodoviária
# - Superfícies silvopastoris de azinheira
    
    
except Exception as e:
    print(f"Aviso: COS placeholder. {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs e Filtro
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df): return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) & (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'], value_vars=disp_cols, var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(start=max(asc_long['date'].min(), desc_long['date'].min()), end=min(asc_long['date'].max(), desc_long['date'].max()), freq='MS')

# ==============================
# 3. Interpolação
# ==============================
def interpolate_ps(df, dates):
    dfs = []
    for (x, y), g in df.groupby(['easting','northing']):
        g = g.sort_values('date')
        interp = np.interp(pd.to_datetime(dates).astype(np.int64), g['date'].astype(np.int64), g['disp'])
        dfs.append(pd.DataFrame({'easting': x, 'northing': y, 'latitude': g['latitude'].iloc[0], 'longitude': g['longitude'].iloc[0], 'date': dates, 'disp': interp, 'incidence_angle': g['incidence_angle'].iloc[0], 'track_angle': g['track_angle'].iloc[0]}))
    return pd.concat(dfs, ignore_index=True)
asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. IDW
# ==============================
def idw(source, target, radius=50, power=2):
    out = []
    for d, src in source.groupby('date'):
        tgt = target[target['date']==d].copy()
        if src.empty or tgt.empty: continue
        tree = cKDTree(list(zip(src['easting'], src['northing'])))
        dist, idx = tree.query(list(zip(tgt['easting'], tgt['northing'])), k=3, distance_upper_bound=radius)
        vals, thetas, alphas = [], [], []
        for d_i, i_i in zip(dist, idx):
            m = np.isfinite(d_i)
            if not np.any(m): vals.append(np.nan); thetas.append(np.nan); alphas.append(np.nan); continue
            w = 1/(d_i[m]**power)
            vals.append(np.sum(w*src.iloc[i_i[m]]['disp'])/np.sum(w))
            thetas.append(np.sum(w*src.iloc[i_i[m]]['incidence_angle'])/np.sum(w))
            alphas.append(np.sum(w*src.iloc[i_i[m]]['track_angle'])/np.sum(w))
        tgt['disp_idw'] = vals; tgt['theta_desc'] = thetas; tgt['alpha_desc'] = alphas
        out.append(tgt)
    return pd.concat(out, ignore_index=True)
asc_interp = idw(desc_interp, asc_interp)
asc_interp = asc_interp.dropna(subset=['disp_idw'])

# ==============================
# 5. dV (ou dH)
# ==============================
orb_inc = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orb_inc) * np.cos(np.deg2rad(asc_interp['latitude'])))

def get_comps(row):
    ta, td, beta = np.deg2rad(row['incidence_angle']), np.deg2rad(row['theta_desc']), row['beta']
    denom = (np.cos(ta)*np.sin(td)*np.cos(beta) + np.cos(td)*np.sin(ta)*np.cos(beta))
    if denom == 0: return np.nan, np.nan
    dV = (row['disp_idw']*np.sin(ta)*np.cos(beta) + row['disp']*np.sin(td)*np.cos(beta))/denom
    dH = (row['disp_idw']*np.cos(ta) - row['disp']*np.cos(td))/denom
    return dV, dH

asc_interp[['dV', 'dH']] = asc_interp.apply(lambda x: pd.Series(get_comps(x)), axis=1)
asc_interp = asc_interp.dropna(subset=['dV', 'dH'])

# ==============================
# 6. Grelha
# ==============================
#grid_size = 100
xe = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
ye = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
xs, ys = xe - grid_size/2, ye - grid_size/2
asc_interp['cx'] = pd.cut(asc_interp['easting'], bins=xs, labels=False)
asc_interp['cy'] = pd.cut(asc_interp['northing'], bins=ys, labels=False)
asc_interp = asc_interp.dropna(subset=['cx','cy'])
asc_interp['cell_id'] = asc_interp['cx'].astype(int).astype(str)+"_"+asc_interp['cy'].astype(int).astype(str)

grid_data = [{'cell_id': f"{ix}_{iy}", 'geometry': box(xs[ix], ys[iy], xs[ix+1], ys[iy+1])} for ix in range(len(xs)-1) for iy in range(len(ys)-1)]
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# Escolha aqui se usa dV ou dH para a agregação
#TARGET_VAR = 'dV' 
agg = asc_interp.groupby(['cell_id','date']).agg(dV=(TARGET_VAR,'mean')).reset_index() # Coluna final chama-se sempre 'dV' para manter compatibilidade com resto do script

# ==============================
# 7. Recorte e Filtro
# ==============================
points_gdf = gpd.GeoDataFrame(asc_interp, geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']), crs="EPSG:3035").to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

poly = grid_recort.geometry.values; ids = grid_recort['cell_id'].values; tree = STRtree(poly)
valid = set()
for pt in points_gdf.geometry:
    for idx in tree.query(pt):
        if poly[idx].contains(pt): valid.add(ids[idx])
grid_barragem = grid_recort[grid_recort['cell_id'].isin(valid)]
agg = agg[agg['cell_id'].isin(valid)]

# ==============================
# 8. CLUSTERING (VERSÃO ROBUSTA CONTRA NaNs)
# ==============================
print(f">>> A calcular DTW (Flexibilidade: {DTW_WINDOW_PCT})...")

# 1. Preparar Matriz (Pivot)
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV')

# --- LIMPEZA DE NaNs (Obrigatório para o Linkage não dar erro) ---
# Interpola pequenos buracos e remove células que ainda tenham falhas críticas
agg_pivot = agg_pivot.interpolate(axis=1, limit_direction='both').dropna()

cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

if n < 2:
    print("Aviso: Células insuficientes após limpeza de dados.")
else:
    # 2. Normalizar
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X.T).T 

    # 3. Calcular DTW com a nova variável
    # max(1, ...) garante que a janela nunca é zero
    window_dtw = max(1, int(DTW_WINDOW_PCT * X_scaled.shape[1])) 
    
    # Cálculo da matriz
    dist_matrix = dtw.distance_matrix_fast(X_scaled, window=window_dtw)

    # --- TRATAMENTO PÓS-CÁLCULO ---
    # Se o DTW gerar algum NaN (acontece se a janela for muito pequena para certas séries)
    if not np.all(np.isfinite(dist_matrix)):
        # Substituímos NaNs pelo valor máximo da matriz (para dizer que são muito diferentes)
        mask_nan = np.isnan(dist_matrix)
        dist_matrix[mask_nan] = np.nanmax(dist_matrix) if not np.all(np.isnan(dist_matrix)) else 0

    # 4. Clustering Hierárquico
    condensed = squareform(dist_matrix)
    Z = linkage(condensed, method=LINKAGE_METHOD)

    # 5. Corte Automático (Elbow)
    last = Z[-10:, 2] # Últimas distâncias de fusão
    acceleration = np.diff(last, 2)
    try:
        k_idx = np.argmax(acceleration) + 2
        cut_distance = (last[k_idx] + last[k_idx-1]) / 2
    except:
        cut_distance = last[len(last)//2]

    # Labels
    cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
    
    # Reajustar labels para começar em 0
    u_labels = np.unique(cluster_labels)
    map_l = {old: new for new, old in enumerate(u_labels)}
    cluster_labels = np.array([map_l[x] for x in cluster_labels])
    
    num_clusters = len(u_labels)
    print(f"DTW Concluído. Corte: {cut_distance:.2f}. Clusters: {num_clusters}")

# DataFrame Final
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})
# Cores Dinâmicas (para N clusters)
palette = sns.color_palette("Set2", num_clusters) if num_clusters <= 8 else sns.color_palette("tab20", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(range(num_clusters), palette)}

grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='inner')

# ==============================
# FIGURA EXTRA: DENDROGRAMA
# ==============================
if Z is not None:
    print("Gerando Dendrograma...")
    plt.figure(figsize=(12, 5))
    dendrogram(Z, leaf_rotation=90, leaf_font_size=8, color_threshold=cut_distance)
    plt.axhline(y=cut_distance, c='k', ls='--', lw=1, label='Corte Automático')
    plt.title('Dendrograma de Clustering Hierárquico (DTW)')
    plt.xlabel('Células'); plt.ylabel('Distância')
    plt.legend(); plt.tight_layout(); plt.show()


# ==============================
# 9. Dados Hidro (IGUAL)
# ==============================
date_range = agg_pivot.columns
win = 13

# Temp
try: df_t = pd.read_excel("data/alqueva_temp.xlsx"); df_t['data']=pd.to_datetime(df_t['data'])
except: df_t = pd.DataFrame({'data': date_range, 'med': 0})
ts = df_t.set_index('data')['med'].reindex(date_range, fill_value=np.nan)
df_temp = pd.DataFrame({'data': ts.index, 'med': ts.values})
df_temp['med_smooth'] = savgol_filter(df_temp['med'].fillna(method='ffill'), win, 2)

# Nível
try: 
    df_n = pd.read_excel("data/alqueva_nivel.xlsx"); df_n['data']=pd.to_datetime(df_n['data'])
    df_n['nivel'] = pd.to_numeric(df_n['nivel'], errors='coerce'); df_n = df_n.dropna(subset=['nivel'])
except: df_n = pd.DataFrame({'data': date_range, 'nivel': 0})
ns = df_n.set_index('data')['nivel'].resample('MS').mean().reindex(date_range).interpolate(limit_direction='both').ffill().bfill()
df_nivel = pd.DataFrame({'data': ns.index, 'nivel': ns.values})
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], win, 2)

# Precipitação
try: df_p = pd.read_excel("data/prec.xlsx"); df_p['data'] = pd.to_datetime(df_p['data'])
except: np.random.seed(42); df_p = pd.DataFrame({'data': date_range, 'prec': 0})
ps = df_p.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': ps.index, 'prec': ps.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
def get_hydro_year(date): return date.year if date.month>=10 else date.year-1
df_prec['ano_hidrologico'] = df_prec['data'].apply(get_hydro_year)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 10. Funções Visuais (IGUAL)
# ==============================
# def scale_vis(d, vmin, vmax, s=0.20, o=0.30):
#     d = np.array(d); mn, mx = np.nanmin(d), np.nanmax(d)
#     span = vmax - vmin
#     norm = np.zeros_like(d) if mx==mn else (d-mn)/(mx-mn)
#     return norm * span * s + (vmax - span * o), mn, mx

# def create_ticks(rmin, rmax, vmin, vmax, s=0.20, o=0.30, n=5):
#     rt = np.linspace(rmin, rmax, n)
#     span = vmax - vmin
#     nt = np.linspace(0,1,n) if rmax==rmin else (rt-rmin)/(rmax-rmin)
#     vt = nt * span * s + (vmax - span * o)
#     return vt, rt


# ==============================
# 10. Funções de escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    visual = norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    if d_max_real == d_min_real: real_ticks = np.linspace(d_min_real-0.5, d_max_real+0.5, num_ticks)
    else: real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    visual_ticks = norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor)
    return visual_ticks, real_ticks


# ==============================
# FIGURA 1: MAPA (ESTILO ORIGINAL)
# ==============================
print("Gerando Figura 1...")
clusters_present = sorted(cluster_df['cluster'].unique())
k_plot = len(clusters_present)

fig_map = plt.figure(figsize=(12, 12))
ax_map = fig_map.add_subplot(1, 1, 1)
grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color='white', lw=0.5, alpha=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', lw=1)

for i in clusters_present:
    sub = grid_sel[grid_sel['cluster'] == i]
    sub.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
    cents = sub.copy(); cents.geometry = cents.geometry.centroid
    cents.plot(ax=ax_map, marker='o', color='white', edgecolor='black', markersize=30, zorder=5)

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

hdl = [plt.Rectangle((0,0),1,1, fc=cluster_colors[i], alpha=0.4) for i in clusters_present]
lbl = [f'Cluster {i+1} ({len(grid_sel[grid_sel["cluster"]==i])} cel)' for i in clusters_present]
hdl.append(Line2D([0],[0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, linestyle='None')); lbl.append('Centróides')
ax_map.legend(hdl, lbl, loc='upper left', fontsize=10, title=f"Clusters DTW (K={k_plot})")
ax_map.set_title(f"Mapa de Clusters ({TARGET_VAR})", fontsize=15)
plt.show()

# ==============================
# FIGURA 2: SÉRIES TEMPORAIS
# ==============================

print("Gerando Figura 2...")

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
n_rows_series = 5

fig_series = plt.figure(figsize=(5.5 * n_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, n_clusters, height_ratios=[3.0] * n_rows_series)

dV_min = agg[TARGET_VAR].min()
dV_max = agg[TARGET_VAR].max()
dV_margin = (dV_max - dV_min) * 0.1
dV_ylim = (dV_min - dV_margin, dV_max + dV_margin)

# --- Séries visuais ---
temp_visual, _, _ = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
nivel_visual, _, _ = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)

# Séries visuais
# tv, tmi, tmx = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
# nivel_visual, nivel_min, nivel_max = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)
# prec_total_acum_visual, _, _ = scale_series_visual(df_prec['prec_acum'], dV_min, dV_max, scale_factor=0.20, offset_factor=0.30)


for idx, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    n_cells = len(cluster_cells)


    # --- Função atualizada ---
    def plot_dV_comparison_series(
        row_idx, ext_data_visual, ext_data_df, ext_col_real,
        ext_label, ext_color,
        plot_as_bar=False, bar_color=None,
        plot_real_scale_line=False,
        combine_annual_prec=False,
        integer_ticks=False, show_background_bars=False):

        ax = fig_series.add_subplot(gs_series[row_idx, idx])

        # --- dV do cluster ---
        for cid in cluster_cells:
            ax.plot(cluster_data.columns, cluster_data.loc[cid],
                    color='lightgray', alpha=0.7, linewidth=1.0)

        ax.plot(cluster_data.columns, cluster_mean_dV,
                color=cluster_color, linewidth=1.0,
                label=f'Média Cluster {cluster_id+1}')

        # --- Eixo Y2 ---
        ax2 = ax.twinx()
        lines2 = []

        # -------------------------
        #   CASOS DE PRECIPITAÇÃO
        # -------------------------
        if combine_annual_prec:

            ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='teal', alpha=0.3)

            for ano in df_prec['ano_hidrologico'].unique():
                grupo = df_prec[df_prec['ano_hidrologico'] == ano]
                ax2.plot(grupo['data'], grupo['prec_acum_anual'],
                         color='teal', linewidth=1.0, alpha=0.9)

            ax2.set_ylim(0, df_prec['prec_acum_anual'].max() * 1.1)
            ax2.set_ylabel("Prec. acumulada anual", color="teal")
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=1.0, label='Prec. acumulada anual (mm)')
            ]

        elif plot_as_bar:

            ax2.bar(ext_data_df['data'], ext_data_df[ext_col_real], width=15,
                    color=bar_color or ext_color, alpha=0.5)

            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            ax2.set_ylabel(ext_label, color='teal')
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc=bar_color or ext_color, alpha=0.5, label=ext_label)
            ]

        elif plot_real_scale_line:

            if show_background_bars:
                ax2.bar(df_prec['data'], df_prec['prec'], width=15,
                        color='teal', alpha=0.3)

            ax2.plot(ext_data_df['data'], ext_data_df[ext_col_real],
                     color='teal', linewidth=1.0, alpha=0.85)

            ax2.set_ylim(0, ext_data_df[ext_col_real].max() * 1.1)
            ax2.set_ylabel(ext_label, color='teal')
            ax2.tick_params(axis='y', colors='teal')

            lines2 = [
                plt.Rectangle((0,0),1,1,fc='teal', alpha=0.3, label='Precipitação mensal (mm)'),
                plt.Line2D([0],[0], color='teal', linewidth=1.0, label=ext_label)
            ]

        # -------------------------
        #      OUTRAS SÉRIES
        # -------------------------
        else:

            ax2.plot(ext_data_df['data'], ext_data_visual,
                     color=ext_color, linewidth=1.0, alpha=0.85)

            visual_ticks, real_ticks = create_visual_ticks(
                ext_data_df[ext_col_real].min(), ext_data_df[ext_col_real].max(),
                dV_min, dV_max, scale_factor=0.20, offset_factor=0.30
            )

            ax2.set_ylim(dV_ylim)
            ax2.set_yticks(visual_ticks)

            if integer_ticks:
                ax2.set_yticklabels([f'{t:.0f}' for t in real_ticks])
            else:
                ax2.set_yticklabels([f'{t:.1f}' for t in real_ticks])

            label_color = 'black' if 'Temperatura' in ext_label else ext_color
            ax2.set_ylabel(ext_label, color=label_color)
            ax2.tick_params(axis='y', colors=label_color)

            lines2 = [ax2.lines[-1]]

        # --- Y1 sempre dV ---
        ax.set_ylim(dV_ylim)
        ax.set_title(f'Cluster {cluster_id + 1} ({n_cells} células)', fontsize=12)

        if idx == 0:
            ax.set_ylabel('dV (mm)')
        else:
            ax.set_yticklabels([])

        # Y2 somente no último cluster
        if idx != n_clusters - 1:
            ax2.set_yticklabels([])

        ax.tick_params(left=(idx==0))
        ax2.tick_params(right=(idx==n_clusters-1))

        # --- X-axis ---
        if row_idx == n_rows_series - 1:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        else:
            ax.set_xticklabels([])

        # --- Legenda somente no primeiro e no último cluster ---
        if idx == 0 or idx == n_clusters - 1:
            lines1, labels1 = ax.get_legend_handles_labels()
            handles2, labels2 = ax2.get_legend_handles_labels()

            ax.legend(
                lines1 + handles2,
                labels1 + labels2,
                fontsize=8,
                loc='best',
                framealpha=1.0,
                facecolor='white',
                edgecolor='lightgray'
            ).set_zorder(100)

        return ax


    # ---- CHAMADA DAS 5 SÉRIES ----
    plot_dV_comparison_series(0, temp_visual, df_temp, 'med_smooth',
                              'Temperatura média (°C)', 'black', integer_ticks=True)

    plot_dV_comparison_series(1, nivel_visual, df_nivel, 'nivel_smooth',
                              'Nível da albufeira (m)', 'navy', integer_ticks=True)

    plot_dV_comparison_series(2, None, df_prec, 'prec',
                              'Precipitação mensal (mm)', 'teal',
                              plot_as_bar=True, bar_color='teal')

    plot_dV_comparison_series(3, None, df_prec, 'prec_acum',
                              'Prec. acumulada total (mm)', 'teal',
                              plot_real_scale_line=True, show_background_bars=True)

    plot_dV_comparison_series(4, None, df_prec, 'prec_acum_anual',
                              'Prec. acumulada anual', 'teal',
                              combine_annual_prec=True)


plt.tight_layout()
plt.show()


# ==============================
# FIGURA 3: DECOMPOSIÇÃO SAZONAL
# ==============================

print("Gerando Figura 3...")

n_rows = 4  # Observed, Trend, Seasonal, Residual
fig_series_decomp, axes = plt.subplots(
    n_rows, n_clusters, figsize=(6 * max(1, n_clusters), 7), sharex=False
)

for col, cluster_id in enumerate(clusters_present):

    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    mean_series = cluster_data.mean(axis=0)

    decomposition = seasonal_decompose(mean_series, model="additive", period=12)

    series_list = [
        ("Observed", decomposition.observed),
        ("Trend", decomposition.trend),
        ("Seasonal", decomposition.seasonal),
        ("Residual", decomposition.resid)
    ]

    cluster_color = cluster_colors.get(cluster_id, "black")

    for row, (label, series) in enumerate(series_list):

        ax = axes[row, col]

        # --------------------------
        #   PLOT (agora com cores)
        # --------------------------
        ax.plot(series.index, series.values,
                color=cluster_color, linewidth=1.0)

        # --------------------------
        #  LIMITE EXTERIOR (spines)
        # --------------------------
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.8)
            spine.set_color("black")

        # --------------------------
        #     LIMITES Y
        # --------------------------
        if label != "Observed":
            ymin = np.nanmin(series)
            ymax = np.nanmax(series)
            if np.isnan(ymin) or np.isnan(ymax):
                ymin, ymax = -1, 1
            margin = (ymax - ymin) * 0.10
            ax.set_ylim(ymin - margin, ymax + margin)

        # --------------------------
        #  TÍTULOS DOS SUBPLOTS
        # --------------------------
        if row == 0:
            ax.set_title(f"Seasonal Decompose – Cluster {cluster_id + 1}",
                         fontsize=12)

        if col == 0:
            ax.set_ylabel(label, fontsize=10)
        else:
            ax.set_yticks([])

        # --------------------------
        #   EIXO X — só no último row
        # --------------------------
        if row == n_rows - 1:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.xaxis.set_major_locator(mdates.YearLocator())
            ax.tick_params(axis='x', labelrotation=0, labelsize=10)
        else:
            ax.set_xticks([])
            ax.set_xticklabels([])

plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import numpy as np

# 1. Definições Globais de Estilo
LINE_WIDTH = 0.4
LINE_COLOR = '#333333' 
MY_CMAP = plt.cm.RdBu # RdBu padrão: Azul é Positivo (+1), Vermelho é Negativo (-1)

sns.set_context("paper", font_scale=0.9)
sns.set_style("white")
plt.rcParams['axes.linewidth'] = LINE_WIDTH

# --- Dados (Certifica-te que estas variáveis existem no teu ambiente) ---
CLUSTER_ALVO_ID = 0 
c_cells = cluster_df[cluster_df['cluster'] == CLUSTER_ALVO_ID]['cell_id']
dV_mean = agg_pivot.loc[c_cells].mean()
df_plot = pd.DataFrame({
    'dV': dV_mean,
    'Temp': df_temp.set_index('data')['med_smooth'].reindex(dV_mean.index),
    'Niv': df_nivel.set_index('data')['nivel_smooth'].reindex(dV_mean.index),
    'Prec': df_prec.set_index('data')['prec'].reindex(dV_mean.index),
    'P_Ac': df_prec.set_index('data')['prec_acum'].reindex(dV_mean.index),
    'P_An': df_prec.set_index('data')['prec_acum_anual'].reindex(dV_mean.index)
}).dropna()

cols = df_plot.columns.tolist()
n_vars = len(cols)

# 2. Criar o PairGrid
g = sns.PairGrid(df_plot, vars=cols, diag_sharey=False, height=1.0, aspect=1.0)

# --- Funções de Desenho ---
def cor_func(x, y, **kwargs):
    r, p = pearsonr(x, y)
    ax = plt.gca()
    # Mapeamento: r=1 -> Azul, r=-1 -> Vermelho
    color_val = (r + 1) / 2
    facecolor = MY_CMAP(color_val)
    ax.set_facecolor((*facecolor[:3], 0.5))
    ax.annotate(f"{r:.2f}", xy=(0.5, 0.5), xycoords=ax.transAxes, 
                ha='center', va='center', fontsize=8, fontweight='normal')

g.map_diag(sns.histplot, kde=True, color="#2c3e50", alpha=0.2, edgecolor='white', linewidth=0.3, line_kws={'linewidth': 0.7})
# --- PARTE INFERIOR: Scatter Plots ---
g.map_lower(sns.regplot, ci=None, 
            scatter_kws={
                's': 3,                # Tamanho do ponto
                'alpha': 1.0,          # Opacidade total (sem transparência)
                'color': 'black',      # Cor base
                'facecolor': 'black',  # Preenchimento preto sólido
                'edgecolor': 'black',  # Contorno preto sólido
                'linewidths': 0        # Remove qualquer largura de linha de bordo para evitar reflexos
            }, 
            line_kws={'color': 'red', 'linewidth': 0.7})
g.map_upper(cor_func)

# 3. Uniformização Total das Linhas e Padding
for i in range(n_vars):
    for j in range(n_vars):
        ax = g.axes[i, j]
        ax.set_xlabel(""); ax.set_ylabel("")
        
        # Ajuste de Padding interno (Respiro de 25%)
        if i != j:
            x_min, x_max = df_plot[cols[j]].min(), df_plot[cols[j]].max()
            y_min, y_max = df_plot[cols[i]].min(), df_plot[cols[i]].max()
            x_range, y_range = x_max - x_min, y_max - y_min
            ax.set_xlim(x_min - 0.3 * x_range, x_max + 0.3 * x_range)
            ax.set_ylim(y_min - 0.3 * y_range, y_max + 0.3 * y_range)

        # Labels apenas nas extremidades
        if j != 0: ax.set_yticklabels([])
        if i != n_vars - 1: ax.set_xticklabels([])
        
        ax.tick_params(labelsize=6, direction='in', pad=1, width=LINE_WIDTH, color=LINE_COLOR)
        
        # Forçar todas as linhas da grelha
        for edge in ['top', 'bottom', 'left', 'right']:
            ax.spines[edge].set_visible(True)
            ax.spines[edge].set_linewidth(LINE_WIDTH)
            ax.spines[edge].set_color(LINE_COLOR)
        
        # Identificação na Diagonal em Vermelho (Sem Bold)
        if i == j:
            ax.annotate(cols[i], xy=(0.05, 0.90), xycoords='axes fraction', 
                        ha='left', va='top', fontsize=8, fontweight='normal', color='red',
                        bbox=dict(facecolor='white', alpha=0.4, edgecolor='none', pad=0))

# 4. Ajuste de Layout para Colagem
plt.subplots_adjust(hspace=0, wspace=0, left=0.1, right=0.85, bottom=0.18, top=0.95)

# 5. Colorbar Corrigida (Azul=Positivo, Vermelho=Negativo)
cax = g.fig.add_axes([0.87, 0.18, 0.02, 0.77]) 
sm = plt.cm.ScalarMappable(cmap=MY_CMAP, norm=plt.Normalize(-1, 1))
cbar = g.fig.colorbar(sm, cax=cax)
cbar.set_ticks([-1, 0, 1])
cbar.outline.set_linewidth(LINE_WIDTH)
cbar.outline.set_edgecolor(LINE_COLOR)
cbar.set_label('Correlation coefficient', rotation=270, labelpad=15, fontsize=9)
cbar.ax.tick_params(labelsize=7, width=LINE_WIDTH, color=LINE_COLOR)

# 6. Legenda Inferior (Recuperada e Colada)
ax_table = g.fig.add_axes([0.1, 0.06, 0.75, 0.12]) 
ax_table.axis('off')

legend_data = [
    ["dV: Vertical Displacement", "Temp: Temperature", "Niv: Reservoir Level"],
    ["Prec: Daily Precipitation", "P_Ac: Accumulated Prec.", "P_An: Annual Acc. Prec."]
]

table = ax_table.table(cellText=legend_data, loc='upper center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.4) 

for (row, col), cell in table.get_celld().items():
    cell.set_linewidth(LINE_WIDTH)
    cell.set_edgecolor(LINE_COLOR)
    cell.get_text().set_fontweight('normal')

# 7. Finalização
plt.savefig("Matriz_Correlacao_Final_Final.png", dpi=600, bbox_inches='tight')
plt.show()

# K-Means

In [ ]:
# Escolha aqui o que quer analisar: 'dV' ou 'dH'
TARGET_VAR = 'dV' 
grid_size = 50
N_CLUSTERS = 4  # No K-means definimos o número de clusters manualmente

# ==============================================================================
# SCRIPT COMPLETO: K-MEANS CLUSTERING PARA INSAR
# ==============================================================================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
import matplotlib.dates as mdates
from shapely.strtree import STRtree
from statsmodels.tsa.seasonal import seasonal_decompose
from matplotlib.lines import Line2D
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import seaborn as sns
from matplotlib.colors import to_hex

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'].isin([
        'Infraestruturas de produção de energia hídrica', 
        'Equipamentos culturais',
        'Florestas de azinheira',
        'Matos',
        'Pastagens melhoradas',
        'Rede rodoviária',
        'Superfícies silvopastoris de azinheira'
    ])]
except Exception as e:
    print(f"Aviso: Usando placeholder para área. {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs e Filtro
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
def filter_area(df): return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) & (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Processamento Temporal
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'], value_vars=disp_cols, var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(start=max(asc_long['date'].min(), desc_long['date'].min()), end=min(asc_long['date'].max(), desc_long['date'].max()), freq='MS')

def interpolate_ps(df, dates):
    dfs = []
    for (x, y), g in df.groupby(['easting','northing']):
        g = g.sort_values('date')
        interp = np.interp(pd.to_datetime(dates).astype(np.int64), g['date'].astype(np.int64), g['disp'])
        dfs.append(pd.DataFrame({'easting': x, 'northing': y, 'latitude': g['latitude'].iloc[0], 'longitude': g['longitude'].iloc[0], 'date': dates, 'disp': interp, 'incidence_angle': g['incidence_angle'].iloc[0], 'track_angle': g['track_angle'].iloc[0]}))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 3. IDW e Decomposição dV/dH
# ==============================
def idw(source, target, radius=50, power=2):
    out = []
    for d, src in source.groupby('date'):
        tgt = target[target['date']==d].copy()
        if src.empty or tgt.empty: continue
        tree = cKDTree(list(zip(src['easting'], src['northing'])))
        dist, idx = tree.query(list(zip(tgt['easting'], tgt['northing'])), k=3, distance_upper_bound=radius)
        vals, thetas, alphas = [], [], []
        for d_i, i_i in zip(dist, idx):
            m = np.isfinite(d_i)
            if not np.any(m): vals.append(np.nan); thetas.append(np.nan); alphas.append(np.nan); continue
            w = 1/(d_i[m]**power)
            vals.append(np.sum(w*src.iloc[i_i[m]]['disp'])/np.sum(w))
            thetas.append(np.sum(w*src.iloc[i_i[m]]['incidence_angle'])/np.sum(w))
            alphas.append(np.sum(w*src.iloc[i_i[m]]['track_angle'])/np.sum(w))
        tgt['disp_idw'] = vals; tgt['theta_desc'] = thetas; tgt['alpha_desc'] = alphas
        out.append(tgt)
    return pd.concat(out, ignore_index=True)

asc_interp = idw(desc_interp, asc_interp).dropna(subset=['disp_idw'])

orb_inc = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orb_inc) * np.cos(np.deg2rad(asc_interp['latitude'])))

def get_comps(row):
    ta, td, beta = np.deg2rad(row['incidence_angle']), np.deg2rad(row['theta_desc']), row['beta']
    denom = (np.cos(ta)*np.sin(td)*np.cos(beta) + np.cos(td)*np.sin(ta)*np.cos(beta))
    if denom == 0: return np.nan, np.nan
    dV = (row['disp_idw']*np.sin(ta)*np.cos(beta) + row['disp']*np.sin(td)*np.cos(beta))/denom
    dH = (row['disp_idw']*np.cos(ta) - row['disp']*np.cos(td))/denom
    return dV, dH

asc_interp[['dV', 'dH']] = asc_interp.apply(lambda x: pd.Series(get_comps(x)), axis=1)
asc_interp = asc_interp.dropna(subset=['dV', 'dH'])

# ==============================
# 4. Grelha e Recorte
# ==============================
xe = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
ye = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
xs, ys = xe - grid_size/2, ye - grid_size/2
asc_interp['cx'] = pd.cut(asc_interp['easting'], bins=xs, labels=False)
asc_interp['cy'] = pd.cut(asc_interp['northing'], bins=ys, labels=False)
asc_interp = asc_interp.dropna(subset=['cx','cy'])
asc_interp['cell_id'] = asc_interp['cx'].astype(int).astype(str)+"_"+asc_interp['cy'].astype(int).astype(str)

grid_data = [{'cell_id': f"{ix}_{iy}", 'geometry': box(xs[ix], ys[iy], xs[ix+1], ys[iy+1])} for ix in range(len(xs)-1) for iy in range(len(ys)-1)]
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')
agg = asc_interp.groupby(['cell_id','date']).agg(dV=(TARGET_VAR,'mean')).reset_index()

grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

poly = grid_recort.geometry.values; ids = grid_recort['cell_id'].values; tree = STRtree(poly)
points_gdf = gpd.GeoDataFrame(asc_interp, geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']), crs="EPSG:3035").to_crs(epsg=3857)
valid = set()
for pt in points_gdf.geometry:
    for idx in tree.query(pt):
        if poly[idx].contains(pt): valid.add(ids[idx])
grid_barragem = grid_recort[grid_recort['cell_id'].isin(valid)]
agg = agg[agg['cell_id'].isin(valid)]

# ==============================
# 5. K-MEANS CLUSTERING
# ==============================
print(f">>> A calcular K-means (K={N_CLUSTERS})...")
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').interpolate(axis=1, limit_direction='both').dropna()

X = agg_pivot.values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X) # Normalização por célula

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
clusters_present = sorted(cluster_df['cluster'].unique())
num_clusters = len(clusters_present)

palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(clusters_present, palette)}
grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='inner')

# ==============================
# 6. Dados Auxiliares (Hidro/Temp)
# ==============================
date_range = agg_pivot.columns
win = 13

# Temp
try: df_t = pd.read_excel("data/alqueva_temp.xlsx"); df_t['data']=pd.to_datetime(df_t['data'])
except: df_t = pd.DataFrame({'data': date_range, 'med': 0})
ts = df_t.set_index('data')['med'].reindex(date_range).interpolate()
df_temp = pd.DataFrame({'data': ts.index, 'med_smooth': savgol_filter(ts.fillna(0), win, 2)})

# Nível
try: df_n = pd.read_excel("data/alqueva_nivel.xlsx"); df_n['data']=pd.to_datetime(df_n['data'])
except: df_n = pd.DataFrame({'data': date_range, 'nivel': 0})
ns = df_n.set_index('data')['nivel'].resample('MS').mean().reindex(date_range).interpolate()
df_nivel = pd.DataFrame({'data': ns.index, 'nivel_smooth': savgol_filter(ns.ffill().bfill(), win, 2)})

# Prec
try: df_p = pd.read_excel("data/prec.xlsx"); df_p['data'] = pd.to_datetime(df_p['data'])
except: df_p = pd.DataFrame({'data': date_range, 'prec': 0})
ps = df_p.set_index('data')['prec'].reindex(date_range, fill_value=0)
df_prec = pd.DataFrame({'data': ps.index, 'prec': ps.values, 'prec_acum': ps.cumsum()})
df_prec['ano_hidrologico'] = df_prec['data'].apply(lambda d: d.year if d.month>=10 else d.year-1)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 7. Funções de Escalonamento Visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data); dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm = np.zeros_like(data) if dmax==dmin else (data-dmin)/(dmax-dmin)
    return norm*dV_span*scale_factor + (dV_max - dV_span*offset_factor), dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    real_ticks = np.linspace(d_min_real, d_max_real, num_ticks)
    dV_span = dV_max - dV_min
    norm_ticks = np.linspace(0,1,num_ticks) if d_max_real==d_min_real else (real_ticks-d_min_real)/(d_max_real-d_min_real)
    return norm_ticks*dV_span*scale_factor + (dV_max - dV_span*offset_factor), real_ticks

# ==============================
# FIGURA 1: MAPA
# ==============================
fig_map, ax_map = plt.subplots(figsize=(12, 12))
grid_sel.plot(ax=ax_map, color=grid_sel['cluster'].map(cluster_colors), alpha=0.5, edgecolor='black', lw=0.5)
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()
ax_map.set_title(f"Clusters K-means ({TARGET_VAR})", fontsize=15)
plt.show()

# ==============================
# FIGURA 2: SÉRIES TEMPORAIS
# ==============================
n_rows_series = 5
fig_series = plt.figure(figsize=(5.5 * num_clusters, 3.0 * n_rows_series))
gs_series = fig_series.add_gridspec(n_rows_series, num_clusters)

dV_min, dV_max = agg['dV'].min(), agg['dV'].max()
dV_ylim = (dV_min - (dV_max-dV_min)*0.1, dV_max + (dV_max-dV_min)*0.1)

temp_vis, _, _ = scale_series_visual(df_temp['med_smooth'], dV_min, dV_max)
niv_vis, _, _ = scale_series_visual(df_nivel['nivel_smooth'], dV_min, dV_max)

for idx, cl_id in enumerate(clusters_present):
    c_cells = cluster_df[cluster_df['cluster']==cl_id]['cell_id']
    c_data = agg_pivot.loc[c_cells]
    c_mean = c_data.mean()
    
    for r in range(5):
        ax = fig_series.add_subplot(gs_series[r, idx])
        for cell in c_cells: ax.plot(c_data.columns, c_data.loc[cell], color='lightgray', lw=0.5, alpha=0.5)
        ax.plot(c_data.columns, c_mean, color=cluster_colors[cl_id], lw=2)
        ax.set_ylim(dV_ylim)
        
        ax2 = ax.twinx()
        if r == 0: # Temp
            ax2.plot(df_temp['data'], temp_vis, color='black', lw=1, ls='--')
            vt, rt = create_visual_ticks(df_temp['med_smooth'].min(), df_temp['med_smooth'].max(), dV_min, dV_max)
            ax2.set_yticks(vt); ax2.set_yticklabels([f"{t:.1f}" for t in rt])
            ax2.set_ylabel("Temp (°C)")
        elif r == 1: # Nivel
            ax2.plot(df_nivel['data'], niv_vis, color='navy', lw=1)
            vt, rt = create_visual_ticks(df_nivel['nivel_smooth'].min(), df_nivel['nivel_smooth'].max(), dV_min, dV_max)
            ax2.set_yticks(vt); ax2.set_yticklabels([f"{t:.0f}" for t in rt])
            ax2.set_ylabel("Nível (m)")
        elif r == 2: # Prec Mensal
            ax2.bar(df_prec['data'], df_prec['prec'], width=20, color='teal', alpha=0.3)
            ax2.set_ylabel("Prec (mm)")
        elif r == 3: # Prec Acum
            ax2.plot(df_prec['data'], df_prec['prec_acum'], color='teal')
            ax2.set_ylabel("Prec Acum (mm)")
        elif r == 4: # Prec Anual
            for a in df_prec['ano_hidrologico'].unique():
                sub_p = df_prec[df_prec['ano_hidrologico']==a]
                ax2.plot(sub_p['data'], sub_p['prec_acum_anual'], color='teal', lw=1)
            ax2.set_ylabel("Prec Anual (mm)")
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        
        if idx != num_clusters -1: ax2.set_yticklabels([])
        if idx != 0: ax.set_yticklabels([])
        if r != 4: ax.set_xticklabels([])
        ax.set_title(f"Cluster {cl_id+1}" if r==0 else "")

plt.tight_layout(); plt.show()

# ==============================
# FIGURA 3: DECOMPOSIÇÃO
# ==============================
fig_decomp, axes = plt.subplots(4, num_clusters, figsize=(15, 8), sharex=True)
for col, cl_id in enumerate(clusters_present):
    series = agg_pivot.loc[cluster_df[cluster_df['cluster']==cl_id]['cell_id']].mean()
    decomp = seasonal_decompose(series, period=12)
    
    plots = [decomp.observed, decomp.trend, decomp.seasonal, decomp.resid]
    labels = ["Obs", "Trend", "Seas", "Res"]
    for row in range(4):
        axes[row, col].plot(plots[row], color=cluster_colors[cl_id])
        if col == 0: axes[row, col].set_ylabel(labels[row])
        if row == 0: axes[row, col].set_title(f"Cluster {cl_id+1}")
plt.tight_layout(); plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import numpy as np

# 1. Definições Globais de Estilo
LINE_WIDTH = 0.4
LINE_COLOR = '#333333' 
MY_CMAP = plt.cm.RdBu # RdBu padrão: Azul é Positivo (+1), Vermelho é Negativo (-1)

sns.set_context("paper", font_scale=0.9)
sns.set_style("white")
plt.rcParams['axes.linewidth'] = LINE_WIDTH

# --- Dados (Certifica-te que estas variáveis existem no teu ambiente) ---
CLUSTER_ALVO_ID = 0 
c_cells = cluster_df[cluster_df['cluster'] == CLUSTER_ALVO_ID]['cell_id']
dV_mean = agg_pivot.loc[c_cells].mean()
df_plot = pd.DataFrame({
    'dV': dV_mean,
    'Temp': df_temp.set_index('data')['med_smooth'].reindex(dV_mean.index),
    'Niv': df_nivel.set_index('data')['nivel_smooth'].reindex(dV_mean.index),
    'Prec': df_prec.set_index('data')['prec'].reindex(dV_mean.index),
    'P_Ac': df_prec.set_index('data')['prec_acum'].reindex(dV_mean.index),
    'P_An': df_prec.set_index('data')['prec_acum_anual'].reindex(dV_mean.index)
}).dropna()

cols = df_plot.columns.tolist()
n_vars = len(cols)

# 2. Criar o PairGrid
g = sns.PairGrid(df_plot, vars=cols, diag_sharey=False, height=1.0, aspect=1.0)

# --- Funções de Desenho ---
def cor_func(x, y, **kwargs):
    r, p = pearsonr(x, y)
    ax = plt.gca()
    # Mapeamento: r=1 -> Azul, r=-1 -> Vermelho
    color_val = (r + 1) / 2
    facecolor = MY_CMAP(color_val)
    ax.set_facecolor((*facecolor[:3], 0.5))
    ax.annotate(f"{r:.2f}", xy=(0.5, 0.5), xycoords=ax.transAxes, 
                ha='center', va='center', fontsize=8, fontweight='normal')

g.map_diag(sns.histplot, kde=True, color="#2c3e50", alpha=0.2, edgecolor='white', linewidth=0.3, line_kws={'linewidth': 0.7})
# --- PARTE INFERIOR: Scatter Plots ---
g.map_lower(sns.regplot, ci=None, 
            scatter_kws={
                's': 3,                # Tamanho do ponto
                'alpha': 1.0,          # Opacidade total (sem transparência)
                'color': 'black',      # Cor base
                'facecolor': 'black',  # Preenchimento preto sólido
                'edgecolor': 'black',  # Contorno preto sólido
                'linewidths': 0        # Remove qualquer largura de linha de bordo para evitar reflexos
            }, 
            line_kws={'color': 'red', 'linewidth': 0.7})
g.map_upper(cor_func)

# 3. Uniformização Total das Linhas e Padding
for i in range(n_vars):
    for j in range(n_vars):
        ax = g.axes[i, j]
        ax.set_xlabel(""); ax.set_ylabel("")
        
        # Ajuste de Padding interno (Respiro de 25%)
        if i != j:
            x_min, x_max = df_plot[cols[j]].min(), df_plot[cols[j]].max()
            y_min, y_max = df_plot[cols[i]].min(), df_plot[cols[i]].max()
            x_range, y_range = x_max - x_min, y_max - y_min
            ax.set_xlim(x_min - 0.3 * x_range, x_max + 0.3 * x_range)
            ax.set_ylim(y_min - 0.3 * y_range, y_max + 0.3 * y_range)

        # Labels apenas nas extremidades
        if j != 0: ax.set_yticklabels([])
        if i != n_vars - 1: ax.set_xticklabels([])
        
        ax.tick_params(labelsize=6, direction='in', pad=1, width=LINE_WIDTH, color=LINE_COLOR)
        
        # Forçar todas as linhas da grelha
        for edge in ['top', 'bottom', 'left', 'right']:
            ax.spines[edge].set_visible(True)
            ax.spines[edge].set_linewidth(LINE_WIDTH)
            ax.spines[edge].set_color(LINE_COLOR)
        
        # Identificação na Diagonal em Vermelho (Sem Bold)
        if i == j:
            ax.annotate(cols[i], xy=(0.05, 0.90), xycoords='axes fraction', 
                        ha='left', va='top', fontsize=8, fontweight='normal', color='red',
                        bbox=dict(facecolor='white', alpha=0.4, edgecolor='none', pad=0))

# 4. Ajuste de Layout para Colagem
plt.subplots_adjust(hspace=0, wspace=0, left=0.1, right=0.85, bottom=0.18, top=0.95)

# 5. Colorbar Corrigida (Azul=Positivo, Vermelho=Negativo)
cax = g.fig.add_axes([0.87, 0.18, 0.02, 0.77]) 
sm = plt.cm.ScalarMappable(cmap=MY_CMAP, norm=plt.Normalize(-1, 1))
cbar = g.fig.colorbar(sm, cax=cax)
cbar.set_ticks([-1, 0, 1])
cbar.outline.set_linewidth(LINE_WIDTH)
cbar.outline.set_edgecolor(LINE_COLOR)
cbar.set_label('Correlation coefficient', rotation=270, labelpad=15, fontsize=9)
cbar.ax.tick_params(labelsize=7, width=LINE_WIDTH, color=LINE_COLOR)

# 6. Legenda Inferior (Recuperada e Colada)
ax_table = g.fig.add_axes([0.1, 0.06, 0.75, 0.12]) 
ax_table.axis('off')

legend_data = [
    ["dV: Vertical Displacement", "Temp: Temperature", "Niv: Reservoir Level"],
    ["Prec: Daily Precipitation", "P_Ac: Accumulated Prec.", "P_An: Annual Acc. Prec."]
]

table = ax_table.table(cellText=legend_data, loc='upper center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.4) 

for (row, col), cell in table.get_celld().items():
    cell.set_linewidth(LINE_WIDTH)
    cell.set_edgecolor(LINE_COLOR)
    cell.get_text().set_fontweight('normal')

# 7. Finalização
plt.savefig("Matriz_Correlacao_Final_Final.png", dpi=600, bbox_inches='tight')
plt.show()